# 1. Pre-baseline fixed equal-fusion reconstruction and training

This standalone notebook performs the complete temporal-leakage sensitivity experiment.

It uses the same already baseline-aligned modality manifests that were used to construct the original model-ready inputs. No longitudinal source reconstruction and no file discovery are performed.

The temporal sensitivity rule is:

- ADAS, MMSE, and FAQ are retained only when `-90 <= DAYS_FROM_BASELINE <= 0`;
- CSF and plasma are retained only when `-180 <= DAYS_FROM_BASELINE <= 0`;
- MRI is retained only when `-90 <= days_from_baseline_mri <= 0`;
- demographics and APOE remain unchanged;
- participants remain in the cohort when a modality becomes unavailable.

After rebuilding the five fold tables, the notebook trains the availability-gated model with fixed 50/50 fusion across all five folds.

## 1.1. A. Create pre-baseline-only model-ready fold tables

The original fold files provide the fixed participant assignments, labels, demographics, APOE encoding, and output schema. Temporally sensitive branches are replaced using the existing baseline-aligned manifests.

In [ ]:
# ============================================================
# A1. Exact project paths and modality definitions
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display
from google.colab import drive


if not Path(
    "/content/drive/MyDrive"
).exists():
    drive.mount(
        "/content/drive"
    )


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

NON_IMAGING_ROOT = (
    PROJECT_ROOT
    / "adni_non_imaging"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

BASELINE_ALIGNED_ROOT = (
    NON_IMAGING_ROOT
    / "manifests"
    / "baseline_aligned_modalities"
)


MANIFEST_PATHS = {
    "adas": (
        BASELINE_ALIGNED_ROOT
        / "adas"
        / "adas_baseline_aligned_cohort.csv"
    ),

    "mmse": (
        BASELINE_ALIGNED_ROOT
        / "mmse"
        / "mmse_baseline_aligned_cohort.csv"
    ),

    "faq": (
        BASELINE_ALIGNED_ROOT
        / "faq"
        / "faq_baseline_aligned_cohort.csv"
    ),

    "csf": (
        BASELINE_ALIGNED_ROOT
        / "csf_core_biomarkers"
        / "csf_core_biomarkers_baseline_aligned_cohort.csv"
    ),

    "plasma": (
        BASELINE_ALIGNED_ROOT
        / "plasma"
        / "plasma_baseline_aligned_cohort.csv"
    ),

    "mri": (
        BASELINE_ALIGNED_ROOT
        / "mri"
        / "mri_baseline_aligned_authoritative_cohort_1057.csv"
    ),
}


ORIGINAL_FOLD_DIR = (
    MODEL_ROOT
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

TEMPORAL_ROOT = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
)

TEMPORAL_FOLD_DIR = (
    TEMPORAL_ROOT
    / "final_task_ready_inputs"
    / "mci_prognosis"
)

TEMPORAL_AUDIT_DIR = (
    TEMPORAL_ROOT
    / "audit"
)

TEMPORAL_FOLD_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEMPORAL_AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


WINDOWS_DAYS = {
    "adas": 90,
    "mmse": 90,
    "faq": 90,
    "csf": 180,
    "plasma": 180,
    "mri": 90,
}


FEATURE_COLUMNS = {
    "adas": [
        "TOTSCORE",
        "TOTAL13",
    ],

    "mmse": [
        "MMSE_TOTAL_SCORE",
        "MMSE_ORIENTATION_SCORE",
        "MMSE_REGISTRATION_SCORE",
        "MMSE_ATTENTION_SCORE",
        "MMSE_DELAYED_RECALL_SCORE",
        "MMSE_LANGUAGE_COMMAND_SCORE",
    ],

    "faq": [
        "FAQTOTAL",
    ],

    "csf": [
        "ABETA40",
        "ABETA42",
        "TAU",
        "PTAU",
        "ABETA42_40_RATIO",
    ],

    "plasma": [
        "pT217_F",
        "AB42_F",
        "AB40_F",
        "AB42_AB40_F",
        "pT217_AB42_F",
        "NfL_Q",
        "GFAP_Q",
        "NfL_F",
        "GFAP_F",
    ],
}


MODEL_Z_COLUMNS = {
    "adas": {
        "TOTSCORE": "ADAS__TOTSCORE__Z",
        "TOTAL13": "ADAS__TOTAL13__Z",
    },

    "mmse": {
        "MMSE_TOTAL_SCORE":
            "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE_ORIENTATION_SCORE":
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE_REGISTRATION_SCORE":
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE_ATTENTION_SCORE":
            "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE_DELAYED_RECALL_SCORE":
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE_LANGUAGE_COMMAND_SCORE":
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
    },

    "faq": {
        "FAQTOTAL": "FAQ__FAQTOTAL__Z",
    },

    "csf": {
        "ABETA40": "CSF__ABETA40__Z",
        "ABETA42": "CSF__ABETA42__Z",
        "TAU": "CSF__TAU__Z",
        "PTAU": "CSF__PTAU__Z",
        "ABETA42_40_RATIO":
            "CSF__ABETA42_40_RATIO__Z",
    },

    "plasma": {
        "pT217_F": "PLASMA__pT217_F__Z",
        "AB42_F": "PLASMA__AB42_F__Z",
        "AB40_F": "PLASMA__AB40_F__Z",
        "AB42_AB40_F":
            "PLASMA__AB42_AB40_F__Z",
        "pT217_AB42_F":
            "PLASMA__pT217_AB42_F__Z",
        "NfL_Q": "PLASMA__NfL_Q__Z",
        "GFAP_Q": "PLASMA__GFAP_Q__Z",
        "NfL_F": "PLASMA__NfL_F__Z",
        "GFAP_F": "PLASMA__GFAP_F__Z",
    },
}


FEATURE_MASK_COLUMNS = {
    "adas": {
        "TOTSCORE":
            "FEATURE_MASK__ADAS_TOTSCORE",
        "TOTAL13":
            "FEATURE_MASK__ADAS_TOTAL13",
    },

    "mmse": {
        "MMSE_TOTAL_SCORE":
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "MMSE_ORIENTATION_SCORE":
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "MMSE_REGISTRATION_SCORE":
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "MMSE_ATTENTION_SCORE":
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "MMSE_DELAYED_RECALL_SCORE":
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "MMSE_LANGUAGE_COMMAND_SCORE":
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
    },

    "faq": {
        "FAQTOTAL":
            "FEATURE_MASK__FAQ_FAQTOTAL",
    },

    "csf": {
        "ABETA40":
            "FEATURE_MASK__CSF_ABETA40",
        "ABETA42":
            "FEATURE_MASK__CSF_ABETA42",
        "TAU":
            "FEATURE_MASK__CSF_TAU",
        "PTAU":
            "FEATURE_MASK__CSF_PTAU",
        "ABETA42_40_RATIO":
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
    },

    "plasma": {
        "pT217_F":
            "FEATURE_MASK__PLASMA_pT217_F",
        "AB42_F":
            "FEATURE_MASK__PLASMA_AB42_F",
        "AB40_F":
            "FEATURE_MASK__PLASMA_AB40_F",
        "AB42_AB40_F":
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "pT217_AB42_F":
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "NfL_Q":
            "FEATURE_MASK__PLASMA_NfL_Q",
        "GFAP_Q":
            "FEATURE_MASK__PLASMA_GFAP_Q",
        "NfL_F":
            "FEATURE_MASK__PLASMA_NfL_F",
        "GFAP_F":
            "FEATURE_MASK__PLASMA_GFAP_F",
    },
}


MRI_PATH_COLUMN = (
    "MRI__NORMALIZED_T1_NPY_PATH"
)


print("=" * 72)
print("PRE-BASELINE MANIFEST FILTERING")
print("=" * 72)

print(
    f"\nPrepared fold files will be written to:\n"
    f"{TEMPORAL_FOLD_DIR}"
)

In [ ]:
# ============================================================
# A2. Load the exact baseline-aligned manifests
# ============================================================

manifest_dfs = {}


for modality, manifest_path in (
    MANIFEST_PATHS.items()
):

    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Missing {modality} manifest:\n"
            f"{manifest_path}"
        )


    manifest_df = pd.read_csv(
        manifest_path,
        low_memory=False,
    )


    manifest_df["RID"] = pd.to_numeric(
        manifest_df["RID"],
        errors="raise",
    ).astype(
        int
    )


    if manifest_df[
        "RID"
    ].duplicated().any():
        raise ValueError(
            f"{modality}: baseline-aligned manifest "
            "contains repeated RID values."
        )


    manifest_dfs[
        modality
    ] = manifest_df


required_manifest_columns = {
    "adas": [
        "RID",
        "DAYS_FROM_BASELINE",
        *FEATURE_COLUMNS[
            "adas"
        ],
    ],

    "mmse": [
        "RID",
        "DAYS_FROM_BASELINE",
        *FEATURE_COLUMNS[
            "mmse"
        ],
    ],

    "faq": [
        "RID",
        "DAYS_FROM_BASELINE",
        *FEATURE_COLUMNS[
            "faq"
        ],
    ],

    "csf": [
        "RID",
        "DAYS_FROM_BASELINE",
        *FEATURE_COLUMNS[
            "csf"
        ],
    ],

    "plasma": [
        "RID",
        "DAYS_FROM_BASELINE",
        *FEATURE_COLUMNS[
            "plasma"
        ],
    ],

    "mri": [
        "RID",
        "days_from_baseline_mri",
        "normalized_t1_npy_path",
    ],
}


for modality, required_columns in (
    required_manifest_columns.items()
):

    missing_columns = [
        column
        for column in required_columns
        if column
        not in manifest_dfs[
            modality
        ].columns
    ]


    if missing_columns:
        raise KeyError(
            f"{modality}: baseline-aligned manifest "
            "is missing required columns: "
            + ", ".join(
                missing_columns
            )
        )


for modality in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
]:

    manifest_dfs[
        modality
    ][
        "DAYS_FROM_BASELINE"
    ] = pd.to_numeric(
        manifest_dfs[
            modality
        ][
            "DAYS_FROM_BASELINE"
        ],
        errors="coerce",
    )


manifest_dfs[
    "mri"
][
    "days_from_baseline_mri"
] = pd.to_numeric(
    manifest_dfs[
        "mri"
    ][
        "days_from_baseline_mri"
    ],
    errors="coerce",
)


print(
    "All exact baseline-aligned manifests loaded."
)

for modality, manifest_df in (
    manifest_dfs.items()
):
    print(
        f"{modality:<8}: "
        f"{len(manifest_df):,} rows"
    )

In [ ]:
# ============================================================
# A3. Apply the pre-baseline-only temporal rule
# ============================================================

retained_manifests = {}
temporal_summary_rows = []


for modality in [
    "adas",
    "mmse",
    "faq",
    "csf",
    "plasma",
]:

    manifest_df = manifest_dfs[
        modality
    ].copy()

    window_days = WINDOWS_DAYS[
        modality
    ]


    retained_df = (
        manifest_df.loc[
            manifest_df[
                "DAYS_FROM_BASELINE"
            ].between(
                -window_days,
                0,
                inclusive="both",
            )
        ]
        .copy()
    )


    retained_manifests[
        modality
    ] = retained_df


    temporal_summary_rows.append(
        {
            "MODALITY":
                modality,

            "ORIGINAL_SELECTED_RECORDS":
                len(
                    manifest_df
                ),

            "PREBASELINE_RECORDS_RETAINED":
                len(
                    retained_df
                ),

            "POSTBASELINE_RECORDS_REMOVED":
                int(
                    (
                        manifest_df[
                            "DAYS_FROM_BASELINE"
                        ]
                        > 0
                    ).sum()
                ),

            "OUTSIDE_NEGATIVE_WINDOW":
                int(
                    (
                        manifest_df[
                            "DAYS_FROM_BASELINE"
                        ]
                        < -window_days
                    ).sum()
                ),

            "MINIMUM_RETAINED_DAYS":
                (
                    retained_df[
                        "DAYS_FROM_BASELINE"
                    ].min()
                    if not retained_df.empty
                    else np.nan
                ),

            "MAXIMUM_RETAINED_DAYS":
                (
                    retained_df[
                        "DAYS_FROM_BASELINE"
                    ].max()
                    if not retained_df.empty
                    else np.nan
                ),
        }
    )


mri_manifest_df = manifest_dfs[
    "mri"
].copy()


retained_mri_df = (
    mri_manifest_df.loc[
        mri_manifest_df[
            "days_from_baseline_mri"
        ].between(
            -WINDOWS_DAYS[
                "mri"
            ],
            0,
            inclusive="both",
        )
        &
        mri_manifest_df[
            "normalized_t1_npy_path"
        ].notna()
    ]
    .copy()
)


retained_manifests[
    "mri"
] = retained_mri_df


temporal_summary_rows.append(
    {
        "MODALITY":
            "mri",

        "ORIGINAL_SELECTED_RECORDS":
            len(
                mri_manifest_df
            ),

        "PREBASELINE_RECORDS_RETAINED":
            len(
                retained_mri_df
            ),

        "POSTBASELINE_RECORDS_REMOVED":
            int(
                (
                    mri_manifest_df[
                        "days_from_baseline_mri"
                    ]
                    > 0
                ).sum()
            ),

        "OUTSIDE_NEGATIVE_WINDOW":
            int(
                (
                    mri_manifest_df[
                        "days_from_baseline_mri"
                    ]
                    < -WINDOWS_DAYS[
                        "mri"
                    ]
                ).sum()
            ),

        "MINIMUM_RETAINED_DAYS":
            retained_mri_df[
                "days_from_baseline_mri"
            ].min(),

        "MAXIMUM_RETAINED_DAYS":
            retained_mri_df[
                "days_from_baseline_mri"
            ].max(),
    }
)


temporal_filter_summary = pd.DataFrame(
    temporal_summary_rows
)


if (
    temporal_filter_summary[
        "MAXIMUM_RETAINED_DAYS"
    ]
    > 0
).any():
    raise ValueError(
        "At least one retained modality record "
        "is post-baseline."
    )


temporal_filter_summary.to_csv(
    TEMPORAL_AUDIT_DIR
    / "temporal_filter_summary.csv",
    index=False,
)


display(
    temporal_filter_summary
)

In [ ]:
# ============================================================
# A4. Rebuild the five fold-specific model-ready tables
# ============================================================

scaler_rows = []
fold_inventory_rows = []


for outer_fold in range(
    5
):

    original_fold_path = (
        ORIGINAL_FOLD_DIR
        / (
            "mci_prognosis_outer_fold_"
            f"{outer_fold}_final_task_ready.csv"
        )
    )


    if not original_fold_path.is_file():
        raise FileNotFoundError(
            f"Missing original fold table:\n"
            f"{original_fold_path}"
        )


    fold_df = pd.read_csv(
        original_fold_path,
        low_memory=False,
    )


    original_columns = (
        fold_df.columns.tolist()
    )


    fold_df["RID"] = pd.to_numeric(
        fold_df["RID"],
        errors="raise",
    ).astype(
        int
    )


    if len(
        fold_df
    ) != 544:
        raise ValueError(
            f"Fold {outer_fold}: expected 544 rows, "
            f"found {len(fold_df)}."
        )


    if fold_df[
        "RID"
    ].duplicated().any():
        raise ValueError(
            f"Fold {outer_fold}: repeated RID values."
        )


    temporary_raw_columns = []


    # --------------------------------------------------------
    # Attach raw retained manifest values
    # --------------------------------------------------------

    for modality in [
        "adas",
        "mmse",
        "faq",
        "csf",
        "plasma",
    ]:

        rename_map = {
            feature: (
                f"RAW_TEMPORAL__"
                f"{modality.upper()}__"
                f"{feature}"
            )
            for feature in FEATURE_COLUMNS[
                modality
            ]
        }


        modality_table = (
            retained_manifests[
                modality
            ][
                [
                    "RID",
                    *FEATURE_COLUMNS[
                        modality
                    ],
                ]
            ]
            .rename(
                columns=rename_map
            )
        )


        fold_df = fold_df.merge(
            modality_table,
            on="RID",
            how="left",
            validate="one_to_one",
        )


        temporary_raw_columns.extend(
            rename_map.values()
        )


    mri_table = (
        retained_manifests[
            "mri"
        ][
            [
                "RID",
                "normalized_t1_npy_path",
            ]
        ]
        .rename(
            columns={
                "normalized_t1_npy_path":
                    "PREBASELINE_MRI_PATH",
            }
        )
    )


    fold_df = fold_df.merge(
        mri_table,
        on="RID",
        how="left",
        validate="one_to_one",
    )


    # --------------------------------------------------------
    # Replace MRI path and MRI branch mask
    # --------------------------------------------------------

    fold_df[
        MRI_PATH_COLUMN
    ] = fold_df[
        "PREBASELINE_MRI_PATH"
    ]


    fold_df[
        "BRANCH_MASK__MRI"
    ] = (
        fold_df[
            MRI_PATH_COLUMN
        ]
        .notna()
        .astype(
            int
        )
    )


    # --------------------------------------------------------
    # Rebuild temporally sensitive feature masks
    # --------------------------------------------------------

    for modality in [
        "adas",
        "mmse",
        "faq",
        "csf",
        "plasma",
    ]:

        for feature in FEATURE_COLUMNS[
            modality
        ]:

            raw_column = (
                f"RAW_TEMPORAL__"
                f"{modality.upper()}__"
                f"{feature}"
            )

            mask_column = (
                FEATURE_MASK_COLUMNS[
                    modality
                ][
                    feature
                ]
            )


            fold_df[
                mask_column
            ] = (
                pd.to_numeric(
                    fold_df[
                        raw_column
                    ],
                    errors="coerce",
                )
                .notna()
                .astype(
                    int
                )
            )


    cognitive_mask_columns = (
        list(
            FEATURE_MASK_COLUMNS[
                "adas"
            ].values()
        )
        +
        list(
            FEATURE_MASK_COLUMNS[
                "mmse"
            ].values()
        )
        +
        list(
            FEATURE_MASK_COLUMNS[
                "faq"
            ].values()
        )
    )


    fold_df[
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL"
    ] = (
        fold_df[
            cognitive_mask_columns
        ]
        .sum(
            axis=1
        )
        > 0
    ).astype(
        int
    )


    fold_df[
        "BRANCH_MASK__CSF"
    ] = (
        fold_df[
            list(
                FEATURE_MASK_COLUMNS[
                    "csf"
                ].values()
            )
        ]
        .sum(
            axis=1
        )
        > 0
    ).astype(
        int
    )


    fold_df[
        "BRANCH_MASK__PLASMA"
    ] = (
        fold_df[
            list(
                FEATURE_MASK_COLUMNS[
                    "plasma"
                ].values()
            )
        ]
        .sum(
            axis=1
        )
        > 0
    ).astype(
        int
    )


    # --------------------------------------------------------
    # Refit continuous scaling on training rows only
    # --------------------------------------------------------

    training_rows = (
        fold_df[
            "DATA_ROLE"
        ]
        == "train"
    )


    for modality in [
        "adas",
        "mmse",
        "faq",
        "csf",
        "plasma",
    ]:

        for feature in FEATURE_COLUMNS[
            modality
        ]:

            raw_column = (
                f"RAW_TEMPORAL__"
                f"{modality.upper()}__"
                f"{feature}"
            )

            output_column = (
                MODEL_Z_COLUMNS[
                    modality
                ][
                    feature
                ]
            )


            training_values = (
                pd.to_numeric(
                    fold_df.loc[
                        training_rows,
                        raw_column,
                    ],
                    errors="coerce",
                )
                .dropna()
                .astype(
                    float
                )
            )


            if training_values.empty:
                raise ValueError(
                    f"Fold {outer_fold}, "
                    f"{modality}, {feature}: "
                    "no observed training values."
                )


            train_mean = float(
                training_values.mean()
            )

            train_std = float(
                training_values.std(
                    ddof=0
                )
            )


            if (
                not np.isfinite(
                    train_std
                )
                or
                train_std == 0.0
            ):
                scaling_denominator = 1.0

            else:
                scaling_denominator = (
                    train_std
                )


            all_values = pd.to_numeric(
                fold_df[
                    raw_column
                ],
                errors="coerce",
            )


            fold_df[
                output_column
            ] = (
                (
                    all_values
                    - train_mean
                )
                /
                scaling_denominator
            ).fillna(
                0.0
            )


            scaler_rows.append(
                {
                    "OUTER_FOLD":
                        outer_fold,

                    "MODALITY":
                        modality,

                    "FEATURE":
                        feature,

                    "OBSERVED_TRAINING_VALUES":
                        len(
                            training_values
                        ),

                    "TRAIN_MEAN":
                        train_mean,

                    "TRAIN_STD":
                        train_std,

                    "SCALING_DENOMINATOR":
                        scaling_denominator,
                }
            )


    # --------------------------------------------------------
    # Return to the exact original model-table schema
    # --------------------------------------------------------

    fold_df = fold_df.drop(
        columns=[
            *temporary_raw_columns,
            "PREBASELINE_MRI_PATH",
        ]
    )


    missing_original_columns = [
        column
        for column in original_columns
        if column
        not in fold_df.columns
    ]


    if missing_original_columns:
        raise KeyError(
            f"Fold {outer_fold}: rebuilt table lost "
            "original columns: "
            + ", ".join(
                missing_original_columns
            )
        )


    fold_df = fold_df[
        original_columns
    ]


    output_path = (
        TEMPORAL_FOLD_DIR
        / (
            "mci_prognosis_outer_fold_"
            f"{outer_fold}_final_task_ready.csv"
        )
    )


    fold_df.to_csv(
        output_path,
        index=False,
    )


    fold_inventory_rows.append(
        {
            "OUTER_FOLD":
                outer_fold,

            "ROWS":
                len(
                    fold_df
                ),

            "TRAIN_ROWS":
                int(
                    (
                        fold_df[
                            "DATA_ROLE"
                        ]
                        == "train"
                    ).sum()
                ),

            "VALIDATION_ROWS":
                int(
                    (
                        fold_df[
                            "DATA_ROLE"
                        ]
                        == "validation"
                    ).sum()
                ),

            "TEST_ROWS":
                int(
                    (
                        fold_df[
                            "DATA_ROLE"
                        ]
                        == "test"
                    ).sum()
                ),

            "COGNITIVE_FUNCTIONAL_AVAILABLE":
                int(
                    fold_df[
                        "BRANCH_MASK__COGNITIVE_FUNCTIONAL"
                    ].sum()
                ),

            "CSF_AVAILABLE":
                int(
                    fold_df[
                        "BRANCH_MASK__CSF"
                    ].sum()
                ),

            "PLASMA_AVAILABLE":
                int(
                    fold_df[
                        "BRANCH_MASK__PLASMA"
                    ].sum()
                ),

            "MRI_AVAILABLE":
                int(
                    fold_df[
                        "BRANCH_MASK__MRI"
                    ].sum()
                ),

            "OUTPUT_PATH":
                str(
                    output_path
                ),
        }
    )


fold_inventory = pd.DataFrame(
    fold_inventory_rows
)

scaler_table = pd.DataFrame(
    scaler_rows
)


fold_inventory.to_csv(
    TEMPORAL_AUDIT_DIR
    / "rebuilt_fold_inventory.csv",
    index=False,
)

scaler_table.to_csv(
    TEMPORAL_AUDIT_DIR
    / "training_only_scaler_parameters.csv",
    index=False,
)


display(
    fold_inventory
)


print("=" * 72)
print("PRE-BASELINE FOLD TABLES COMPLETE")
print("=" * 72)

print(
    f"\nFive prepared fold tables saved under:\n"
    f"{TEMPORAL_FOLD_DIR}"
)

## 1.2. B. Train the fixed 50/50 model across all five folds

The following training sections use the five fold files created immediately above.

## 1.3. Running the experiment

Run the notebook from top to bottom. Every fold starts in `fresh` mode and uses a newly initialised model. Its outputs are written to:

```text
models/3mt_tmc_evidential/experiments/
    temporal_prebaseline_fixed_equal_fusion_md050/
        mci_prognosis/fold_X/
```

A fold in `fresh` mode stops before training if its output directory already contains files. After an interrupted run, change only that fold's `FOLD_RUN_MODE` to `"resume"`.


## 1.4. Fold 0

This section trains, validates, and evaluates fold 0. It uses the exact prepared table `mci_prognosis_outer_fold_0_final_task_ready.csv` and writes only to the experiment's `fold_0` directory.


### 1.4.1. Loading the prepared fold-0 input

The model-input columns are defined explicitly in this notebook, matching the original fold-0 implementation. No schema file or output from an earlier experiment is loaded.


In [ ]:
# ============================================================
# Loading the prepared inputs for fold 0
# ============================================================

from pathlib import Path
import json
import pandas as pd

from google.colab import drive


# ------------------------------------------------------------
# Experiment identity
# ------------------------------------------------------------

EXPERIMENT_NAME = "temporal_prebaseline_fixed_equal_fusion_md050"
SELECTED_TASK = "mci_prognosis"
SELECTED_FOLD = 0

# Use "fresh" for the formal first run.
# Change this to "resume" only after an interrupted run of this
# same experiment and fold.
FOLD_RUN_MODE = "fresh"


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():
    drive.mount(
        "/content/drive"
    )


# ------------------------------------------------------------
# Exact project and prepared-input paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

SELECTED_INPUT_PATH = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / "mci_prognosis"
    / f"mci_prognosis_outer_fold_{SELECTED_FOLD}_final_task_ready.csv"
)


# ------------------------------------------------------------
# Fixed model-input columns from the prepared pipeline
# ------------------------------------------------------------

final_model_schema = {
    "identifier_columns": [
        "RID",
        "PTID",
    ],

    "audit_label_columns": [
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ],

    "model_target_column":
        "MODEL_TARGET",

    "split_columns": [
        "OUTER_FOLD",
        "DATA_ROLE",
    ],

    "scaled_continuous_columns": [
        "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
        "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        "ADAS__TOTSCORE__Z",
        "ADAS__TOTAL13__Z",
        "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
        "FAQ__FAQTOTAL__Z",
        "CSF__ABETA40__Z",
        "CSF__ABETA42__Z",
        "CSF__TAU__Z",
        "CSF__PTAU__Z",
        "CSF__ABETA42_40_RATIO__Z",
        "PLASMA__pT217_F__Z",
        "PLASMA__AB42_F__Z",
        "PLASMA__AB40_F__Z",
        "PLASMA__AB42_AB40_F__Z",
        "PLASMA__pT217_AB42_F__Z",
        "PLASMA__NfL_Q__Z",
        "PLASMA__GFAP_Q__Z",
        "PLASMA__NfL_F__Z",
        "PLASMA__GFAP_F__Z",
    ],

    "encoded_categorical_columns": [
        "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
        "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        "APOE__APOE4_ALLELE_COUNT__IDX",
    ],

    "mri_path_columns": [
        "MRI__NORMALIZED_T1_NPY_PATH",
    ],

    "branch_mask_columns": [
        "BRANCH_MASK__DEMOGRAPHICS",
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
        "BRANCH_MASK__CSF",
        "BRANCH_MASK__PLASMA",
        "BRANCH_MASK__APOE",
        "BRANCH_MASK__MRI",
    ],

    "feature_mask_columns": [
        "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
        "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        "FEATURE_MASK__ADAS_TOTSCORE",
        "FEATURE_MASK__ADAS_TOTAL13",
        "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
        "FEATURE_MASK__FAQ_FAQTOTAL",
        "FEATURE_MASK__CSF_ABETA40",
        "FEATURE_MASK__CSF_ABETA42",
        "FEATURE_MASK__CSF_TAU",
        "FEATURE_MASK__CSF_PTAU",
        "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        "FEATURE_MASK__PLASMA_pT217_F",
        "FEATURE_MASK__PLASMA_AB42_F",
        "FEATURE_MASK__PLASMA_AB40_F",
        "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "FEATURE_MASK__PLASMA_NfL_Q",
        "FEATURE_MASK__PLASMA_GFAP_Q",
        "FEATURE_MASK__PLASMA_NfL_F",
        "FEATURE_MASK__PLASMA_GFAP_F",
        "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
    ],
}



# ------------------------------------------------------------
# Load the prepared fold table
# ------------------------------------------------------------

fold_table = pd.read_csv(
    SELECTED_INPUT_PATH
)


print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} INPUT")
print("=" * 72)

print(
    f"\nTask-ready table:\n{SELECTED_INPUT_PATH}"
)

print(
    f"\nLoaded shape: "
    f"{fold_table.shape[0]} rows × "
    f"{fold_table.shape[1]} columns"
)


### 1.4.2. Preparing the fold-0 model-input groups

The continuous, categorical, MRI-path, branch-mask, feature-mask, identifier, target, and split columns are taken from the fixed definitions loaded above.


In [ ]:
# ============================================================
# Preparing the model-input groups for fold 0
# ============================================================

# ------------------------------------------------------------
# Prepared model-input column groups
# ------------------------------------------------------------

identifier_columns = final_model_schema[
    "identifier_columns"
]

audit_label_columns = final_model_schema[
    "audit_label_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

split_columns = final_model_schema[
    "split_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]


# ------------------------------------------------------------
# Fold-0 structure
# ------------------------------------------------------------

print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} EXPERIMENT")
print("=" * 72)

print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Task: {SELECTED_TASK}")
print(f"Outer fold: {SELECTED_FOLD}")

print("\nPrepared data roles:")
print(
    fold_table["DATA_ROLE"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared target counts by role:")
display(
    pd.crosstab(
        fold_table["DATA_ROLE"],
        fold_table[target_column],
    ).reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared input groups:")
print(
    f"- scaled continuous features: "
    f"{len(scaled_continuous_columns)}"
)
print(
    f"- encoded categorical features: "
    f"{len(encoded_categorical_columns)}"
)
print(
    f"- MRI path columns: "
    f"{len(mri_path_columns)}"
)
print(
    f"- branch masks: "
    f"{len(branch_mask_columns)}"
)
print(
    f"- feature masks: "
    f"{len(feature_mask_columns)}"
)

preview_columns = (
    identifier_columns
    + ["CLINICAL_GROUP"]
    + split_columns
    + [target_column]
    + branch_mask_columns
    + mri_path_columns
)

print("\nExample prepared rows:")
display(
    fold_table[
        preview_columns
    ].head(5)
)


### 1.4.3. Defining the multimodal dataset-output contract

The model will receive each modality as a separate input branch rather than as one combined feature vector.

Continuous and categorical variables are kept separate because they require different encoder operations. Continuous variables will enter small numerical encoders, while categorical variables will later be represented through trainable embeddings.

Each sample will also contain:

- the participant identifier;
- the binary prognosis target;
- branch-level availability masks;
- feature-level observation masks;
- the prepared MRI path.

The dataset will not load MRI arrays yet. At this stage, I define the column organisation and the exact sample structure that the PyTorch dataset will later return.

For participants without MRI, the MRI path remains unavailable and the MRI branch mask remains zero. The participant is retained in the dataset.

In [ ]:
# ============================================================
# 3. Defining the multimodal dataset-output contract
# ============================================================

# ------------------------------------------------------------
# Branch-specific predictor columns
# ------------------------------------------------------------

# I organise the prepared continuous and categorical columns into
# the six modality branches used by the architecture.

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


# ------------------------------------------------------------
# Dataset sample structure
# ------------------------------------------------------------

# One participant will later be returned by the PyTorch dataset
# using the following nested structure.
#
# Continuous features will become float32 tensors.
# Categorical indices will become int64 tensors for embeddings.
# Masks will become float32 tensors containing 0 or 1.
# The target will become an int64 class index.

dataset_output_contract = {
    "rid": "Participant RID as an integer",
    "ptid": "Participant PTID as a string",
    "target": "Binary class index: 0 for sMCI and 1 for pMCI",

    "modalities": {
        "demographics": {
            "continuous": "Shape (2,), float32",
            "categorical": "Shape (2,), int64",
            "feature_mask": "Shape (4,), float32",
            "branch_mask": "Scalar, float32",
        },

        "cognitive_functional": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "csf": {
            "continuous": "Shape (5,), float32",
            "categorical": None,
            "feature_mask": "Shape (5,), float32",
            "branch_mask": "Scalar, float32",
        },

        "plasma": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "apoe": {
            "continuous": None,
            "categorical": "Shape (1,), int64",
            "feature_mask": "Shape (1,), float32",
            "branch_mask": "Scalar, float32",
        },

        "mri": {
            "path": "Prepared NumPy path or None",
            "image": "Later: shape (1, 177, 213, 183), float32",
            "branch_mask": "Scalar, float32",
        },
    },

    "branch_masks": (
        "Shape (6,), float32, ordered as "
        "demographics, cognitive-functional, CSF, "
        "plasma, APOE, MRI"
    ),

    "feature_masks": (
        "Shape (28,), float32, using the authoritative "
        "feature-mask order"
    ),
}


# ------------------------------------------------------------
# Display the agreed contract
# ------------------------------------------------------------

print("=" * 72)
print("MULTIMODAL DATASET CONTRACT")
print("=" * 72)

print("\nBranch-specific input dimensions:")

for branch_name, branch_definition in dataset_column_contract.items():

    continuous_count = len(
        branch_definition.get("continuous", [])
    )

    categorical_count = len(
        branch_definition.get("categorical", [])
    )

    feature_mask_count = len(
        branch_definition.get("feature_masks", [])
    )

    has_mri_path = "path" in branch_definition

    print(
        f"- {branch_name}: "
        f"{continuous_count} continuous, "
        f"{categorical_count} categorical, "
        f"{feature_mask_count} feature masks"
        + (", 1 MRI path" if has_mri_path else "")
    )


print("\nPlanned sample output:")
print(
    json.dumps(
        dataset_output_contract,
        indent=2,
    )
)

print(
    "\nThis contract will be used in the next step to "
    "implement the PyTorch dataset."
)

### 1.4.4. Implementing the multimodal PyTorch dataset

implement a PyTorch dataset that converts each prepared participant row into the agreed multimodal structure.

The dataset preserves the six modality branches and returns continuous variables, categorical indices, observation masks, participant identifiers, and the prognosis target separately.

MRI volumes are loaded only when a sample is requested. The existing preprocessed NumPy array is used directly, and a channel dimension is added to produce the shape required by a three-dimensional neural network.

When MRI is unavailable, the participant remains in the dataset. The dataset returns a zero placeholder volume together with an MRI branch mask of zero, allowing the model to distinguish an unavailable scan from an observed image.

In [ ]:
# ============================================================
# 4. Implementing the multimodal PyTorch dataset
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset


# ------------------------------------------------------------
# Expected prepared MRI shape
# ------------------------------------------------------------

# I preserve the spatial dimensions produced by the completed
# MRI preprocessing pipeline.
MRI_SPATIAL_SHAPE = (177, 213, 183)

# I add one channel dimension when returning an MRI tensor.
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)


# ------------------------------------------------------------
# Multimodal PyTorch dataset
# ------------------------------------------------------------

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # I reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # I retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # I preserve the authoritative mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri


    def __len__(self):
        return len(self.dataframe)


    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # I load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # I ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)


    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # --------------------------------------------------------
        # Demographics
        # --------------------------------------------------------

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Cognitive and functional measures
        # --------------------------------------------------------

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # CSF
        # --------------------------------------------------------

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Plasma
        # --------------------------------------------------------

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # APOE
        # --------------------------------------------------------

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # MRI
        # --------------------------------------------------------

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Complete sample
        # --------------------------------------------------------

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample


# ------------------------------------------------------------
# Create role-specific datasets
# ------------------------------------------------------------

# I retain the prepared role assignments exactly as stored in
# the selected outer-fold table.
train_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "train"
    ]
    .reset_index(drop=True)
)

validation_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "validation"
    ]
    .reset_index(drop=True)
)

test_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "test"
    ]
    .reset_index(drop=True)
)


# I initially disable MRI disk loading so that I can inspect the
# dataset structure quickly before constructing the DataLoaders.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)


# ------------------------------------------------------------
# Concise dataset summary
# ------------------------------------------------------------

print("=" * 72)
print("PYTORCH DATASETS")
print("=" * 72)

print(f"\nTraining participants: {len(train_dataset)}")
print(f"Validation participants: {len(validation_dataset)}")
print(f"Test participants: {len(test_dataset)}")

print(
    "\nThe datasets preserve the prepared role assignments "
    "and return separate modality inputs."
)

print(
    "MRI loading is temporarily disabled for structural "
    "inspection and will be enabled for the DataLoaders."
)

### 1.4.5. Inspecting one multimodal sample and one prepared MRI volume

Before constructing the DataLoaders, The notebook inspects the structure returned for one participant.

use the dataset with MRI loading disabled to confirm the scalar tensors, categorical indices, targets, and masks. I then create a temporary MRI-enabled dataset and load one participant whose MRI branch is available.

This checks the dataset interface required by the model while avoiding unnecessary loading of multiple MRI volumes at this stage.

In [ ]:
# ============================================================
# 5. Inspecting one multimodal sample and one MRI volume
# ============================================================

# ------------------------------------------------------------
# Inspect one scalar-only training sample
# ------------------------------------------------------------

# I retrieve one participant while MRI disk loading remains
# disabled. The returned MRI tensor is therefore only the
# temporary placeholder defined in the current dataset class.
sample = train_dataset[0]

print("=" * 72)
print("EXAMPLE MULTIMODAL SAMPLE")
print("=" * 72)

print(f"\nRID: {sample['rid']}")
print(f"PTID: {sample['ptid']}")
print(f"Target: {sample['target'].item()}")

print("\nModality tensor structure:")

for modality_name, modality_data in sample["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"{value}"
            )

print(
    f"\nComplete branch-mask shape: "
    f"{tuple(sample['branch_masks'].shape)}"
)

print(
    f"Complete feature-mask shape: "
    f"{tuple(sample['feature_masks'].shape)}"
)


# ------------------------------------------------------------
# Find one participant with an available MRI
# ------------------------------------------------------------

# I select the first training participant whose prepared MRI
# branch mask is one.
example_mri_index = train_table.index[
    train_table["BRANCH_MASK__MRI"] == 1
][0]


# ------------------------------------------------------------
# Create a temporary MRI-enabled dataset
# ------------------------------------------------------------

# I enable MRI loading only for this temporary inspection
# dataset. The main role-specific datasets remain unchanged.
mri_inspection_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

mri_sample = mri_inspection_dataset[
    example_mri_index
]

mri_tensor = mri_sample[
    "modalities"
]["mri"]["image"]

mri_branch_mask = mri_sample[
    "modalities"
]["mri"]["branch_mask"]


# ------------------------------------------------------------
# Display the prepared MRI tensor information
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXAMPLE PREPARED MRI")
print("=" * 72)

print(f"\nRID: {mri_sample['rid']}")
print(f"PTID: {mri_sample['ptid']}")

print(
    f"MRI branch mask: "
    f"{mri_branch_mask.item():.0f}"
)

print(
    f"MRI tensor shape: "
    f"{tuple(mri_tensor.shape)}"
)

print(
    f"MRI tensor dtype: "
    f"{mri_tensor.dtype}"
)

print(
    f"MRI intensity minimum: "
    f"{mri_tensor.min().item():.6f}"
)

print(
    f"MRI intensity maximum: "
    f"{mri_tensor.max().item():.6f}"
)

print(
    f"MRI intensity mean: "
    f"{mri_tensor.mean().item():.6f}"
)

print(
    "\nThe sample structure and full prepared MRI volume "
    "are ready for DataLoader construction."
)

### 1.4.6. Constructing the multimodal DataLoaders

create separate DataLoaders for the fixed training, validation, and test subsets.

MRI loading is enabled, so an available scan is read lazily from its prepared NumPy path when its participant enters a batch. Participants without MRI receive a zero placeholder volume and retain an MRI branch mask of zero.

I begin with a small batch size because each sample contains a full three-dimensional MRI volume. The final training batch size will be selected later according to the memory requirements of the complete model.

The training DataLoader shuffles participants. Validation and test DataLoaders preserve a deterministic order.

In [ ]:
# ============================================================
# 6. Constructing the multimodal DataLoaders
# ============================================================

from torch.utils.data import DataLoader


# ------------------------------------------------------------
# Initial DataLoader settings
# ------------------------------------------------------------

# I begin with a small batch because each participant may contain
# a full MRI volume with shape (1, 177, 213, 183).
INITIAL_BATCH_SIZE = 2

# I initially use the main process for data loading. This is the
# most reliable starting configuration when reading NumPy files
# from mounted Google Drive.
NUM_WORKERS = 0

# Pinned memory can speed transfers to a CUDA device.
PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# Recreate the datasets with MRI loading enabled
# ------------------------------------------------------------

# Available MRI volumes will now be read lazily when requested.
# Missing MRI branches will retain their zero placeholders and
# branch masks of zero.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)


# ------------------------------------------------------------
# Create role-specific DataLoaders
# ------------------------------------------------------------

# I shuffle only the training subset.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# Validation order does not need to be shuffled.
validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# The test subset also retains a deterministic order.
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)


# ------------------------------------------------------------
# Retrieve one complete training batch
# ------------------------------------------------------------

example_batch = next(iter(train_loader))


# ------------------------------------------------------------
# Display the batched tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("EXAMPLE MULTIMODAL BATCH")
print("=" * 72)

print(f"\nBatch size: {example_batch['target'].shape[0]}")
print(f"RID values: {example_batch['rid']}")
print(f"PTID values: {example_batch['ptid']}")
print(f"Targets: {example_batch['target']}")

print("\nModality tensors:")

for modality_name, modality_data in example_batch["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"type={type(value).__name__}"
            )


print("\nCombined masks:")

print(
    "  branch_masks: "
    f"shape={tuple(example_batch['branch_masks'].shape)}, "
    f"dtype={example_batch['branch_masks'].dtype}"
)

print(
    "  feature_masks: "
    f"shape={tuple(example_batch['feature_masks'].shape)}, "
    f"dtype={example_batch['feature_masks'].dtype}"
)


# ------------------------------------------------------------
# Show MRI availability in this batch
# ------------------------------------------------------------

batch_mri = example_batch[
    "modalities"
]["mri"]["image"]

batch_mri_masks = example_batch[
    "modalities"
]["mri"]["branch_mask"]

print("\nMRI batch:")

print(
    f"  image shape: {tuple(batch_mri.shape)}"
)

print(
    f"  branch masks: {batch_mri_masks}"
)

print(
    f"  approximate raw MRI batch size: "
    f"{batch_mri.numel() * batch_mri.element_size() / (1024 ** 2):.2f} MB"
)


# ------------------------------------------------------------
# DataLoader summary
# ------------------------------------------------------------

print("\nDataLoader batches:")

print(
    f"  training: {len(train_loader)} batches"
)

print(
    f"  validation: {len(validation_loader)} batches"
)

print(
    f"  test: {len(test_loader)} batches"
)

print(
    "\nThe complete multimodal batch is ready for "
    "modality-specific encoder construction."
)

### 1.4.7. Building the modality-specific encoders

Each modality has a different input structure, so I encode the six branches separately before multimodal interaction.

For the scalar branches, the prepared feature values are combined with their feature-observation masks. This allows the encoders to distinguish an observed standardised value close to zero from a missing-value placeholder.

Categorical variables use trainable embeddings. Encoded index zero remains reserved for missing or unseen values and is handled through embedding padding behaviour.

### 1.4.8. Common latent dimension

Every branch is projected into the same latent dimension:

$$
d_{\mathrm{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore produces:

$$
\mathbf{H}^{(m)}
\in
\mathbb{R}^{B \times 64}.
$$

The shared dimension is required so that all modality representations can later enter the same cascaded cross-modal transformer architecture.

The original interaction pathway implementation used a dimension of \(512\). In this thesis, I begin with a more compact dimension of \(64\) because the main prognosis training folds contain approximately \(369\) participants and the complete model will additionally include independent evidential heads, modality-specific evidence fusion, auxiliary outputs, and the cascaded interaction pathway interaction path.

The embedding dimension remains a model-capacity hyperparameter and may later be compared with larger values using only training and validation data.

### 1.4.9. MRI encoder

The MRI branch follows the general image-encoding structure used by interaction pathway:

1. initial three-dimensional convolutions;
2. residual three-dimensional downsampling blocks;
3. conversion of the final feature map into patch tokens;
4. addition of learned positional embeddings;
5. transformer encoding of patch-wise relationships;
6. global averaging across patch tokens;
7. projection into the shared modality dimension.

The original paper used a 512-dimensional MRI output. Here, the final MRI representation is projected to the common 64-dimensional latent space used by the other branches.

The prepared MRI volumes are larger than those used in the original interaction pathway experiments. I therefore apply stride-two downsampling in the initial convolutional stem before the four residual blocks. This preserves the CNN-transformer design while keeping the number of transformer patch tokens computationally manageable.

No new MRI preprocessing is performed. The encoder receives the complete prepared volume with shape:

$$
(1, 177, 213, 183).
$$

In [ ]:
# ============================================================
# 7. Building the modality-specific encoders
# ============================================================

import math

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Shared representation dimension
# ------------------------------------------------------------

# I project every modality into one common latent space so that
# all branches can later enter the same cross-modal transformers.
MODALITY_EMBEDDING_DIM = 64


# ------------------------------------------------------------
# Reusable scalar encoder
# ------------------------------------------------------------

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        values,
        feature_mask,
    ):
        # I supply both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Demographics encoder
# ------------------------------------------------------------

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# APOE encoder
# ------------------------------------------------------------

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Residual 3D downsampling block
# ------------------------------------------------------------

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original interaction pathway image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()


    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )


# ------------------------------------------------------------
# interaction pathway-style CNN-transformer MRI encoder
# ------------------------------------------------------------

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # --------------------------------------------------------
        # Initial convolutional stem
        # --------------------------------------------------------

        # I use two initial 3D convolutions, following the broad
        # structure shown in the interaction pathway image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original interaction pathway inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )


        # --------------------------------------------------------
        # Four residual downsampling blocks
        # --------------------------------------------------------

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )


        # --------------------------------------------------------
        # Determine the resulting patch grid
        # --------------------------------------------------------

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )


        # --------------------------------------------------------
        # Learned positional embeddings
        # --------------------------------------------------------

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )


        # --------------------------------------------------------
        # Patch-wise transformer encoder
        # --------------------------------------------------------

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )


        # --------------------------------------------------------
        # Projection to the shared modality dimension
        # --------------------------------------------------------

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )


    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # I flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # I add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )


# ------------------------------------------------------------
# Complete set of six modality encoders
# ------------------------------------------------------------

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )


    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations


# ------------------------------------------------------------
# Instantiate the revised encoders
# ------------------------------------------------------------

modality_encoders = ADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Count trainable parameters
# ------------------------------------------------------------

def count_trainable_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print("=" * 72)
print("MODALITY-SPECIFIC ENCODERS")
print("=" * 72)

print(
    f"\nShared modality embedding dimension: "
    f"{MODALITY_EMBEDDING_DIM}"
)

print(
    "\nMRI transformer patch grid: "
    f"{modality_encoders.mri.patch_grid_shape}"
)

print(
    "MRI transformer patch count: "
    f"{modality_encoders.mri.number_of_patches}"
)

print("\nTrainable parameters by encoder:")

for encoder_name in [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]:
    encoder = getattr(
        modality_encoders,
        encoder_name,
    )

    print(
        f"- {encoder_name}: "
        f"{count_trainable_parameters(encoder):,}"
    )

print(
    "\nTotal trainable encoder parameters: "
    f"{count_trainable_parameters(modality_encoders):,}"
)

print(
    "\nThe revised MRI branch now uses a 3D CNN, patch tokens, "
    "positional embeddings, and a transformer encoder."
)

print(
    "No multimodal fusion or classification head has been "
    "added yet."
)

### 1.4.10. Applying branch-availability masks to the encoded modalities

Each modality has a different original input structure, but every modality-specific encoder projects its input into the same latent dimensionality.

In this implementation, each branch produces a representation of size:

$$
d_{\text{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore returns:

$$
\mathbf{H}^{(m)} \in \mathbb{R}^{B \times 64},
$$

where \(m\) denotes one of the six modalities:

- demographics;
- cognitive-functional measures;
- CSF;
- plasma;
- APOE;
- MRI.

The shared dimensionality does not mean that the modalities contain the same information or use equally complex encoders. Each branch has its own input-specific encoder, but the final representations must have a common size so that they can later participate in cross-modal attention and evidential fusion.

After stacking the six modality representations, the model obtains:

$$
\mathbf{H}
\in
\mathbb{R}^{B \times 6 \times 64}.
$$

This follows the general design principle used by interaction pathway, in which heterogeneous modality inputs are first projected into a common transformer embedding space before cross-modal interaction. The original interaction pathway implementation used a larger embedding dimension of \(512\), but \(512\) is an architectural hyperparameter rather than a methodological requirement.

A compact starting dimension of \(64\) is used here because the main MCI prognosis training folds contain only approximately \(369\) participants, while the complete planned model will also contain:

- six modality-specific encoders;
- cascaded cross-modal attention;
- independent evidential heads;
- evidence pathway-style evidential fusion;
- a joint evidential prediction path.

The number of parameters in transformer projections grows approximately with the square of the embedding dimension. For example:

$$
64^2 = 4{,}096,
$$

whereas:

$$
512^2 = 262{,}144.
$$

Thus, increasing the embedding dimension from \(64\) to \(512\) can make several attention and feed-forward parameter blocks approximately \(64\) times larger. A \(512\)-dimensional model would therefore introduce substantially greater overfitting and memory risk for the available prognosis cohort.

The value \(64\) is treated as a compact initial configuration rather than as a permanently fixed optimum. The latent dimensionality can later be compared with alternatives such as \(128\) or \(256\), using only the training and validation subsets.

### 1.4.11. Branch-availability masking

The prepared zero placeholders make missing inputs computationally compatible with neural-network layers, but they do not guarantee that an unavailable modality will produce a zero encoder output.

Linear layers, embeddings, normalisation parameters, and learned biases can generate a non-zero representation even when all supplied inputs are zero. I therefore apply the prepared branch mask after each modality encoder.

For participant \(i\) and modality \(m\), the masked representation is:

$$
\widetilde{\mathbf{h}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{h}_{i}^{(m)},
$$

where:

- \(\mathbf{h}_{i}^{(m)} \in \mathbb{R}^{64}\) is the raw modality representation;
- \(a_{i}^{(m)} \in \{0,1\}\) is the prepared branch-availability mask;
- \(\widetilde{\mathbf{h}}_{i}^{(m)} \in \mathbb{R}^{64}\) is the masked representation passed to later model components.

Therefore:

$$
a_{i}^{(m)} = 1
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{h}_{i}^{(m)},
$$

and:

$$
a_{i}^{(m)} = 0
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{0}.
$$

An unavailable modality consequently contributes an exact zero representation rather than a learned bias-derived vector. The original branch masks are also retained separately so that the later cross-modal attention and evidential-fusion components can explicitly identify which modalities are available for each participant.

In [ ]:
# ============================================================
# 8. Applying branch masks to the encoded modalities
# ============================================================

# ------------------------------------------------------------
# Fixed modality order
# ------------------------------------------------------------

# I use one explicit modality order throughout the architecture.
# This order matches the prepared branch-mask columns.
MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]


# ------------------------------------------------------------
# Mask-aware encoder wrapper
# ------------------------------------------------------------

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )


    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # I add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask


    def forward(self, modalities):
        # I first obtain the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # I then suppress every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # I return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }


# ------------------------------------------------------------
# Instantiate the mask-aware encoder collection
# ------------------------------------------------------------

masked_modality_encoders = MaskedADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Apply the encoders to one complete batch
# ------------------------------------------------------------

# I keep this inspection on the CPU. The training device will be
# configured later when the full model and optimisation loop exist.
masked_modality_encoders.eval()

with torch.no_grad():
    encoded_batch = masked_modality_encoders(
        example_batch["modalities"]
    )


# ------------------------------------------------------------
# Stack modality representations
# ------------------------------------------------------------

# I stack the representations in the fixed modality order.
#
# Resulting shape:
# (batch_size, number_of_modalities, embedding_dimension)
stacked_masked_representations = torch.stack(
    [
        encoded_batch["masked"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_raw_representations = torch.stack(
    [
        encoded_batch["raw"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the encoded structure
# ------------------------------------------------------------

print("=" * 72)
print("MASKED MODALITY REPRESENTATIONS")
print("=" * 72)

print(
    f"\nStacked raw representation shape: "
    f"{tuple(stacked_raw_representations.shape)}"
)

print(
    f"Stacked masked representation shape: "
    f"{tuple(stacked_masked_representations.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, embedding)"
)

print("\nBranch masks in this batch:")

branch_mask_table = pd.DataFrame(
    example_batch["branch_masks"].numpy(),
    columns=MODALITY_ORDER,
)

display(branch_mask_table)


# ------------------------------------------------------------
# Representation norms before and after masking
# ------------------------------------------------------------

# A representation norm summarises the magnitude of each branch
# vector. Missing branches may have non-zero raw norms because of
# learned biases, but their masked norms must be exactly zero.
raw_norms = torch.linalg.vector_norm(
    stacked_raw_representations,
    dim=-1,
)

masked_norms = torch.linalg.vector_norm(
    stacked_masked_representations,
    dim=-1,
)

norm_summary = []

for participant_index in range(
    stacked_masked_representations.shape[0]
):
    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):
        norm_summary.append(
            {
                "BATCH_ROW": participant_index,
                "RID": int(
                    example_batch["rid"][
                        participant_index
                    ].item()
                ),
                "MODALITY": modality_name,
                "BRANCH_MASK": float(
                    example_batch["branch_masks"][
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "RAW_REPRESENTATION_NORM": float(
                    raw_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "MASKED_REPRESENTATION_NORM": float(
                    masked_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
            }
        )

norm_summary = pd.DataFrame(norm_summary)

print("\nRepresentation norms before and after masking:")

display(
    norm_summary.round(6)
)


# ------------------------------------------------------------
# Confirm the representation dimensions
# ------------------------------------------------------------

print("\nEncoded modality shapes:")

for modality_name in MODALITY_ORDER:
    print(
        f"- {modality_name}: "
        f"{tuple(encoded_batch['masked'][modality_name].shape)}"
    )

print(
    "\nOnly the masked representations will enter the "
    "multimodal interaction and evidential-fusion paths."
)

### 1.4.12. Building the availability-gated interaction pathway cascade

The cascade order remains unchanged:

$$
\text{demographics}
\rightarrow
\text{APOE}
\rightarrow
\text{cognitive/functional}
\rightarrow
\text{CSF}
\rightarrow
\text{plasma}
\rightarrow
\text{MRI}.
$$

Each cross-modal interaction block still computes a candidate update using query self-attention followed by cross-attention to the encoded modality token. The branch mask then determines whether that candidate becomes the next cumulative query:

$$
\mathbf{q}_m
=
a_m\mathbf{q}^{\mathrm{candidate}}_m
+
(1-a_m)\mathbf{q}_{m-1}.
$$

For an available modality, $a_m=1$ and the candidate update is used. For an unavailable modality, $a_m=0$ and the stage becomes an exact identity update.

The branch-mask tensor follows `MODALITY_ORDER`, while the cross-modal interaction blocks follow `THREE_MT_CASCADE_ORDER`. The class therefore uses the explicit modality-to-mask index mapping already defined by the fold-0 architecture.

In [ ]:
# ============================================================
# 9. Building the availability-gated interaction pathway cascade
# ============================================================

# ------------------------------------------------------------
# Fixed cascade order
# ------------------------------------------------------------

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]


# ------------------------------------------------------------
# One Cascaded Modality Transformer
# ------------------------------------------------------------

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }


# ------------------------------------------------------------
# Complete six-stage availability-gated cascade
# ------------------------------------------------------------

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )


    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }


# ------------------------------------------------------------
# Instantiate and inspect the gated cascade
# ------------------------------------------------------------

three_mt_cascade = ThreeMTCascade(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    number_of_heads=4,
    dropout=0.10,
)

three_mt_cascade.eval()

with torch.no_grad():
    three_mt_output = three_mt_cascade(
        masked_representations=encoded_batch[
            "masked"
        ],
        branch_masks=example_batch[
            "branch_masks"
        ],
    )


print("=" * 72)
print("AVAILABILITY-GATED 3MT CASCADE")
print("=" * 72)

print(
    f"\nCascade order:\n"
    f"{THREE_MT_CASCADE_ORDER}"
)

print(
    "\nFinal joint representation shape: "
    f"{tuple(three_mt_output['joint_representation'].shape)}"
)


# ------------------------------------------------------------
# Show the actual query update at every stage
# ------------------------------------------------------------

query_change_rows = []

previous_query = (
    three_mt_cascade
    .learned_latent_query
    .expand(
        example_batch["target"].shape[0],
        -1,
        -1,
    )
)

for modality_name in THREE_MT_CASCADE_ORDER:
    current_query = three_mt_output[
        "stage_queries"
    ][modality_name]

    query_change_norm = torch.linalg.vector_norm(
        current_query - previous_query,
        dim=-1,
    ).squeeze(1)

    modality_index = MODALITY_ORDER.index(
        modality_name
    )

    branch_mask = example_batch[
        "branch_masks"
    ][
        :,
        modality_index,
    ]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):
        query_change_rows.append(
            {
                "BATCH_ROW": batch_row,
                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),
                "CASCADE_STAGE": modality_name,
                "BRANCH_MASK": float(
                    branch_mask[
                        batch_row
                    ].item()
                ),
                "QUERY_CHANGE_NORM": float(
                    query_change_norm[
                        batch_row
                    ].item()
                ),
            }
        )

    previous_query = current_query

query_change_summary = pd.DataFrame(
    query_change_rows
)

print(
    "\nQuery changes after availability gating:"
)

display(
    query_change_summary.round(6)
)

print(
    "\nTrainable cascade parameters: "
    f"{count_trainable_parameters(three_mt_cascade):,}"
)


### 1.4.13. Producing independent modality-specific evidential opinions

The interaction pathway cascade produces a cumulative interaction-aware representation, but its intermediate states are not independent modality opinions because every stage contains information inherited from earlier stages.

Trusted Multi-View Classification requires each modality to produce its own class evidence before cross-modal interaction.

For participant \(i\), modality \(m\), and class \(k\), the modality-specific evidential head produces non-negative evidence:

$$
e_{ik}^{(m)}
=
\operatorname{Softplus}
\left(
\mathbf{W}_{m}
\mathbf{z}_{i}^{(m)}
+
\mathbf{b}_{m}
\right),
$$

where:

- \(\mathbf{z}_{i}^{(m)} \in \mathbb{R}^{64}\) is the independently encoded modality representation;
- \(e_{ik}^{(m)} \geq 0\) is the evidence assigned to class \(k\);
- each modality has its own evidential head.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{(m)}
=
e_{ik}^{(m)} + 1.
$$

For the binary prognosis task:

$$
K = 2,
$$

with class order:

$$
[\mathrm{sMCI},\mathrm{pMCI}].
$$

The expected class probabilities are:

$$
p_{ik}^{(m)}
=
\frac{
\alpha_{ik}^{(m)}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{(m)}
}.
$$

The Dirichlet strength is:

$$
S_{i}^{(m)}
=
\sum_{k=1}^{K}
\alpha_{ik}^{(m)}.
$$

The standard evidential uncertainty mass is:

$$
u_{i}^{(m)}
=
\frac{K}{
S_{i}^{(m)}
}.
$$

Low total evidence produces high uncertainty, while stronger evidence produces lower uncertainty.

### 1.4.14. Treatment of unavailable modalities

Only available modalities should contribute an opinion to modality-specific evidence fusion.

For an unavailable branch, I do not interpret the evidential head output as a genuine prediction. Instead, its effective evidence is set to zero:

$$
\widetilde{\mathbf{e}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{e}_{i}^{(m)},
$$

where \(a_{i}^{(m)}\) is the prepared branch mask.

Therefore, an unavailable modality receives:

$$
\widetilde{\boldsymbol{\alpha}}_{i}^{(m)}
=
\mathbf{1},
$$

which is the uniform Dirichlet opinion with:

$$
u_{i}^{(m)} = 1.
$$

The original branch mask is retained so that the next step can exclude unavailable opinions explicitly during evidence pathway/Dempster--Shafer fusion.

At this stage, I construct and inspect the six independent evidential opinions. I do not yet fuse them or combine them with the interaction pathway joint representation.

In [ ]:
# ============================================================
# 10. Producing independent modality-specific evidential opinions
# ============================================================

# ------------------------------------------------------------
# Binary prognosis class definition
# ------------------------------------------------------------

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]


# ------------------------------------------------------------
# One modality-specific evidential head
# ------------------------------------------------------------

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(
        self,
        representation,
        branch_mask,
    ):
        # I use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }


# ------------------------------------------------------------
# Independent evidential heads for all six modalities
# ------------------------------------------------------------

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )


    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions


# ------------------------------------------------------------
# Instantiate the independent evidential path
# ------------------------------------------------------------

independent_evidential_heads = (
    IndependentModalityEvidentialHeads(
        modality_order=MODALITY_ORDER,
        input_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
    )
)


# ------------------------------------------------------------
# Produce one opinion per modality
# ------------------------------------------------------------

independent_evidential_heads.eval()

with torch.no_grad():

    modality_opinions = (
        independent_evidential_heads(
            # I use the independently encoded branch outputs before
            # they enter the interaction pathway cascade.
            modality_representations=encoded_batch[
                "masked"
            ],

            modalities=example_batch[
                "modalities"
            ],
        )
    )


# ------------------------------------------------------------
# Stack the opinion tensors
# ------------------------------------------------------------

stacked_modality_evidence = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["evidence"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_alpha = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["alpha"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_probabilities = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["probabilities"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_uncertainty = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["uncertainty"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the evidential tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("INDEPENDENT MODALITY EVIDENTIAL OPINIONS")
print("=" * 72)

print(
    f"\nClass order: "
    f"{PROGNOSIS_CLASS_ORDER}"
)

print(
    "\nStacked evidence shape: "
    f"{tuple(stacked_modality_evidence.shape)}"
)

print(
    "Stacked Dirichlet-alpha shape: "
    f"{tuple(stacked_modality_alpha.shape)}"
)

print(
    "Stacked probability shape: "
    f"{tuple(stacked_modality_probabilities.shape)}"
)

print(
    "Stacked uncertainty shape: "
    f"{tuple(stacked_modality_uncertainty.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, class)"
)


# ------------------------------------------------------------
# Create a readable modality-opinion summary
# ------------------------------------------------------------

opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):

        branch_mask = float(
            example_batch["branch_masks"][
                batch_row,
                modality_index,
            ].item()
        )

        alpha_values = stacked_modality_alpha[
            batch_row,
            modality_index,
        ]

        probability_values = (
            stacked_modality_probabilities[
                batch_row,
                modality_index,
            ]
        )

        uncertainty_value = (
            stacked_modality_uncertainty[
                batch_row,
                modality_index,
                0,
            ]
        )

        opinion_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "MODALITY": modality_name,

                "BRANCH_MASK": branch_mask,

                "ALPHA_sMCI": float(
                    alpha_values[0].item()
                ),

                "ALPHA_pMCI": float(
                    alpha_values[1].item()
                ),

                "P_sMCI": float(
                    probability_values[0].item()
                ),

                "P_pMCI": float(
                    probability_values[1].item()
                ),

                "UNCERTAINTY": float(
                    uncertainty_value.item()
                ),
            }
        )


modality_opinion_summary = pd.DataFrame(
    opinion_rows
)

print("\nIndependent modality opinions:")

display(
    modality_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable independent evidential-head parameters: "
    f"{count_trainable_parameters(independent_evidential_heads):,}"
)

print(
    "\nUnavailable modalities should have alpha=[1, 1], "
    "probabilities=[0.5, 0.5], and uncertainty=1."
)

print(
    "\nThe available modality opinions are ready for "
    "TMC/Dempster-Shafer fusion."
)

### 1.4.15. Fusing the independent modality opinions with evidence pathway

combine the six independent modality opinions using the reduced Dempster--Shafer rule adopted by Trusted Multi-View Classification.

For modality \(m\), the Dirichlet parameters are:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{e}^{(m)} + \mathbf{1}.
$$

The Dirichlet strength is:

$$
S^{(m)}
=
\sum_{k=1}^{K}
\alpha_{k}^{(m)}.
$$

The class-specific belief masses are:

$$
b_{k}^{(m)}
=
\frac{
\alpha_{k}^{(m)} - 1
}{
S^{(m)}
}
=
\frac{
e_{k}^{(m)}
}{
S^{(m)}
}.
$$

The uncertainty mass is:

$$
u^{(m)}
=
\frac{K}{
S^{(m)}
}.
$$

These masses satisfy:

$$
\sum_{k=1}^{K}
b_{k}^{(m)}
+
u^{(m)}
=
1.
$$

### 1.4.16. Combining two opinions

Consider two opinions, \(A\) and \(B\). Their conflict mass is:

$$
C
=
\sum_{i \neq j}
b_{i}^{A}
b_{j}^{B}.
$$

For each class \(k\), the combined belief mass is:

$$
b_{k}^{A \oplus B}
=
\frac{
b_{k}^{A}b_{k}^{B}
+
b_{k}^{A}u^{B}
+
b_{k}^{B}u^{A}
}{
1-C
}.
$$

The combined uncertainty mass is:

$$
u^{A \oplus B}
=
\frac{
u^{A}u^{B}
}{
1-C
}.
$$

The fused Dirichlet strength is recovered from the fused uncertainty:

$$
S^{A \oplus B}
=
\frac{K}{
u^{A \oplus B}
}.
$$

The fused evidence and Dirichlet parameters are then:

$$
e_{k}^{A \oplus B}
=
b_{k}^{A \oplus B}
S^{A \oplus B},
$$

and:

$$
\alpha_{k}^{A \oplus B}
=
e_{k}^{A \oplus B} + 1.
$$

The rule is applied repeatedly until all six modality opinions have been considered.

### 1.4.17. Missing modalities

An unavailable modality was assigned the vacuous Dirichlet opinion:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{1}.
$$

For this opinion:

$$
\mathbf{b}^{(m)}
=
\mathbf{0},
\qquad
u^{(m)} = 1.
$$

A vacuous opinion acts as an identity element in this combination rule. It adds no class evidence, creates no conflict, and leaves the available opinion unchanged.

Therefore, the same fusion procedure can process all participants without complete-case filtering or synthetic modality imputation.

At this stage, I construct only the evidence pathway-fused independent opinion. The interaction-aware interaction pathway query will receive its own evidential head in a later step.

In [ ]:
# ============================================================
# 11. Fusing the independent modality opinions with evidence pathway
# ============================================================

# ------------------------------------------------------------
# Reduced Dempster-Shafer combination rule
# ------------------------------------------------------------

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )


    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }


    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]


        # --------------------------------------------------------
        # Conflict mass
        # --------------------------------------------------------

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)


        # --------------------------------------------------------
        # Fused belief and uncertainty masses
        # --------------------------------------------------------

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation


        # --------------------------------------------------------
        # Recover the fused Dirichlet opinion
        # --------------------------------------------------------

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }


    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # I retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }


        # --------------------------------------------------------
        # Final fused opinion
        # --------------------------------------------------------

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }


# ------------------------------------------------------------
# Instantiate the modality-specific evidence fusion module
# ------------------------------------------------------------

tmc_fusion = TMCFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
)


# ------------------------------------------------------------
# Fuse the current independent modality opinions
# ------------------------------------------------------------

with torch.no_grad():

    tmc_output = tmc_fusion(
        modality_opinions=modality_opinions
    )


# ------------------------------------------------------------
# Display final fused tensor shapes
# ------------------------------------------------------------

print("=" * 72)
print("TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    f"\nFusion order: "
    f"{MODALITY_ORDER}"
)

print(
    "\nFused evidence shape: "
    f"{tuple(tmc_output['evidence'].shape)}"
)

print(
    "Fused alpha shape: "
    f"{tuple(tmc_output['alpha'].shape)}"
)

print(
    "Fused probability shape: "
    f"{tuple(tmc_output['probabilities'].shape)}"
)

print(
    "Fused uncertainty shape: "
    f"{tuple(tmc_output['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Final participant-level fused opinions
# ------------------------------------------------------------

fused_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    fused_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "AVAILABLE_MODALITIES": int(
                example_batch["branch_masks"][
                    batch_row
                ].sum()
                .item()
            ),

            "ALPHA_sMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                tmc_output["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


fused_opinion_summary = pd.DataFrame(
    fused_opinion_rows
)

print("\nFinal TMC-fused opinions:")

display(
    fused_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Inspect the sequential fusion history
# ------------------------------------------------------------

fusion_history_rows = []

for modality_name in MODALITY_ORDER:

    stage_output = tmc_output[
        "fusion_history"
    ][modality_name]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):

        modality_index = MODALITY_ORDER.index(
            modality_name
        )

        fusion_history_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "FUSED_THROUGH": modality_name,

                "CURRENT_MODALITY_MASK": float(
                    example_batch["branch_masks"][
                        batch_row,
                        modality_index,
                    ].item()
                ),

                "STAGE_CONFLICT": float(
                    stage_output["conflict"][
                        batch_row,
                        0,
                    ].item()
                ),

                "STAGE_UNCERTAINTY": float(
                    stage_output["uncertainty"][
                        batch_row,
                        0,
                    ].item()
                ),
            }
        )


fusion_history_summary = pd.DataFrame(
    fusion_history_rows
)

print("\nSequential fusion history:")

display(
    fusion_history_summary.round(6)
)


print(
    "\nUnavailable modalities should introduce zero conflict "
    "and leave the accumulated opinion unchanged."
)

print(
    "The TMC-fused independent opinion is ready for later "
    "combination with the interaction-aware 3MT opinion."
)

### 1.4.18. Producing the interaction-aware interaction pathway evidential opinion

The modality-specific evidence pathway combines independent modality opinions produced before cross-modal interaction. construct a separate evidential head for the final query produced by the interaction pathway cascade.

The final interaction-pathway representation is:

$$
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\in
\mathbb{R}^{64}.
$$

Unlike the independent modality representations, this vector contains cumulative information learned through the ordered sequence of Cascaded Modality Transformers.

The joint evidential head produces non-negative class evidence:

$$
e_{ik}^{\mathrm{joint}}
=
\operatorname{Softplus}
\left(
f_{\mathrm{joint}}
\left(
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\right)
\right),
$$

where \(k\) denotes either sMCI or pMCI.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{\mathrm{joint}}
=
e_{ik}^{\mathrm{joint}} + 1.
$$

The expected class probabilities are:

$$
p_{ik}^{\mathrm{joint}}
=
\frac{
\alpha_{ik}^{\mathrm{joint}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{joint}}
}.
$$

The joint uncertainty is:

$$
u_{i}^{\mathrm{joint}}
=
\frac{K}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{joint}}
}.
$$

This opinion serves a different purpose from the evidence pathway-fused independent opinion:

- the modality-specific opinion represents agreement and conflict between modality-specific predictions;
- the joint interaction pathway opinion represents the prediction obtained after learning cross-modal interactions.

### 1.4.19. Auxiliary outputs

The original interaction pathway architecture places an auxiliary classifier after each intermediate cross-modal interaction to provide direct training signals to earlier cascade stages.

The notebook preserves that principle by attaching one auxiliary classifier to every intermediate query except the final MRI stage. These auxiliary heads produce ordinary logits for the prognosis classes and are used only during training.

The auxiliary classifiers are not treated as independent modality-specific opinions because each intermediate query already contains information accumulated from all preceding modalities. They support gradient flow through the cascade but do not represent isolated modality evidence.

The final MRI-stage query receives the joint evidential head and produces the interaction-aware Dirichlet opinion.

In [ ]:
# ============================================================
# 12. Producing the interaction-aware interaction pathway evidential opinion
# ============================================================

# ------------------------------------------------------------
# Auxiliary classifier for one intermediate interaction pathway query
# ------------------------------------------------------------

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(self, representation):
        return self.network(
            representation
        )


# ------------------------------------------------------------
# Joint interaction pathway evidential and auxiliary heads
# ------------------------------------------------------------

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )


    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # --------------------------------------------------------
        # Intermediate auxiliary predictions
        # --------------------------------------------------------

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )


        # --------------------------------------------------------
        # Final joint evidential opinion
        # --------------------------------------------------------

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final interaction pathway query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }


# ------------------------------------------------------------
# Instantiate the interaction-pathway prediction heads
# ------------------------------------------------------------

three_mt_prediction_heads = ThreeMTPredictionHeads(
    cascade_order=THREE_MT_CASCADE_ORDER,
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
)


# ------------------------------------------------------------
# Produce the auxiliary and joint outputs
# ------------------------------------------------------------

three_mt_prediction_heads.eval()

with torch.no_grad():

    three_mt_predictions = (
        three_mt_prediction_heads(
            three_mt_output=three_mt_output
        )
    )


joint_opinion = three_mt_predictions[
    "joint_opinion"
]


# ------------------------------------------------------------
# Display the output structure
# ------------------------------------------------------------

print("=" * 72)
print("3MT PREDICTION HEADS")
print("=" * 72)

print(
    f"\nAuxiliary stages: "
    f"{three_mt_prediction_heads.auxiliary_stages}"
)

print(
    f"Final evidential stage: "
    f"{three_mt_prediction_heads.final_stage}"
)

print("\nAuxiliary-logit shapes:")

for stage_name, stage_logits in (
    three_mt_predictions[
        "auxiliary_logits"
    ].items()
):
    print(
        f"- after {stage_name}: "
        f"{tuple(stage_logits.shape)}"
    )


print("\nJoint evidential shapes:")

print(
    "  evidence: "
    f"{tuple(joint_opinion['evidence'].shape)}"
)

print(
    "  alpha: "
    f"{tuple(joint_opinion['alpha'].shape)}"
)

print(
    "  probabilities: "
    f"{tuple(joint_opinion['probabilities'].shape)}"
)

print(
    "  uncertainty: "
    f"{tuple(joint_opinion['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Participant-level joint opinions
# ------------------------------------------------------------

joint_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    joint_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ALPHA_sMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                joint_opinion["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


joint_opinion_summary = pd.DataFrame(
    joint_opinion_rows
)

print("\nInteraction-aware 3MT opinions:")

display(
    joint_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable 3MT prediction-head parameters: "
    f"{count_trainable_parameters(three_mt_prediction_heads):,}"
)

print(
    "\nThe auxiliary outputs will support training-time "
    "gradient flow through the cascade."
)

print(
    "The final joint opinion is ready for combination with "
    "the TMC-fused independent opinion."
)

### 1.4.20. Combining the interaction pathway and modality-specific evidence pathways with a constrained reliability gate

The model currently produces two complementary evidential outputs:

1. the interaction-aware interaction pathway opinion;
2. the independently fused modality-specific opinion.

These opinions are derived from the same underlying participant data and therefore should not be combined using Dempster--Shafer fusion as though they were independent evidence sources.

Instead, This notebook uses a participant-specific convex mixture of their calibrated evidence vectors.

### 1.4.21. Pathway calibration

The evidence magnitudes produced by the two pathways may have different numerical scales. In particular, the modality-specific evidence pathway accumulates evidence across several available modalities, whereas the cross-modal interaction pathway produces evidence from one joint head.

I therefore introduce one positive scalar calibration parameter for each pathway:

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
=
\tau_{\mathrm{interaction pathway}}
\mathbf{e}_{i}^{\mathrm{interaction pathway}},
$$

and

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}
=
\tau_{\mathrm{evidence pathway}}
\mathbf{e}_{i}^{\mathrm{evidence pathway}}.
$$

Both calibration factors are constrained to be positive using the Softplus function:

$$
\tau_r
=
\operatorname{Softplus}(\rho_r),
\qquad
r \in
\{\mathrm{interaction pathway},\mathrm{evidence pathway}\}.
$$

The parameters are initialised so that both calibration factors begin at approximately one.

### 1.4.22. Reliability-gate input

The reliability gate receives only uncertainty, conflict, and modality-availability information. It does not receive the original participant features or the full interaction-pathway representation.

For participant \(i\), the gate input is:

$$
\mathbf{r}_i
=
\left[
u_i^{\mathrm{interaction pathway}},
u_i^{\mathrm{evidence pathway}},
\bar{C}_i^{\mathrm{evidence pathway}},
\frac{n_i^{\mathrm{available}}}{M},
\mathbf{a}_i
\right],
$$

where:

- \(u_i^{\mathrm{interaction pathway}}\) is the uncertainty of the joint interaction pathway opinion;
- \(u_i^{\mathrm{evidence pathway}}\) is the uncertainty of the evidence pathway-fused opinion;
- \(\bar{C}_i^{\mathrm{evidence pathway}}\) is the mean conflict encountered when combining available modality opinions;
- \(n_i^{\mathrm{available}}\) is the number of available branches;
- \(M=6\) is the total number of branches;
- \(\mathbf{a}_i\) is the six-element branch-availability vector.

The gate produces one scalar weight:

$$
w_i
=
\sigma
\left(
\mathbf{w}^{\top}
\mathbf{r}_i+b
\right).
$$

The interpretation is:

$$
w_i \rightarrow 1
\quad
\Longrightarrow
\quad
\text{greater reliance on interaction pathway},
$$

and

$$
w_i \rightarrow 0
\quad
\Longrightarrow
\quad
\text{greater reliance on evidence pathway}.
$$

The gate is initialised with zero weights and zero bias. Therefore, before training:

$$
w_i = 0.5.
$$

This prevents either pathway from being preferred arbitrarily at model initialisation.

### 1.4.23. Final evidential opinion

The final evidence is:

$$
\mathbf{e}_{i}^{\mathrm{final}}
=
w_i
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
+
(1-w_i)
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}.
$$

Because \(w_i \in [0,1]\), this is a convex mixture rather than an addition of two supposedly independent evidence sources.

The final Dirichlet parameters are:

$$
\boldsymbol{\alpha}_{i}^{\mathrm{final}}
=
\mathbf{e}_{i}^{\mathrm{final}}
+
\mathbf{1}.
$$

The final class probabilities and uncertainty are:

$$
p_{ik}^{\mathrm{final}}
=
\frac{
\alpha_{ik}^{\mathrm{final}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{final}}
},
$$

and

$$
u_i^{\mathrm{final}}
=
\frac{
K
}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{final}}
}.
$$

The reliability statistics supplied to the gate are detached from the computational graph. This prevents the upstream pathways from manipulating their uncertainty or conflict values merely to obtain a larger gate weight. The final loss can still train both pathways through their evidence contributions.

In [ ]:
# ============================================================
# 13. Combining interaction pathway and evidence pathway with fixed equal fusion
# ============================================================

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )


class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )


    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )


    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


hybrid_fusion = FixedEqualHybridFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    number_of_modalities=len(
        MODALITY_ORDER
    ),
    initial_three_mt_scale=1.0,
    initial_tmc_scale=1.0,
)


hybrid_fusion.eval()

with torch.no_grad():
    hybrid_output = hybrid_fusion(
        joint_opinion=joint_opinion,
        tmc_output=tmc_output,
        branch_masks=example_batch[
            "branch_masks"
        ],
        modality_order=MODALITY_ORDER,
    )


print("=" * 72)
print("FIXED 50/50 3MT-TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    "\nFinal probability shape: "
    f"{tuple(hybrid_output['probabilities'].shape)}"
)

print(
    "Initial 3MT evidence scale: "
    f"{hybrid_output['three_mt_scale'].item():.6f}"
)

print(
    "Initial TMC evidence scale: "
    f"{hybrid_output['tmc_scale'].item():.6f}"
)

print(
    "Trainable fixed-fusion parameters: "
    f"{count_trainable_parameters(hybrid_fusion):,}"
)

maximum_three_mt_weight_error = (
    hybrid_output[
        "three_mt_weight"
    ]
    - 0.5
).abs().max().item()

maximum_tmc_weight_error = (
    hybrid_output[
        "tmc_weight"
    ]
    - 0.5
).abs().max().item()

probability_sum_error = (
    hybrid_output[
        "probabilities"
    ].sum(
        dim=-1
    )
    - 1.0
).abs().max().item()

print(
    "\nMaximum 3MT-weight deviation from 0.5: "
    f"{maximum_three_mt_weight_error:.10f}"
)

print(
    "Maximum TMC-weight deviation from 0.5: "
    f"{maximum_tmc_weight_error:.10f}"
)

print(
    "Maximum final probability-sum error: "
    f"{probability_sum_error:.10f}"
)

assert maximum_three_mt_weight_error == 0.0
assert maximum_tmc_weight_error == 0.0

print(
    "\nThe participant-specific reliability gate is absent. "
    "Only the two positive pathway evidence scales remain trainable."
)


### 1.4.24. Assembling the complete availability-gated interaction-evidence model

The complete model keeps the same six components used in the original fold-0 run:

1. modality-specific encoders;
2. training-time modality dropout;
3. independent modality evidential heads;
4. modality-specific evidence fusion;
5. the interaction pathway interaction pathway;
6. fixed-equal hybrid evidence fusion.

The effective branch masks are created before encoding. They contain both natural missingness and any additional modality removed by training-time dropout. These exact masks are now passed into the interaction pathway cascade, so every unavailable cross-modal interaction stage preserves the preceding cumulative query.

The modality-specific evidence pathway and fixed equal fusion are unchanged. This isolates the effect of availability-gating the cross-modal interaction updates.

In [ ]:
# ============================================================
# 14. Assembling the complete end-to-end interaction-evidence model
# ============================================================

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )


        # --------------------------------------------------------
        # Modality-specific encoders
        # --------------------------------------------------------

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )


        # --------------------------------------------------------
        # Independent modality evidential pathway
        # --------------------------------------------------------

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )


        # --------------------------------------------------------
        # Final fixed 50/50 hybrid fusion
        # --------------------------------------------------------

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )


    # ------------------------------------------------------------
    # Training-time modality dropout
    # ------------------------------------------------------------

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )


        # --------------------------------------------------------
        # Sample branch-retention indicators
        # --------------------------------------------------------

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )


        # --------------------------------------------------------
        # Prevent complete information removal
        # --------------------------------------------------------

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # I randomly restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0


        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )


    # ------------------------------------------------------------
    # Construct effective modality dictionaries
    # ------------------------------------------------------------

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities


    # ------------------------------------------------------------
    # Complete forward pass
    # ------------------------------------------------------------

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # --------------------------------------------------------
        # Apply training-time modality dropout
        # --------------------------------------------------------

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )


        # --------------------------------------------------------
        # Encode all six modalities
        # --------------------------------------------------------

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. I use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]


        # --------------------------------------------------------
        # Independent modality opinions and modality-specific evidence fusion
        # --------------------------------------------------------

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]


        # --------------------------------------------------------
        # Final fixed-equal hybrid opinion
        # --------------------------------------------------------

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )


        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


# ------------------------------------------------------------
# Instantiate the complete model
# ------------------------------------------------------------

complete_model = ADNIEvidential3MTTMCModel(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    modality_dropout_probability=0.50,
)


# ------------------------------------------------------------
# Evaluation-mode end-to-end forward pass
# ------------------------------------------------------------

# In evaluation mode, modality dropout is disabled.
complete_model.eval()

with torch.no_grad():

    complete_model_output = complete_model(
        modalities=example_batch[
            "modalities"
        ],

        original_branch_masks=example_batch[
            "branch_masks"
        ],
    )


# ------------------------------------------------------------
# Inspect final outputs
# ------------------------------------------------------------

final_output = complete_model_output[
    "final_output"
]

print("=" * 72)
print("COMPLETE AVAILABILITY-GATED 3MT-TMC MODEL")
print("=" * 72)

print(
    "\nModel mode: "
    f"{'training' if complete_model.training else 'evaluation'}"
)

print(
    "Modality-dropout probability: "
    f"{complete_model.modality_dropout_probability:.2f}"
)

print(
    "\nOriginal branch-mask shape: "
    f"{tuple(complete_model_output['original_branch_masks'].shape)}"
)

print(
    "Effective branch-mask shape: "
    f"{tuple(complete_model_output['effective_branch_masks'].shape)}"
)

print(
    "\nFinal alpha shape: "
    f"{tuple(final_output['alpha'].shape)}"
)

print(
    "Final probability shape: "
    f"{tuple(final_output['probabilities'].shape)}"
)

print(
    "Final uncertainty shape: "
    f"{tuple(final_output['uncertainty'].shape)}"
)

print(
    "\nTotal trainable model parameters: "
    f"{count_trainable_parameters(complete_model):,}"
)


# ------------------------------------------------------------
# Verify that evaluation mode preserves genuine availability
# ------------------------------------------------------------

evaluation_mask_difference = (
    complete_model_output[
        "effective_branch_masks"
    ]
    - complete_model_output[
        "original_branch_masks"
    ]
).abs().max().item()

print(
    "\nMaximum evaluation-mode difference between original "
    f"and effective branch masks: "
    f"{evaluation_mask_difference:.10f}"
)


# ------------------------------------------------------------
# Participant-level output summary
# ------------------------------------------------------------

complete_model_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    original_available_count = int(
        complete_model_output[
            "original_branch_masks"
        ][batch_row].sum().item()
    )

    effective_available_count = int(
        complete_model_output[
            "effective_branch_masks"
        ][batch_row].sum().item()
    )

    complete_model_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ORIGINAL_MODALITIES":
                original_available_count,

            "EFFECTIVE_MODALITIES":
                effective_available_count,

            "W_3MT": float(
                final_output[
                    "three_mt_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "W_TMC": float(
                final_output[
                    "tmc_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_sMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    1,
                ].item()
            ),

            "FINAL_UNCERTAINTY": float(
                final_output[
                    "uncertainty"
                ][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


complete_model_summary = pd.DataFrame(
    complete_model_rows
)

print("\nComplete-model evaluation-mode outputs:")

display(
    complete_model_summary.round(6)
)

print(
    "\nThe complete model now passes the effective branch "
    "masks into every CMT stage."
)


### 1.4.25. Defining the joint training objective

The complete architecture produces several supervised outputs with different purposes:

1. the final fixed-equal hybrid opinion;
2. the interaction-aware interaction pathway joint opinion;
3. the evidence pathway-fused independent-modality opinion;
4. six independent modality-specific opinions;
5. five intermediate interaction pathway auxiliary predictions.

These outputs are trained jointly, but the final hybrid prediction remains the primary objective.

The total loss is:

$$
\mathcal{L}_{\mathrm{total}}
=
\mathcal{L}_{\mathrm{final}}
+
\lambda_{\mathrm{joint}}
\mathcal{L}_{\mathrm{joint}}
+
\lambda_{\mathrm{evidence pathway}}
\mathcal{L}_{\mathrm{evidence pathway}}
+
\lambda_{\mathrm{mod}}
\mathcal{L}_{\mathrm{mod}}
+
\lambda_{\mathrm{aux}}
\mathcal{L}_{\mathrm{aux}}
+
\lambda_{\mathrm{gate}}
\mathcal{L}_{\mathrm{gate}}.
$$

The initial loss weights are:

$$
\lambda_{\mathrm{joint}} = 0.5,
\qquad
\lambda_{\mathrm{evidence pathway}} = 0.5,
\qquad
\lambda_{\mathrm{mod}} = 0.1,
$$

$$
\lambda_{\mathrm{aux}} = 0.1,
\qquad
\lambda_{\mathrm{gate}} = 0.01.
$$

The final hybrid loss has coefficient one and therefore remains the dominant objective. The remaining terms provide direct supervision to the two pathways and their intermediate components.

These coefficients are initial modelling choices. Any comparison of alternative values must use only the training and validation partitions.

### 1.4.26. Evidential classification loss

For a Dirichlet prediction:

$$
\boldsymbol{\alpha}_i
=
\mathbf{e}_i+\mathbf{1},
$$

the expected cross-entropy loss is:

$$
\mathcal{L}_{\mathrm{ECE},i}
=
\sum_{k=1}^{K}
y_{ik}
\left[
\psi(S_i)
-
\psi(\alpha_{ik})
\right],
$$

where:

$$
S_i
=
\sum_{k=1}^{K}
\alpha_{ik},
$$

and \(\psi(\cdot)\) denotes the digamma function.

This objective minimises the expected negative log-likelihood under the predicted Dirichlet distribution.

### 1.4.27. Evidence regularisation

An evidential network may become unjustifiably confident by assigning strong evidence to an incorrect class. I therefore add a Kullback--Leibler regularisation term that discourages unsupported evidence.

The adjusted Dirichlet parameters are:

$$
\widetilde{\boldsymbol{\alpha}}_i
=
\mathbf{y}_i
+
(1-\mathbf{y}_i)
\odot
\boldsymbol{\alpha}_i.
$$

This construction removes the evidence assigned to the correct class from the regularisation term while penalising evidence assigned to incorrect classes.

The regularisation term is:

$$
\mathcal{L}_{\mathrm{KL},i}
=
D_{\mathrm{KL}}
\left[
\operatorname{Dir}
\left(
\widetilde{\boldsymbol{\alpha}}_i
\right)
\parallel
\operatorname{Dir}
\left(
\mathbf{1}
\right)
\right].
$$

The complete evidential loss is:

$$
\mathcal{L}_{\mathrm{EDL},i}
=
\mathcal{L}_{\mathrm{ECE},i}
+
\beta_t
\mathcal{L}_{\mathrm{KL},i}.
$$

The regularisation coefficient is annealed during the first training epochs:

$$
\beta_t
=
\min
\left(
1,
\frac{t}{T_{\mathrm{anneal}}}
\right),
$$

where \(t\) is the current epoch and \(T_{\mathrm{anneal}}\) is initially set to ten epochs.

This allows the model to begin learning the classification task before the full evidence penalty is applied.

### 1.4.28. Modality-specific loss

Each independent modality opinion is supervised only when that modality is effectively available after training-time modality dropout.

For branch \(m\), let:

$$
\widetilde{a}_i^{(m)}
\in
\{0,1\}
$$

denote the effective branch mask.

The modality-specific loss is:

$$
\mathcal{L}_{\mathrm{mod}}
=
\frac{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
\mathcal{L}_{\mathrm{EDL},i}^{(m)}
}{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
+
\varepsilon
}.
$$

Naturally unavailable or deliberately dropped branches therefore contribute no modality-specific classification loss.

### 1.4.29. Auxiliary interaction pathway loss

The intermediate interaction pathway heads produce ordinary class logits rather than Dirichlet opinions. Their loss is the mean cross-entropy across the five intermediate cascade stages:

$$
\mathcal{L}_{\mathrm{aux}}
=
\frac{1}{J}
\sum_{j=1}^{J}
\operatorname{CE}
\left(
\mathbf{z}^{(j)},
y
\right),
$$

where \(J=5\).

These losses provide direct gradient signals to earlier stages of the cascaded transformer.

### 1.4.30. Gate regularisation

The reliability gate is initialised at:

$$
w_i=0.5.
$$

A weak early-training regulariser discourages immediate collapse to a single pathway:

$$
\mathcal{L}_{\mathrm{gate}}
=
\left(
\frac{1}{N}
\sum_{i=1}^{N}
w_i
-
0.5
\right)^2.
$$

The gate regularisation is annealed to zero after the initial training period. It therefore stabilises early optimisation without forcing the final trained gate to remain balanced.

This cell defines and validates the training objective only. Parameter updates begin after gradient flow is checked in the following step.

In [ ]:
# ============================================================
# 15. Defining the complete joint training objective
# ============================================================

# ------------------------------------------------------------
# KL divergence between a predicted Dirichlet distribution and
# a uniform Dirichlet distribution
# ------------------------------------------------------------

def dirichlet_kl_to_uniform(
    alpha,
):
    """
    Calculate:

        KL(Dir(alpha) || Dir(1))

    for every participant in the batch.

    Parameters
    ----------
    alpha:
        Positive Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    Returns
    -------
    Tensor with shape:
        (batch_size,)
    """

    number_of_classes = alpha.shape[-1]

    uniform_alpha = torch.ones_like(
        alpha
    )

    alpha_strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    uniform_strength = uniform_alpha.sum(
        dim=-1,
        keepdim=True,
    )

    log_normalisation_ratio = (
        torch.lgamma(alpha_strength)
        - torch.lgamma(uniform_strength)
        - torch.lgamma(alpha).sum(
            dim=-1,
            keepdim=True,
        )
        + torch.lgamma(uniform_alpha).sum(
            dim=-1,
            keepdim=True,
        )
    )

    digamma_difference = (
        torch.digamma(alpha)
        - torch.digamma(alpha_strength)
    )

    parameter_difference = (
        alpha
        - uniform_alpha
    )

    expectation_term = (
        parameter_difference
        * digamma_difference
    ).sum(
        dim=-1,
        keepdim=True,
    )

    kl_divergence = (
        log_normalisation_ratio
        + expectation_term
    )

    return kl_divergence.squeeze(-1)


# ------------------------------------------------------------
# Evidential classification loss
# ------------------------------------------------------------

def evidential_classification_loss(
    alpha,
    targets,
    number_of_classes,
    annealing_coefficient,
    class_weights=None,
    reduction="mean",
):
    """
    Calculate the expected cross-entropy under a Dirichlet
    distribution together with annealed KL regularisation.

    Parameters
    ----------
    alpha:
        Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    targets:
        Integer class labels with shape:
        (batch_size,)

    number_of_classes:
        Number of prognosis classes.

    annealing_coefficient:
        Current coefficient applied to the KL term.

    class_weights:
        Optional class-weight tensor with shape:
        (number_of_classes,)

    reduction:
        "none", "mean", or "sum".
    """

    targets = targets.long()

    one_hot_targets = F.one_hot(
        targets,
        num_classes=number_of_classes,
    ).to(
        dtype=alpha.dtype
    )

    strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )


    # --------------------------------------------------------
    # Expected cross-entropy under the Dirichlet distribution
    # --------------------------------------------------------

    expected_cross_entropy_by_class = (
        torch.digamma(strength)
        - torch.digamma(alpha)
    )

    expected_cross_entropy = (
        one_hot_targets
        * expected_cross_entropy_by_class
    ).sum(
        dim=-1
    )


    # --------------------------------------------------------
    # Optional class weighting
    # --------------------------------------------------------

    if class_weights is not None:

        sample_weights = class_weights[
            targets
        ].to(
            dtype=alpha.dtype,
            device=alpha.device,
        )

        expected_cross_entropy = (
            expected_cross_entropy
            * sample_weights
        )


    # --------------------------------------------------------
    # Remove correct-class evidence from the KL penalty
    # --------------------------------------------------------

    adjusted_alpha = (
        one_hot_targets
        +
        (
            1.0
            - one_hot_targets
        )
        * alpha
    )

    kl_regularisation = (
        dirichlet_kl_to_uniform(
            adjusted_alpha
        )
    )

    per_sample_loss = (
        expected_cross_entropy
        +
        annealing_coefficient
        * kl_regularisation
    )


    # --------------------------------------------------------
    # Requested reduction
    # --------------------------------------------------------

    if reduction == "none":
        reduced_loss = per_sample_loss

    elif reduction == "mean":
        reduced_loss = per_sample_loss.mean()

    elif reduction == "sum":
        reduced_loss = per_sample_loss.sum()

    else:
        raise ValueError(
            "reduction must be 'none', 'mean', or 'sum'."
        )


    return {
        "loss":
            reduced_loss,

        "per_sample_loss":
            per_sample_loss,

        "expected_cross_entropy":
            expected_cross_entropy,

        "kl_regularisation":
            kl_regularisation,
    }


# ------------------------------------------------------------
# Complete multi-output objective
# ------------------------------------------------------------

class HybridEvidentialTrainingLoss(nn.Module):
    """
    Jointly supervise the final hybrid output, both main pathways,
    the individual available modality opinions, and the auxiliary
    3MT classifiers.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        evidential_annealing_epochs=10,
        gate_regularisation_epochs=0,
        joint_loss_weight=0.50,
        tmc_loss_weight=0.50,
        modality_loss_weight=0.10,
        auxiliary_loss_weight=0.10,
        gate_loss_weight=0.0,
        class_weights=None,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.evidential_annealing_epochs = (
            evidential_annealing_epochs
        )

        self.gate_regularisation_epochs = (
            gate_regularisation_epochs
        )

        self.joint_loss_weight = (
            joint_loss_weight
        )

        self.tmc_loss_weight = (
            tmc_loss_weight
        )

        self.modality_loss_weight = (
            modality_loss_weight
        )

        self.auxiliary_loss_weight = (
            auxiliary_loss_weight
        )

        self.gate_loss_weight = (
            gate_loss_weight
        )


        # --------------------------------------------------------
        # Optional training-fold class weights
        # --------------------------------------------------------

        if class_weights is None:

            self.register_buffer(
                "class_weights",
                None,
            )

        else:

            class_weights = torch.as_tensor(
                class_weights,
                dtype=torch.float32,
            )

            if class_weights.shape != (
                number_of_classes,
            ):
                raise ValueError(
                    "class_weights must contain one value "
                    "for each class."
                )

            self.register_buffer(
                "class_weights",
                class_weights,
            )


    # ------------------------------------------------------------
    # Annealing schedules
    # ------------------------------------------------------------

    def _evidential_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Increase the KL coefficient linearly from zero to one.
        """

        if self.evidential_annealing_epochs <= 0:
            return 1.0

        return min(
            1.0,
            float(epoch)
            / float(
                self.evidential_annealing_epochs
            ),
        )


    def _gate_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Reduce the gate-balance penalty to zero after the initial
        optimisation period.
        """

        if self.gate_regularisation_epochs <= 0:
            return 0.0

        return max(
            0.0,
            1.0
            -
            (
                float(epoch - 1)
                /
                float(
                    self.gate_regularisation_epochs
                )
            ),
        )


    # ------------------------------------------------------------
    # Complete loss calculation
    # ------------------------------------------------------------

    def forward(
        self,
        model_output,
        targets,
        epoch,
    ):
        targets = targets.long()

        evidential_annealing = (
            self._evidential_annealing_coefficient(
                epoch
            )
        )

        gate_annealing = (
            self._gate_annealing_coefficient(
                epoch
            )
        )


        # --------------------------------------------------------
        # Final hybrid evidential loss
        # --------------------------------------------------------

        final_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "final_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        final_loss = final_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Interaction-aware interaction pathway evidential loss
        # --------------------------------------------------------

        joint_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "joint_opinion"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        joint_loss = joint_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # evidence pathway-fused evidential loss
        # --------------------------------------------------------

        tmc_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "tmc_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        tmc_loss = tmc_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Independent available-modality evidential loss
        # --------------------------------------------------------

        effective_branch_masks = model_output[
            "effective_branch_masks"
        ]

        weighted_modality_loss_sum = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        available_opinion_count = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        modality_loss_by_name = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):

            modality_alpha = model_output[
                "modality_opinions"
            ][modality_name]["alpha"]

            modality_loss_components = (
                evidential_classification_loss(
                    alpha=modality_alpha,

                    targets=targets,

                    number_of_classes=
                        self.number_of_classes,

                    annealing_coefficient=
                        evidential_annealing,

                    class_weights=
                        self.class_weights,

                    reduction="none",
                )
            )

            per_sample_modality_loss = (
                modality_loss_components[
                    "per_sample_loss"
                ]
            )

            modality_mask = effective_branch_masks[
                :,
                modality_index,
            ].to(
                dtype=per_sample_modality_loss.dtype
            )

            masked_modality_loss_sum = (
                per_sample_modality_loss
                * modality_mask
            ).sum()

            modality_available_count = (
                modality_mask.sum()
            )

            weighted_modality_loss_sum = (
                weighted_modality_loss_sum
                + masked_modality_loss_sum
            )

            available_opinion_count = (
                available_opinion_count
                + modality_available_count
            )

            modality_loss_by_name[
                modality_name
            ] = (
                masked_modality_loss_sum
                /
                modality_available_count.clamp_min(
                    1.0
                )
            )

        modality_loss = (
            weighted_modality_loss_sum
            /
            available_opinion_count.clamp_min(
                1.0
            )
        )


        # --------------------------------------------------------
        # Intermediate interaction pathway auxiliary cross-entropy loss
        # --------------------------------------------------------

        auxiliary_logits = model_output[
            "three_mt_predictions"
        ]["auxiliary_logits"]

        auxiliary_loss_by_stage = {}

        auxiliary_losses = []

        for stage_name, stage_logits in (
            auxiliary_logits.items()
        ):

            stage_loss = F.cross_entropy(
                input=stage_logits,
                target=targets,
                weight=self.class_weights,
            )

            auxiliary_loss_by_stage[
                stage_name
            ] = stage_loss

            auxiliary_losses.append(
                stage_loss
            )

        if auxiliary_losses:

            auxiliary_loss = torch.stack(
                auxiliary_losses
            ).mean()

        else:

            auxiliary_loss = torch.zeros(
                (),
                dtype=final_loss.dtype,
                device=final_loss.device,
            )


        # --------------------------------------------------------
        # Early gate-balance regularisation
        # --------------------------------------------------------

        three_mt_weights = model_output[
            "final_output"
        ]["three_mt_weight"]

        raw_gate_loss = (
            three_mt_weights.mean()
            - 0.5
        ).pow(2)

        annealed_gate_loss = (
            gate_annealing
            * raw_gate_loss
        )


        # --------------------------------------------------------
        # Weighted total objective
        # --------------------------------------------------------

        total_loss = (
            final_loss
            +
            self.joint_loss_weight
            * joint_loss
            +
            self.tmc_loss_weight
            * tmc_loss
            +
            self.modality_loss_weight
            * modality_loss
            +
            self.auxiliary_loss_weight
            * auxiliary_loss
            +
            self.gate_loss_weight
            * annealed_gate_loss
        )


        return {
            "total_loss":
                total_loss,

            "final_loss":
                final_loss,

            "joint_loss":
                joint_loss,

            "tmc_loss":
                tmc_loss,

            "modality_loss":
                modality_loss,

            "auxiliary_loss":
                auxiliary_loss,

            "raw_gate_loss":
                raw_gate_loss,

            "annealed_gate_loss":
                annealed_gate_loss,

            "evidential_annealing":
                torch.tensor(
                    evidential_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "gate_annealing":
                torch.tensor(
                    gate_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "available_opinion_count":
                available_opinion_count,

            "modality_loss_by_name":
                modality_loss_by_name,

            "auxiliary_loss_by_stage":
                auxiliary_loss_by_stage,

            "final_expected_cross_entropy":
                final_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "final_kl_regularisation":
                final_loss_components[
                    "kl_regularisation"
                ].mean(),

            "joint_expected_cross_entropy":
                joint_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "joint_kl_regularisation":
                joint_loss_components[
                    "kl_regularisation"
                ].mean(),

            "tmc_expected_cross_entropy":
                tmc_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "tmc_kl_regularisation":
                tmc_loss_components[
                    "kl_regularisation"
                ].mean(),
        }


# ------------------------------------------------------------
# Instantiate the training objective
# ------------------------------------------------------------

training_objective = HybridEvidentialTrainingLoss(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,

    evidential_annealing_epochs=10,
    gate_regularisation_epochs=0,

    joint_loss_weight=0.50,
    tmc_loss_weight=0.50,
    modality_loss_weight=0.10,
    auxiliary_loss_weight=0.10,
    gate_loss_weight=0.0,

    # I initially leave class weighting disabled. It can be added
    # using weights calculated from each training fold only.
    class_weights=None,
)


# ------------------------------------------------------------
# Test the objective on the complete untrained forward pass
# ------------------------------------------------------------

example_loss_output = training_objective(
    model_output=complete_model_output,

    targets=example_batch[
        "target"
    ],

    epoch=1,
)


# ------------------------------------------------------------
# Display the initial loss structure
# ------------------------------------------------------------

print("=" * 72)
print("JOINT 3MT-TMC TRAINING OBJECTIVE")
print("=" * 72)

print(
    "\nEvidential KL annealing coefficient at epoch 1: "
    f"{example_loss_output['evidential_annealing'].item():.6f}"
)

print(
    "Gate regularisation coefficient at epoch 1: "
    f"{example_loss_output['gate_annealing'].item():.6f}"
)

print("\nMain loss components:")

for loss_name in [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]:
    print(
        f"- {loss_name}: "
        f"{example_loss_output[loss_name].item():.6f}"
    )


print("\nFinal evidential-loss decomposition:")

print(
    "- expected cross-entropy: "
    f"{example_loss_output['final_expected_cross_entropy'].item():.6f}"
)

print(
    "- KL regularisation: "
    f"{example_loss_output['final_kl_regularisation'].item():.6f}"
)


print("\nAvailable modality opinions in this batch: "
      f"{int(example_loss_output['available_opinion_count'].item())}")


# ------------------------------------------------------------
# Modality-specific loss summary
# ------------------------------------------------------------

modality_loss_rows = []

for modality_name in MODALITY_ORDER:

    modality_loss_rows.append(
        {
            "MODALITY": modality_name,

            "MEAN_AVAILABLE_LOSS": float(
                example_loss_output[
                    "modality_loss_by_name"
                ][modality_name].item()
            ),
        }
    )


print("\nModality-specific evidential losses:")

display(
    pd.DataFrame(
        modality_loss_rows
    ).round(6)
)


# ------------------------------------------------------------
# Auxiliary-stage loss summary
# ------------------------------------------------------------

auxiliary_loss_rows = []

for stage_name in THREE_MT_CASCADE_ORDER[:-1]:

    auxiliary_loss_rows.append(
        {
            "AUXILIARY_STAGE": stage_name,

            "CROSS_ENTROPY_LOSS": float(
                example_loss_output[
                    "auxiliary_loss_by_stage"
                ][stage_name].item()
            ),
        }
    )


print("\nIntermediate 3MT auxiliary losses:")

display(
    pd.DataFrame(
        auxiliary_loss_rows
    ).round(6)
)

print(
    "\nThe joint objective is ready for one end-to-end "
    "backward pass."
)


### 1.4.31. Running one end-to-end backward pass

Before configuring the optimiser, I run one training batch through the complete gated model and joint objective.

For this single diagnostic pass, modality dropout is set to zero so that the natural fold-0 availability pattern is used without additional random branch removal. The total loss is backpropagated, but no optimiser step is taken. The cell reports the total loss and the global gradient norm, then restores the configured modality-dropout probability of `0.50`.

In [ ]:
# ============================================================
# 16. Running one end-to-end backward pass
# ============================================================

diagnostic_batch = next(
    iter(train_loader)
)

original_modality_dropout_probability = (
    complete_model.modality_dropout_probability
)

complete_model.modality_dropout_probability = 0.0
complete_model.train()
complete_model.zero_grad(
    set_to_none=True
)


diagnostic_output = complete_model(
    modalities=diagnostic_batch[
        "modalities"
    ],
    original_branch_masks=diagnostic_batch[
        "branch_masks"
    ],
)

diagnostic_losses = training_objective(
    model_output=diagnostic_output,
    targets=diagnostic_batch[
        "target"
    ],
    epoch=1,
)

diagnostic_total_loss = diagnostic_losses[
    "total_loss"
]

diagnostic_total_loss.backward()


global_gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().float().pow(2).sum()
        for parameter in complete_model.parameters()
        if parameter.grad is not None
    )
)

print("=" * 72)
print("END-TO-END BACKWARD PASS")
print("=" * 72)

print(
    "\nTotal loss: "
    f"{diagnostic_total_loss.item():.6f}"
)

print(
    "Global gradient norm: "
    f"{global_gradient_norm.item():.6f}"
)

complete_model.zero_grad(
    set_to_none=True
)

complete_model.modality_dropout_probability = (
    original_modality_dropout_probability
)

complete_model.eval()

print(
    "Modality-dropout probability restored to: "
    f"{complete_model.modality_dropout_probability:.2f}"
)


### 1.4.32. Configuring optimisation and experiment-specific output paths

The original training hyperparameters remain unchanged. Every checkpoint, history file, validation prediction, and test prediction is written only under:

```text
models/3mt_tmc_evidential/experiments/
    gated_cmt_learned_gate_md050/
        mci_prognosis/fold_0/
```

This notebook does not use the older shared `training/mci_prognosis/fold_0/` directory.

In [ ]:
# ============================================================
# 17. Configuring optimisation, checkpoints, and metrics
# ============================================================

import os
import random
from datetime import datetime

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# Current experiment
# ------------------------------------------------------------

# EXPERIMENT_NAME, SELECTED_TASK, and SELECTED_FOLD were fixed when this fold's prepared input was loaded.


# ------------------------------------------------------------
# Reproducibility configuration
# ------------------------------------------------------------

GLOBAL_RANDOM_SEED = 42


def set_global_random_seed(seed):
    """
    Set the random seed used by Python, NumPy, and PyTorch.
    """

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_random_seed(
    GLOBAL_RANDOM_SEED
)


# ------------------------------------------------------------
# PyTorch numerical configuration
# ------------------------------------------------------------

# I allow cuDNN to choose efficient convolution algorithms.
#
# This is appropriate for the computationally expensive 3D MRI
# encoder. Exact bitwise reproducibility can still depend on the
# installed PyTorch, CUDA, cuDNN, and GPU versions.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 72)
print("TRAINING CONFIGURATION")
print("=" * 72)

print(
    f"\nExperiment: {EXPERIMENT_NAME}"
)

print(
    f"Task: {SELECTED_TASK}"
)

print(
    f"Selected fold: {SELECTED_FOLD}"
)

print(
    f"Device: {DEVICE}"
)

if torch.cuda.is_available():

    print(
        "CUDA device: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        "CUDA memory allocated before model transfer: "
        f"{torch.cuda.memory_allocated(0) / (1024 ** 3):.3f} GB"
    )


# ------------------------------------------------------------
# Move the complete model and objective to the selected device
# ------------------------------------------------------------

complete_model = complete_model.to(
    DEVICE
)

training_objective = training_objective.to(
    DEVICE
)


# ------------------------------------------------------------
# Optimisation hyperparameters
# ------------------------------------------------------------

MAXIMUM_EPOCHS = 50

INITIAL_LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAXIMUM_GRADIENT_NORM = 5.0

EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 3

SCHEDULER_REDUCTION_FACTOR = 0.5

MINIMUM_LEARNING_RATE = 1e-6

CLASSIFICATION_THRESHOLD = 0.50


# ------------------------------------------------------------
# AdamW optimiser
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    params=complete_model.parameters(),
    lr=INITIAL_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ------------------------------------------------------------
# Validation-AUC learning-rate scheduler
# ------------------------------------------------------------

learning_rate_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="max",
        factor=SCHEDULER_REDUCTION_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)


# ------------------------------------------------------------
# Checkpoint and history directories
# ------------------------------------------------------------

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
)

FOLD_TRAINING_DIR = (
    EXPERIMENT_ROOT
    / SELECTED_TASK
    / f"fold_{SELECTED_FOLD}"
)

CHECKPOINT_DIR = (
    FOLD_TRAINING_DIR
    / "checkpoints"
)

HISTORY_DIR = (
    FOLD_TRAINING_DIR
    / "history"
)

PREDICTION_DIR = (
    FOLD_TRAINING_DIR
    / "predictions"
)


BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_validation_auc_checkpoint.pt"
)

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last_epoch_checkpoint.pt"
)


# ------------------------------------------------------------
# Fold-output safety policy
# ------------------------------------------------------------

existing_fold_files = []

if FOLD_TRAINING_DIR.exists():
    existing_fold_files = [
        path
        for path in FOLD_TRAINING_DIR.rglob("*")
        if path.is_file()
    ]


if FOLD_RUN_MODE == "fresh" and existing_fold_files:
    raise FileExistsError(
        "Fresh training was requested, but this fold directory "
        "already contains files. Nothing was overwritten. "
        "Use a different experiment name, remove the intentionally "
        "discarded fold directory, or select resume mode only for "
        "an interrupted run of this exact fold.\n"
        f"Fold directory: {FOLD_TRAINING_DIR}"
    )


if FOLD_RUN_MODE == "resume" and not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Resume mode was requested, but this fold has no latest "
        "checkpoint. Nothing was changed.\n"
        f"Expected checkpoint: {LAST_CHECKPOINT_PATH}"
    )


for directory_path in [
    EXPERIMENT_ROOT,
    FOLD_TRAINING_DIR,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    PREDICTION_DIR,
]:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

TRAINING_HISTORY_PATH = (
    HISTORY_DIR
    / "training_history.csv"
)

TRAINING_CONFIGURATION_PATH = (
    HISTORY_DIR
    / "training_configuration.json"
)

VALIDATION_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "best_validation_predictions.csv"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "test_predictions.csv"
)


# ------------------------------------------------------------
# Record the training configuration
# ------------------------------------------------------------

training_configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "fixed_equal_fusion": True,
    "three_mt_weight": 0.5,
    "tmc_weight": 0.5,
    "task": SELECTED_TASK,
    "fold": int(SELECTED_FOLD),
    "random_seed": int(GLOBAL_RANDOM_SEED),

    "maximum_epochs": int(
        MAXIMUM_EPOCHS
    ),

    "batch_size": int(
        train_loader.batch_size
    ),

    "initial_learning_rate": float(
        INITIAL_LEARNING_RATE
    ),

    "weight_decay": float(
        WEIGHT_DECAY
    ),

    "maximum_gradient_norm": float(
        MAXIMUM_GRADIENT_NORM
    ),

    "early_stopping_patience": int(
        EARLY_STOPPING_PATIENCE
    ),

    "scheduler_patience": int(
        SCHEDULER_PATIENCE
    ),

    "scheduler_reduction_factor": float(
        SCHEDULER_REDUCTION_FACTOR
    ),

    "minimum_learning_rate": float(
        MINIMUM_LEARNING_RATE
    ),

    "classification_threshold": float(
        CLASSIFICATION_THRESHOLD
    ),

    "modality_dropout_probability": float(
        complete_model.modality_dropout_probability
    ),

    "embedding_dimension": int(
        MODALITY_EMBEDDING_DIM
    ),

    "number_of_classes": int(
        NUMBER_OF_CLASSES
    ),

    "class_order": list(
        PROGNOSIS_CLASS_ORDER
    ),

    "modality_order": list(
        MODALITY_ORDER
    ),

    "three_mt_cascade_order": list(
        THREE_MT_CASCADE_ORDER
    ),

    "trainable_parameters": int(
        count_trainable_parameters(
            complete_model
        )
    ),

    "device": str(
        DEVICE
    ),

    "cuda_device_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}


with open(
    TRAINING_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        training_configuration,
        configuration_file,
        indent=2,
    )


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def calculate_binary_expected_calibration_error(
    targets,
    positive_class_probabilities,
    number_of_bins=10,
):
    """
    Calculate equal-width binary expected calibration error.

    Confidence is the probability assigned to the predicted class.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        return float("nan")

    predicted_classes = (
        positive_class_probabilities
        >= 0.5
    ).astype(
        np.int64
    )

    predicted_confidences = np.where(
        predicted_classes == 1,
        positive_class_probabilities,
        1.0 - positive_class_probabilities,
    )

    prediction_correctness = (
        predicted_classes
        == targets
    ).astype(
        np.float64
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    expected_calibration_error = 0.0

    sample_count = targets.size

    for bin_index in range(
        number_of_bins
    ):

        lower_edge = bin_edges[
            bin_index
        ]

        upper_edge = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:

            in_bin = (
                predicted_confidences
                >= lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        else:

            in_bin = (
                predicted_confidences
                > lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        bin_count = int(
            in_bin.sum()
        )

        if bin_count == 0:
            continue

        mean_confidence = float(
            predicted_confidences[
                in_bin
            ].mean()
        )

        mean_accuracy = float(
            prediction_correctness[
                in_bin
            ].mean()
        )

        expected_calibration_error += (
            bin_count
            / sample_count
        ) * abs(
            mean_accuracy
            - mean_confidence
        )

    return float(
        expected_calibration_error
    )


# ------------------------------------------------------------
# Safe metric helpers
# ------------------------------------------------------------

def safely_calculate_roc_auc(
    targets,
    probabilities,
):
    """
    Return NaN when ROC AUC is undefined because only one class
    is present in the supplied targets.
    """

    if np.unique(targets).size < 2:
        return float("nan")

    return float(
        roc_auc_score(
            targets,
            probabilities,
        )
    )


def safely_calculate_average_precision(
    targets,
    probabilities,
):
    """
    Return NaN when average precision is not meaningful because
    the supplied targets contain no positive examples.
    """

    if np.sum(targets == 1) == 0:
        return float("nan")

    return float(
        average_precision_score(
            targets,
            probabilities,
        )
    )


# ------------------------------------------------------------
# Complete binary prognosis metrics
# ------------------------------------------------------------

def calculate_binary_classification_metrics(
    targets,
    positive_class_probabilities,
    uncertainties=None,
    three_mt_weights=None,
    classification_threshold=0.50,
):
    """
    Calculate discrimination, classification, calibration, and
    uncertainty summaries for the pMCI-positive prognosis task.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        raise ValueError(
            "At least one target is required to calculate metrics."
        )

    if (
        targets.shape[0]
        != positive_class_probabilities.shape[0]
    ):
        raise ValueError(
            "Targets and probabilities must contain the same "
            "number of participants."
        )

    positive_class_probabilities = np.clip(
        positive_class_probabilities,
        0.0,
        1.0,
    )

    predicted_classes = (
        positive_class_probabilities
        >= classification_threshold
    ).astype(
        np.int64
    )

    (
        true_negative,
        false_positive,
        false_negative,
        true_positive,
    ) = confusion_matrix(
        targets,
        predicted_classes,
        labels=[0, 1],
    ).ravel()


    sensitivity_denominator = (
        true_positive
        + false_negative
    )

    specificity_denominator = (
        true_negative
        + false_positive
    )

    sensitivity = (
        true_positive
        / sensitivity_denominator
        if sensitivity_denominator > 0
        else float("nan")
    )

    specificity = (
        true_negative
        / specificity_denominator
        if specificity_denominator > 0
        else float("nan")
    )


    clipped_probabilities = np.clip(
        positive_class_probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    metrics = {
        "roc_auc":
            safely_calculate_roc_auc(
                targets,
                positive_class_probabilities,
            ),

        "average_precision":
            safely_calculate_average_precision(
                targets,
                positive_class_probabilities,
            ),

        "accuracy":
            float(
                accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "precision":
            float(
                precision_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "f1":
            float(
                f1_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "brier_score":
            float(
                brier_score_loss(
                    targets,
                    positive_class_probabilities,
                )
            ),

        "negative_log_likelihood":
            float(
                log_loss(
                    targets,
                    np.column_stack(
                        [
                            1.0
                            - clipped_probabilities,

                            clipped_probabilities,
                        ]
                    ),
                    labels=[0, 1],
                )
            ),

        "expected_calibration_error":
            calculate_binary_expected_calibration_error(
                targets=targets,

                positive_class_probabilities=
                    positive_class_probabilities,

                number_of_bins=10,
            ),

        "classification_threshold":
            float(
                classification_threshold
            ),

        "true_negative":
            int(
                true_negative
            ),

        "false_positive":
            int(
                false_positive
            ),

        "false_negative":
            int(
                false_negative
            ),

        "true_positive":
            int(
                true_positive
            ),
    }


    if uncertainties is not None:

        uncertainties = np.asarray(
            uncertainties,
            dtype=np.float64,
        )

        metrics[
            "mean_uncertainty"
        ] = float(
            uncertainties.mean()
        )

        metrics[
            "std_uncertainty"
        ] = float(
            uncertainties.std()
        )


    if three_mt_weights is not None:

        three_mt_weights = np.asarray(
            three_mt_weights,
            dtype=np.float64,
        )

        metrics[
            "mean_three_mt_weight"
        ] = float(
            three_mt_weights.mean()
        )

        metrics[
            "std_three_mt_weight"
        ] = float(
            three_mt_weights.std()
        )

        metrics[
            "minimum_three_mt_weight"
        ] = float(
            three_mt_weights.min()
        )

        metrics[
            "maximum_three_mt_weight"
        ] = float(
            three_mt_weights.max()
        )


    return metrics

# ------------------------------------------------------------
# Exact fold-0 parameter and target summaries
# ------------------------------------------------------------

model_parameter_count = sum(
    parameter.numel()
    for parameter in complete_model.parameters()
    if parameter.requires_grad
)

optimised_parameter_count = sum(
    parameter.numel()
    for parameter_group in optimizer.param_groups
    for parameter in parameter_group["params"]
    if parameter.requires_grad
)

training_target_counts = (
    train_loader
    .dataset
    .dataframe[target_column]
    .value_counts()
    .sort_index()
)

training_target_summary = pd.DataFrame(
    {
        "CLASS_INDEX": [0, 1],
        "CLASS_NAME": ["sMCI", "pMCI"],
        "TRAINING_COUNT": [
            int(training_target_counts.loc[0]),
            int(training_target_counts.loc[1]),
        ],
    }
)

training_target_summary[
    "TRAINING_PROPORTION"
] = (
    training_target_summary[
        "TRAINING_COUNT"
    ]
    /
    training_target_summary[
        "TRAINING_COUNT"
    ].sum()
)


print("\nOptimiser: AdamW")
print(
    "Initial learning rate: "
    f"{INITIAL_LEARNING_RATE:.6f}"
)
print(
    "Weight decay: "
    f"{WEIGHT_DECAY:.6f}"
)
print(
    "Maximum epochs: "
    f"{MAXIMUM_EPOCHS}"
)
print(
    "Early-stopping patience: "
    f"{EARLY_STOPPING_PATIENCE} epochs"
)
print(
    "Scheduler patience: "
    f"{SCHEDULER_PATIENCE} epochs"
)
print(
    "Checkpoint-selection metric: validation ROC AUC"
)

print(
    "\nExperiment output directory:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "\nBest-checkpoint path:\n"
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    "\nTrainable model parameters: "
    f"{model_parameter_count:,}"
)

print(
    "Parameters included in optimiser: "
    f"{optimised_parameter_count:,}"
)

print(
    "\nFold-0 training target balance:"
)

display(
    training_target_summary.round(6)
)


### 1.4.33. Defining reusable training and validation epoch functions

define the reusable functions that will execute one complete training epoch and one complete validation epoch.

A single training epoch performs the following operations for every mini-batch:

1. move the nested multimodal batch to the selected device;
2. run the complete interaction pathway--evidence pathway forward pass in training mode;
3. apply training-time modality dropout;
4. calculate the joint objective;
5. backpropagate the total loss;
6. clip the global gradient norm;
7. update all model parameters using AdamW;
8. accumulate predictions, uncertainty estimates, gate weights, and losses.

The validation epoch uses the same complete model but differs in three important ways:

- the model is placed in evaluation mode;
- modality dropout and ordinary neural-network dropout are disabled;
- no gradients or parameter updates are calculated.

For both training and validation, pMCI is treated as the positive class. The epoch functions collect:

- participant identifiers;
- binary targets;
- final pMCI probabilities;
- final uncertainty values;
- interaction pathway and modality-specific evidence pathway weights;
- interaction pathway-only pMCI probabilities;
- evidence pathway-only pMCI probabilities;
- the original and effective numbers of available modalities.

The accumulated participant-level outputs are passed to the previously defined metric function after the entire epoch has completed.

### 1.4.34. Gradient clipping

For every training batch, the total gradient norm is calculated and clipped before the optimiser step:

$$
\left\|
\nabla_{\boldsymbol{\theta}}
\mathcal{L}_{\mathrm{total}}
\right\|_2
\leq 5.
$$

The unclipped norm is retained for monitoring.

### 1.4.35. Epoch-level loss aggregation

For loss component \(\ell\), the epoch-level mean is calculated using the number of participants in each mini-batch:

$$
\overline{\mathcal{L}}_{\ell}
=
\frac{
\sum_{b=1}^{B}
n_b
\mathcal{L}_{\ell,b}
}{
\sum_{b=1}^{B}
n_b
},
$$

where \(n_b\) is the batch size.

This avoids giving the final incomplete mini-batch the same weight as a full mini-batch.

The functions defined in this step do not yet train the model across multiple epochs. The full checkpointed training loop is constructed in the following step.

In [ ]:
# ============================================================
# 18. Defining reusable training and validation epoch functions
# ============================================================

from collections import defaultdict


# ------------------------------------------------------------
# Initial numerical-precision policy
# ------------------------------------------------------------

# I initially train in full float32 precision.
#
# This is computationally feasible on the available A100 GPU and
# avoids introducing mixed-precision instability into the Dirichlet
# digamma, log-gamma, and Dempster-Shafer calculations.
USE_MIXED_PRECISION = False


# ------------------------------------------------------------
# Recursively move a nested batch to the selected device
# ------------------------------------------------------------

def move_nested_batch_to_device(
    value,
    device,
):
    """
    Recursively move tensors in dictionaries, lists, and tuples
    to the selected PyTorch device.

    Non-tensor values are preserved unchanged.
    """

    if isinstance(
        value,
        torch.Tensor,
    ):
        return value.to(
            device,
            non_blocking=True,
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: move_nested_batch_to_device(
                nested_value,
                device,
            )
            for key, nested_value in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        )

    return value


# ------------------------------------------------------------
# Convert one completed forward pass into stored predictions
# ------------------------------------------------------------

def extract_batch_prediction_arrays(
    batch,
    model_output,
):
    """
    Extract participant-level targets, predictions, uncertainty,
    pathway outputs, and modality counts from one mini-batch.
    """

    final_output = model_output[
        "final_output"
    ]

    joint_opinion = model_output[
        "joint_opinion"
    ]

    tmc_output = model_output[
        "tmc_output"
    ]


    extracted = {
        "rid":
            batch["rid"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "target":
            batch["target"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_p_pMCI":
            final_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_uncertainty":
            final_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_weight":
            final_output[
                "three_mt_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_weight":
            final_output[
                "tmc_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_p_pMCI":
            joint_opinion[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_uncertainty":
            joint_opinion[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_p_pMCI":
            tmc_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_uncertainty":
            tmc_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "original_modality_count":
            model_output[
                "original_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "effective_modality_count":
            model_output[
                "effective_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),
    }

    return extracted


# ------------------------------------------------------------
# Concatenate the participant outputs collected across an epoch
# ------------------------------------------------------------

def concatenate_epoch_prediction_storage(
    prediction_storage,
):
    """
    Concatenate a dictionary of mini-batch NumPy arrays.
    """

    concatenated = {}

    for key, value_list in prediction_storage.items():

        if len(value_list) == 0:
            concatenated[key] = np.asarray([])

        else:
            concatenated[key] = np.concatenate(
                value_list,
                axis=0,
            )

    return concatenated


# ------------------------------------------------------------
# Convert stored predictions into a participant-level table
# ------------------------------------------------------------

def build_epoch_prediction_table(
    concatenated_predictions,
    split_name,
    epoch,
):
    """
    Build one participant-level DataFrame for an epoch.
    """

    prediction_table = pd.DataFrame(
        {
            "RID":
                concatenated_predictions[
                    "rid"
                ].astype(
                    np.int64
                ),

            "TARGET":
                concatenated_predictions[
                    "target"
                ].astype(
                    np.int64
                ),

            "FINAL_P_pMCI":
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            "FINAL_UNCERTAINTY":
                concatenated_predictions[
                    "final_uncertainty"
                ],

            "W_3MT":
                concatenated_predictions[
                    "three_mt_weight"
                ],

            "W_TMC":
                concatenated_predictions[
                    "tmc_weight"
                ],

            "THREE_MT_P_pMCI":
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            "THREE_MT_UNCERTAINTY":
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            "TMC_P_pMCI":
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            "TMC_UNCERTAINTY":
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            "ORIGINAL_MODALITY_COUNT":
                concatenated_predictions[
                    "original_modality_count"
                ].astype(
                    np.int64
                ),

            "EFFECTIVE_MODALITY_COUNT":
                concatenated_predictions[
                    "effective_modality_count"
                ].astype(
                    np.int64
                ),
        }
    )

    prediction_table.insert(
        loc=0,
        column="EPOCH",
        value=int(epoch),
    )

    prediction_table.insert(
        loc=1,
        column="SPLIT",
        value=str(split_name),
    )

    prediction_table[
        "FINAL_PREDICTED_CLASS"
    ] = (
        prediction_table[
            "FINAL_P_pMCI"
        ]
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        np.int64
    )

    return prediction_table


# ------------------------------------------------------------
# Calculate metrics for all three prediction outputs
# ------------------------------------------------------------

def calculate_epoch_prediction_metrics(
    concatenated_predictions,
):
    """
    Calculate metrics for:

    1. the final fixed-equal hybrid output;
    2. the 3MT-only joint output;
    3. the TMC-only fused output.
    """

    targets = concatenated_predictions[
        "target"
    ]

    final_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "final_uncertainty"
                ],

            three_mt_weights=
                concatenated_predictions[
                    "three_mt_weight"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    three_mt_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    tmc_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    return {
        "final": final_metrics,
        "three_mt": three_mt_metrics,
        "tmc": tmc_metrics,
    }


# ------------------------------------------------------------
# Initialise the loss accumulator used within an epoch
# ------------------------------------------------------------

EPOCH_LOSS_NAMES = [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]


def initialise_epoch_loss_storage():
    """
    Create participant-weighted loss totals.
    """

    return {
        loss_name: 0.0
        for loss_name in EPOCH_LOSS_NAMES
    }


def update_epoch_loss_storage(
    storage,
    loss_output,
    batch_size,
):
    """
    Add one batch's losses, weighted by the number of participants.
    """

    for loss_name in EPOCH_LOSS_NAMES:

        storage[loss_name] += (
            float(
                loss_output[
                    loss_name
                ]
                .detach()
                .float()
                .item()
            )
            * batch_size
        )


def finalise_epoch_loss_storage(
    storage,
    participant_count,
):
    """
    Convert accumulated loss sums into participant-weighted means.
    """

    if participant_count <= 0:
        raise ValueError(
            "The epoch contained no participants."
        )

    return {
        loss_name:
            loss_sum
            / participant_count

        for loss_name, loss_sum in storage.items()
    }


# ------------------------------------------------------------
# Run one complete training epoch
# ------------------------------------------------------------

def run_training_epoch(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_gradient_norm,
):
    """
    Train the complete model for one epoch.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []


    for batch_index, batch in enumerate(
        data_loader
    ):

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = targets.shape[0]

        participant_count += batch_size


        # --------------------------------------------------------
        # Clear gradients from the previous mini-batch
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # --------------------------------------------------------
        # Complete forward pass
        # --------------------------------------------------------

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )


        # --------------------------------------------------------
        # Complete multi-output objective
        # --------------------------------------------------------

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )


        # --------------------------------------------------------
        # Backpropagation
        # --------------------------------------------------------

        total_loss.backward()


        # --------------------------------------------------------
        # Global gradient clipping
        # --------------------------------------------------------

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )


        # --------------------------------------------------------
        # Parameter update
        # --------------------------------------------------------

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate losses and predictions
        # --------------------------------------------------------

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[key].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


    # ------------------------------------------------------------
    # Finalise the complete training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# Run one complete validation epoch
# ------------------------------------------------------------

def run_validation_epoch(
    model,
    data_loader,
    objective,
    device,
    epoch,
):
    """
    Evaluate the complete model for one epoch without gradients
    or training-time modality dropout.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    with torch.no_grad():

        for batch_index, batch in enumerate(
            data_loader
        ):

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = targets.shape[0]

            participant_count += batch_size


            # ----------------------------------------------------
            # Complete evaluation forward pass
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ----------------------------------------------------
            # Validation objective
            # ----------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_index}."
                )


            # ----------------------------------------------------
            # Accumulate losses and predictions
            # ----------------------------------------------------

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[key].append(
                    values
                )


    # ------------------------------------------------------------
    # Finalise the complete validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Display the configured epoch-function summary
# ------------------------------------------------------------

print("=" * 72)
print("TRAINING AND VALIDATION EPOCH FUNCTIONS")
print("=" * 72)

print(
    f"\nMixed precision enabled: "
    f"{USE_MIXED_PRECISION}"
)

print(
    "Training batches per epoch: "
    f"{len(train_loader)}"
)

print(
    "Validation batches per epoch: "
    f"{len(validation_loader)}"
)

print(
    "Training participants: "
    f"{len(train_loader.dataset)}"
)

print(
    "Validation participants: "
    f"{len(validation_loader.dataset)}"
)

print(
    "\nTraining epoch operations:"
)

print(
    "- forward pass with modality dropout;"
)

print(
    "- complete joint loss calculation;"
)

print(
    "- backpropagation;"
)

print(
    "- global gradient clipping;"
)

print(
    "- AdamW parameter update;"
)

print(
    "- prediction and uncertainty accumulation."
)

print(
    "\nValidation epoch operations:"
)

print(
    "- evaluation mode;"
)

print(
    "- no modality dropout;"
)

print(
    "- no gradient calculation;"
)

print(
    "- complete validation loss and metric calculation."
)

print(
    "\nThe epoch functions are defined."
)

print(
    "No complete training or validation epoch has been run yet."
)

print(
    "The next step will create the checkpointed multi-epoch "
    "training loop and begin model optimisation."
)

### 1.4.36. Training a newly initialised model with validation-based checkpointing

This standalone experiment starts from epoch 1 using the model initialised in this notebook. It does not load a checkpoint from the previous ungated baseline or from an earlier gated run.

The validation partition controls learning-rate reduction, early stopping, and best-checkpoint selection. The test partition remains untouched during training.

In [ ]:
# ============================================================
# 19. Training with live batch and epoch progress
# ============================================================

import time
import traceback
from collections import defaultdict
from datetime import datetime

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Resume and progress-display behaviour
# ------------------------------------------------------------

# The formal first run begins from epoch 1.
# Resume is enabled only when FOLD_RUN_MODE was explicitly set
# to "resume" for this exact experiment and fold.
RESUME_FROM_LAST_CHECKPOINT = (
    FOLD_RUN_MODE == "resume"
)

# I refresh the live progress display after every batch.
PROGRESS_UPDATE_INTERVAL = 1


# ------------------------------------------------------------
# Loss-configuration serialisation
# ------------------------------------------------------------

def obtain_training_objective_configuration(
    objective,
):
    """
    Return the principal loss settings in a checkpoint-safe form.
    """

    return {
        "evidential_annealing_epochs": int(
            objective.evidential_annealing_epochs
        ),

        "gate_regularisation_epochs": int(
            objective.gate_regularisation_epochs
        ),

        "joint_loss_weight": float(
            objective.joint_loss_weight
        ),

        "tmc_loss_weight": float(
            objective.tmc_loss_weight
        ),

        "modality_loss_weight": float(
            objective.modality_loss_weight
        ),

        "auxiliary_loss_weight": float(
            objective.auxiliary_loss_weight
        ),

        "gate_loss_weight": float(
            objective.gate_loss_weight
        ),

        "class_weights": (
            None
            if objective.class_weights is None
            else
            objective.class_weights
            .detach()
            .cpu()
            .tolist()
        ),
    }


# ------------------------------------------------------------
# Flatten one epoch into one history row
# ------------------------------------------------------------

def build_training_history_row(
    epoch,
    learning_rate,
    epoch_duration_seconds,
    training_result,
    validation_result,
    best_validation_auc,
    epochs_without_improvement,
    checkpoint_improved,
):
    """
    Create one flat row containing losses, metrics, optimisation
    diagnostics, and checkpoint information.
    """

    history_row = {
        "epoch":
            int(epoch),

        "learning_rate":
            float(learning_rate),

        "epoch_duration_seconds":
            float(epoch_duration_seconds),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "checkpoint_improved":
            bool(checkpoint_improved),

        "train_participants":
            int(
                training_result[
                    "participant_count"
                ]
            ),

        "validation_participants":
            int(
                validation_result[
                    "participant_count"
                ]
            ),

        "train_batches":
            int(
                training_result[
                    "batch_count"
                ]
            ),

        "validation_batches":
            int(
                validation_result[
                    "batch_count"
                ]
            ),

        "train_mean_gradient_norm":
            float(
                training_result[
                    "gradient_summary"
                ]["mean_gradient_norm"]
            ),

        "train_maximum_gradient_norm_before_clipping":
            float(
                training_result[
                    "gradient_summary"
                ][
                    "maximum_gradient_norm_before_clipping"
                ]
            ),

        "train_mean_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "mean_effective_modality_count"
                ]
            ),

        "train_minimum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "minimum_effective_modality_count"
                ]
            ),

        "train_maximum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "maximum_effective_modality_count"
                ]
            ),

        "validation_maximum_modality_count_difference":
            float(
                validation_result[
                    "maximum_evaluation_modality_count_difference"
                ]
            ),
    }


    # --------------------------------------------------------
    # Add all training and validation losses
    # --------------------------------------------------------

    for loss_name, loss_value in (
        training_result[
            "losses"
        ].items()
    ):
        history_row[
            f"train_{loss_name}"
        ] = float(
            loss_value
        )

    for loss_name, loss_value in (
        validation_result[
            "losses"
        ].items()
    ):
        history_row[
            f"validation_{loss_name}"
        ] = float(
            loss_value
        )


    # --------------------------------------------------------
    # Add metrics for all three prediction outputs
    # --------------------------------------------------------

    for output_name in [
        "final",
        "three_mt",
        "tmc",
    ]:

        for metric_name, metric_value in (
            training_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"train_{output_name}_{metric_name}"
            ] = metric_value

        for metric_name, metric_value in (
            validation_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"validation_{output_name}_{metric_name}"
            ] = metric_value


    return history_row


# ------------------------------------------------------------
# Save a fully recoverable checkpoint
# ------------------------------------------------------------

def save_training_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_validation_auc,
    epochs_without_improvement,
    validation_metrics,
    training_history,
):
    """
    Save the state required to reproduce or continue training.
    """

    checkpoint = {
        "epoch":
            int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "validation_metrics":
            validation_metrics,

        "training_configuration":
            training_configuration,

        "training_objective_configuration":
            obtain_training_objective_configuration(
                training_objective
            ),

        "training_history":
            training_history,

        "random_seed":
            int(GLOBAL_RANDOM_SEED),

        "task":
            SELECTED_TASK,

        "fold":
            int(SELECTED_FOLD),

        "saved_at":
            datetime.now().isoformat(
                timespec="seconds"
            ),
    }

    torch.save(
        checkpoint,
        checkpoint_path,
    )


# ------------------------------------------------------------
# GPU-memory helper for the progress display
# ------------------------------------------------------------

def current_cuda_memory_gb():
    """
    Return currently allocated CUDA memory in gigabytes.
    """

    if not torch.cuda.is_available():
        return 0.0

    return float(
        torch.cuda.memory_allocated(
            DEVICE
        )
        / (1024 ** 3)
    )


# ------------------------------------------------------------
# One training epoch with a live batch progress bar
# ------------------------------------------------------------

def run_training_epoch_with_progress(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_epochs,
    maximum_gradient_norm,
):
    """
    Train for one epoch while displaying batch-level progress.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | training"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    for batch_number, batch in progress_bar:

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = int(
            targets.shape[0]
        )

        participant_count += (
            batch_size
        )


        # --------------------------------------------------------
        # Forward pass and loss
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )


        # --------------------------------------------------------
        # Backpropagation, clipping, and update
        # --------------------------------------------------------

        total_loss.backward()

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate diagnostics
        # --------------------------------------------------------

        current_total_loss = float(
            total_loss.detach().item()
        )

        running_total_loss_sum += (
            current_total_loss
            * batch_size
        )

        running_mean_total_loss = (
            running_total_loss_sum
            / participant_count
        )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[
                key
            ].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


        # --------------------------------------------------------
        # Update the visible progress information
        # --------------------------------------------------------

        if (
            batch_number
            % PROGRESS_UPDATE_INTERVAL
            == 0
            or batch_number
            == len(data_loader)
        ):

            elapsed_minutes = (
                time.time()
                - phase_start_time
            ) / 60.0

            progress_bar.set_postfix(
                {
                    "loss":
                        f"{current_total_loss:.3f}",

                    "avg":
                        f"{running_mean_total_loss:.3f}",

                    "grad":
                        f"{float(gradient_norm):.2f}",

                    "lr":
                        f"{optimizer.param_groups[0]['lr']:.1e}",

                    "GPU":
                        f"{current_cuda_memory_gb():.1f}GB",

                    "elapsed":
                        f"{elapsed_minutes:.1f}m",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise the training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# One validation epoch with a live batch progress bar
# ------------------------------------------------------------

def run_validation_epoch_with_progress(
    model,
    data_loader,
    objective,
    device,
    epoch,
    maximum_epochs,
):
    """
    Validate for one epoch while displaying batch-level progress.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | validation"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ----------------------------------------------------
            # Forward pass and validation loss
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            total_loss = loss_output[
                "total_loss"
            ]


            if not torch.isfinite(
                total_loss
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_number}."
                )


            # ----------------------------------------------------
            # Accumulate diagnostics and predictions
            # ----------------------------------------------------

            current_total_loss = float(
                total_loss.detach().item()
            )

            running_total_loss_sum += (
                current_total_loss
                * batch_size
            )

            running_mean_total_loss = (
                running_total_loss_sum
                / participant_count
            )

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[
                    key
                ].append(
                    values
                )


            # ----------------------------------------------------
            # Update the visible progress information
            # ----------------------------------------------------

            if (
                batch_number
                % PROGRESS_UPDATE_INTERVAL
                == 0
                or batch_number
                == len(data_loader)
            ):

                elapsed_minutes = (
                    time.time()
                    - phase_start_time
                ) / 60.0

                progress_bar.set_postfix(
                    {
                        "loss":
                            f"{current_total_loss:.3f}",

                        "avg":
                            f"{running_mean_total_loss:.3f}",

                        "GPU":
                            f"{current_cuda_memory_gb():.1f}GB",

                        "elapsed":
                            f"{elapsed_minutes:.1f}m",
                    },
                    refresh=True,
                )


    # ------------------------------------------------------------
    # Finalise the validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                concatenated_predictions[
                    "original_modality_count"
                ]
                -
                concatenated_predictions[
                    "effective_modality_count"
                ]
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Initial training state
# ------------------------------------------------------------

training_history = []

starting_epoch = 1

best_validation_auc = float(
    "-inf"
)

epochs_without_improvement = 0


# ------------------------------------------------------------
# Resume from the latest fully completed epoch
# ------------------------------------------------------------

if (
    RESUME_FROM_LAST_CHECKPOINT
    and LAST_CHECKPOINT_PATH.exists()
):

    print(
        "Loading the latest fully completed checkpoint:",
        flush=True,
    )

    print(
        LAST_CHECKPOINT_PATH,
        flush=True,
    )

    resumed_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    complete_model.load_state_dict(
        resumed_checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        resumed_checkpoint[
            "optimizer_state_dict"
        ]
    )

    learning_rate_scheduler.load_state_dict(
        resumed_checkpoint[
            "scheduler_state_dict"
        ]
    )

    completed_epoch = int(
        resumed_checkpoint[
            "epoch"
        ]
    )

    starting_epoch = (
        completed_epoch
        + 1
    )

    best_validation_auc = float(
        resumed_checkpoint[
            "best_validation_auc"
        ]
    )

    epochs_without_improvement = int(
        resumed_checkpoint[
            "epochs_without_improvement"
        ]
    )

    training_history = list(
        resumed_checkpoint.get(
            "training_history",
            [],
        )
    )

    print(
        f"\nResuming after epoch {completed_epoch}.",
        flush=True,
    )

    print(
        f"Next epoch: {starting_epoch}",
        flush=True,
    )

    print(
        "Best validation ROC AUC so far: "
        f"{best_validation_auc:.6f}",
        flush=True,
    )

else:

    print(
        "No fully completed checkpoint was loaded.",
        flush=True,
    )

    print(
        "Fresh training begins from the newly initialised model state.",
        flush=True,
    )


# ------------------------------------------------------------
# Main multi-epoch training loop
# ------------------------------------------------------------

if starting_epoch > MAXIMUM_EPOCHS:

    print(
        "\nTraining has already reached the configured maximum "
        f"of {MAXIMUM_EPOCHS} epochs.",
        flush=True,
    )

else:

    print("\n" + "=" * 72, flush=True)
    print("BEGINNING MODEL TRAINING", flush=True)
    print("=" * 72, flush=True)

    print(
        f"\nEpoch range: {starting_epoch}--{MAXIMUM_EPOCHS}",
        flush=True,
    )

    print(
        f"Training batches per epoch: {len(train_loader)}",
        flush=True,
    )

    print(
        f"Validation batches per epoch: {len(validation_loader)}",
        flush=True,
    )

    print(
        "Each epoch displays separate live training and "
        "validation progress bars.",
        flush=True,
    )

    print(
        "The test partition will not be evaluated.",
        flush=True,
    )


    try:

        for epoch in range(
            starting_epoch,
            MAXIMUM_EPOCHS + 1,
        ):

            epoch_start_time = time.time()

            current_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )


            print("\n" + "=" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d}/{MAXIMUM_EPOCHS}",
                flush=True,
            )

            print("=" * 72, flush=True)

            print(
                "\nPhase 1/4: training batches",
                flush=True,
            )


            # ------------------------------------------------
            # Train on all training participants
            # ------------------------------------------------

            training_result = (
                run_training_epoch_with_progress(
                    model=complete_model,
                    data_loader=train_loader,
                    objective=training_objective,
                    optimizer=optimizer,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                    maximum_gradient_norm=
                        MAXIMUM_GRADIENT_NORM,
                )
            )


            print(
                "\nPhase 2/4: validation batches",
                flush=True,
            )


            # ------------------------------------------------
            # Validate on all validation participants
            # ------------------------------------------------

            validation_result = (
                run_validation_epoch_with_progress(
                    model=complete_model,
                    data_loader=validation_loader,
                    objective=training_objective,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                )
            )


            print(
                "\nPhase 3/4: calculating metrics and "
                "updating the scheduler",
                flush=True,
            )


            # ------------------------------------------------
            # Validation AUC and checkpoint decision
            # ------------------------------------------------

            validation_auc = float(
                validation_result[
                    "metrics"
                ]["final"]["roc_auc"]
            )

            validation_auc_is_valid = bool(
                np.isfinite(
                    validation_auc
                )
            )

            checkpoint_improved = (
                validation_auc_is_valid
                and
                validation_auc
                > best_validation_auc
            )

            if checkpoint_improved:

                best_validation_auc = (
                    validation_auc
                )

                epochs_without_improvement = 0

            else:

                epochs_without_improvement += 1


            scheduler_score = (
                validation_auc
                if validation_auc_is_valid
                else -1.0
            )

            learning_rate_scheduler.step(
                scheduler_score
            )

            updated_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            epoch_duration_seconds = (
                time.time()
                - epoch_start_time
            )


            # ------------------------------------------------
            # Persistent training history
            # ------------------------------------------------

            history_row = build_training_history_row(
                epoch=epoch,

                learning_rate=
                    current_learning_rate,

                epoch_duration_seconds=
                    epoch_duration_seconds,

                training_result=
                    training_result,

                validation_result=
                    validation_result,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                checkpoint_improved=
                    checkpoint_improved,
            )

            history_row[
                "learning_rate_after_scheduler"
            ] = updated_learning_rate

            training_history.append(
                history_row
            )

            pd.DataFrame(
                training_history
            ).to_csv(
                TRAINING_HISTORY_PATH,
                index=False,
            )


            print(
                "\nPhase 4/4: saving checkpoints and history",
                flush=True,
            )


            # ------------------------------------------------
            # Save the latest completed epoch
            # ------------------------------------------------

            save_training_checkpoint(
                checkpoint_path=
                    LAST_CHECKPOINT_PATH,

                epoch=epoch,

                model=complete_model,

                optimizer=optimizer,

                scheduler=
                    learning_rate_scheduler,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                validation_metrics=
                    validation_result[
                        "metrics"
                    ],

                training_history=
                    training_history,
            )


            # ------------------------------------------------
            # Save the best validation checkpoint
            # ------------------------------------------------

            if checkpoint_improved:

                save_training_checkpoint(
                    checkpoint_path=
                        BEST_CHECKPOINT_PATH,

                    epoch=epoch,

                    model=complete_model,

                    optimizer=optimizer,

                    scheduler=
                        learning_rate_scheduler,

                    best_validation_auc=
                        best_validation_auc,

                    epochs_without_improvement=
                        epochs_without_improvement,

                    validation_metrics=
                        validation_result[
                            "metrics"
                        ],

                    training_history=
                        training_history,
                )

                validation_result[
                    "predictions"
                ].to_csv(
                    VALIDATION_PREDICTIONS_PATH,
                    index=False,
                )


            # ------------------------------------------------
            # Readable completed-epoch summary
            # ------------------------------------------------

            train_metrics = training_result[
                "metrics"
            ]["final"]

            validation_metrics = validation_result[
                "metrics"
            ]["final"]


            print("\n" + "-" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d} COMPLETE",
                flush=True,
            )

            print(
                "Duration: "
                f"{epoch_duration_seconds / 60.0:.2f} minutes",
                flush=True,
            )

            print(
                "Learning rate: "
                f"{current_learning_rate:.8f}"
                f" -> {updated_learning_rate:.8f}",
                flush=True,
            )


            print("\nTraining:", flush=True)

            print(
                "  total loss: "
                f"{training_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{train_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{train_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{train_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{train_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )

            print(
                "  mean effective modalities: "
                f"{training_result['modality_dropout_summary']['mean_effective_modality_count']:.3f}",
                flush=True,
            )

            print(
                "  maximum pre-clipping gradient norm: "
                f"{training_result['gradient_summary']['maximum_gradient_norm_before_clipping']:.6f}",
                flush=True,
            )


            print("\nValidation:", flush=True)

            print(
                "  total loss: "
                f"{validation_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{validation_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  average precision: "
                f"{validation_metrics['average_precision']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{validation_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  sensitivity: "
                f"{validation_metrics['sensitivity']:.6f}",
                flush=True,
            )

            print(
                "  specificity: "
                f"{validation_metrics['specificity']:.6f}",
                flush=True,
            )

            print(
                "  Brier score: "
                f"{validation_metrics['brier_score']:.6f}",
                flush=True,
            )

            print(
                "  calibration error: "
                f"{validation_metrics['expected_calibration_error']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{validation_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{validation_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )


            print("\nCheckpoint status:", flush=True)

            print(
                "  improved this epoch: "
                f"{checkpoint_improved}",
                flush=True,
            )

            print(
                "  best validation ROC AUC: "
                f"{best_validation_auc:.6f}",
                flush=True,
            )

            print(
                "  epochs without improvement: "
                f"{epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}",
                flush=True,
            )

            print(
                "  latest completed epoch saved: True",
                flush=True,
            )

            if torch.cuda.is_available():

                print(
                    "  peak CUDA memory: "
                    f"{torch.cuda.max_memory_allocated(0) / (1024 ** 3):.3f} GB",
                    flush=True,
                )


            # ------------------------------------------------
            # Early stopping
            # ------------------------------------------------

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):

                print(
                    "\nEarly stopping activated because "
                    "validation ROC AUC did not improve for "
                    f"{EARLY_STOPPING_PATIENCE} consecutive epochs.",
                    flush=True,
                )

                break


    # --------------------------------------------------------
    # Interruption and error handling
    # --------------------------------------------------------

    except KeyboardInterrupt:

        print(
            "\nTraining was interrupted manually.",
            flush=True,
        )

        print(
            "Only fully completed epochs are recoverable from "
            "the last-epoch checkpoint.",
            flush=True,
        )


    except Exception:

        print(
            "\nTraining stopped because an exception occurred.",
            flush=True,
        )

        print(
            "The latest fully completed epoch remains saved.",
            flush=True,
        )

        traceback.print_exc()

        raise


    # --------------------------------------------------------
    # Final training summary
    # --------------------------------------------------------

    if len(
        training_history
    ) > 0:

        final_history_table = pd.DataFrame(
            training_history
        )

        completed_epochs = int(
            final_history_table[
                "epoch"
            ].max()
        )

        print("\n" + "=" * 72, flush=True)

        print(
            "TRAINING RUN COMPLETE",
            flush=True,
        )

        print("=" * 72, flush=True)

        print(
            f"\nLast completed epoch: {completed_epochs}",
            flush=True,
        )

        print(
            "Best validation ROC AUC: "
            f"{best_validation_auc:.6f}",
            flush=True,
        )

        print(
            "\nBest checkpoint:\n"
            f"{BEST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nLatest checkpoint:\n"
            f"{LAST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nTraining history:\n"
            f"{TRAINING_HISTORY_PATH}",
            flush=True,
        )

        print(
            "\nThe test partition has not been evaluated.",
            flush=True,
        )

### 1.4.37. Evaluating the best gated checkpoint on the untouched test set

After training and validation-based checkpoint selection, the best checkpoint from this named experiment is loaded from its experiment-specific fold directory and evaluated once on the held-out test partition.

The evaluation cell does not update model parameters, the optimiser, the scheduler, or modality-dropout state.

In [ ]:
from datetime import datetime


# ------------------------------------------------------------
# Test-output paths
# ------------------------------------------------------------

TEST_METRICS_PATH = (
    PREDICTION_DIR
    / "test_metrics.json"
)

TEST_SUMMARY_PATH = (
    PREDICTION_DIR
    / "test_metrics_summary.csv"
)


print("=" * 72)
print("FIXED-EQUAL-FUSION TEST-SET EVALUATION")
print("=" * 72)

print(
    "\nLoading the best validation checkpoint:\n"
    f"{BEST_CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# Load the best validation checkpoint
# ------------------------------------------------------------


best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_checkpoint_epoch = int(
    best_checkpoint[
        "epoch"
    ]
)

best_checkpoint_validation_auc = float(
    best_checkpoint[
        "best_validation_auc"
    ]
)


complete_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


print(
    f"\nBest checkpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Best validation ROC AUC stored in checkpoint: "
    f"{best_checkpoint_validation_auc:.6f}"
)


# ------------------------------------------------------------
# Test evaluation function
# ------------------------------------------------------------

def run_test_epoch(
    model,
    data_loader,
    objective,
    device,
    checkpoint_epoch,
):
    """
    Evaluate the frozen model on the untouched test partition.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc="Test evaluation",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ------------------------------------------------
            # Frozen forward pass
            # ------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ------------------------------------------------
            # Test loss
            # ------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=checkpoint_epoch,
            )


            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):

                raise FloatingPointError(
                    "A non-finite test loss was encountered "
                    f"at batch {batch_number}."
                )


            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )


            # ------------------------------------------------
            # Store participant-level outputs
            # ------------------------------------------------

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):

                prediction_storage[
                    key
                ].append(
                    values
                )


            running_average_loss = (
                loss_storage[
                    "total_loss"
                ]
                / participant_count
            )

            progress_bar.set_postfix(
                {
                    "avg_loss":
                        f"{running_average_loss:.3f}",

                    "participants":
                        f"{participant_count}/{len(data_loader.dataset)}",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise test losses and predictions
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="test",

            epoch=checkpoint_epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_modality_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_modality_count_difference,
    }


# ------------------------------------------------------------
# Run the untouched test evaluation
# ------------------------------------------------------------

test_result = run_test_epoch(
    model=complete_model,
    data_loader=test_loader,
    objective=training_objective,
    device=DEVICE,
    checkpoint_epoch=best_checkpoint_epoch,
)


# ------------------------------------------------------------
# Evaluation-mode modality count
# ------------------------------------------------------------

maximum_test_mask_difference = (
    test_result[
        "maximum_evaluation_modality_count_difference"
    ]
)


# ------------------------------------------------------------
# Save participant-level test predictions
# ------------------------------------------------------------


test_prediction_table = test_result[
    "predictions"
]

test_prediction_table.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save all test metrics in JSON format
# ------------------------------------------------------------

serialisable_test_result = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "task":
        SELECTED_TASK,

    "fold":
        int(
            SELECTED_FOLD
        ),

    "checkpoint_epoch":
        int(
            best_checkpoint_epoch
        ),

    "checkpoint_validation_auc":
        float(
            best_checkpoint_validation_auc
        ),

    "classification_threshold":
        float(
            CLASSIFICATION_THRESHOLD
        ),

    "test_participants":
        int(
            test_result[
                "participant_count"
            ]
        ),

    "test_batches":
        int(
            test_result[
                "batch_count"
            ]
        ),

    "test_losses": {
        key:
            float(value)

        for key, value in (
            test_result[
                "losses"
            ].items()
        )
    },

    "test_metrics":
        test_result[
            "metrics"
        ],

    "maximum_modality_count_difference":
        float(
            maximum_test_mask_difference
        ),

    "evaluated_at":
        datetime.now().isoformat(
            timespec="seconds"
        ),
}


with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as test_metrics_file:

    json.dump(
        serialisable_test_result,
        test_metrics_file,
        indent=2,
    )


# ------------------------------------------------------------
# Build a compact comparison table
# ------------------------------------------------------------

test_metric_rows = []

for output_name, display_name in [
    (
        "final",
        "Hybrid",
    ),
    (
        "three_mt",
        "3MT-only",
    ),
    (
        "tmc",
        "TMC-only",
    ),
]:

    output_metrics = test_result[
        "metrics"
    ][
        output_name
    ]

    test_metric_rows.append(
        {
            "OUTPUT":
                display_name,

            "ROC_AUC":
                output_metrics[
                    "roc_auc"
                ],

            "AVERAGE_PRECISION":
                output_metrics[
                    "average_precision"
                ],

            "ACCURACY":
                output_metrics[
                    "accuracy"
                ],

            "BALANCED_ACCURACY":
                output_metrics[
                    "balanced_accuracy"
                ],

            "SENSITIVITY":
                output_metrics[
                    "sensitivity"
                ],

            "SPECIFICITY":
                output_metrics[
                    "specificity"
                ],

            "PRECISION":
                output_metrics[
                    "precision"
                ],

            "F1":
                output_metrics[
                    "f1"
                ],

            "BRIER_SCORE":
                output_metrics[
                    "brier_score"
                ],

            "NEGATIVE_LOG_LIKELIHOOD":
                output_metrics[
                    "negative_log_likelihood"
                ],

            "EXPECTED_CALIBRATION_ERROR":
                output_metrics[
                    "expected_calibration_error"
                ],

            "MEAN_UNCERTAINTY":
                output_metrics[
                    "mean_uncertainty"
                ],
        }
    )


test_metrics_summary = pd.DataFrame(
    test_metric_rows
)

test_metrics_summary.to_csv(
    TEST_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display the final test results
# ------------------------------------------------------------

hybrid_test_metrics = test_result[
    "metrics"
][
    "final"
]


print("\n" + "=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} TEST RESULTS")
print("=" * 72)

print(
    f"\nCheckpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Test participants: "
    f"{test_result['participant_count']}"
)

print(
    "Test batches: "
    f"{test_result['batch_count']}"
)

print(
    "Maximum difference between original and effective "
    "test modality counts: "
    f"{maximum_test_mask_difference:.1f}"
)


print(
    "\nFinal hybrid test performance:"
)

print(
    "  ROC AUC: "
    f"{hybrid_test_metrics['roc_auc']:.6f}"
)

print(
    "  average precision: "
    f"{hybrid_test_metrics['average_precision']:.6f}"
)

print(
    "  accuracy: "
    f"{hybrid_test_metrics['accuracy']:.6f}"
)

print(
    "  balanced accuracy: "
    f"{hybrid_test_metrics['balanced_accuracy']:.6f}"
)

print(
    "  sensitivity: "
    f"{hybrid_test_metrics['sensitivity']:.6f}"
)

print(
    "  specificity: "
    f"{hybrid_test_metrics['specificity']:.6f}"
)

print(
    "  precision: "
    f"{hybrid_test_metrics['precision']:.6f}"
)

print(
    "  F1 score: "
    f"{hybrid_test_metrics['f1']:.6f}"
)

print(
    "  Brier score: "
    f"{hybrid_test_metrics['brier_score']:.6f}"
)

print(
    "  negative log-likelihood: "
    f"{hybrid_test_metrics['negative_log_likelihood']:.6f}"
)

print(
    "  expected calibration error: "
    f"{hybrid_test_metrics['expected_calibration_error']:.6f}"
)

print(
    "  mean uncertainty: "
    f"{hybrid_test_metrics['mean_uncertainty']:.6f}"
)

print(
    "  mean 3MT weight: "
    f"{hybrid_test_metrics['mean_three_mt_weight']:.6f}"
)

print(
    "  standard deviation of 3MT weight: "
    f"{hybrid_test_metrics['std_three_mt_weight']:.6f}"
)


print(
    "\nConfusion matrix counts:"
)

print(
    "  true negatives: "
    f"{hybrid_test_metrics['true_negative']}"
)

print(
    "  false positives: "
    f"{hybrid_test_metrics['false_positive']}"
)

print(
    "  false negatives: "
    f"{hybrid_test_metrics['false_negative']}"
)

print(
    "  true positives: "
    f"{hybrid_test_metrics['true_positive']}"
)


print(
    "\nHybrid, 3MT-only, and TMC-only comparison:"
)

display(
    test_metrics_summary.round(
        6
    )
)


print(
    "\nParticipant-level predictions saved to:\n"
    f"{TEST_PREDICTIONS_PATH}"
)

print(
    "\nComplete test metrics saved to:\n"
    f"{TEST_METRICS_PATH}"
)

print(
    "\nCompact metric summary saved to:\n"
    f"{TEST_SUMMARY_PATH}"
)

print(
    "\nThe model was evaluated without gradient updates, "
    "scheduler changes, or test-time modality dropout."
)

### 1.4.38. Releasing fold-specific memory

The fold-0 checkpoints, histories, validation predictions, test predictions, and metrics are already stored in its own directory. This cleanup removes the in-memory model and DataLoaders before the next fold is created.

In [ ]:
# ============================================================
# 21. Releasing fold-0 memory before the next fold
# ============================================================

import gc

objects_to_release = [
    "complete_model",
    "optimizer",
    "scheduler",
    "training_objective",
    "train_loader",
    "validation_loader",
    "test_loader",
    "train_dataset",
    "validation_dataset",
    "test_dataset",
    "example_batch",
    "diagnostic_batch",
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Fold 0 outputs remain saved under:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "Fold-specific GPU cache has been released."
)


## 1.5. Fold 1

This section trains, validates, and evaluates fold 1. It uses the exact prepared table `mci_prognosis_outer_fold_1_final_task_ready.csv` and writes only to the experiment's `fold_1` directory.


### 1.5.1. Loading the prepared fold-1 input

The model-input columns are defined explicitly in this notebook, matching the original fold-0 implementation. No schema file or output from an earlier experiment is loaded.


In [ ]:
# ============================================================
# Loading the prepared inputs for fold 1
# ============================================================

from pathlib import Path
import json
import pandas as pd

from google.colab import drive


# ------------------------------------------------------------
# Experiment identity
# ------------------------------------------------------------

EXPERIMENT_NAME = "temporal_prebaseline_fixed_equal_fusion_md050"
SELECTED_TASK = "mci_prognosis"
SELECTED_FOLD = 1

# Use "fresh" for the formal first run.
# Change this to "resume" only after an interrupted run of this
# same experiment and fold.
FOLD_RUN_MODE = "fresh"


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():
    drive.mount(
        "/content/drive"
    )


# ------------------------------------------------------------
# Exact project and prepared-input paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

SELECTED_INPUT_PATH = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / "mci_prognosis"
    / f"mci_prognosis_outer_fold_{SELECTED_FOLD}_final_task_ready.csv"
)


# ------------------------------------------------------------
# Fixed model-input columns from the prepared pipeline
# ------------------------------------------------------------

final_model_schema = {
    "identifier_columns": [
        "RID",
        "PTID",
    ],

    "audit_label_columns": [
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ],

    "model_target_column":
        "MODEL_TARGET",

    "split_columns": [
        "OUTER_FOLD",
        "DATA_ROLE",
    ],

    "scaled_continuous_columns": [
        "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
        "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        "ADAS__TOTSCORE__Z",
        "ADAS__TOTAL13__Z",
        "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
        "FAQ__FAQTOTAL__Z",
        "CSF__ABETA40__Z",
        "CSF__ABETA42__Z",
        "CSF__TAU__Z",
        "CSF__PTAU__Z",
        "CSF__ABETA42_40_RATIO__Z",
        "PLASMA__pT217_F__Z",
        "PLASMA__AB42_F__Z",
        "PLASMA__AB40_F__Z",
        "PLASMA__AB42_AB40_F__Z",
        "PLASMA__pT217_AB42_F__Z",
        "PLASMA__NfL_Q__Z",
        "PLASMA__GFAP_Q__Z",
        "PLASMA__NfL_F__Z",
        "PLASMA__GFAP_F__Z",
    ],

    "encoded_categorical_columns": [
        "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
        "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        "APOE__APOE4_ALLELE_COUNT__IDX",
    ],

    "mri_path_columns": [
        "MRI__NORMALIZED_T1_NPY_PATH",
    ],

    "branch_mask_columns": [
        "BRANCH_MASK__DEMOGRAPHICS",
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
        "BRANCH_MASK__CSF",
        "BRANCH_MASK__PLASMA",
        "BRANCH_MASK__APOE",
        "BRANCH_MASK__MRI",
    ],

    "feature_mask_columns": [
        "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
        "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        "FEATURE_MASK__ADAS_TOTSCORE",
        "FEATURE_MASK__ADAS_TOTAL13",
        "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
        "FEATURE_MASK__FAQ_FAQTOTAL",
        "FEATURE_MASK__CSF_ABETA40",
        "FEATURE_MASK__CSF_ABETA42",
        "FEATURE_MASK__CSF_TAU",
        "FEATURE_MASK__CSF_PTAU",
        "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        "FEATURE_MASK__PLASMA_pT217_F",
        "FEATURE_MASK__PLASMA_AB42_F",
        "FEATURE_MASK__PLASMA_AB40_F",
        "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "FEATURE_MASK__PLASMA_NfL_Q",
        "FEATURE_MASK__PLASMA_GFAP_Q",
        "FEATURE_MASK__PLASMA_NfL_F",
        "FEATURE_MASK__PLASMA_GFAP_F",
        "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
    ],
}



# ------------------------------------------------------------
# Load the prepared fold table
# ------------------------------------------------------------

fold_table = pd.read_csv(
    SELECTED_INPUT_PATH
)


print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} INPUT")
print("=" * 72)

print(
    f"\nTask-ready table:\n{SELECTED_INPUT_PATH}"
)

print(
    f"\nLoaded shape: "
    f"{fold_table.shape[0]} rows × "
    f"{fold_table.shape[1]} columns"
)


### 1.5.2. Preparing the fold-1 model-input groups

The continuous, categorical, MRI-path, branch-mask, feature-mask, identifier, target, and split columns are taken from the fixed definitions loaded above.


In [ ]:
# ============================================================
# 2. Reading the prepared schema and describing fold 0
# ============================================================

# ------------------------------------------------------------
# Prepared model-input column groups
# ------------------------------------------------------------

identifier_columns = final_model_schema[
    "identifier_columns"
]

audit_label_columns = final_model_schema[
    "audit_label_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

split_columns = final_model_schema[
    "split_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]


# ------------------------------------------------------------
# Fold-0 structure
# ------------------------------------------------------------

print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} EXPERIMENT")
print("=" * 72)

print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Task: {SELECTED_TASK}")
print(f"Outer fold: {SELECTED_FOLD}")

print("\nPrepared data roles:")
print(
    fold_table["DATA_ROLE"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared target counts by role:")
display(
    pd.crosstab(
        fold_table["DATA_ROLE"],
        fold_table[target_column],
    ).reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared input groups:")
print(
    f"- scaled continuous features: "
    f"{len(scaled_continuous_columns)}"
)
print(
    f"- encoded categorical features: "
    f"{len(encoded_categorical_columns)}"
)
print(
    f"- MRI path columns: "
    f"{len(mri_path_columns)}"
)
print(
    f"- branch masks: "
    f"{len(branch_mask_columns)}"
)
print(
    f"- feature masks: "
    f"{len(feature_mask_columns)}"
)

preview_columns = (
    identifier_columns
    + ["CLINICAL_GROUP"]
    + split_columns
    + [target_column]
    + branch_mask_columns
    + mri_path_columns
)

print("\nExample prepared rows:")
display(
    fold_table[
        preview_columns
    ].head(5)
)


### 1.5.3. Defining the multimodal dataset-output contract

The model will receive each modality as a separate input branch rather than as one combined feature vector.

Continuous and categorical variables are kept separate because they require different encoder operations. Continuous variables will enter small numerical encoders, while categorical variables will later be represented through trainable embeddings.

Each sample will also contain:

- the participant identifier;
- the binary prognosis target;
- branch-level availability masks;
- feature-level observation masks;
- the prepared MRI path.

The dataset will not load MRI arrays yet. At this stage, I define the column organisation and the exact sample structure that the PyTorch dataset will later return.

For participants without MRI, the MRI path remains unavailable and the MRI branch mask remains zero. The participant is retained in the dataset.

In [ ]:
# ============================================================
# 3. Defining the multimodal dataset-output contract
# ============================================================

# ------------------------------------------------------------
# Branch-specific predictor columns
# ------------------------------------------------------------

# I organise the prepared continuous and categorical columns into
# the six modality branches used by the architecture.

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


# ------------------------------------------------------------
# Dataset sample structure
# ------------------------------------------------------------

# One participant will later be returned by the PyTorch dataset
# using the following nested structure.
#
# Continuous features will become float32 tensors.
# Categorical indices will become int64 tensors for embeddings.
# Masks will become float32 tensors containing 0 or 1.
# The target will become an int64 class index.

dataset_output_contract = {
    "rid": "Participant RID as an integer",
    "ptid": "Participant PTID as a string",
    "target": "Binary class index: 0 for sMCI and 1 for pMCI",

    "modalities": {
        "demographics": {
            "continuous": "Shape (2,), float32",
            "categorical": "Shape (2,), int64",
            "feature_mask": "Shape (4,), float32",
            "branch_mask": "Scalar, float32",
        },

        "cognitive_functional": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "csf": {
            "continuous": "Shape (5,), float32",
            "categorical": None,
            "feature_mask": "Shape (5,), float32",
            "branch_mask": "Scalar, float32",
        },

        "plasma": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "apoe": {
            "continuous": None,
            "categorical": "Shape (1,), int64",
            "feature_mask": "Shape (1,), float32",
            "branch_mask": "Scalar, float32",
        },

        "mri": {
            "path": "Prepared NumPy path or None",
            "image": "Later: shape (1, 177, 213, 183), float32",
            "branch_mask": "Scalar, float32",
        },
    },

    "branch_masks": (
        "Shape (6,), float32, ordered as "
        "demographics, cognitive-functional, CSF, "
        "plasma, APOE, MRI"
    ),

    "feature_masks": (
        "Shape (28,), float32, using the authoritative "
        "feature-mask order"
    ),
}


# ------------------------------------------------------------
# Display the agreed contract
# ------------------------------------------------------------

print("=" * 72)
print("MULTIMODAL DATASET CONTRACT")
print("=" * 72)

print("\nBranch-specific input dimensions:")

for branch_name, branch_definition in dataset_column_contract.items():

    continuous_count = len(
        branch_definition.get("continuous", [])
    )

    categorical_count = len(
        branch_definition.get("categorical", [])
    )

    feature_mask_count = len(
        branch_definition.get("feature_masks", [])
    )

    has_mri_path = "path" in branch_definition

    print(
        f"- {branch_name}: "
        f"{continuous_count} continuous, "
        f"{categorical_count} categorical, "
        f"{feature_mask_count} feature masks"
        + (", 1 MRI path" if has_mri_path else "")
    )


print("\nPlanned sample output:")
print(
    json.dumps(
        dataset_output_contract,
        indent=2,
    )
)

print(
    "\nThis contract will be used in the next step to "
    "implement the PyTorch dataset."
)

### 1.5.4. Implementing the multimodal PyTorch dataset

implement a PyTorch dataset that converts each prepared participant row into the agreed multimodal structure.

The dataset preserves the six modality branches and returns continuous variables, categorical indices, observation masks, participant identifiers, and the prognosis target separately.

MRI volumes are loaded only when a sample is requested. The existing preprocessed NumPy array is used directly, and a channel dimension is added to produce the shape required by a three-dimensional neural network.

When MRI is unavailable, the participant remains in the dataset. The dataset returns a zero placeholder volume together with an MRI branch mask of zero, allowing the model to distinguish an unavailable scan from an observed image.

In [ ]:
# ============================================================
# 4. Implementing the multimodal PyTorch dataset
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset


# ------------------------------------------------------------
# Expected prepared MRI shape
# ------------------------------------------------------------

# I preserve the spatial dimensions produced by the completed
# MRI preprocessing pipeline.
MRI_SPATIAL_SHAPE = (177, 213, 183)

# I add one channel dimension when returning an MRI tensor.
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)


# ------------------------------------------------------------
# Multimodal PyTorch dataset
# ------------------------------------------------------------

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # I reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # I retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # I preserve the authoritative mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri


    def __len__(self):
        return len(self.dataframe)


    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # I load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # I ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)


    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # --------------------------------------------------------
        # Demographics
        # --------------------------------------------------------

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Cognitive and functional measures
        # --------------------------------------------------------

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # CSF
        # --------------------------------------------------------

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Plasma
        # --------------------------------------------------------

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # APOE
        # --------------------------------------------------------

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # MRI
        # --------------------------------------------------------

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Complete sample
        # --------------------------------------------------------

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample


# ------------------------------------------------------------
# Create role-specific datasets
# ------------------------------------------------------------

# I retain the prepared role assignments exactly as stored in
# the selected outer-fold table.
train_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "train"
    ]
    .reset_index(drop=True)
)

validation_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "validation"
    ]
    .reset_index(drop=True)
)

test_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "test"
    ]
    .reset_index(drop=True)
)


# I initially disable MRI disk loading so that I can inspect the
# dataset structure quickly before constructing the DataLoaders.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)


# ------------------------------------------------------------
# Concise dataset summary
# ------------------------------------------------------------

print("=" * 72)
print("PYTORCH DATASETS")
print("=" * 72)

print(f"\nTraining participants: {len(train_dataset)}")
print(f"Validation participants: {len(validation_dataset)}")
print(f"Test participants: {len(test_dataset)}")

print(
    "\nThe datasets preserve the prepared role assignments "
    "and return separate modality inputs."
)

print(
    "MRI loading is temporarily disabled for structural "
    "inspection and will be enabled for the DataLoaders."
)

### 1.5.5. Inspecting one multimodal sample and one prepared MRI volume

Before constructing the DataLoaders, The notebook inspects the structure returned for one participant.

use the dataset with MRI loading disabled to confirm the scalar tensors, categorical indices, targets, and masks. I then create a temporary MRI-enabled dataset and load one participant whose MRI branch is available.

This checks the dataset interface required by the model while avoiding unnecessary loading of multiple MRI volumes at this stage.

In [ ]:
# ============================================================
# 5. Inspecting one multimodal sample and one MRI volume
# ============================================================

# ------------------------------------------------------------
# Inspect one scalar-only training sample
# ------------------------------------------------------------

# I retrieve one participant while MRI disk loading remains
# disabled. The returned MRI tensor is therefore only the
# temporary placeholder defined in the current dataset class.
sample = train_dataset[0]

print("=" * 72)
print("EXAMPLE MULTIMODAL SAMPLE")
print("=" * 72)

print(f"\nRID: {sample['rid']}")
print(f"PTID: {sample['ptid']}")
print(f"Target: {sample['target'].item()}")

print("\nModality tensor structure:")

for modality_name, modality_data in sample["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"{value}"
            )

print(
    f"\nComplete branch-mask shape: "
    f"{tuple(sample['branch_masks'].shape)}"
)

print(
    f"Complete feature-mask shape: "
    f"{tuple(sample['feature_masks'].shape)}"
)


# ------------------------------------------------------------
# Find one participant with an available MRI
# ------------------------------------------------------------

# I select the first training participant whose prepared MRI
# branch mask is one.
example_mri_index = train_table.index[
    train_table["BRANCH_MASK__MRI"] == 1
][0]


# ------------------------------------------------------------
# Create a temporary MRI-enabled dataset
# ------------------------------------------------------------

# I enable MRI loading only for this temporary inspection
# dataset. The main role-specific datasets remain unchanged.
mri_inspection_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

mri_sample = mri_inspection_dataset[
    example_mri_index
]

mri_tensor = mri_sample[
    "modalities"
]["mri"]["image"]

mri_branch_mask = mri_sample[
    "modalities"
]["mri"]["branch_mask"]


# ------------------------------------------------------------
# Display the prepared MRI tensor information
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXAMPLE PREPARED MRI")
print("=" * 72)

print(f"\nRID: {mri_sample['rid']}")
print(f"PTID: {mri_sample['ptid']}")

print(
    f"MRI branch mask: "
    f"{mri_branch_mask.item():.0f}"
)

print(
    f"MRI tensor shape: "
    f"{tuple(mri_tensor.shape)}"
)

print(
    f"MRI tensor dtype: "
    f"{mri_tensor.dtype}"
)

print(
    f"MRI intensity minimum: "
    f"{mri_tensor.min().item():.6f}"
)

print(
    f"MRI intensity maximum: "
    f"{mri_tensor.max().item():.6f}"
)

print(
    f"MRI intensity mean: "
    f"{mri_tensor.mean().item():.6f}"
)

print(
    "\nThe sample structure and full prepared MRI volume "
    "are ready for DataLoader construction."
)

### 1.5.6. Constructing the multimodal DataLoaders

create separate DataLoaders for the fixed training, validation, and test subsets.

MRI loading is enabled, so an available scan is read lazily from its prepared NumPy path when its participant enters a batch. Participants without MRI receive a zero placeholder volume and retain an MRI branch mask of zero.

I begin with a small batch size because each sample contains a full three-dimensional MRI volume. The final training batch size will be selected later according to the memory requirements of the complete model.

The training DataLoader shuffles participants. Validation and test DataLoaders preserve a deterministic order.

In [ ]:
# ============================================================
# 6. Constructing the multimodal DataLoaders
# ============================================================

from torch.utils.data import DataLoader


# ------------------------------------------------------------
# Initial DataLoader settings
# ------------------------------------------------------------

# I begin with a small batch because each participant may contain
# a full MRI volume with shape (1, 177, 213, 183).
INITIAL_BATCH_SIZE = 2

# I initially use the main process for data loading. This is the
# most reliable starting configuration when reading NumPy files
# from mounted Google Drive.
NUM_WORKERS = 0

# Pinned memory can speed transfers to a CUDA device.
PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# Recreate the datasets with MRI loading enabled
# ------------------------------------------------------------

# Available MRI volumes will now be read lazily when requested.
# Missing MRI branches will retain their zero placeholders and
# branch masks of zero.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)


# ------------------------------------------------------------
# Create role-specific DataLoaders
# ------------------------------------------------------------

# I shuffle only the training subset.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# Validation order does not need to be shuffled.
validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# The test subset also retains a deterministic order.
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)


# ------------------------------------------------------------
# Retrieve one complete training batch
# ------------------------------------------------------------

example_batch = next(iter(train_loader))


# ------------------------------------------------------------
# Display the batched tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("EXAMPLE MULTIMODAL BATCH")
print("=" * 72)

print(f"\nBatch size: {example_batch['target'].shape[0]}")
print(f"RID values: {example_batch['rid']}")
print(f"PTID values: {example_batch['ptid']}")
print(f"Targets: {example_batch['target']}")

print("\nModality tensors:")

for modality_name, modality_data in example_batch["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"type={type(value).__name__}"
            )


print("\nCombined masks:")

print(
    "  branch_masks: "
    f"shape={tuple(example_batch['branch_masks'].shape)}, "
    f"dtype={example_batch['branch_masks'].dtype}"
)

print(
    "  feature_masks: "
    f"shape={tuple(example_batch['feature_masks'].shape)}, "
    f"dtype={example_batch['feature_masks'].dtype}"
)


# ------------------------------------------------------------
# Show MRI availability in this batch
# ------------------------------------------------------------

batch_mri = example_batch[
    "modalities"
]["mri"]["image"]

batch_mri_masks = example_batch[
    "modalities"
]["mri"]["branch_mask"]

print("\nMRI batch:")

print(
    f"  image shape: {tuple(batch_mri.shape)}"
)

print(
    f"  branch masks: {batch_mri_masks}"
)

print(
    f"  approximate raw MRI batch size: "
    f"{batch_mri.numel() * batch_mri.element_size() / (1024 ** 2):.2f} MB"
)


# ------------------------------------------------------------
# DataLoader summary
# ------------------------------------------------------------

print("\nDataLoader batches:")

print(
    f"  training: {len(train_loader)} batches"
)

print(
    f"  validation: {len(validation_loader)} batches"
)

print(
    f"  test: {len(test_loader)} batches"
)

print(
    "\nThe complete multimodal batch is ready for "
    "modality-specific encoder construction."
)

### 1.5.7. Building the modality-specific encoders

Each modality has a different input structure, so I encode the six branches separately before multimodal interaction.

For the scalar branches, the prepared feature values are combined with their feature-observation masks. This allows the encoders to distinguish an observed standardised value close to zero from a missing-value placeholder.

Categorical variables use trainable embeddings. Encoded index zero remains reserved for missing or unseen values and is handled through embedding padding behaviour.

### 1.5.8. Common latent dimension

Every branch is projected into the same latent dimension:

$$
d_{\mathrm{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore produces:

$$
\mathbf{H}^{(m)}
\in
\mathbb{R}^{B \times 64}.
$$

The shared dimension is required so that all modality representations can later enter the same cascaded cross-modal transformer architecture.

The original interaction pathway implementation used a dimension of \(512\). In this thesis, I begin with a more compact dimension of \(64\) because the main prognosis training folds contain approximately \(369\) participants and the complete model will additionally include independent evidential heads, modality-specific evidence fusion, auxiliary outputs, and the cascaded interaction pathway interaction path.

The embedding dimension remains a model-capacity hyperparameter and may later be compared with larger values using only training and validation data.

### 1.5.9. MRI encoder

The MRI branch follows the general image-encoding structure used by interaction pathway:

1. initial three-dimensional convolutions;
2. residual three-dimensional downsampling blocks;
3. conversion of the final feature map into patch tokens;
4. addition of learned positional embeddings;
5. transformer encoding of patch-wise relationships;
6. global averaging across patch tokens;
7. projection into the shared modality dimension.

The original paper used a 512-dimensional MRI output. Here, the final MRI representation is projected to the common 64-dimensional latent space used by the other branches.

The prepared MRI volumes are larger than those used in the original interaction pathway experiments. I therefore apply stride-two downsampling in the initial convolutional stem before the four residual blocks. This preserves the CNN-transformer design while keeping the number of transformer patch tokens computationally manageable.

No new MRI preprocessing is performed. The encoder receives the complete prepared volume with shape:

$$
(1, 177, 213, 183).
$$

In [ ]:
# ============================================================
# 7. Building the modality-specific encoders
# ============================================================

import math

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Shared representation dimension
# ------------------------------------------------------------

# I project every modality into one common latent space so that
# all branches can later enter the same cross-modal transformers.
MODALITY_EMBEDDING_DIM = 64


# ------------------------------------------------------------
# Reusable scalar encoder
# ------------------------------------------------------------

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        values,
        feature_mask,
    ):
        # I supply both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Demographics encoder
# ------------------------------------------------------------

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# APOE encoder
# ------------------------------------------------------------

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Residual 3D downsampling block
# ------------------------------------------------------------

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original interaction pathway image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()


    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )


# ------------------------------------------------------------
# interaction pathway-style CNN-transformer MRI encoder
# ------------------------------------------------------------

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # --------------------------------------------------------
        # Initial convolutional stem
        # --------------------------------------------------------

        # I use two initial 3D convolutions, following the broad
        # structure shown in the interaction pathway image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original interaction pathway inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )


        # --------------------------------------------------------
        # Four residual downsampling blocks
        # --------------------------------------------------------

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )


        # --------------------------------------------------------
        # Determine the resulting patch grid
        # --------------------------------------------------------

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )


        # --------------------------------------------------------
        # Learned positional embeddings
        # --------------------------------------------------------

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )


        # --------------------------------------------------------
        # Patch-wise transformer encoder
        # --------------------------------------------------------

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )


        # --------------------------------------------------------
        # Projection to the shared modality dimension
        # --------------------------------------------------------

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )


    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # I flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # I add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )


# ------------------------------------------------------------
# Complete set of six modality encoders
# ------------------------------------------------------------

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )


    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations


# ------------------------------------------------------------
# Instantiate the revised encoders
# ------------------------------------------------------------

modality_encoders = ADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Count trainable parameters
# ------------------------------------------------------------

def count_trainable_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print("=" * 72)
print("MODALITY-SPECIFIC ENCODERS")
print("=" * 72)

print(
    f"\nShared modality embedding dimension: "
    f"{MODALITY_EMBEDDING_DIM}"
)

print(
    "\nMRI transformer patch grid: "
    f"{modality_encoders.mri.patch_grid_shape}"
)

print(
    "MRI transformer patch count: "
    f"{modality_encoders.mri.number_of_patches}"
)

print("\nTrainable parameters by encoder:")

for encoder_name in [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]:
    encoder = getattr(
        modality_encoders,
        encoder_name,
    )

    print(
        f"- {encoder_name}: "
        f"{count_trainable_parameters(encoder):,}"
    )

print(
    "\nTotal trainable encoder parameters: "
    f"{count_trainable_parameters(modality_encoders):,}"
)

print(
    "\nThe revised MRI branch now uses a 3D CNN, patch tokens, "
    "positional embeddings, and a transformer encoder."
)

print(
    "No multimodal fusion or classification head has been "
    "added yet."
)

### 1.5.10. Applying branch-availability masks to the encoded modalities

Each modality has a different original input structure, but every modality-specific encoder projects its input into the same latent dimensionality.

In this implementation, each branch produces a representation of size:

$$
d_{\text{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore returns:

$$
\mathbf{H}^{(m)} \in \mathbb{R}^{B \times 64},
$$

where \(m\) denotes one of the six modalities:

- demographics;
- cognitive-functional measures;
- CSF;
- plasma;
- APOE;
- MRI.

The shared dimensionality does not mean that the modalities contain the same information or use equally complex encoders. Each branch has its own input-specific encoder, but the final representations must have a common size so that they can later participate in cross-modal attention and evidential fusion.

After stacking the six modality representations, the model obtains:

$$
\mathbf{H}
\in
\mathbb{R}^{B \times 6 \times 64}.
$$

This follows the general design principle used by interaction pathway, in which heterogeneous modality inputs are first projected into a common transformer embedding space before cross-modal interaction. The original interaction pathway implementation used a larger embedding dimension of \(512\), but \(512\) is an architectural hyperparameter rather than a methodological requirement.

A compact starting dimension of \(64\) is used here because the main MCI prognosis training folds contain only approximately \(369\) participants, while the complete planned model will also contain:

- six modality-specific encoders;
- cascaded cross-modal attention;
- independent evidential heads;
- evidence pathway-style evidential fusion;
- a joint evidential prediction path.

The number of parameters in transformer projections grows approximately with the square of the embedding dimension. For example:

$$
64^2 = 4{,}096,
$$

whereas:

$$
512^2 = 262{,}144.
$$

Thus, increasing the embedding dimension from \(64\) to \(512\) can make several attention and feed-forward parameter blocks approximately \(64\) times larger. A \(512\)-dimensional model would therefore introduce substantially greater overfitting and memory risk for the available prognosis cohort.

The value \(64\) is treated as a compact initial configuration rather than as a permanently fixed optimum. The latent dimensionality can later be compared with alternatives such as \(128\) or \(256\), using only the training and validation subsets.

### 1.5.11. Branch-availability masking

The prepared zero placeholders make missing inputs computationally compatible with neural-network layers, but they do not guarantee that an unavailable modality will produce a zero encoder output.

Linear layers, embeddings, normalisation parameters, and learned biases can generate a non-zero representation even when all supplied inputs are zero. I therefore apply the prepared branch mask after each modality encoder.

For participant \(i\) and modality \(m\), the masked representation is:

$$
\widetilde{\mathbf{h}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{h}_{i}^{(m)},
$$

where:

- \(\mathbf{h}_{i}^{(m)} \in \mathbb{R}^{64}\) is the raw modality representation;
- \(a_{i}^{(m)} \in \{0,1\}\) is the prepared branch-availability mask;
- \(\widetilde{\mathbf{h}}_{i}^{(m)} \in \mathbb{R}^{64}\) is the masked representation passed to later model components.

Therefore:

$$
a_{i}^{(m)} = 1
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{h}_{i}^{(m)},
$$

and:

$$
a_{i}^{(m)} = 0
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{0}.
$$

An unavailable modality consequently contributes an exact zero representation rather than a learned bias-derived vector. The original branch masks are also retained separately so that the later cross-modal attention and evidential-fusion components can explicitly identify which modalities are available for each participant.

In [ ]:
# ============================================================
# 8. Applying branch masks to the encoded modalities
# ============================================================

# ------------------------------------------------------------
# Fixed modality order
# ------------------------------------------------------------

# I use one explicit modality order throughout the architecture.
# This order matches the prepared branch-mask columns.
MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]


# ------------------------------------------------------------
# Mask-aware encoder wrapper
# ------------------------------------------------------------

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )


    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # I add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask


    def forward(self, modalities):
        # I first obtain the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # I then suppress every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # I return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }


# ------------------------------------------------------------
# Instantiate the mask-aware encoder collection
# ------------------------------------------------------------

masked_modality_encoders = MaskedADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Apply the encoders to one complete batch
# ------------------------------------------------------------

# I keep this inspection on the CPU. The training device will be
# configured later when the full model and optimisation loop exist.
masked_modality_encoders.eval()

with torch.no_grad():
    encoded_batch = masked_modality_encoders(
        example_batch["modalities"]
    )


# ------------------------------------------------------------
# Stack modality representations
# ------------------------------------------------------------

# I stack the representations in the fixed modality order.
#
# Resulting shape:
# (batch_size, number_of_modalities, embedding_dimension)
stacked_masked_representations = torch.stack(
    [
        encoded_batch["masked"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_raw_representations = torch.stack(
    [
        encoded_batch["raw"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the encoded structure
# ------------------------------------------------------------

print("=" * 72)
print("MASKED MODALITY REPRESENTATIONS")
print("=" * 72)

print(
    f"\nStacked raw representation shape: "
    f"{tuple(stacked_raw_representations.shape)}"
)

print(
    f"Stacked masked representation shape: "
    f"{tuple(stacked_masked_representations.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, embedding)"
)

print("\nBranch masks in this batch:")

branch_mask_table = pd.DataFrame(
    example_batch["branch_masks"].numpy(),
    columns=MODALITY_ORDER,
)

display(branch_mask_table)


# ------------------------------------------------------------
# Representation norms before and after masking
# ------------------------------------------------------------

# A representation norm summarises the magnitude of each branch
# vector. Missing branches may have non-zero raw norms because of
# learned biases, but their masked norms must be exactly zero.
raw_norms = torch.linalg.vector_norm(
    stacked_raw_representations,
    dim=-1,
)

masked_norms = torch.linalg.vector_norm(
    stacked_masked_representations,
    dim=-1,
)

norm_summary = []

for participant_index in range(
    stacked_masked_representations.shape[0]
):
    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):
        norm_summary.append(
            {
                "BATCH_ROW": participant_index,
                "RID": int(
                    example_batch["rid"][
                        participant_index
                    ].item()
                ),
                "MODALITY": modality_name,
                "BRANCH_MASK": float(
                    example_batch["branch_masks"][
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "RAW_REPRESENTATION_NORM": float(
                    raw_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "MASKED_REPRESENTATION_NORM": float(
                    masked_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
            }
        )

norm_summary = pd.DataFrame(norm_summary)

print("\nRepresentation norms before and after masking:")

display(
    norm_summary.round(6)
)


# ------------------------------------------------------------
# Confirm the representation dimensions
# ------------------------------------------------------------

print("\nEncoded modality shapes:")

for modality_name in MODALITY_ORDER:
    print(
        f"- {modality_name}: "
        f"{tuple(encoded_batch['masked'][modality_name].shape)}"
    )

print(
    "\nOnly the masked representations will enter the "
    "multimodal interaction and evidential-fusion paths."
)

### 1.5.12. Building the availability-gated interaction pathway cascade

The cascade order remains unchanged:

$$
\text{demographics}
\rightarrow
\text{APOE}
\rightarrow
\text{cognitive/functional}
\rightarrow
\text{CSF}
\rightarrow
\text{plasma}
\rightarrow
\text{MRI}.
$$

Each cross-modal interaction block still computes a candidate update using query self-attention followed by cross-attention to the encoded modality token. The branch mask then determines whether that candidate becomes the next cumulative query:

$$
\mathbf{q}_m
=
a_m\mathbf{q}^{\mathrm{candidate}}_m
+
(1-a_m)\mathbf{q}_{m-1}.
$$

For an available modality, $a_m=1$ and the candidate update is used. For an unavailable modality, $a_m=0$ and the stage becomes an exact identity update.

The branch-mask tensor follows `MODALITY_ORDER`, while the cross-modal interaction blocks follow `THREE_MT_CASCADE_ORDER`. The class therefore uses the explicit modality-to-mask index mapping already defined by the fold-0 architecture.

In [ ]:
# ============================================================
# 9. Building the availability-gated interaction pathway cascade
# ============================================================

# ------------------------------------------------------------
# Fixed cascade order
# ------------------------------------------------------------

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]


# ------------------------------------------------------------
# One Cascaded Modality Transformer
# ------------------------------------------------------------

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }


# ------------------------------------------------------------
# Complete six-stage availability-gated cascade
# ------------------------------------------------------------

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )


    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }


# ------------------------------------------------------------
# Instantiate and inspect the gated cascade
# ------------------------------------------------------------

three_mt_cascade = ThreeMTCascade(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    number_of_heads=4,
    dropout=0.10,
)

three_mt_cascade.eval()

with torch.no_grad():
    three_mt_output = three_mt_cascade(
        masked_representations=encoded_batch[
            "masked"
        ],
        branch_masks=example_batch[
            "branch_masks"
        ],
    )


print("=" * 72)
print("AVAILABILITY-GATED 3MT CASCADE")
print("=" * 72)

print(
    f"\nCascade order:\n"
    f"{THREE_MT_CASCADE_ORDER}"
)

print(
    "\nFinal joint representation shape: "
    f"{tuple(three_mt_output['joint_representation'].shape)}"
)


# ------------------------------------------------------------
# Show the actual query update at every stage
# ------------------------------------------------------------

query_change_rows = []

previous_query = (
    three_mt_cascade
    .learned_latent_query
    .expand(
        example_batch["target"].shape[0],
        -1,
        -1,
    )
)

for modality_name in THREE_MT_CASCADE_ORDER:
    current_query = three_mt_output[
        "stage_queries"
    ][modality_name]

    query_change_norm = torch.linalg.vector_norm(
        current_query - previous_query,
        dim=-1,
    ).squeeze(1)

    modality_index = MODALITY_ORDER.index(
        modality_name
    )

    branch_mask = example_batch[
        "branch_masks"
    ][
        :,
        modality_index,
    ]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):
        query_change_rows.append(
            {
                "BATCH_ROW": batch_row,
                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),
                "CASCADE_STAGE": modality_name,
                "BRANCH_MASK": float(
                    branch_mask[
                        batch_row
                    ].item()
                ),
                "QUERY_CHANGE_NORM": float(
                    query_change_norm[
                        batch_row
                    ].item()
                ),
            }
        )

    previous_query = current_query

query_change_summary = pd.DataFrame(
    query_change_rows
)

print(
    "\nQuery changes after availability gating:"
)

display(
    query_change_summary.round(6)
)

print(
    "\nTrainable cascade parameters: "
    f"{count_trainable_parameters(three_mt_cascade):,}"
)


### 1.5.13. Producing independent modality-specific evidential opinions

The interaction pathway cascade produces a cumulative interaction-aware representation, but its intermediate states are not independent modality opinions because every stage contains information inherited from earlier stages.

Trusted Multi-View Classification requires each modality to produce its own class evidence before cross-modal interaction.

For participant \(i\), modality \(m\), and class \(k\), the modality-specific evidential head produces non-negative evidence:

$$
e_{ik}^{(m)}
=
\operatorname{Softplus}
\left(
\mathbf{W}_{m}
\mathbf{z}_{i}^{(m)}
+
\mathbf{b}_{m}
\right),
$$

where:

- \(\mathbf{z}_{i}^{(m)} \in \mathbb{R}^{64}\) is the independently encoded modality representation;
- \(e_{ik}^{(m)} \geq 0\) is the evidence assigned to class \(k\);
- each modality has its own evidential head.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{(m)}
=
e_{ik}^{(m)} + 1.
$$

For the binary prognosis task:

$$
K = 2,
$$

with class order:

$$
[\mathrm{sMCI},\mathrm{pMCI}].
$$

The expected class probabilities are:

$$
p_{ik}^{(m)}
=
\frac{
\alpha_{ik}^{(m)}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{(m)}
}.
$$

The Dirichlet strength is:

$$
S_{i}^{(m)}
=
\sum_{k=1}^{K}
\alpha_{ik}^{(m)}.
$$

The standard evidential uncertainty mass is:

$$
u_{i}^{(m)}
=
\frac{K}{
S_{i}^{(m)}
}.
$$

Low total evidence produces high uncertainty, while stronger evidence produces lower uncertainty.

### 1.5.14. Treatment of unavailable modalities

Only available modalities should contribute an opinion to modality-specific evidence fusion.

For an unavailable branch, I do not interpret the evidential head output as a genuine prediction. Instead, its effective evidence is set to zero:

$$
\widetilde{\mathbf{e}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{e}_{i}^{(m)},
$$

where \(a_{i}^{(m)}\) is the prepared branch mask.

Therefore, an unavailable modality receives:

$$
\widetilde{\boldsymbol{\alpha}}_{i}^{(m)}
=
\mathbf{1},
$$

which is the uniform Dirichlet opinion with:

$$
u_{i}^{(m)} = 1.
$$

The original branch mask is retained so that the next step can exclude unavailable opinions explicitly during evidence pathway/Dempster--Shafer fusion.

At this stage, I construct and inspect the six independent evidential opinions. I do not yet fuse them or combine them with the interaction pathway joint representation.

In [ ]:
# ============================================================
# 10. Producing independent modality-specific evidential opinions
# ============================================================

# ------------------------------------------------------------
# Binary prognosis class definition
# ------------------------------------------------------------

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]


# ------------------------------------------------------------
# One modality-specific evidential head
# ------------------------------------------------------------

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(
        self,
        representation,
        branch_mask,
    ):
        # I use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }


# ------------------------------------------------------------
# Independent evidential heads for all six modalities
# ------------------------------------------------------------

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )


    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions


# ------------------------------------------------------------
# Instantiate the independent evidential path
# ------------------------------------------------------------

independent_evidential_heads = (
    IndependentModalityEvidentialHeads(
        modality_order=MODALITY_ORDER,
        input_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
    )
)


# ------------------------------------------------------------
# Produce one opinion per modality
# ------------------------------------------------------------

independent_evidential_heads.eval()

with torch.no_grad():

    modality_opinions = (
        independent_evidential_heads(
            # I use the independently encoded branch outputs before
            # they enter the interaction pathway cascade.
            modality_representations=encoded_batch[
                "masked"
            ],

            modalities=example_batch[
                "modalities"
            ],
        )
    )


# ------------------------------------------------------------
# Stack the opinion tensors
# ------------------------------------------------------------

stacked_modality_evidence = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["evidence"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_alpha = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["alpha"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_probabilities = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["probabilities"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_uncertainty = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["uncertainty"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the evidential tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("INDEPENDENT MODALITY EVIDENTIAL OPINIONS")
print("=" * 72)

print(
    f"\nClass order: "
    f"{PROGNOSIS_CLASS_ORDER}"
)

print(
    "\nStacked evidence shape: "
    f"{tuple(stacked_modality_evidence.shape)}"
)

print(
    "Stacked Dirichlet-alpha shape: "
    f"{tuple(stacked_modality_alpha.shape)}"
)

print(
    "Stacked probability shape: "
    f"{tuple(stacked_modality_probabilities.shape)}"
)

print(
    "Stacked uncertainty shape: "
    f"{tuple(stacked_modality_uncertainty.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, class)"
)


# ------------------------------------------------------------
# Create a readable modality-opinion summary
# ------------------------------------------------------------

opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):

        branch_mask = float(
            example_batch["branch_masks"][
                batch_row,
                modality_index,
            ].item()
        )

        alpha_values = stacked_modality_alpha[
            batch_row,
            modality_index,
        ]

        probability_values = (
            stacked_modality_probabilities[
                batch_row,
                modality_index,
            ]
        )

        uncertainty_value = (
            stacked_modality_uncertainty[
                batch_row,
                modality_index,
                0,
            ]
        )

        opinion_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "MODALITY": modality_name,

                "BRANCH_MASK": branch_mask,

                "ALPHA_sMCI": float(
                    alpha_values[0].item()
                ),

                "ALPHA_pMCI": float(
                    alpha_values[1].item()
                ),

                "P_sMCI": float(
                    probability_values[0].item()
                ),

                "P_pMCI": float(
                    probability_values[1].item()
                ),

                "UNCERTAINTY": float(
                    uncertainty_value.item()
                ),
            }
        )


modality_opinion_summary = pd.DataFrame(
    opinion_rows
)

print("\nIndependent modality opinions:")

display(
    modality_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable independent evidential-head parameters: "
    f"{count_trainable_parameters(independent_evidential_heads):,}"
)

print(
    "\nUnavailable modalities should have alpha=[1, 1], "
    "probabilities=[0.5, 0.5], and uncertainty=1."
)

print(
    "\nThe available modality opinions are ready for "
    "TMC/Dempster-Shafer fusion."
)

### 1.5.15. Fusing the independent modality opinions with evidence pathway

combine the six independent modality opinions using the reduced Dempster--Shafer rule adopted by Trusted Multi-View Classification.

For modality \(m\), the Dirichlet parameters are:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{e}^{(m)} + \mathbf{1}.
$$

The Dirichlet strength is:

$$
S^{(m)}
=
\sum_{k=1}^{K}
\alpha_{k}^{(m)}.
$$

The class-specific belief masses are:

$$
b_{k}^{(m)}
=
\frac{
\alpha_{k}^{(m)} - 1
}{
S^{(m)}
}
=
\frac{
e_{k}^{(m)}
}{
S^{(m)}
}.
$$

The uncertainty mass is:

$$
u^{(m)}
=
\frac{K}{
S^{(m)}
}.
$$

These masses satisfy:

$$
\sum_{k=1}^{K}
b_{k}^{(m)}
+
u^{(m)}
=
1.
$$

### 1.5.16. Combining two opinions

Consider two opinions, \(A\) and \(B\). Their conflict mass is:

$$
C
=
\sum_{i \neq j}
b_{i}^{A}
b_{j}^{B}.
$$

For each class \(k\), the combined belief mass is:

$$
b_{k}^{A \oplus B}
=
\frac{
b_{k}^{A}b_{k}^{B}
+
b_{k}^{A}u^{B}
+
b_{k}^{B}u^{A}
}{
1-C
}.
$$

The combined uncertainty mass is:

$$
u^{A \oplus B}
=
\frac{
u^{A}u^{B}
}{
1-C
}.
$$

The fused Dirichlet strength is recovered from the fused uncertainty:

$$
S^{A \oplus B}
=
\frac{K}{
u^{A \oplus B}
}.
$$

The fused evidence and Dirichlet parameters are then:

$$
e_{k}^{A \oplus B}
=
b_{k}^{A \oplus B}
S^{A \oplus B},
$$

and:

$$
\alpha_{k}^{A \oplus B}
=
e_{k}^{A \oplus B} + 1.
$$

The rule is applied repeatedly until all six modality opinions have been considered.

### 1.5.17. Missing modalities

An unavailable modality was assigned the vacuous Dirichlet opinion:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{1}.
$$

For this opinion:

$$
\mathbf{b}^{(m)}
=
\mathbf{0},
\qquad
u^{(m)} = 1.
$$

A vacuous opinion acts as an identity element in this combination rule. It adds no class evidence, creates no conflict, and leaves the available opinion unchanged.

Therefore, the same fusion procedure can process all participants without complete-case filtering or synthetic modality imputation.

At this stage, I construct only the evidence pathway-fused independent opinion. The interaction-aware interaction pathway query will receive its own evidential head in a later step.

In [ ]:
# ============================================================
# 11. Fusing the independent modality opinions with evidence pathway
# ============================================================

# ------------------------------------------------------------
# Reduced Dempster-Shafer combination rule
# ------------------------------------------------------------

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )


    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }


    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]


        # --------------------------------------------------------
        # Conflict mass
        # --------------------------------------------------------

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)


        # --------------------------------------------------------
        # Fused belief and uncertainty masses
        # --------------------------------------------------------

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation


        # --------------------------------------------------------
        # Recover the fused Dirichlet opinion
        # --------------------------------------------------------

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }


    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # I retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }


        # --------------------------------------------------------
        # Final fused opinion
        # --------------------------------------------------------

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }


# ------------------------------------------------------------
# Instantiate the modality-specific evidence fusion module
# ------------------------------------------------------------

tmc_fusion = TMCFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
)


# ------------------------------------------------------------
# Fuse the current independent modality opinions
# ------------------------------------------------------------

with torch.no_grad():

    tmc_output = tmc_fusion(
        modality_opinions=modality_opinions
    )


# ------------------------------------------------------------
# Display final fused tensor shapes
# ------------------------------------------------------------

print("=" * 72)
print("TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    f"\nFusion order: "
    f"{MODALITY_ORDER}"
)

print(
    "\nFused evidence shape: "
    f"{tuple(tmc_output['evidence'].shape)}"
)

print(
    "Fused alpha shape: "
    f"{tuple(tmc_output['alpha'].shape)}"
)

print(
    "Fused probability shape: "
    f"{tuple(tmc_output['probabilities'].shape)}"
)

print(
    "Fused uncertainty shape: "
    f"{tuple(tmc_output['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Final participant-level fused opinions
# ------------------------------------------------------------

fused_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    fused_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "AVAILABLE_MODALITIES": int(
                example_batch["branch_masks"][
                    batch_row
                ].sum()
                .item()
            ),

            "ALPHA_sMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                tmc_output["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


fused_opinion_summary = pd.DataFrame(
    fused_opinion_rows
)

print("\nFinal TMC-fused opinions:")

display(
    fused_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Inspect the sequential fusion history
# ------------------------------------------------------------

fusion_history_rows = []

for modality_name in MODALITY_ORDER:

    stage_output = tmc_output[
        "fusion_history"
    ][modality_name]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):

        modality_index = MODALITY_ORDER.index(
            modality_name
        )

        fusion_history_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "FUSED_THROUGH": modality_name,

                "CURRENT_MODALITY_MASK": float(
                    example_batch["branch_masks"][
                        batch_row,
                        modality_index,
                    ].item()
                ),

                "STAGE_CONFLICT": float(
                    stage_output["conflict"][
                        batch_row,
                        0,
                    ].item()
                ),

                "STAGE_UNCERTAINTY": float(
                    stage_output["uncertainty"][
                        batch_row,
                        0,
                    ].item()
                ),
            }
        )


fusion_history_summary = pd.DataFrame(
    fusion_history_rows
)

print("\nSequential fusion history:")

display(
    fusion_history_summary.round(6)
)


print(
    "\nUnavailable modalities should introduce zero conflict "
    "and leave the accumulated opinion unchanged."
)

print(
    "The TMC-fused independent opinion is ready for later "
    "combination with the interaction-aware 3MT opinion."
)

### 1.5.18. Producing the interaction-aware interaction pathway evidential opinion

The modality-specific evidence pathway combines independent modality opinions produced before cross-modal interaction. construct a separate evidential head for the final query produced by the interaction pathway cascade.

The final interaction-pathway representation is:

$$
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\in
\mathbb{R}^{64}.
$$

Unlike the independent modality representations, this vector contains cumulative information learned through the ordered sequence of Cascaded Modality Transformers.

The joint evidential head produces non-negative class evidence:

$$
e_{ik}^{\mathrm{joint}}
=
\operatorname{Softplus}
\left(
f_{\mathrm{joint}}
\left(
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\right)
\right),
$$

where \(k\) denotes either sMCI or pMCI.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{\mathrm{joint}}
=
e_{ik}^{\mathrm{joint}} + 1.
$$

The expected class probabilities are:

$$
p_{ik}^{\mathrm{joint}}
=
\frac{
\alpha_{ik}^{\mathrm{joint}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{joint}}
}.
$$

The joint uncertainty is:

$$
u_{i}^{\mathrm{joint}}
=
\frac{K}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{joint}}
}.
$$

This opinion serves a different purpose from the evidence pathway-fused independent opinion:

- the modality-specific opinion represents agreement and conflict between modality-specific predictions;
- the joint interaction pathway opinion represents the prediction obtained after learning cross-modal interactions.

### 1.5.19. Auxiliary outputs

The original interaction pathway architecture places an auxiliary classifier after each intermediate cross-modal interaction to provide direct training signals to earlier cascade stages.

The notebook preserves that principle by attaching one auxiliary classifier to every intermediate query except the final MRI stage. These auxiliary heads produce ordinary logits for the prognosis classes and are used only during training.

The auxiliary classifiers are not treated as independent modality-specific opinions because each intermediate query already contains information accumulated from all preceding modalities. They support gradient flow through the cascade but do not represent isolated modality evidence.

The final MRI-stage query receives the joint evidential head and produces the interaction-aware Dirichlet opinion.

In [ ]:
# ============================================================
# 12. Producing the interaction-aware interaction pathway evidential opinion
# ============================================================

# ------------------------------------------------------------
# Auxiliary classifier for one intermediate interaction pathway query
# ------------------------------------------------------------

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(self, representation):
        return self.network(
            representation
        )


# ------------------------------------------------------------
# Joint interaction pathway evidential and auxiliary heads
# ------------------------------------------------------------

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )


    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # --------------------------------------------------------
        # Intermediate auxiliary predictions
        # --------------------------------------------------------

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )


        # --------------------------------------------------------
        # Final joint evidential opinion
        # --------------------------------------------------------

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final interaction pathway query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }


# ------------------------------------------------------------
# Instantiate the interaction-pathway prediction heads
# ------------------------------------------------------------

three_mt_prediction_heads = ThreeMTPredictionHeads(
    cascade_order=THREE_MT_CASCADE_ORDER,
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
)


# ------------------------------------------------------------
# Produce the auxiliary and joint outputs
# ------------------------------------------------------------

three_mt_prediction_heads.eval()

with torch.no_grad():

    three_mt_predictions = (
        three_mt_prediction_heads(
            three_mt_output=three_mt_output
        )
    )


joint_opinion = three_mt_predictions[
    "joint_opinion"
]


# ------------------------------------------------------------
# Display the output structure
# ------------------------------------------------------------

print("=" * 72)
print("3MT PREDICTION HEADS")
print("=" * 72)

print(
    f"\nAuxiliary stages: "
    f"{three_mt_prediction_heads.auxiliary_stages}"
)

print(
    f"Final evidential stage: "
    f"{three_mt_prediction_heads.final_stage}"
)

print("\nAuxiliary-logit shapes:")

for stage_name, stage_logits in (
    three_mt_predictions[
        "auxiliary_logits"
    ].items()
):
    print(
        f"- after {stage_name}: "
        f"{tuple(stage_logits.shape)}"
    )


print("\nJoint evidential shapes:")

print(
    "  evidence: "
    f"{tuple(joint_opinion['evidence'].shape)}"
)

print(
    "  alpha: "
    f"{tuple(joint_opinion['alpha'].shape)}"
)

print(
    "  probabilities: "
    f"{tuple(joint_opinion['probabilities'].shape)}"
)

print(
    "  uncertainty: "
    f"{tuple(joint_opinion['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Participant-level joint opinions
# ------------------------------------------------------------

joint_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    joint_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ALPHA_sMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                joint_opinion["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


joint_opinion_summary = pd.DataFrame(
    joint_opinion_rows
)

print("\nInteraction-aware 3MT opinions:")

display(
    joint_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable 3MT prediction-head parameters: "
    f"{count_trainable_parameters(three_mt_prediction_heads):,}"
)

print(
    "\nThe auxiliary outputs will support training-time "
    "gradient flow through the cascade."
)

print(
    "The final joint opinion is ready for combination with "
    "the TMC-fused independent opinion."
)

### 1.5.20. Combining the interaction pathway and modality-specific evidence pathways with a constrained reliability gate

The model currently produces two complementary evidential outputs:

1. the interaction-aware interaction pathway opinion;
2. the independently fused modality-specific opinion.

These opinions are derived from the same underlying participant data and therefore should not be combined using Dempster--Shafer fusion as though they were independent evidence sources.

Instead, This notebook uses a participant-specific convex mixture of their calibrated evidence vectors.

### 1.5.21. Pathway calibration

The evidence magnitudes produced by the two pathways may have different numerical scales. In particular, the modality-specific evidence pathway accumulates evidence across several available modalities, whereas the cross-modal interaction pathway produces evidence from one joint head.

I therefore introduce one positive scalar calibration parameter for each pathway:

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
=
\tau_{\mathrm{interaction pathway}}
\mathbf{e}_{i}^{\mathrm{interaction pathway}},
$$

and

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}
=
\tau_{\mathrm{evidence pathway}}
\mathbf{e}_{i}^{\mathrm{evidence pathway}}.
$$

Both calibration factors are constrained to be positive using the Softplus function:

$$
\tau_r
=
\operatorname{Softplus}(\rho_r),
\qquad
r \in
\{\mathrm{interaction pathway},\mathrm{evidence pathway}\}.
$$

The parameters are initialised so that both calibration factors begin at approximately one.

### 1.5.22. Reliability-gate input

The reliability gate receives only uncertainty, conflict, and modality-availability information. It does not receive the original participant features or the full interaction-pathway representation.

For participant \(i\), the gate input is:

$$
\mathbf{r}_i
=
\left[
u_i^{\mathrm{interaction pathway}},
u_i^{\mathrm{evidence pathway}},
\bar{C}_i^{\mathrm{evidence pathway}},
\frac{n_i^{\mathrm{available}}}{M},
\mathbf{a}_i
\right],
$$

where:

- \(u_i^{\mathrm{interaction pathway}}\) is the uncertainty of the joint interaction pathway opinion;
- \(u_i^{\mathrm{evidence pathway}}\) is the uncertainty of the evidence pathway-fused opinion;
- \(\bar{C}_i^{\mathrm{evidence pathway}}\) is the mean conflict encountered when combining available modality opinions;
- \(n_i^{\mathrm{available}}\) is the number of available branches;
- \(M=6\) is the total number of branches;
- \(\mathbf{a}_i\) is the six-element branch-availability vector.

The gate produces one scalar weight:

$$
w_i
=
\sigma
\left(
\mathbf{w}^{\top}
\mathbf{r}_i+b
\right).
$$

The interpretation is:

$$
w_i \rightarrow 1
\quad
\Longrightarrow
\quad
\text{greater reliance on interaction pathway},
$$

and

$$
w_i \rightarrow 0
\quad
\Longrightarrow
\quad
\text{greater reliance on evidence pathway}.
$$

The gate is initialised with zero weights and zero bias. Therefore, before training:

$$
w_i = 0.5.
$$

This prevents either pathway from being preferred arbitrarily at model initialisation.

### 1.5.23. Final evidential opinion

The final evidence is:

$$
\mathbf{e}_{i}^{\mathrm{final}}
=
w_i
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
+
(1-w_i)
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}.
$$

Because \(w_i \in [0,1]\), this is a convex mixture rather than an addition of two supposedly independent evidence sources.

The final Dirichlet parameters are:

$$
\boldsymbol{\alpha}_{i}^{\mathrm{final}}
=
\mathbf{e}_{i}^{\mathrm{final}}
+
\mathbf{1}.
$$

The final class probabilities and uncertainty are:

$$
p_{ik}^{\mathrm{final}}
=
\frac{
\alpha_{ik}^{\mathrm{final}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{final}}
},
$$

and

$$
u_i^{\mathrm{final}}
=
\frac{
K
}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{final}}
}.
$$

The reliability statistics supplied to the gate are detached from the computational graph. This prevents the upstream pathways from manipulating their uncertainty or conflict values merely to obtain a larger gate weight. The final loss can still train both pathways through their evidence contributions.

In [ ]:
# ============================================================
# 13. Combining interaction pathway and evidence pathway with fixed equal fusion
# ============================================================

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )


class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )


    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )


    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


hybrid_fusion = FixedEqualHybridFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    number_of_modalities=len(
        MODALITY_ORDER
    ),
    initial_three_mt_scale=1.0,
    initial_tmc_scale=1.0,
)


hybrid_fusion.eval()

with torch.no_grad():
    hybrid_output = hybrid_fusion(
        joint_opinion=joint_opinion,
        tmc_output=tmc_output,
        branch_masks=example_batch[
            "branch_masks"
        ],
        modality_order=MODALITY_ORDER,
    )


print("=" * 72)
print("FIXED 50/50 3MT-TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    "\nFinal probability shape: "
    f"{tuple(hybrid_output['probabilities'].shape)}"
)

print(
    "Initial 3MT evidence scale: "
    f"{hybrid_output['three_mt_scale'].item():.6f}"
)

print(
    "Initial TMC evidence scale: "
    f"{hybrid_output['tmc_scale'].item():.6f}"
)

print(
    "Trainable fixed-fusion parameters: "
    f"{count_trainable_parameters(hybrid_fusion):,}"
)

maximum_three_mt_weight_error = (
    hybrid_output[
        "three_mt_weight"
    ]
    - 0.5
).abs().max().item()

maximum_tmc_weight_error = (
    hybrid_output[
        "tmc_weight"
    ]
    - 0.5
).abs().max().item()

probability_sum_error = (
    hybrid_output[
        "probabilities"
    ].sum(
        dim=-1
    )
    - 1.0
).abs().max().item()

print(
    "\nMaximum 3MT-weight deviation from 0.5: "
    f"{maximum_three_mt_weight_error:.10f}"
)

print(
    "Maximum TMC-weight deviation from 0.5: "
    f"{maximum_tmc_weight_error:.10f}"
)

print(
    "Maximum final probability-sum error: "
    f"{probability_sum_error:.10f}"
)

assert maximum_three_mt_weight_error == 0.0
assert maximum_tmc_weight_error == 0.0

print(
    "\nThe participant-specific reliability gate is absent. "
    "Only the two positive pathway evidence scales remain trainable."
)


### 1.5.24. Assembling the complete availability-gated interaction-evidence model

The complete model keeps the same six components used in the original fold-0 run:

1. modality-specific encoders;
2. training-time modality dropout;
3. independent modality evidential heads;
4. modality-specific evidence fusion;
5. the interaction pathway interaction pathway;
6. fixed-equal hybrid evidence fusion.

The effective branch masks are created before encoding. They contain both natural missingness and any additional modality removed by training-time dropout. These exact masks are now passed into the interaction pathway cascade, so every unavailable cross-modal interaction stage preserves the preceding cumulative query.

The modality-specific evidence pathway and fixed equal fusion are unchanged. This isolates the effect of availability-gating the cross-modal interaction updates.

In [ ]:
# ============================================================
# 14. Assembling the complete end-to-end interaction-evidence model
# ============================================================

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )


        # --------------------------------------------------------
        # Modality-specific encoders
        # --------------------------------------------------------

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )


        # --------------------------------------------------------
        # Independent modality evidential pathway
        # --------------------------------------------------------

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )


        # --------------------------------------------------------
        # Final fixed 50/50 hybrid fusion
        # --------------------------------------------------------

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )


    # ------------------------------------------------------------
    # Training-time modality dropout
    # ------------------------------------------------------------

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )


        # --------------------------------------------------------
        # Sample branch-retention indicators
        # --------------------------------------------------------

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )


        # --------------------------------------------------------
        # Prevent complete information removal
        # --------------------------------------------------------

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # I randomly restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0


        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )


    # ------------------------------------------------------------
    # Construct effective modality dictionaries
    # ------------------------------------------------------------

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities


    # ------------------------------------------------------------
    # Complete forward pass
    # ------------------------------------------------------------

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # --------------------------------------------------------
        # Apply training-time modality dropout
        # --------------------------------------------------------

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )


        # --------------------------------------------------------
        # Encode all six modalities
        # --------------------------------------------------------

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. I use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]


        # --------------------------------------------------------
        # Independent modality opinions and modality-specific evidence fusion
        # --------------------------------------------------------

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]


        # --------------------------------------------------------
        # Final fixed-equal hybrid opinion
        # --------------------------------------------------------

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )


        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


# ------------------------------------------------------------
# Instantiate the complete model
# ------------------------------------------------------------

complete_model = ADNIEvidential3MTTMCModel(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    modality_dropout_probability=0.50,
)


# ------------------------------------------------------------
# Evaluation-mode end-to-end forward pass
# ------------------------------------------------------------

# In evaluation mode, modality dropout is disabled.
complete_model.eval()

with torch.no_grad():

    complete_model_output = complete_model(
        modalities=example_batch[
            "modalities"
        ],

        original_branch_masks=example_batch[
            "branch_masks"
        ],
    )


# ------------------------------------------------------------
# Inspect final outputs
# ------------------------------------------------------------

final_output = complete_model_output[
    "final_output"
]

print("=" * 72)
print("COMPLETE AVAILABILITY-GATED 3MT-TMC MODEL")
print("=" * 72)

print(
    "\nModel mode: "
    f"{'training' if complete_model.training else 'evaluation'}"
)

print(
    "Modality-dropout probability: "
    f"{complete_model.modality_dropout_probability:.2f}"
)

print(
    "\nOriginal branch-mask shape: "
    f"{tuple(complete_model_output['original_branch_masks'].shape)}"
)

print(
    "Effective branch-mask shape: "
    f"{tuple(complete_model_output['effective_branch_masks'].shape)}"
)

print(
    "\nFinal alpha shape: "
    f"{tuple(final_output['alpha'].shape)}"
)

print(
    "Final probability shape: "
    f"{tuple(final_output['probabilities'].shape)}"
)

print(
    "Final uncertainty shape: "
    f"{tuple(final_output['uncertainty'].shape)}"
)

print(
    "\nTotal trainable model parameters: "
    f"{count_trainable_parameters(complete_model):,}"
)


# ------------------------------------------------------------
# Verify that evaluation mode preserves genuine availability
# ------------------------------------------------------------

evaluation_mask_difference = (
    complete_model_output[
        "effective_branch_masks"
    ]
    - complete_model_output[
        "original_branch_masks"
    ]
).abs().max().item()

print(
    "\nMaximum evaluation-mode difference between original "
    f"and effective branch masks: "
    f"{evaluation_mask_difference:.10f}"
)


# ------------------------------------------------------------
# Participant-level output summary
# ------------------------------------------------------------

complete_model_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    original_available_count = int(
        complete_model_output[
            "original_branch_masks"
        ][batch_row].sum().item()
    )

    effective_available_count = int(
        complete_model_output[
            "effective_branch_masks"
        ][batch_row].sum().item()
    )

    complete_model_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ORIGINAL_MODALITIES":
                original_available_count,

            "EFFECTIVE_MODALITIES":
                effective_available_count,

            "W_3MT": float(
                final_output[
                    "three_mt_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "W_TMC": float(
                final_output[
                    "tmc_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_sMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    1,
                ].item()
            ),

            "FINAL_UNCERTAINTY": float(
                final_output[
                    "uncertainty"
                ][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


complete_model_summary = pd.DataFrame(
    complete_model_rows
)

print("\nComplete-model evaluation-mode outputs:")

display(
    complete_model_summary.round(6)
)

print(
    "\nThe complete model now passes the effective branch "
    "masks into every CMT stage."
)


### 1.5.25. Defining the joint training objective

The complete architecture produces several supervised outputs with different purposes:

1. the final fixed-equal hybrid opinion;
2. the interaction-aware interaction pathway joint opinion;
3. the evidence pathway-fused independent-modality opinion;
4. six independent modality-specific opinions;
5. five intermediate interaction pathway auxiliary predictions.

These outputs are trained jointly, but the final hybrid prediction remains the primary objective.

The total loss is:

$$
\mathcal{L}_{\mathrm{total}}
=
\mathcal{L}_{\mathrm{final}}
+
\lambda_{\mathrm{joint}}
\mathcal{L}_{\mathrm{joint}}
+
\lambda_{\mathrm{evidence pathway}}
\mathcal{L}_{\mathrm{evidence pathway}}
+
\lambda_{\mathrm{mod}}
\mathcal{L}_{\mathrm{mod}}
+
\lambda_{\mathrm{aux}}
\mathcal{L}_{\mathrm{aux}}
+
\lambda_{\mathrm{gate}}
\mathcal{L}_{\mathrm{gate}}.
$$

The initial loss weights are:

$$
\lambda_{\mathrm{joint}} = 0.5,
\qquad
\lambda_{\mathrm{evidence pathway}} = 0.5,
\qquad
\lambda_{\mathrm{mod}} = 0.1,
$$

$$
\lambda_{\mathrm{aux}} = 0.1,
\qquad
\lambda_{\mathrm{gate}} = 0.01.
$$

The final hybrid loss has coefficient one and therefore remains the dominant objective. The remaining terms provide direct supervision to the two pathways and their intermediate components.

These coefficients are initial modelling choices. Any comparison of alternative values must use only the training and validation partitions.

### 1.5.26. Evidential classification loss

For a Dirichlet prediction:

$$
\boldsymbol{\alpha}_i
=
\mathbf{e}_i+\mathbf{1},
$$

the expected cross-entropy loss is:

$$
\mathcal{L}_{\mathrm{ECE},i}
=
\sum_{k=1}^{K}
y_{ik}
\left[
\psi(S_i)
-
\psi(\alpha_{ik})
\right],
$$

where:

$$
S_i
=
\sum_{k=1}^{K}
\alpha_{ik},
$$

and \(\psi(\cdot)\) denotes the digamma function.

This objective minimises the expected negative log-likelihood under the predicted Dirichlet distribution.

### 1.5.27. Evidence regularisation

An evidential network may become unjustifiably confident by assigning strong evidence to an incorrect class. I therefore add a Kullback--Leibler regularisation term that discourages unsupported evidence.

The adjusted Dirichlet parameters are:

$$
\widetilde{\boldsymbol{\alpha}}_i
=
\mathbf{y}_i
+
(1-\mathbf{y}_i)
\odot
\boldsymbol{\alpha}_i.
$$

This construction removes the evidence assigned to the correct class from the regularisation term while penalising evidence assigned to incorrect classes.

The regularisation term is:

$$
\mathcal{L}_{\mathrm{KL},i}
=
D_{\mathrm{KL}}
\left[
\operatorname{Dir}
\left(
\widetilde{\boldsymbol{\alpha}}_i
\right)
\parallel
\operatorname{Dir}
\left(
\mathbf{1}
\right)
\right].
$$

The complete evidential loss is:

$$
\mathcal{L}_{\mathrm{EDL},i}
=
\mathcal{L}_{\mathrm{ECE},i}
+
\beta_t
\mathcal{L}_{\mathrm{KL},i}.
$$

The regularisation coefficient is annealed during the first training epochs:

$$
\beta_t
=
\min
\left(
1,
\frac{t}{T_{\mathrm{anneal}}}
\right),
$$

where \(t\) is the current epoch and \(T_{\mathrm{anneal}}\) is initially set to ten epochs.

This allows the model to begin learning the classification task before the full evidence penalty is applied.

### 1.5.28. Modality-specific loss

Each independent modality opinion is supervised only when that modality is effectively available after training-time modality dropout.

For branch \(m\), let:

$$
\widetilde{a}_i^{(m)}
\in
\{0,1\}
$$

denote the effective branch mask.

The modality-specific loss is:

$$
\mathcal{L}_{\mathrm{mod}}
=
\frac{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
\mathcal{L}_{\mathrm{EDL},i}^{(m)}
}{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
+
\varepsilon
}.
$$

Naturally unavailable or deliberately dropped branches therefore contribute no modality-specific classification loss.

### 1.5.29. Auxiliary interaction pathway loss

The intermediate interaction pathway heads produce ordinary class logits rather than Dirichlet opinions. Their loss is the mean cross-entropy across the five intermediate cascade stages:

$$
\mathcal{L}_{\mathrm{aux}}
=
\frac{1}{J}
\sum_{j=1}^{J}
\operatorname{CE}
\left(
\mathbf{z}^{(j)},
y
\right),
$$

where \(J=5\).

These losses provide direct gradient signals to earlier stages of the cascaded transformer.

### 1.5.30. Gate regularisation

The reliability gate is initialised at:

$$
w_i=0.5.
$$

A weak early-training regulariser discourages immediate collapse to a single pathway:

$$
\mathcal{L}_{\mathrm{gate}}
=
\left(
\frac{1}{N}
\sum_{i=1}^{N}
w_i
-
0.5
\right)^2.
$$

The gate regularisation is annealed to zero after the initial training period. It therefore stabilises early optimisation without forcing the final trained gate to remain balanced.

This cell defines and validates the training objective only. Parameter updates begin after gradient flow is checked in the following step.

In [ ]:
# ============================================================
# 15. Defining the complete joint training objective
# ============================================================

# ------------------------------------------------------------
# KL divergence between a predicted Dirichlet distribution and
# a uniform Dirichlet distribution
# ------------------------------------------------------------

def dirichlet_kl_to_uniform(
    alpha,
):
    """
    Calculate:

        KL(Dir(alpha) || Dir(1))

    for every participant in the batch.

    Parameters
    ----------
    alpha:
        Positive Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    Returns
    -------
    Tensor with shape:
        (batch_size,)
    """

    number_of_classes = alpha.shape[-1]

    uniform_alpha = torch.ones_like(
        alpha
    )

    alpha_strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    uniform_strength = uniform_alpha.sum(
        dim=-1,
        keepdim=True,
    )

    log_normalisation_ratio = (
        torch.lgamma(alpha_strength)
        - torch.lgamma(uniform_strength)
        - torch.lgamma(alpha).sum(
            dim=-1,
            keepdim=True,
        )
        + torch.lgamma(uniform_alpha).sum(
            dim=-1,
            keepdim=True,
        )
    )

    digamma_difference = (
        torch.digamma(alpha)
        - torch.digamma(alpha_strength)
    )

    parameter_difference = (
        alpha
        - uniform_alpha
    )

    expectation_term = (
        parameter_difference
        * digamma_difference
    ).sum(
        dim=-1,
        keepdim=True,
    )

    kl_divergence = (
        log_normalisation_ratio
        + expectation_term
    )

    return kl_divergence.squeeze(-1)


# ------------------------------------------------------------
# Evidential classification loss
# ------------------------------------------------------------

def evidential_classification_loss(
    alpha,
    targets,
    number_of_classes,
    annealing_coefficient,
    class_weights=None,
    reduction="mean",
):
    """
    Calculate the expected cross-entropy under a Dirichlet
    distribution together with annealed KL regularisation.

    Parameters
    ----------
    alpha:
        Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    targets:
        Integer class labels with shape:
        (batch_size,)

    number_of_classes:
        Number of prognosis classes.

    annealing_coefficient:
        Current coefficient applied to the KL term.

    class_weights:
        Optional class-weight tensor with shape:
        (number_of_classes,)

    reduction:
        "none", "mean", or "sum".
    """

    targets = targets.long()

    one_hot_targets = F.one_hot(
        targets,
        num_classes=number_of_classes,
    ).to(
        dtype=alpha.dtype
    )

    strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )


    # --------------------------------------------------------
    # Expected cross-entropy under the Dirichlet distribution
    # --------------------------------------------------------

    expected_cross_entropy_by_class = (
        torch.digamma(strength)
        - torch.digamma(alpha)
    )

    expected_cross_entropy = (
        one_hot_targets
        * expected_cross_entropy_by_class
    ).sum(
        dim=-1
    )


    # --------------------------------------------------------
    # Optional class weighting
    # --------------------------------------------------------

    if class_weights is not None:

        sample_weights = class_weights[
            targets
        ].to(
            dtype=alpha.dtype,
            device=alpha.device,
        )

        expected_cross_entropy = (
            expected_cross_entropy
            * sample_weights
        )


    # --------------------------------------------------------
    # Remove correct-class evidence from the KL penalty
    # --------------------------------------------------------

    adjusted_alpha = (
        one_hot_targets
        +
        (
            1.0
            - one_hot_targets
        )
        * alpha
    )

    kl_regularisation = (
        dirichlet_kl_to_uniform(
            adjusted_alpha
        )
    )

    per_sample_loss = (
        expected_cross_entropy
        +
        annealing_coefficient
        * kl_regularisation
    )


    # --------------------------------------------------------
    # Requested reduction
    # --------------------------------------------------------

    if reduction == "none":
        reduced_loss = per_sample_loss

    elif reduction == "mean":
        reduced_loss = per_sample_loss.mean()

    elif reduction == "sum":
        reduced_loss = per_sample_loss.sum()

    else:
        raise ValueError(
            "reduction must be 'none', 'mean', or 'sum'."
        )


    return {
        "loss":
            reduced_loss,

        "per_sample_loss":
            per_sample_loss,

        "expected_cross_entropy":
            expected_cross_entropy,

        "kl_regularisation":
            kl_regularisation,
    }


# ------------------------------------------------------------
# Complete multi-output objective
# ------------------------------------------------------------

class HybridEvidentialTrainingLoss(nn.Module):
    """
    Jointly supervise the final hybrid output, both main pathways,
    the individual available modality opinions, and the auxiliary
    3MT classifiers.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        evidential_annealing_epochs=10,
        gate_regularisation_epochs=0,
        joint_loss_weight=0.50,
        tmc_loss_weight=0.50,
        modality_loss_weight=0.10,
        auxiliary_loss_weight=0.10,
        gate_loss_weight=0.0,
        class_weights=None,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.evidential_annealing_epochs = (
            evidential_annealing_epochs
        )

        self.gate_regularisation_epochs = (
            gate_regularisation_epochs
        )

        self.joint_loss_weight = (
            joint_loss_weight
        )

        self.tmc_loss_weight = (
            tmc_loss_weight
        )

        self.modality_loss_weight = (
            modality_loss_weight
        )

        self.auxiliary_loss_weight = (
            auxiliary_loss_weight
        )

        self.gate_loss_weight = (
            gate_loss_weight
        )


        # --------------------------------------------------------
        # Optional training-fold class weights
        # --------------------------------------------------------

        if class_weights is None:

            self.register_buffer(
                "class_weights",
                None,
            )

        else:

            class_weights = torch.as_tensor(
                class_weights,
                dtype=torch.float32,
            )

            if class_weights.shape != (
                number_of_classes,
            ):
                raise ValueError(
                    "class_weights must contain one value "
                    "for each class."
                )

            self.register_buffer(
                "class_weights",
                class_weights,
            )


    # ------------------------------------------------------------
    # Annealing schedules
    # ------------------------------------------------------------

    def _evidential_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Increase the KL coefficient linearly from zero to one.
        """

        if self.evidential_annealing_epochs <= 0:
            return 1.0

        return min(
            1.0,
            float(epoch)
            / float(
                self.evidential_annealing_epochs
            ),
        )


    def _gate_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Reduce the gate-balance penalty to zero after the initial
        optimisation period.
        """

        if self.gate_regularisation_epochs <= 0:
            return 0.0

        return max(
            0.0,
            1.0
            -
            (
                float(epoch - 1)
                /
                float(
                    self.gate_regularisation_epochs
                )
            ),
        )


    # ------------------------------------------------------------
    # Complete loss calculation
    # ------------------------------------------------------------

    def forward(
        self,
        model_output,
        targets,
        epoch,
    ):
        targets = targets.long()

        evidential_annealing = (
            self._evidential_annealing_coefficient(
                epoch
            )
        )

        gate_annealing = (
            self._gate_annealing_coefficient(
                epoch
            )
        )


        # --------------------------------------------------------
        # Final hybrid evidential loss
        # --------------------------------------------------------

        final_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "final_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        final_loss = final_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Interaction-aware interaction pathway evidential loss
        # --------------------------------------------------------

        joint_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "joint_opinion"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        joint_loss = joint_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # evidence pathway-fused evidential loss
        # --------------------------------------------------------

        tmc_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "tmc_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        tmc_loss = tmc_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Independent available-modality evidential loss
        # --------------------------------------------------------

        effective_branch_masks = model_output[
            "effective_branch_masks"
        ]

        weighted_modality_loss_sum = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        available_opinion_count = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        modality_loss_by_name = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):

            modality_alpha = model_output[
                "modality_opinions"
            ][modality_name]["alpha"]

            modality_loss_components = (
                evidential_classification_loss(
                    alpha=modality_alpha,

                    targets=targets,

                    number_of_classes=
                        self.number_of_classes,

                    annealing_coefficient=
                        evidential_annealing,

                    class_weights=
                        self.class_weights,

                    reduction="none",
                )
            )

            per_sample_modality_loss = (
                modality_loss_components[
                    "per_sample_loss"
                ]
            )

            modality_mask = effective_branch_masks[
                :,
                modality_index,
            ].to(
                dtype=per_sample_modality_loss.dtype
            )

            masked_modality_loss_sum = (
                per_sample_modality_loss
                * modality_mask
            ).sum()

            modality_available_count = (
                modality_mask.sum()
            )

            weighted_modality_loss_sum = (
                weighted_modality_loss_sum
                + masked_modality_loss_sum
            )

            available_opinion_count = (
                available_opinion_count
                + modality_available_count
            )

            modality_loss_by_name[
                modality_name
            ] = (
                masked_modality_loss_sum
                /
                modality_available_count.clamp_min(
                    1.0
                )
            )

        modality_loss = (
            weighted_modality_loss_sum
            /
            available_opinion_count.clamp_min(
                1.0
            )
        )


        # --------------------------------------------------------
        # Intermediate interaction pathway auxiliary cross-entropy loss
        # --------------------------------------------------------

        auxiliary_logits = model_output[
            "three_mt_predictions"
        ]["auxiliary_logits"]

        auxiliary_loss_by_stage = {}

        auxiliary_losses = []

        for stage_name, stage_logits in (
            auxiliary_logits.items()
        ):

            stage_loss = F.cross_entropy(
                input=stage_logits,
                target=targets,
                weight=self.class_weights,
            )

            auxiliary_loss_by_stage[
                stage_name
            ] = stage_loss

            auxiliary_losses.append(
                stage_loss
            )

        if auxiliary_losses:

            auxiliary_loss = torch.stack(
                auxiliary_losses
            ).mean()

        else:

            auxiliary_loss = torch.zeros(
                (),
                dtype=final_loss.dtype,
                device=final_loss.device,
            )


        # --------------------------------------------------------
        # Early gate-balance regularisation
        # --------------------------------------------------------

        three_mt_weights = model_output[
            "final_output"
        ]["three_mt_weight"]

        raw_gate_loss = (
            three_mt_weights.mean()
            - 0.5
        ).pow(2)

        annealed_gate_loss = (
            gate_annealing
            * raw_gate_loss
        )


        # --------------------------------------------------------
        # Weighted total objective
        # --------------------------------------------------------

        total_loss = (
            final_loss
            +
            self.joint_loss_weight
            * joint_loss
            +
            self.tmc_loss_weight
            * tmc_loss
            +
            self.modality_loss_weight
            * modality_loss
            +
            self.auxiliary_loss_weight
            * auxiliary_loss
            +
            self.gate_loss_weight
            * annealed_gate_loss
        )


        return {
            "total_loss":
                total_loss,

            "final_loss":
                final_loss,

            "joint_loss":
                joint_loss,

            "tmc_loss":
                tmc_loss,

            "modality_loss":
                modality_loss,

            "auxiliary_loss":
                auxiliary_loss,

            "raw_gate_loss":
                raw_gate_loss,

            "annealed_gate_loss":
                annealed_gate_loss,

            "evidential_annealing":
                torch.tensor(
                    evidential_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "gate_annealing":
                torch.tensor(
                    gate_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "available_opinion_count":
                available_opinion_count,

            "modality_loss_by_name":
                modality_loss_by_name,

            "auxiliary_loss_by_stage":
                auxiliary_loss_by_stage,

            "final_expected_cross_entropy":
                final_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "final_kl_regularisation":
                final_loss_components[
                    "kl_regularisation"
                ].mean(),

            "joint_expected_cross_entropy":
                joint_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "joint_kl_regularisation":
                joint_loss_components[
                    "kl_regularisation"
                ].mean(),

            "tmc_expected_cross_entropy":
                tmc_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "tmc_kl_regularisation":
                tmc_loss_components[
                    "kl_regularisation"
                ].mean(),
        }


# ------------------------------------------------------------
# Instantiate the training objective
# ------------------------------------------------------------

training_objective = HybridEvidentialTrainingLoss(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,

    evidential_annealing_epochs=10,
    gate_regularisation_epochs=0,

    joint_loss_weight=0.50,
    tmc_loss_weight=0.50,
    modality_loss_weight=0.10,
    auxiliary_loss_weight=0.10,
    gate_loss_weight=0.0,

    # I initially leave class weighting disabled. It can be added
    # using weights calculated from each training fold only.
    class_weights=None,
)


# ------------------------------------------------------------
# Test the objective on the complete untrained forward pass
# ------------------------------------------------------------

example_loss_output = training_objective(
    model_output=complete_model_output,

    targets=example_batch[
        "target"
    ],

    epoch=1,
)


# ------------------------------------------------------------
# Display the initial loss structure
# ------------------------------------------------------------

print("=" * 72)
print("JOINT 3MT-TMC TRAINING OBJECTIVE")
print("=" * 72)

print(
    "\nEvidential KL annealing coefficient at epoch 1: "
    f"{example_loss_output['evidential_annealing'].item():.6f}"
)

print(
    "Gate regularisation coefficient at epoch 1: "
    f"{example_loss_output['gate_annealing'].item():.6f}"
)

print("\nMain loss components:")

for loss_name in [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]:
    print(
        f"- {loss_name}: "
        f"{example_loss_output[loss_name].item():.6f}"
    )


print("\nFinal evidential-loss decomposition:")

print(
    "- expected cross-entropy: "
    f"{example_loss_output['final_expected_cross_entropy'].item():.6f}"
)

print(
    "- KL regularisation: "
    f"{example_loss_output['final_kl_regularisation'].item():.6f}"
)


print("\nAvailable modality opinions in this batch: "
      f"{int(example_loss_output['available_opinion_count'].item())}")


# ------------------------------------------------------------
# Modality-specific loss summary
# ------------------------------------------------------------

modality_loss_rows = []

for modality_name in MODALITY_ORDER:

    modality_loss_rows.append(
        {
            "MODALITY": modality_name,

            "MEAN_AVAILABLE_LOSS": float(
                example_loss_output[
                    "modality_loss_by_name"
                ][modality_name].item()
            ),
        }
    )


print("\nModality-specific evidential losses:")

display(
    pd.DataFrame(
        modality_loss_rows
    ).round(6)
)


# ------------------------------------------------------------
# Auxiliary-stage loss summary
# ------------------------------------------------------------

auxiliary_loss_rows = []

for stage_name in THREE_MT_CASCADE_ORDER[:-1]:

    auxiliary_loss_rows.append(
        {
            "AUXILIARY_STAGE": stage_name,

            "CROSS_ENTROPY_LOSS": float(
                example_loss_output[
                    "auxiliary_loss_by_stage"
                ][stage_name].item()
            ),
        }
    )


print("\nIntermediate 3MT auxiliary losses:")

display(
    pd.DataFrame(
        auxiliary_loss_rows
    ).round(6)
)

print(
    "\nThe joint objective is ready for one end-to-end "
    "backward pass."
)


### 1.5.31. Running one end-to-end backward pass

Before configuring the optimiser, I run one training batch through the complete gated model and joint objective.

For this single diagnostic pass, modality dropout is set to zero so that the natural fold-0 availability pattern is used without additional random branch removal. The total loss is backpropagated, but no optimiser step is taken. The cell reports the total loss and the global gradient norm, then restores the configured modality-dropout probability of `0.50`.

In [ ]:
# ============================================================
# 16. Running one end-to-end backward pass
# ============================================================

diagnostic_batch = next(
    iter(train_loader)
)

original_modality_dropout_probability = (
    complete_model.modality_dropout_probability
)

complete_model.modality_dropout_probability = 0.0
complete_model.train()
complete_model.zero_grad(
    set_to_none=True
)


diagnostic_output = complete_model(
    modalities=diagnostic_batch[
        "modalities"
    ],
    original_branch_masks=diagnostic_batch[
        "branch_masks"
    ],
)

diagnostic_losses = training_objective(
    model_output=diagnostic_output,
    targets=diagnostic_batch[
        "target"
    ],
    epoch=1,
)

diagnostic_total_loss = diagnostic_losses[
    "total_loss"
]

diagnostic_total_loss.backward()


global_gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().float().pow(2).sum()
        for parameter in complete_model.parameters()
        if parameter.grad is not None
    )
)

print("=" * 72)
print("END-TO-END BACKWARD PASS")
print("=" * 72)

print(
    "\nTotal loss: "
    f"{diagnostic_total_loss.item():.6f}"
)

print(
    "Global gradient norm: "
    f"{global_gradient_norm.item():.6f}"
)

complete_model.zero_grad(
    set_to_none=True
)

complete_model.modality_dropout_probability = (
    original_modality_dropout_probability
)

complete_model.eval()

print(
    "Modality-dropout probability restored to: "
    f"{complete_model.modality_dropout_probability:.2f}"
)


### 1.5.32. Configuring optimisation and experiment-specific output paths

The original training hyperparameters remain unchanged. Every checkpoint, history file, validation prediction, and test prediction is written only under:

```text
models/3mt_tmc_evidential/experiments/
    gated_cmt_learned_gate_md050/
        mci_prognosis/fold_0/
```

This notebook does not use the older shared `training/mci_prognosis/fold_0/` directory.

In [ ]:
# ============================================================
# 17. Configuring optimisation, checkpoints, and metrics
# ============================================================

import os
import random
from datetime import datetime

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# Current experiment
# ------------------------------------------------------------

# EXPERIMENT_NAME, SELECTED_TASK, and SELECTED_FOLD were fixed when this fold's prepared input was loaded.


# ------------------------------------------------------------
# Reproducibility configuration
# ------------------------------------------------------------

GLOBAL_RANDOM_SEED = 42


def set_global_random_seed(seed):
    """
    Set the random seed used by Python, NumPy, and PyTorch.
    """

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_random_seed(
    GLOBAL_RANDOM_SEED
)


# ------------------------------------------------------------
# PyTorch numerical configuration
# ------------------------------------------------------------

# I allow cuDNN to choose efficient convolution algorithms.
#
# This is appropriate for the computationally expensive 3D MRI
# encoder. Exact bitwise reproducibility can still depend on the
# installed PyTorch, CUDA, cuDNN, and GPU versions.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 72)
print("TRAINING CONFIGURATION")
print("=" * 72)

print(
    f"\nExperiment: {EXPERIMENT_NAME}"
)

print(
    f"Task: {SELECTED_TASK}"
)

print(
    f"Selected fold: {SELECTED_FOLD}"
)

print(
    f"Device: {DEVICE}"
)

if torch.cuda.is_available():

    print(
        "CUDA device: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        "CUDA memory allocated before model transfer: "
        f"{torch.cuda.memory_allocated(0) / (1024 ** 3):.3f} GB"
    )


# ------------------------------------------------------------
# Move the complete model and objective to the selected device
# ------------------------------------------------------------

complete_model = complete_model.to(
    DEVICE
)

training_objective = training_objective.to(
    DEVICE
)


# ------------------------------------------------------------
# Optimisation hyperparameters
# ------------------------------------------------------------

MAXIMUM_EPOCHS = 50

INITIAL_LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAXIMUM_GRADIENT_NORM = 5.0

EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 3

SCHEDULER_REDUCTION_FACTOR = 0.5

MINIMUM_LEARNING_RATE = 1e-6

CLASSIFICATION_THRESHOLD = 0.50


# ------------------------------------------------------------
# AdamW optimiser
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    params=complete_model.parameters(),
    lr=INITIAL_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ------------------------------------------------------------
# Validation-AUC learning-rate scheduler
# ------------------------------------------------------------

learning_rate_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="max",
        factor=SCHEDULER_REDUCTION_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)


# ------------------------------------------------------------
# Checkpoint and history directories
# ------------------------------------------------------------

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
)

FOLD_TRAINING_DIR = (
    EXPERIMENT_ROOT
    / SELECTED_TASK
    / f"fold_{SELECTED_FOLD}"
)

CHECKPOINT_DIR = (
    FOLD_TRAINING_DIR
    / "checkpoints"
)

HISTORY_DIR = (
    FOLD_TRAINING_DIR
    / "history"
)

PREDICTION_DIR = (
    FOLD_TRAINING_DIR
    / "predictions"
)


BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_validation_auc_checkpoint.pt"
)

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last_epoch_checkpoint.pt"
)


# ------------------------------------------------------------
# Fold-output safety policy
# ------------------------------------------------------------

existing_fold_files = []

if FOLD_TRAINING_DIR.exists():
    existing_fold_files = [
        path
        for path in FOLD_TRAINING_DIR.rglob("*")
        if path.is_file()
    ]


if FOLD_RUN_MODE == "fresh" and existing_fold_files:
    raise FileExistsError(
        "Fresh training was requested, but this fold directory "
        "already contains files. Nothing was overwritten. "
        "Use a different experiment name, remove the intentionally "
        "discarded fold directory, or select resume mode only for "
        "an interrupted run of this exact fold.\n"
        f"Fold directory: {FOLD_TRAINING_DIR}"
    )


if FOLD_RUN_MODE == "resume" and not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Resume mode was requested, but this fold has no latest "
        "checkpoint. Nothing was changed.\n"
        f"Expected checkpoint: {LAST_CHECKPOINT_PATH}"
    )


for directory_path in [
    EXPERIMENT_ROOT,
    FOLD_TRAINING_DIR,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    PREDICTION_DIR,
]:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

TRAINING_HISTORY_PATH = (
    HISTORY_DIR
    / "training_history.csv"
)

TRAINING_CONFIGURATION_PATH = (
    HISTORY_DIR
    / "training_configuration.json"
)

VALIDATION_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "best_validation_predictions.csv"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "test_predictions.csv"
)


# ------------------------------------------------------------
# Record the training configuration
# ------------------------------------------------------------

training_configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "fixed_equal_fusion": True,
    "three_mt_weight": 0.5,
    "tmc_weight": 0.5,
    "task": SELECTED_TASK,
    "fold": int(SELECTED_FOLD),
    "random_seed": int(GLOBAL_RANDOM_SEED),

    "maximum_epochs": int(
        MAXIMUM_EPOCHS
    ),

    "batch_size": int(
        train_loader.batch_size
    ),

    "initial_learning_rate": float(
        INITIAL_LEARNING_RATE
    ),

    "weight_decay": float(
        WEIGHT_DECAY
    ),

    "maximum_gradient_norm": float(
        MAXIMUM_GRADIENT_NORM
    ),

    "early_stopping_patience": int(
        EARLY_STOPPING_PATIENCE
    ),

    "scheduler_patience": int(
        SCHEDULER_PATIENCE
    ),

    "scheduler_reduction_factor": float(
        SCHEDULER_REDUCTION_FACTOR
    ),

    "minimum_learning_rate": float(
        MINIMUM_LEARNING_RATE
    ),

    "classification_threshold": float(
        CLASSIFICATION_THRESHOLD
    ),

    "modality_dropout_probability": float(
        complete_model.modality_dropout_probability
    ),

    "embedding_dimension": int(
        MODALITY_EMBEDDING_DIM
    ),

    "number_of_classes": int(
        NUMBER_OF_CLASSES
    ),

    "class_order": list(
        PROGNOSIS_CLASS_ORDER
    ),

    "modality_order": list(
        MODALITY_ORDER
    ),

    "three_mt_cascade_order": list(
        THREE_MT_CASCADE_ORDER
    ),

    "trainable_parameters": int(
        count_trainable_parameters(
            complete_model
        )
    ),

    "device": str(
        DEVICE
    ),

    "cuda_device_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}


with open(
    TRAINING_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        training_configuration,
        configuration_file,
        indent=2,
    )


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def calculate_binary_expected_calibration_error(
    targets,
    positive_class_probabilities,
    number_of_bins=10,
):
    """
    Calculate equal-width binary expected calibration error.

    Confidence is the probability assigned to the predicted class.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        return float("nan")

    predicted_classes = (
        positive_class_probabilities
        >= 0.5
    ).astype(
        np.int64
    )

    predicted_confidences = np.where(
        predicted_classes == 1,
        positive_class_probabilities,
        1.0 - positive_class_probabilities,
    )

    prediction_correctness = (
        predicted_classes
        == targets
    ).astype(
        np.float64
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    expected_calibration_error = 0.0

    sample_count = targets.size

    for bin_index in range(
        number_of_bins
    ):

        lower_edge = bin_edges[
            bin_index
        ]

        upper_edge = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:

            in_bin = (
                predicted_confidences
                >= lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        else:

            in_bin = (
                predicted_confidences
                > lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        bin_count = int(
            in_bin.sum()
        )

        if bin_count == 0:
            continue

        mean_confidence = float(
            predicted_confidences[
                in_bin
            ].mean()
        )

        mean_accuracy = float(
            prediction_correctness[
                in_bin
            ].mean()
        )

        expected_calibration_error += (
            bin_count
            / sample_count
        ) * abs(
            mean_accuracy
            - mean_confidence
        )

    return float(
        expected_calibration_error
    )


# ------------------------------------------------------------
# Safe metric helpers
# ------------------------------------------------------------

def safely_calculate_roc_auc(
    targets,
    probabilities,
):
    """
    Return NaN when ROC AUC is undefined because only one class
    is present in the supplied targets.
    """

    if np.unique(targets).size < 2:
        return float("nan")

    return float(
        roc_auc_score(
            targets,
            probabilities,
        )
    )


def safely_calculate_average_precision(
    targets,
    probabilities,
):
    """
    Return NaN when average precision is not meaningful because
    the supplied targets contain no positive examples.
    """

    if np.sum(targets == 1) == 0:
        return float("nan")

    return float(
        average_precision_score(
            targets,
            probabilities,
        )
    )


# ------------------------------------------------------------
# Complete binary prognosis metrics
# ------------------------------------------------------------

def calculate_binary_classification_metrics(
    targets,
    positive_class_probabilities,
    uncertainties=None,
    three_mt_weights=None,
    classification_threshold=0.50,
):
    """
    Calculate discrimination, classification, calibration, and
    uncertainty summaries for the pMCI-positive prognosis task.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        raise ValueError(
            "At least one target is required to calculate metrics."
        )

    if (
        targets.shape[0]
        != positive_class_probabilities.shape[0]
    ):
        raise ValueError(
            "Targets and probabilities must contain the same "
            "number of participants."
        )

    positive_class_probabilities = np.clip(
        positive_class_probabilities,
        0.0,
        1.0,
    )

    predicted_classes = (
        positive_class_probabilities
        >= classification_threshold
    ).astype(
        np.int64
    )

    (
        true_negative,
        false_positive,
        false_negative,
        true_positive,
    ) = confusion_matrix(
        targets,
        predicted_classes,
        labels=[0, 1],
    ).ravel()


    sensitivity_denominator = (
        true_positive
        + false_negative
    )

    specificity_denominator = (
        true_negative
        + false_positive
    )

    sensitivity = (
        true_positive
        / sensitivity_denominator
        if sensitivity_denominator > 0
        else float("nan")
    )

    specificity = (
        true_negative
        / specificity_denominator
        if specificity_denominator > 0
        else float("nan")
    )


    clipped_probabilities = np.clip(
        positive_class_probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    metrics = {
        "roc_auc":
            safely_calculate_roc_auc(
                targets,
                positive_class_probabilities,
            ),

        "average_precision":
            safely_calculate_average_precision(
                targets,
                positive_class_probabilities,
            ),

        "accuracy":
            float(
                accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "precision":
            float(
                precision_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "f1":
            float(
                f1_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "brier_score":
            float(
                brier_score_loss(
                    targets,
                    positive_class_probabilities,
                )
            ),

        "negative_log_likelihood":
            float(
                log_loss(
                    targets,
                    np.column_stack(
                        [
                            1.0
                            - clipped_probabilities,

                            clipped_probabilities,
                        ]
                    ),
                    labels=[0, 1],
                )
            ),

        "expected_calibration_error":
            calculate_binary_expected_calibration_error(
                targets=targets,

                positive_class_probabilities=
                    positive_class_probabilities,

                number_of_bins=10,
            ),

        "classification_threshold":
            float(
                classification_threshold
            ),

        "true_negative":
            int(
                true_negative
            ),

        "false_positive":
            int(
                false_positive
            ),

        "false_negative":
            int(
                false_negative
            ),

        "true_positive":
            int(
                true_positive
            ),
    }


    if uncertainties is not None:

        uncertainties = np.asarray(
            uncertainties,
            dtype=np.float64,
        )

        metrics[
            "mean_uncertainty"
        ] = float(
            uncertainties.mean()
        )

        metrics[
            "std_uncertainty"
        ] = float(
            uncertainties.std()
        )


    if three_mt_weights is not None:

        three_mt_weights = np.asarray(
            three_mt_weights,
            dtype=np.float64,
        )

        metrics[
            "mean_three_mt_weight"
        ] = float(
            three_mt_weights.mean()
        )

        metrics[
            "std_three_mt_weight"
        ] = float(
            three_mt_weights.std()
        )

        metrics[
            "minimum_three_mt_weight"
        ] = float(
            three_mt_weights.min()
        )

        metrics[
            "maximum_three_mt_weight"
        ] = float(
            three_mt_weights.max()
        )


    return metrics

# ------------------------------------------------------------
# Exact fold-0 parameter and target summaries
# ------------------------------------------------------------

model_parameter_count = sum(
    parameter.numel()
    for parameter in complete_model.parameters()
    if parameter.requires_grad
)

optimised_parameter_count = sum(
    parameter.numel()
    for parameter_group in optimizer.param_groups
    for parameter in parameter_group["params"]
    if parameter.requires_grad
)

training_target_counts = (
    train_loader
    .dataset
    .dataframe[target_column]
    .value_counts()
    .sort_index()
)

training_target_summary = pd.DataFrame(
    {
        "CLASS_INDEX": [0, 1],
        "CLASS_NAME": ["sMCI", "pMCI"],
        "TRAINING_COUNT": [
            int(training_target_counts.loc[0]),
            int(training_target_counts.loc[1]),
        ],
    }
)

training_target_summary[
    "TRAINING_PROPORTION"
] = (
    training_target_summary[
        "TRAINING_COUNT"
    ]
    /
    training_target_summary[
        "TRAINING_COUNT"
    ].sum()
)


print("\nOptimiser: AdamW")
print(
    "Initial learning rate: "
    f"{INITIAL_LEARNING_RATE:.6f}"
)
print(
    "Weight decay: "
    f"{WEIGHT_DECAY:.6f}"
)
print(
    "Maximum epochs: "
    f"{MAXIMUM_EPOCHS}"
)
print(
    "Early-stopping patience: "
    f"{EARLY_STOPPING_PATIENCE} epochs"
)
print(
    "Scheduler patience: "
    f"{SCHEDULER_PATIENCE} epochs"
)
print(
    "Checkpoint-selection metric: validation ROC AUC"
)

print(
    "\nExperiment output directory:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "\nBest-checkpoint path:\n"
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    "\nTrainable model parameters: "
    f"{model_parameter_count:,}"
)

print(
    "Parameters included in optimiser: "
    f"{optimised_parameter_count:,}"
)

print(
    "\nFold-0 training target balance:"
)

display(
    training_target_summary.round(6)
)


### 1.5.33. Defining reusable training and validation epoch functions

define the reusable functions that will execute one complete training epoch and one complete validation epoch.

A single training epoch performs the following operations for every mini-batch:

1. move the nested multimodal batch to the selected device;
2. run the complete interaction pathway--evidence pathway forward pass in training mode;
3. apply training-time modality dropout;
4. calculate the joint objective;
5. backpropagate the total loss;
6. clip the global gradient norm;
7. update all model parameters using AdamW;
8. accumulate predictions, uncertainty estimates, gate weights, and losses.

The validation epoch uses the same complete model but differs in three important ways:

- the model is placed in evaluation mode;
- modality dropout and ordinary neural-network dropout are disabled;
- no gradients or parameter updates are calculated.

For both training and validation, pMCI is treated as the positive class. The epoch functions collect:

- participant identifiers;
- binary targets;
- final pMCI probabilities;
- final uncertainty values;
- interaction pathway and modality-specific evidence pathway weights;
- interaction pathway-only pMCI probabilities;
- evidence pathway-only pMCI probabilities;
- the original and effective numbers of available modalities.

The accumulated participant-level outputs are passed to the previously defined metric function after the entire epoch has completed.

### 1.5.34. Gradient clipping

For every training batch, the total gradient norm is calculated and clipped before the optimiser step:

$$
\left\|
\nabla_{\boldsymbol{\theta}}
\mathcal{L}_{\mathrm{total}}
\right\|_2
\leq 5.
$$

The unclipped norm is retained for monitoring.

### 1.5.35. Epoch-level loss aggregation

For loss component \(\ell\), the epoch-level mean is calculated using the number of participants in each mini-batch:

$$
\overline{\mathcal{L}}_{\ell}
=
\frac{
\sum_{b=1}^{B}
n_b
\mathcal{L}_{\ell,b}
}{
\sum_{b=1}^{B}
n_b
},
$$

where \(n_b\) is the batch size.

This avoids giving the final incomplete mini-batch the same weight as a full mini-batch.

The functions defined in this step do not yet train the model across multiple epochs. The full checkpointed training loop is constructed in the following step.

In [ ]:
# ============================================================
# 18. Defining reusable training and validation epoch functions
# ============================================================

from collections import defaultdict


# ------------------------------------------------------------
# Initial numerical-precision policy
# ------------------------------------------------------------

# I initially train in full float32 precision.
#
# This is computationally feasible on the available A100 GPU and
# avoids introducing mixed-precision instability into the Dirichlet
# digamma, log-gamma, and Dempster-Shafer calculations.
USE_MIXED_PRECISION = False


# ------------------------------------------------------------
# Recursively move a nested batch to the selected device
# ------------------------------------------------------------

def move_nested_batch_to_device(
    value,
    device,
):
    """
    Recursively move tensors in dictionaries, lists, and tuples
    to the selected PyTorch device.

    Non-tensor values are preserved unchanged.
    """

    if isinstance(
        value,
        torch.Tensor,
    ):
        return value.to(
            device,
            non_blocking=True,
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: move_nested_batch_to_device(
                nested_value,
                device,
            )
            for key, nested_value in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        )

    return value


# ------------------------------------------------------------
# Convert one completed forward pass into stored predictions
# ------------------------------------------------------------

def extract_batch_prediction_arrays(
    batch,
    model_output,
):
    """
    Extract participant-level targets, predictions, uncertainty,
    pathway outputs, and modality counts from one mini-batch.
    """

    final_output = model_output[
        "final_output"
    ]

    joint_opinion = model_output[
        "joint_opinion"
    ]

    tmc_output = model_output[
        "tmc_output"
    ]


    extracted = {
        "rid":
            batch["rid"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "target":
            batch["target"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_p_pMCI":
            final_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_uncertainty":
            final_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_weight":
            final_output[
                "three_mt_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_weight":
            final_output[
                "tmc_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_p_pMCI":
            joint_opinion[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_uncertainty":
            joint_opinion[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_p_pMCI":
            tmc_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_uncertainty":
            tmc_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "original_modality_count":
            model_output[
                "original_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "effective_modality_count":
            model_output[
                "effective_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),
    }

    return extracted


# ------------------------------------------------------------
# Concatenate the participant outputs collected across an epoch
# ------------------------------------------------------------

def concatenate_epoch_prediction_storage(
    prediction_storage,
):
    """
    Concatenate a dictionary of mini-batch NumPy arrays.
    """

    concatenated = {}

    for key, value_list in prediction_storage.items():

        if len(value_list) == 0:
            concatenated[key] = np.asarray([])

        else:
            concatenated[key] = np.concatenate(
                value_list,
                axis=0,
            )

    return concatenated


# ------------------------------------------------------------
# Convert stored predictions into a participant-level table
# ------------------------------------------------------------

def build_epoch_prediction_table(
    concatenated_predictions,
    split_name,
    epoch,
):
    """
    Build one participant-level DataFrame for an epoch.
    """

    prediction_table = pd.DataFrame(
        {
            "RID":
                concatenated_predictions[
                    "rid"
                ].astype(
                    np.int64
                ),

            "TARGET":
                concatenated_predictions[
                    "target"
                ].astype(
                    np.int64
                ),

            "FINAL_P_pMCI":
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            "FINAL_UNCERTAINTY":
                concatenated_predictions[
                    "final_uncertainty"
                ],

            "W_3MT":
                concatenated_predictions[
                    "three_mt_weight"
                ],

            "W_TMC":
                concatenated_predictions[
                    "tmc_weight"
                ],

            "THREE_MT_P_pMCI":
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            "THREE_MT_UNCERTAINTY":
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            "TMC_P_pMCI":
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            "TMC_UNCERTAINTY":
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            "ORIGINAL_MODALITY_COUNT":
                concatenated_predictions[
                    "original_modality_count"
                ].astype(
                    np.int64
                ),

            "EFFECTIVE_MODALITY_COUNT":
                concatenated_predictions[
                    "effective_modality_count"
                ].astype(
                    np.int64
                ),
        }
    )

    prediction_table.insert(
        loc=0,
        column="EPOCH",
        value=int(epoch),
    )

    prediction_table.insert(
        loc=1,
        column="SPLIT",
        value=str(split_name),
    )

    prediction_table[
        "FINAL_PREDICTED_CLASS"
    ] = (
        prediction_table[
            "FINAL_P_pMCI"
        ]
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        np.int64
    )

    return prediction_table


# ------------------------------------------------------------
# Calculate metrics for all three prediction outputs
# ------------------------------------------------------------

def calculate_epoch_prediction_metrics(
    concatenated_predictions,
):
    """
    Calculate metrics for:

    1. the final fixed-equal hybrid output;
    2. the 3MT-only joint output;
    3. the TMC-only fused output.
    """

    targets = concatenated_predictions[
        "target"
    ]

    final_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "final_uncertainty"
                ],

            three_mt_weights=
                concatenated_predictions[
                    "three_mt_weight"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    three_mt_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    tmc_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    return {
        "final": final_metrics,
        "three_mt": three_mt_metrics,
        "tmc": tmc_metrics,
    }


# ------------------------------------------------------------
# Initialise the loss accumulator used within an epoch
# ------------------------------------------------------------

EPOCH_LOSS_NAMES = [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]


def initialise_epoch_loss_storage():
    """
    Create participant-weighted loss totals.
    """

    return {
        loss_name: 0.0
        for loss_name in EPOCH_LOSS_NAMES
    }


def update_epoch_loss_storage(
    storage,
    loss_output,
    batch_size,
):
    """
    Add one batch's losses, weighted by the number of participants.
    """

    for loss_name in EPOCH_LOSS_NAMES:

        storage[loss_name] += (
            float(
                loss_output[
                    loss_name
                ]
                .detach()
                .float()
                .item()
            )
            * batch_size
        )


def finalise_epoch_loss_storage(
    storage,
    participant_count,
):
    """
    Convert accumulated loss sums into participant-weighted means.
    """

    if participant_count <= 0:
        raise ValueError(
            "The epoch contained no participants."
        )

    return {
        loss_name:
            loss_sum
            / participant_count

        for loss_name, loss_sum in storage.items()
    }


# ------------------------------------------------------------
# Run one complete training epoch
# ------------------------------------------------------------

def run_training_epoch(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_gradient_norm,
):
    """
    Train the complete model for one epoch.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []


    for batch_index, batch in enumerate(
        data_loader
    ):

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = targets.shape[0]

        participant_count += batch_size


        # --------------------------------------------------------
        # Clear gradients from the previous mini-batch
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # --------------------------------------------------------
        # Complete forward pass
        # --------------------------------------------------------

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )


        # --------------------------------------------------------
        # Complete multi-output objective
        # --------------------------------------------------------

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )


        # --------------------------------------------------------
        # Backpropagation
        # --------------------------------------------------------

        total_loss.backward()


        # --------------------------------------------------------
        # Global gradient clipping
        # --------------------------------------------------------

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )


        # --------------------------------------------------------
        # Parameter update
        # --------------------------------------------------------

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate losses and predictions
        # --------------------------------------------------------

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[key].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


    # ------------------------------------------------------------
    # Finalise the complete training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# Run one complete validation epoch
# ------------------------------------------------------------

def run_validation_epoch(
    model,
    data_loader,
    objective,
    device,
    epoch,
):
    """
    Evaluate the complete model for one epoch without gradients
    or training-time modality dropout.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    with torch.no_grad():

        for batch_index, batch in enumerate(
            data_loader
        ):

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = targets.shape[0]

            participant_count += batch_size


            # ----------------------------------------------------
            # Complete evaluation forward pass
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ----------------------------------------------------
            # Validation objective
            # ----------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_index}."
                )


            # ----------------------------------------------------
            # Accumulate losses and predictions
            # ----------------------------------------------------

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[key].append(
                    values
                )


    # ------------------------------------------------------------
    # Finalise the complete validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Display the configured epoch-function summary
# ------------------------------------------------------------

print("=" * 72)
print("TRAINING AND VALIDATION EPOCH FUNCTIONS")
print("=" * 72)

print(
    f"\nMixed precision enabled: "
    f"{USE_MIXED_PRECISION}"
)

print(
    "Training batches per epoch: "
    f"{len(train_loader)}"
)

print(
    "Validation batches per epoch: "
    f"{len(validation_loader)}"
)

print(
    "Training participants: "
    f"{len(train_loader.dataset)}"
)

print(
    "Validation participants: "
    f"{len(validation_loader.dataset)}"
)

print(
    "\nTraining epoch operations:"
)

print(
    "- forward pass with modality dropout;"
)

print(
    "- complete joint loss calculation;"
)

print(
    "- backpropagation;"
)

print(
    "- global gradient clipping;"
)

print(
    "- AdamW parameter update;"
)

print(
    "- prediction and uncertainty accumulation."
)

print(
    "\nValidation epoch operations:"
)

print(
    "- evaluation mode;"
)

print(
    "- no modality dropout;"
)

print(
    "- no gradient calculation;"
)

print(
    "- complete validation loss and metric calculation."
)

print(
    "\nThe epoch functions are defined."
)

print(
    "No complete training or validation epoch has been run yet."
)

print(
    "The next step will create the checkpointed multi-epoch "
    "training loop and begin model optimisation."
)

### 1.5.36. Training a newly initialised model with validation-based checkpointing

This standalone experiment starts from epoch 1 using the model initialised in this notebook. It does not load a checkpoint from the previous ungated baseline or from an earlier gated run.

The validation partition controls learning-rate reduction, early stopping, and best-checkpoint selection. The test partition remains untouched during training.

In [ ]:
# ============================================================
# 19. Training with live batch and epoch progress
# ============================================================

import time
import traceback
from collections import defaultdict
from datetime import datetime

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Resume and progress-display behaviour
# ------------------------------------------------------------

# The formal first run begins from epoch 1.
# Resume is enabled only when FOLD_RUN_MODE was explicitly set
# to "resume" for this exact experiment and fold.
RESUME_FROM_LAST_CHECKPOINT = (
    FOLD_RUN_MODE == "resume"
)

# I refresh the live progress display after every batch.
PROGRESS_UPDATE_INTERVAL = 1


# ------------------------------------------------------------
# Loss-configuration serialisation
# ------------------------------------------------------------

def obtain_training_objective_configuration(
    objective,
):
    """
    Return the principal loss settings in a checkpoint-safe form.
    """

    return {
        "evidential_annealing_epochs": int(
            objective.evidential_annealing_epochs
        ),

        "gate_regularisation_epochs": int(
            objective.gate_regularisation_epochs
        ),

        "joint_loss_weight": float(
            objective.joint_loss_weight
        ),

        "tmc_loss_weight": float(
            objective.tmc_loss_weight
        ),

        "modality_loss_weight": float(
            objective.modality_loss_weight
        ),

        "auxiliary_loss_weight": float(
            objective.auxiliary_loss_weight
        ),

        "gate_loss_weight": float(
            objective.gate_loss_weight
        ),

        "class_weights": (
            None
            if objective.class_weights is None
            else
            objective.class_weights
            .detach()
            .cpu()
            .tolist()
        ),
    }


# ------------------------------------------------------------
# Flatten one epoch into one history row
# ------------------------------------------------------------

def build_training_history_row(
    epoch,
    learning_rate,
    epoch_duration_seconds,
    training_result,
    validation_result,
    best_validation_auc,
    epochs_without_improvement,
    checkpoint_improved,
):
    """
    Create one flat row containing losses, metrics, optimisation
    diagnostics, and checkpoint information.
    """

    history_row = {
        "epoch":
            int(epoch),

        "learning_rate":
            float(learning_rate),

        "epoch_duration_seconds":
            float(epoch_duration_seconds),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "checkpoint_improved":
            bool(checkpoint_improved),

        "train_participants":
            int(
                training_result[
                    "participant_count"
                ]
            ),

        "validation_participants":
            int(
                validation_result[
                    "participant_count"
                ]
            ),

        "train_batches":
            int(
                training_result[
                    "batch_count"
                ]
            ),

        "validation_batches":
            int(
                validation_result[
                    "batch_count"
                ]
            ),

        "train_mean_gradient_norm":
            float(
                training_result[
                    "gradient_summary"
                ]["mean_gradient_norm"]
            ),

        "train_maximum_gradient_norm_before_clipping":
            float(
                training_result[
                    "gradient_summary"
                ][
                    "maximum_gradient_norm_before_clipping"
                ]
            ),

        "train_mean_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "mean_effective_modality_count"
                ]
            ),

        "train_minimum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "minimum_effective_modality_count"
                ]
            ),

        "train_maximum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "maximum_effective_modality_count"
                ]
            ),

        "validation_maximum_modality_count_difference":
            float(
                validation_result[
                    "maximum_evaluation_modality_count_difference"
                ]
            ),
    }


    # --------------------------------------------------------
    # Add all training and validation losses
    # --------------------------------------------------------

    for loss_name, loss_value in (
        training_result[
            "losses"
        ].items()
    ):
        history_row[
            f"train_{loss_name}"
        ] = float(
            loss_value
        )

    for loss_name, loss_value in (
        validation_result[
            "losses"
        ].items()
    ):
        history_row[
            f"validation_{loss_name}"
        ] = float(
            loss_value
        )


    # --------------------------------------------------------
    # Add metrics for all three prediction outputs
    # --------------------------------------------------------

    for output_name in [
        "final",
        "three_mt",
        "tmc",
    ]:

        for metric_name, metric_value in (
            training_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"train_{output_name}_{metric_name}"
            ] = metric_value

        for metric_name, metric_value in (
            validation_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"validation_{output_name}_{metric_name}"
            ] = metric_value


    return history_row


# ------------------------------------------------------------
# Save a fully recoverable checkpoint
# ------------------------------------------------------------

def save_training_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_validation_auc,
    epochs_without_improvement,
    validation_metrics,
    training_history,
):
    """
    Save the state required to reproduce or continue training.
    """

    checkpoint = {
        "epoch":
            int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "validation_metrics":
            validation_metrics,

        "training_configuration":
            training_configuration,

        "training_objective_configuration":
            obtain_training_objective_configuration(
                training_objective
            ),

        "training_history":
            training_history,

        "random_seed":
            int(GLOBAL_RANDOM_SEED),

        "task":
            SELECTED_TASK,

        "fold":
            int(SELECTED_FOLD),

        "saved_at":
            datetime.now().isoformat(
                timespec="seconds"
            ),
    }

    torch.save(
        checkpoint,
        checkpoint_path,
    )


# ------------------------------------------------------------
# GPU-memory helper for the progress display
# ------------------------------------------------------------

def current_cuda_memory_gb():
    """
    Return currently allocated CUDA memory in gigabytes.
    """

    if not torch.cuda.is_available():
        return 0.0

    return float(
        torch.cuda.memory_allocated(
            DEVICE
        )
        / (1024 ** 3)
    )


# ------------------------------------------------------------
# One training epoch with a live batch progress bar
# ------------------------------------------------------------

def run_training_epoch_with_progress(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_epochs,
    maximum_gradient_norm,
):
    """
    Train for one epoch while displaying batch-level progress.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | training"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    for batch_number, batch in progress_bar:

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = int(
            targets.shape[0]
        )

        participant_count += (
            batch_size
        )


        # --------------------------------------------------------
        # Forward pass and loss
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )


        # --------------------------------------------------------
        # Backpropagation, clipping, and update
        # --------------------------------------------------------

        total_loss.backward()

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate diagnostics
        # --------------------------------------------------------

        current_total_loss = float(
            total_loss.detach().item()
        )

        running_total_loss_sum += (
            current_total_loss
            * batch_size
        )

        running_mean_total_loss = (
            running_total_loss_sum
            / participant_count
        )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[
                key
            ].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


        # --------------------------------------------------------
        # Update the visible progress information
        # --------------------------------------------------------

        if (
            batch_number
            % PROGRESS_UPDATE_INTERVAL
            == 0
            or batch_number
            == len(data_loader)
        ):

            elapsed_minutes = (
                time.time()
                - phase_start_time
            ) / 60.0

            progress_bar.set_postfix(
                {
                    "loss":
                        f"{current_total_loss:.3f}",

                    "avg":
                        f"{running_mean_total_loss:.3f}",

                    "grad":
                        f"{float(gradient_norm):.2f}",

                    "lr":
                        f"{optimizer.param_groups[0]['lr']:.1e}",

                    "GPU":
                        f"{current_cuda_memory_gb():.1f}GB",

                    "elapsed":
                        f"{elapsed_minutes:.1f}m",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise the training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# One validation epoch with a live batch progress bar
# ------------------------------------------------------------

def run_validation_epoch_with_progress(
    model,
    data_loader,
    objective,
    device,
    epoch,
    maximum_epochs,
):
    """
    Validate for one epoch while displaying batch-level progress.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | validation"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ----------------------------------------------------
            # Forward pass and validation loss
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            total_loss = loss_output[
                "total_loss"
            ]


            if not torch.isfinite(
                total_loss
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_number}."
                )


            # ----------------------------------------------------
            # Accumulate diagnostics and predictions
            # ----------------------------------------------------

            current_total_loss = float(
                total_loss.detach().item()
            )

            running_total_loss_sum += (
                current_total_loss
                * batch_size
            )

            running_mean_total_loss = (
                running_total_loss_sum
                / participant_count
            )

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[
                    key
                ].append(
                    values
                )


            # ----------------------------------------------------
            # Update the visible progress information
            # ----------------------------------------------------

            if (
                batch_number
                % PROGRESS_UPDATE_INTERVAL
                == 0
                or batch_number
                == len(data_loader)
            ):

                elapsed_minutes = (
                    time.time()
                    - phase_start_time
                ) / 60.0

                progress_bar.set_postfix(
                    {
                        "loss":
                            f"{current_total_loss:.3f}",

                        "avg":
                            f"{running_mean_total_loss:.3f}",

                        "GPU":
                            f"{current_cuda_memory_gb():.1f}GB",

                        "elapsed":
                            f"{elapsed_minutes:.1f}m",
                    },
                    refresh=True,
                )


    # ------------------------------------------------------------
    # Finalise the validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                concatenated_predictions[
                    "original_modality_count"
                ]
                -
                concatenated_predictions[
                    "effective_modality_count"
                ]
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Initial training state
# ------------------------------------------------------------

training_history = []

starting_epoch = 1

best_validation_auc = float(
    "-inf"
)

epochs_without_improvement = 0


# ------------------------------------------------------------
# Resume from the latest fully completed epoch
# ------------------------------------------------------------

if (
    RESUME_FROM_LAST_CHECKPOINT
    and LAST_CHECKPOINT_PATH.exists()
):

    print(
        "Loading the latest fully completed checkpoint:",
        flush=True,
    )

    print(
        LAST_CHECKPOINT_PATH,
        flush=True,
    )

    resumed_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    complete_model.load_state_dict(
        resumed_checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        resumed_checkpoint[
            "optimizer_state_dict"
        ]
    )

    learning_rate_scheduler.load_state_dict(
        resumed_checkpoint[
            "scheduler_state_dict"
        ]
    )

    completed_epoch = int(
        resumed_checkpoint[
            "epoch"
        ]
    )

    starting_epoch = (
        completed_epoch
        + 1
    )

    best_validation_auc = float(
        resumed_checkpoint[
            "best_validation_auc"
        ]
    )

    epochs_without_improvement = int(
        resumed_checkpoint[
            "epochs_without_improvement"
        ]
    )

    training_history = list(
        resumed_checkpoint.get(
            "training_history",
            [],
        )
    )

    print(
        f"\nResuming after epoch {completed_epoch}.",
        flush=True,
    )

    print(
        f"Next epoch: {starting_epoch}",
        flush=True,
    )

    print(
        "Best validation ROC AUC so far: "
        f"{best_validation_auc:.6f}",
        flush=True,
    )

else:

    print(
        "No fully completed checkpoint was loaded.",
        flush=True,
    )

    print(
        "Fresh training begins from the newly initialised model state.",
        flush=True,
    )


# ------------------------------------------------------------
# Main multi-epoch training loop
# ------------------------------------------------------------

if starting_epoch > MAXIMUM_EPOCHS:

    print(
        "\nTraining has already reached the configured maximum "
        f"of {MAXIMUM_EPOCHS} epochs.",
        flush=True,
    )

else:

    print("\n" + "=" * 72, flush=True)
    print("BEGINNING MODEL TRAINING", flush=True)
    print("=" * 72, flush=True)

    print(
        f"\nEpoch range: {starting_epoch}--{MAXIMUM_EPOCHS}",
        flush=True,
    )

    print(
        f"Training batches per epoch: {len(train_loader)}",
        flush=True,
    )

    print(
        f"Validation batches per epoch: {len(validation_loader)}",
        flush=True,
    )

    print(
        "Each epoch displays separate live training and "
        "validation progress bars.",
        flush=True,
    )

    print(
        "The test partition will not be evaluated.",
        flush=True,
    )


    try:

        for epoch in range(
            starting_epoch,
            MAXIMUM_EPOCHS + 1,
        ):

            epoch_start_time = time.time()

            current_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )


            print("\n" + "=" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d}/{MAXIMUM_EPOCHS}",
                flush=True,
            )

            print("=" * 72, flush=True)

            print(
                "\nPhase 1/4: training batches",
                flush=True,
            )


            # ------------------------------------------------
            # Train on all training participants
            # ------------------------------------------------

            training_result = (
                run_training_epoch_with_progress(
                    model=complete_model,
                    data_loader=train_loader,
                    objective=training_objective,
                    optimizer=optimizer,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                    maximum_gradient_norm=
                        MAXIMUM_GRADIENT_NORM,
                )
            )


            print(
                "\nPhase 2/4: validation batches",
                flush=True,
            )


            # ------------------------------------------------
            # Validate on all validation participants
            # ------------------------------------------------

            validation_result = (
                run_validation_epoch_with_progress(
                    model=complete_model,
                    data_loader=validation_loader,
                    objective=training_objective,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                )
            )


            print(
                "\nPhase 3/4: calculating metrics and "
                "updating the scheduler",
                flush=True,
            )


            # ------------------------------------------------
            # Validation AUC and checkpoint decision
            # ------------------------------------------------

            validation_auc = float(
                validation_result[
                    "metrics"
                ]["final"]["roc_auc"]
            )

            validation_auc_is_valid = bool(
                np.isfinite(
                    validation_auc
                )
            )

            checkpoint_improved = (
                validation_auc_is_valid
                and
                validation_auc
                > best_validation_auc
            )

            if checkpoint_improved:

                best_validation_auc = (
                    validation_auc
                )

                epochs_without_improvement = 0

            else:

                epochs_without_improvement += 1


            scheduler_score = (
                validation_auc
                if validation_auc_is_valid
                else -1.0
            )

            learning_rate_scheduler.step(
                scheduler_score
            )

            updated_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            epoch_duration_seconds = (
                time.time()
                - epoch_start_time
            )


            # ------------------------------------------------
            # Persistent training history
            # ------------------------------------------------

            history_row = build_training_history_row(
                epoch=epoch,

                learning_rate=
                    current_learning_rate,

                epoch_duration_seconds=
                    epoch_duration_seconds,

                training_result=
                    training_result,

                validation_result=
                    validation_result,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                checkpoint_improved=
                    checkpoint_improved,
            )

            history_row[
                "learning_rate_after_scheduler"
            ] = updated_learning_rate

            training_history.append(
                history_row
            )

            pd.DataFrame(
                training_history
            ).to_csv(
                TRAINING_HISTORY_PATH,
                index=False,
            )


            print(
                "\nPhase 4/4: saving checkpoints and history",
                flush=True,
            )


            # ------------------------------------------------
            # Save the latest completed epoch
            # ------------------------------------------------

            save_training_checkpoint(
                checkpoint_path=
                    LAST_CHECKPOINT_PATH,

                epoch=epoch,

                model=complete_model,

                optimizer=optimizer,

                scheduler=
                    learning_rate_scheduler,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                validation_metrics=
                    validation_result[
                        "metrics"
                    ],

                training_history=
                    training_history,
            )


            # ------------------------------------------------
            # Save the best validation checkpoint
            # ------------------------------------------------

            if checkpoint_improved:

                save_training_checkpoint(
                    checkpoint_path=
                        BEST_CHECKPOINT_PATH,

                    epoch=epoch,

                    model=complete_model,

                    optimizer=optimizer,

                    scheduler=
                        learning_rate_scheduler,

                    best_validation_auc=
                        best_validation_auc,

                    epochs_without_improvement=
                        epochs_without_improvement,

                    validation_metrics=
                        validation_result[
                            "metrics"
                        ],

                    training_history=
                        training_history,
                )

                validation_result[
                    "predictions"
                ].to_csv(
                    VALIDATION_PREDICTIONS_PATH,
                    index=False,
                )


            # ------------------------------------------------
            # Readable completed-epoch summary
            # ------------------------------------------------

            train_metrics = training_result[
                "metrics"
            ]["final"]

            validation_metrics = validation_result[
                "metrics"
            ]["final"]


            print("\n" + "-" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d} COMPLETE",
                flush=True,
            )

            print(
                "Duration: "
                f"{epoch_duration_seconds / 60.0:.2f} minutes",
                flush=True,
            )

            print(
                "Learning rate: "
                f"{current_learning_rate:.8f}"
                f" -> {updated_learning_rate:.8f}",
                flush=True,
            )


            print("\nTraining:", flush=True)

            print(
                "  total loss: "
                f"{training_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{train_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{train_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{train_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{train_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )

            print(
                "  mean effective modalities: "
                f"{training_result['modality_dropout_summary']['mean_effective_modality_count']:.3f}",
                flush=True,
            )

            print(
                "  maximum pre-clipping gradient norm: "
                f"{training_result['gradient_summary']['maximum_gradient_norm_before_clipping']:.6f}",
                flush=True,
            )


            print("\nValidation:", flush=True)

            print(
                "  total loss: "
                f"{validation_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{validation_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  average precision: "
                f"{validation_metrics['average_precision']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{validation_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  sensitivity: "
                f"{validation_metrics['sensitivity']:.6f}",
                flush=True,
            )

            print(
                "  specificity: "
                f"{validation_metrics['specificity']:.6f}",
                flush=True,
            )

            print(
                "  Brier score: "
                f"{validation_metrics['brier_score']:.6f}",
                flush=True,
            )

            print(
                "  calibration error: "
                f"{validation_metrics['expected_calibration_error']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{validation_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{validation_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )


            print("\nCheckpoint status:", flush=True)

            print(
                "  improved this epoch: "
                f"{checkpoint_improved}",
                flush=True,
            )

            print(
                "  best validation ROC AUC: "
                f"{best_validation_auc:.6f}",
                flush=True,
            )

            print(
                "  epochs without improvement: "
                f"{epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}",
                flush=True,
            )

            print(
                "  latest completed epoch saved: True",
                flush=True,
            )

            if torch.cuda.is_available():

                print(
                    "  peak CUDA memory: "
                    f"{torch.cuda.max_memory_allocated(0) / (1024 ** 3):.3f} GB",
                    flush=True,
                )


            # ------------------------------------------------
            # Early stopping
            # ------------------------------------------------

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):

                print(
                    "\nEarly stopping activated because "
                    "validation ROC AUC did not improve for "
                    f"{EARLY_STOPPING_PATIENCE} consecutive epochs.",
                    flush=True,
                )

                break


    # --------------------------------------------------------
    # Interruption and error handling
    # --------------------------------------------------------

    except KeyboardInterrupt:

        print(
            "\nTraining was interrupted manually.",
            flush=True,
        )

        print(
            "Only fully completed epochs are recoverable from "
            "the last-epoch checkpoint.",
            flush=True,
        )


    except Exception:

        print(
            "\nTraining stopped because an exception occurred.",
            flush=True,
        )

        print(
            "The latest fully completed epoch remains saved.",
            flush=True,
        )

        traceback.print_exc()

        raise


    # --------------------------------------------------------
    # Final training summary
    # --------------------------------------------------------

    if len(
        training_history
    ) > 0:

        final_history_table = pd.DataFrame(
            training_history
        )

        completed_epochs = int(
            final_history_table[
                "epoch"
            ].max()
        )

        print("\n" + "=" * 72, flush=True)

        print(
            "TRAINING RUN COMPLETE",
            flush=True,
        )

        print("=" * 72, flush=True)

        print(
            f"\nLast completed epoch: {completed_epochs}",
            flush=True,
        )

        print(
            "Best validation ROC AUC: "
            f"{best_validation_auc:.6f}",
            flush=True,
        )

        print(
            "\nBest checkpoint:\n"
            f"{BEST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nLatest checkpoint:\n"
            f"{LAST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nTraining history:\n"
            f"{TRAINING_HISTORY_PATH}",
            flush=True,
        )

        print(
            "\nThe test partition has not been evaluated.",
            flush=True,
        )

### 1.5.37. Evaluating the best gated checkpoint on the untouched test set

After training and validation-based checkpoint selection, the best checkpoint from this named experiment is loaded from its experiment-specific fold directory and evaluated once on the held-out test partition.

The evaluation cell does not update model parameters, the optimiser, the scheduler, or modality-dropout state.

In [ ]:
from datetime import datetime


# ------------------------------------------------------------
# Test-output paths
# ------------------------------------------------------------

TEST_METRICS_PATH = (
    PREDICTION_DIR
    / "test_metrics.json"
)

TEST_SUMMARY_PATH = (
    PREDICTION_DIR
    / "test_metrics_summary.csv"
)


print("=" * 72)
print("FIXED-EQUAL-FUSION TEST-SET EVALUATION")
print("=" * 72)

print(
    "\nLoading the best validation checkpoint:\n"
    f"{BEST_CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# Load the best validation checkpoint
# ------------------------------------------------------------


best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_checkpoint_epoch = int(
    best_checkpoint[
        "epoch"
    ]
)

best_checkpoint_validation_auc = float(
    best_checkpoint[
        "best_validation_auc"
    ]
)


complete_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


print(
    f"\nBest checkpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Best validation ROC AUC stored in checkpoint: "
    f"{best_checkpoint_validation_auc:.6f}"
)


# ------------------------------------------------------------
# Test evaluation function
# ------------------------------------------------------------

def run_test_epoch(
    model,
    data_loader,
    objective,
    device,
    checkpoint_epoch,
):
    """
    Evaluate the frozen model on the untouched test partition.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc="Test evaluation",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ------------------------------------------------
            # Frozen forward pass
            # ------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ------------------------------------------------
            # Test loss
            # ------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=checkpoint_epoch,
            )


            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):

                raise FloatingPointError(
                    "A non-finite test loss was encountered "
                    f"at batch {batch_number}."
                )


            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )


            # ------------------------------------------------
            # Store participant-level outputs
            # ------------------------------------------------

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):

                prediction_storage[
                    key
                ].append(
                    values
                )


            running_average_loss = (
                loss_storage[
                    "total_loss"
                ]
                / participant_count
            )

            progress_bar.set_postfix(
                {
                    "avg_loss":
                        f"{running_average_loss:.3f}",

                    "participants":
                        f"{participant_count}/{len(data_loader.dataset)}",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise test losses and predictions
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="test",

            epoch=checkpoint_epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_modality_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_modality_count_difference,
    }


# ------------------------------------------------------------
# Run the untouched test evaluation
# ------------------------------------------------------------

test_result = run_test_epoch(
    model=complete_model,
    data_loader=test_loader,
    objective=training_objective,
    device=DEVICE,
    checkpoint_epoch=best_checkpoint_epoch,
)


# ------------------------------------------------------------
# Evaluation-mode modality count
# ------------------------------------------------------------

maximum_test_mask_difference = (
    test_result[
        "maximum_evaluation_modality_count_difference"
    ]
)


# ------------------------------------------------------------
# Save participant-level test predictions
# ------------------------------------------------------------


test_prediction_table = test_result[
    "predictions"
]

test_prediction_table.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save all test metrics in JSON format
# ------------------------------------------------------------

serialisable_test_result = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "task":
        SELECTED_TASK,

    "fold":
        int(
            SELECTED_FOLD
        ),

    "checkpoint_epoch":
        int(
            best_checkpoint_epoch
        ),

    "checkpoint_validation_auc":
        float(
            best_checkpoint_validation_auc
        ),

    "classification_threshold":
        float(
            CLASSIFICATION_THRESHOLD
        ),

    "test_participants":
        int(
            test_result[
                "participant_count"
            ]
        ),

    "test_batches":
        int(
            test_result[
                "batch_count"
            ]
        ),

    "test_losses": {
        key:
            float(value)

        for key, value in (
            test_result[
                "losses"
            ].items()
        )
    },

    "test_metrics":
        test_result[
            "metrics"
        ],

    "maximum_modality_count_difference":
        float(
            maximum_test_mask_difference
        ),

    "evaluated_at":
        datetime.now().isoformat(
            timespec="seconds"
        ),
}


with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as test_metrics_file:

    json.dump(
        serialisable_test_result,
        test_metrics_file,
        indent=2,
    )


# ------------------------------------------------------------
# Build a compact comparison table
# ------------------------------------------------------------

test_metric_rows = []

for output_name, display_name in [
    (
        "final",
        "Hybrid",
    ),
    (
        "three_mt",
        "3MT-only",
    ),
    (
        "tmc",
        "TMC-only",
    ),
]:

    output_metrics = test_result[
        "metrics"
    ][
        output_name
    ]

    test_metric_rows.append(
        {
            "OUTPUT":
                display_name,

            "ROC_AUC":
                output_metrics[
                    "roc_auc"
                ],

            "AVERAGE_PRECISION":
                output_metrics[
                    "average_precision"
                ],

            "ACCURACY":
                output_metrics[
                    "accuracy"
                ],

            "BALANCED_ACCURACY":
                output_metrics[
                    "balanced_accuracy"
                ],

            "SENSITIVITY":
                output_metrics[
                    "sensitivity"
                ],

            "SPECIFICITY":
                output_metrics[
                    "specificity"
                ],

            "PRECISION":
                output_metrics[
                    "precision"
                ],

            "F1":
                output_metrics[
                    "f1"
                ],

            "BRIER_SCORE":
                output_metrics[
                    "brier_score"
                ],

            "NEGATIVE_LOG_LIKELIHOOD":
                output_metrics[
                    "negative_log_likelihood"
                ],

            "EXPECTED_CALIBRATION_ERROR":
                output_metrics[
                    "expected_calibration_error"
                ],

            "MEAN_UNCERTAINTY":
                output_metrics[
                    "mean_uncertainty"
                ],
        }
    )


test_metrics_summary = pd.DataFrame(
    test_metric_rows
)

test_metrics_summary.to_csv(
    TEST_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display the final test results
# ------------------------------------------------------------

hybrid_test_metrics = test_result[
    "metrics"
][
    "final"
]


print("\n" + "=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} TEST RESULTS")
print("=" * 72)

print(
    f"\nCheckpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Test participants: "
    f"{test_result['participant_count']}"
)

print(
    "Test batches: "
    f"{test_result['batch_count']}"
)

print(
    "Maximum difference between original and effective "
    "test modality counts: "
    f"{maximum_test_mask_difference:.1f}"
)


print(
    "\nFinal hybrid test performance:"
)

print(
    "  ROC AUC: "
    f"{hybrid_test_metrics['roc_auc']:.6f}"
)

print(
    "  average precision: "
    f"{hybrid_test_metrics['average_precision']:.6f}"
)

print(
    "  accuracy: "
    f"{hybrid_test_metrics['accuracy']:.6f}"
)

print(
    "  balanced accuracy: "
    f"{hybrid_test_metrics['balanced_accuracy']:.6f}"
)

print(
    "  sensitivity: "
    f"{hybrid_test_metrics['sensitivity']:.6f}"
)

print(
    "  specificity: "
    f"{hybrid_test_metrics['specificity']:.6f}"
)

print(
    "  precision: "
    f"{hybrid_test_metrics['precision']:.6f}"
)

print(
    "  F1 score: "
    f"{hybrid_test_metrics['f1']:.6f}"
)

print(
    "  Brier score: "
    f"{hybrid_test_metrics['brier_score']:.6f}"
)

print(
    "  negative log-likelihood: "
    f"{hybrid_test_metrics['negative_log_likelihood']:.6f}"
)

print(
    "  expected calibration error: "
    f"{hybrid_test_metrics['expected_calibration_error']:.6f}"
)

print(
    "  mean uncertainty: "
    f"{hybrid_test_metrics['mean_uncertainty']:.6f}"
)

print(
    "  mean 3MT weight: "
    f"{hybrid_test_metrics['mean_three_mt_weight']:.6f}"
)

print(
    "  standard deviation of 3MT weight: "
    f"{hybrid_test_metrics['std_three_mt_weight']:.6f}"
)


print(
    "\nConfusion matrix counts:"
)

print(
    "  true negatives: "
    f"{hybrid_test_metrics['true_negative']}"
)

print(
    "  false positives: "
    f"{hybrid_test_metrics['false_positive']}"
)

print(
    "  false negatives: "
    f"{hybrid_test_metrics['false_negative']}"
)

print(
    "  true positives: "
    f"{hybrid_test_metrics['true_positive']}"
)


print(
    "\nHybrid, 3MT-only, and TMC-only comparison:"
)

display(
    test_metrics_summary.round(
        6
    )
)


print(
    "\nParticipant-level predictions saved to:\n"
    f"{TEST_PREDICTIONS_PATH}"
)

print(
    "\nComplete test metrics saved to:\n"
    f"{TEST_METRICS_PATH}"
)

print(
    "\nCompact metric summary saved to:\n"
    f"{TEST_SUMMARY_PATH}"
)

print(
    "\nThe model was evaluated without gradient updates, "
    "scheduler changes, or test-time modality dropout."
)

### 1.5.38. Releasing fold-specific memory

The fold-1 checkpoints, histories, validation predictions, test predictions, and metrics are already stored in its own directory. This cleanup removes the in-memory model and DataLoaders before the next fold is created.

In [ ]:
# ============================================================
# 21. Releasing fold-1 memory before the next fold
# ============================================================

import gc

objects_to_release = [
    "complete_model",
    "optimizer",
    "scheduler",
    "training_objective",
    "train_loader",
    "validation_loader",
    "test_loader",
    "train_dataset",
    "validation_dataset",
    "test_dataset",
    "example_batch",
    "diagnostic_batch",
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Fold 1 outputs remain saved under:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "Fold-specific GPU cache has been released."
)


## 1.6. Fold 2

This section trains, validates, and evaluates fold 2. It uses the exact prepared table `mci_prognosis_outer_fold_2_final_task_ready.csv` and writes only to the experiment's `fold_2` directory.


### 1.6.1. Loading the prepared fold-2 input

The model-input columns are defined explicitly in this notebook, matching the original fold-0 implementation. No schema file or output from an earlier experiment is loaded.


In [ ]:
# ============================================================
# Loading the prepared inputs for fold 2
# ============================================================

from pathlib import Path
import json
import pandas as pd

from google.colab import drive


# ------------------------------------------------------------
# Experiment identity
# ------------------------------------------------------------

EXPERIMENT_NAME = "temporal_prebaseline_fixed_equal_fusion_md050"
SELECTED_TASK = "mci_prognosis"
SELECTED_FOLD = 2

# Use "fresh" for the formal first run.
# Change this to "resume" only after an interrupted run of this
# same experiment and fold.
FOLD_RUN_MODE = "fresh"


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():
    drive.mount(
        "/content/drive"
    )


# ------------------------------------------------------------
# Exact project and prepared-input paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

SELECTED_INPUT_PATH = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / "mci_prognosis"
    / f"mci_prognosis_outer_fold_{SELECTED_FOLD}_final_task_ready.csv"
)


# ------------------------------------------------------------
# Fixed model-input columns from the prepared pipeline
# ------------------------------------------------------------

final_model_schema = {
    "identifier_columns": [
        "RID",
        "PTID",
    ],

    "audit_label_columns": [
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ],

    "model_target_column":
        "MODEL_TARGET",

    "split_columns": [
        "OUTER_FOLD",
        "DATA_ROLE",
    ],

    "scaled_continuous_columns": [
        "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
        "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        "ADAS__TOTSCORE__Z",
        "ADAS__TOTAL13__Z",
        "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
        "FAQ__FAQTOTAL__Z",
        "CSF__ABETA40__Z",
        "CSF__ABETA42__Z",
        "CSF__TAU__Z",
        "CSF__PTAU__Z",
        "CSF__ABETA42_40_RATIO__Z",
        "PLASMA__pT217_F__Z",
        "PLASMA__AB42_F__Z",
        "PLASMA__AB40_F__Z",
        "PLASMA__AB42_AB40_F__Z",
        "PLASMA__pT217_AB42_F__Z",
        "PLASMA__NfL_Q__Z",
        "PLASMA__GFAP_Q__Z",
        "PLASMA__NfL_F__Z",
        "PLASMA__GFAP_F__Z",
    ],

    "encoded_categorical_columns": [
        "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
        "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        "APOE__APOE4_ALLELE_COUNT__IDX",
    ],

    "mri_path_columns": [
        "MRI__NORMALIZED_T1_NPY_PATH",
    ],

    "branch_mask_columns": [
        "BRANCH_MASK__DEMOGRAPHICS",
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
        "BRANCH_MASK__CSF",
        "BRANCH_MASK__PLASMA",
        "BRANCH_MASK__APOE",
        "BRANCH_MASK__MRI",
    ],

    "feature_mask_columns": [
        "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
        "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        "FEATURE_MASK__ADAS_TOTSCORE",
        "FEATURE_MASK__ADAS_TOTAL13",
        "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
        "FEATURE_MASK__FAQ_FAQTOTAL",
        "FEATURE_MASK__CSF_ABETA40",
        "FEATURE_MASK__CSF_ABETA42",
        "FEATURE_MASK__CSF_TAU",
        "FEATURE_MASK__CSF_PTAU",
        "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        "FEATURE_MASK__PLASMA_pT217_F",
        "FEATURE_MASK__PLASMA_AB42_F",
        "FEATURE_MASK__PLASMA_AB40_F",
        "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "FEATURE_MASK__PLASMA_NfL_Q",
        "FEATURE_MASK__PLASMA_GFAP_Q",
        "FEATURE_MASK__PLASMA_NfL_F",
        "FEATURE_MASK__PLASMA_GFAP_F",
        "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
    ],
}



# ------------------------------------------------------------
# Load the prepared fold table
# ------------------------------------------------------------

fold_table = pd.read_csv(
    SELECTED_INPUT_PATH
)


print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} INPUT")
print("=" * 72)

print(
    f"\nTask-ready table:\n{SELECTED_INPUT_PATH}"
)

print(
    f"\nLoaded shape: "
    f"{fold_table.shape[0]} rows × "
    f"{fold_table.shape[1]} columns"
)


### 1.6.2. Preparing the fold-2 model-input groups

The continuous, categorical, MRI-path, branch-mask, feature-mask, identifier, target, and split columns are taken from the fixed definitions loaded above.


In [ ]:
# ============================================================
# 2. Reading the prepared schema and describing fold 0
# ============================================================

# ------------------------------------------------------------
# Prepared model-input column groups
# ------------------------------------------------------------

identifier_columns = final_model_schema[
    "identifier_columns"
]

audit_label_columns = final_model_schema[
    "audit_label_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

split_columns = final_model_schema[
    "split_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]


# ------------------------------------------------------------
# Fold-0 structure
# ------------------------------------------------------------

print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} EXPERIMENT")
print("=" * 72)

print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Task: {SELECTED_TASK}")
print(f"Outer fold: {SELECTED_FOLD}")

print("\nPrepared data roles:")
print(
    fold_table["DATA_ROLE"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared target counts by role:")
display(
    pd.crosstab(
        fold_table["DATA_ROLE"],
        fold_table[target_column],
    ).reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared input groups:")
print(
    f"- scaled continuous features: "
    f"{len(scaled_continuous_columns)}"
)
print(
    f"- encoded categorical features: "
    f"{len(encoded_categorical_columns)}"
)
print(
    f"- MRI path columns: "
    f"{len(mri_path_columns)}"
)
print(
    f"- branch masks: "
    f"{len(branch_mask_columns)}"
)
print(
    f"- feature masks: "
    f"{len(feature_mask_columns)}"
)

preview_columns = (
    identifier_columns
    + ["CLINICAL_GROUP"]
    + split_columns
    + [target_column]
    + branch_mask_columns
    + mri_path_columns
)

print("\nExample prepared rows:")
display(
    fold_table[
        preview_columns
    ].head(5)
)


### 1.6.3. Defining the multimodal dataset-output contract

The model will receive each modality as a separate input branch rather than as one combined feature vector.

Continuous and categorical variables are kept separate because they require different encoder operations. Continuous variables will enter small numerical encoders, while categorical variables will later be represented through trainable embeddings.

Each sample will also contain:

- the participant identifier;
- the binary prognosis target;
- branch-level availability masks;
- feature-level observation masks;
- the prepared MRI path.

The dataset will not load MRI arrays yet. At this stage, I define the column organisation and the exact sample structure that the PyTorch dataset will later return.

For participants without MRI, the MRI path remains unavailable and the MRI branch mask remains zero. The participant is retained in the dataset.

In [ ]:
# ============================================================
# 3. Defining the multimodal dataset-output contract
# ============================================================

# ------------------------------------------------------------
# Branch-specific predictor columns
# ------------------------------------------------------------

# I organise the prepared continuous and categorical columns into
# the six modality branches used by the architecture.

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


# ------------------------------------------------------------
# Dataset sample structure
# ------------------------------------------------------------

# One participant will later be returned by the PyTorch dataset
# using the following nested structure.
#
# Continuous features will become float32 tensors.
# Categorical indices will become int64 tensors for embeddings.
# Masks will become float32 tensors containing 0 or 1.
# The target will become an int64 class index.

dataset_output_contract = {
    "rid": "Participant RID as an integer",
    "ptid": "Participant PTID as a string",
    "target": "Binary class index: 0 for sMCI and 1 for pMCI",

    "modalities": {
        "demographics": {
            "continuous": "Shape (2,), float32",
            "categorical": "Shape (2,), int64",
            "feature_mask": "Shape (4,), float32",
            "branch_mask": "Scalar, float32",
        },

        "cognitive_functional": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "csf": {
            "continuous": "Shape (5,), float32",
            "categorical": None,
            "feature_mask": "Shape (5,), float32",
            "branch_mask": "Scalar, float32",
        },

        "plasma": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "apoe": {
            "continuous": None,
            "categorical": "Shape (1,), int64",
            "feature_mask": "Shape (1,), float32",
            "branch_mask": "Scalar, float32",
        },

        "mri": {
            "path": "Prepared NumPy path or None",
            "image": "Later: shape (1, 177, 213, 183), float32",
            "branch_mask": "Scalar, float32",
        },
    },

    "branch_masks": (
        "Shape (6,), float32, ordered as "
        "demographics, cognitive-functional, CSF, "
        "plasma, APOE, MRI"
    ),

    "feature_masks": (
        "Shape (28,), float32, using the authoritative "
        "feature-mask order"
    ),
}


# ------------------------------------------------------------
# Display the agreed contract
# ------------------------------------------------------------

print("=" * 72)
print("MULTIMODAL DATASET CONTRACT")
print("=" * 72)

print("\nBranch-specific input dimensions:")

for branch_name, branch_definition in dataset_column_contract.items():

    continuous_count = len(
        branch_definition.get("continuous", [])
    )

    categorical_count = len(
        branch_definition.get("categorical", [])
    )

    feature_mask_count = len(
        branch_definition.get("feature_masks", [])
    )

    has_mri_path = "path" in branch_definition

    print(
        f"- {branch_name}: "
        f"{continuous_count} continuous, "
        f"{categorical_count} categorical, "
        f"{feature_mask_count} feature masks"
        + (", 1 MRI path" if has_mri_path else "")
    )


print("\nPlanned sample output:")
print(
    json.dumps(
        dataset_output_contract,
        indent=2,
    )
)

print(
    "\nThis contract will be used in the next step to "
    "implement the PyTorch dataset."
)

### 1.6.4. Implementing the multimodal PyTorch dataset

implement a PyTorch dataset that converts each prepared participant row into the agreed multimodal structure.

The dataset preserves the six modality branches and returns continuous variables, categorical indices, observation masks, participant identifiers, and the prognosis target separately.

MRI volumes are loaded only when a sample is requested. The existing preprocessed NumPy array is used directly, and a channel dimension is added to produce the shape required by a three-dimensional neural network.

When MRI is unavailable, the participant remains in the dataset. The dataset returns a zero placeholder volume together with an MRI branch mask of zero, allowing the model to distinguish an unavailable scan from an observed image.

In [ ]:
# ============================================================
# 4. Implementing the multimodal PyTorch dataset
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset


# ------------------------------------------------------------
# Expected prepared MRI shape
# ------------------------------------------------------------

# I preserve the spatial dimensions produced by the completed
# MRI preprocessing pipeline.
MRI_SPATIAL_SHAPE = (177, 213, 183)

# I add one channel dimension when returning an MRI tensor.
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)


# ------------------------------------------------------------
# Multimodal PyTorch dataset
# ------------------------------------------------------------

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # I reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # I retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # I preserve the authoritative mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri


    def __len__(self):
        return len(self.dataframe)


    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # I load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # I ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)


    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # --------------------------------------------------------
        # Demographics
        # --------------------------------------------------------

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Cognitive and functional measures
        # --------------------------------------------------------

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # CSF
        # --------------------------------------------------------

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Plasma
        # --------------------------------------------------------

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # APOE
        # --------------------------------------------------------

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # MRI
        # --------------------------------------------------------

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Complete sample
        # --------------------------------------------------------

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample


# ------------------------------------------------------------
# Create role-specific datasets
# ------------------------------------------------------------

# I retain the prepared role assignments exactly as stored in
# the selected outer-fold table.
train_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "train"
    ]
    .reset_index(drop=True)
)

validation_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "validation"
    ]
    .reset_index(drop=True)
)

test_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "test"
    ]
    .reset_index(drop=True)
)


# I initially disable MRI disk loading so that I can inspect the
# dataset structure quickly before constructing the DataLoaders.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)


# ------------------------------------------------------------
# Concise dataset summary
# ------------------------------------------------------------

print("=" * 72)
print("PYTORCH DATASETS")
print("=" * 72)

print(f"\nTraining participants: {len(train_dataset)}")
print(f"Validation participants: {len(validation_dataset)}")
print(f"Test participants: {len(test_dataset)}")

print(
    "\nThe datasets preserve the prepared role assignments "
    "and return separate modality inputs."
)

print(
    "MRI loading is temporarily disabled for structural "
    "inspection and will be enabled for the DataLoaders."
)

### 1.6.5. Inspecting one multimodal sample and one prepared MRI volume

Before constructing the DataLoaders, The notebook inspects the structure returned for one participant.

use the dataset with MRI loading disabled to confirm the scalar tensors, categorical indices, targets, and masks. I then create a temporary MRI-enabled dataset and load one participant whose MRI branch is available.

This checks the dataset interface required by the model while avoiding unnecessary loading of multiple MRI volumes at this stage.

In [ ]:
# ============================================================
# 5. Inspecting one multimodal sample and one MRI volume
# ============================================================

# ------------------------------------------------------------
# Inspect one scalar-only training sample
# ------------------------------------------------------------

# I retrieve one participant while MRI disk loading remains
# disabled. The returned MRI tensor is therefore only the
# temporary placeholder defined in the current dataset class.
sample = train_dataset[0]

print("=" * 72)
print("EXAMPLE MULTIMODAL SAMPLE")
print("=" * 72)

print(f"\nRID: {sample['rid']}")
print(f"PTID: {sample['ptid']}")
print(f"Target: {sample['target'].item()}")

print("\nModality tensor structure:")

for modality_name, modality_data in sample["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"{value}"
            )

print(
    f"\nComplete branch-mask shape: "
    f"{tuple(sample['branch_masks'].shape)}"
)

print(
    f"Complete feature-mask shape: "
    f"{tuple(sample['feature_masks'].shape)}"
)


# ------------------------------------------------------------
# Find one participant with an available MRI
# ------------------------------------------------------------

# I select the first training participant whose prepared MRI
# branch mask is one.
example_mri_index = train_table.index[
    train_table["BRANCH_MASK__MRI"] == 1
][0]


# ------------------------------------------------------------
# Create a temporary MRI-enabled dataset
# ------------------------------------------------------------

# I enable MRI loading only for this temporary inspection
# dataset. The main role-specific datasets remain unchanged.
mri_inspection_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

mri_sample = mri_inspection_dataset[
    example_mri_index
]

mri_tensor = mri_sample[
    "modalities"
]["mri"]["image"]

mri_branch_mask = mri_sample[
    "modalities"
]["mri"]["branch_mask"]


# ------------------------------------------------------------
# Display the prepared MRI tensor information
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXAMPLE PREPARED MRI")
print("=" * 72)

print(f"\nRID: {mri_sample['rid']}")
print(f"PTID: {mri_sample['ptid']}")

print(
    f"MRI branch mask: "
    f"{mri_branch_mask.item():.0f}"
)

print(
    f"MRI tensor shape: "
    f"{tuple(mri_tensor.shape)}"
)

print(
    f"MRI tensor dtype: "
    f"{mri_tensor.dtype}"
)

print(
    f"MRI intensity minimum: "
    f"{mri_tensor.min().item():.6f}"
)

print(
    f"MRI intensity maximum: "
    f"{mri_tensor.max().item():.6f}"
)

print(
    f"MRI intensity mean: "
    f"{mri_tensor.mean().item():.6f}"
)

print(
    "\nThe sample structure and full prepared MRI volume "
    "are ready for DataLoader construction."
)

### 1.6.6. Constructing the multimodal DataLoaders

create separate DataLoaders for the fixed training, validation, and test subsets.

MRI loading is enabled, so an available scan is read lazily from its prepared NumPy path when its participant enters a batch. Participants without MRI receive a zero placeholder volume and retain an MRI branch mask of zero.

I begin with a small batch size because each sample contains a full three-dimensional MRI volume. The final training batch size will be selected later according to the memory requirements of the complete model.

The training DataLoader shuffles participants. Validation and test DataLoaders preserve a deterministic order.

In [ ]:
# ============================================================
# 6. Constructing the multimodal DataLoaders
# ============================================================

from torch.utils.data import DataLoader


# ------------------------------------------------------------
# Initial DataLoader settings
# ------------------------------------------------------------

# I begin with a small batch because each participant may contain
# a full MRI volume with shape (1, 177, 213, 183).
INITIAL_BATCH_SIZE = 2

# I initially use the main process for data loading. This is the
# most reliable starting configuration when reading NumPy files
# from mounted Google Drive.
NUM_WORKERS = 0

# Pinned memory can speed transfers to a CUDA device.
PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# Recreate the datasets with MRI loading enabled
# ------------------------------------------------------------

# Available MRI volumes will now be read lazily when requested.
# Missing MRI branches will retain their zero placeholders and
# branch masks of zero.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)


# ------------------------------------------------------------
# Create role-specific DataLoaders
# ------------------------------------------------------------

# I shuffle only the training subset.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# Validation order does not need to be shuffled.
validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# The test subset also retains a deterministic order.
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)


# ------------------------------------------------------------
# Retrieve one complete training batch
# ------------------------------------------------------------

example_batch = next(iter(train_loader))


# ------------------------------------------------------------
# Display the batched tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("EXAMPLE MULTIMODAL BATCH")
print("=" * 72)

print(f"\nBatch size: {example_batch['target'].shape[0]}")
print(f"RID values: {example_batch['rid']}")
print(f"PTID values: {example_batch['ptid']}")
print(f"Targets: {example_batch['target']}")

print("\nModality tensors:")

for modality_name, modality_data in example_batch["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"type={type(value).__name__}"
            )


print("\nCombined masks:")

print(
    "  branch_masks: "
    f"shape={tuple(example_batch['branch_masks'].shape)}, "
    f"dtype={example_batch['branch_masks'].dtype}"
)

print(
    "  feature_masks: "
    f"shape={tuple(example_batch['feature_masks'].shape)}, "
    f"dtype={example_batch['feature_masks'].dtype}"
)


# ------------------------------------------------------------
# Show MRI availability in this batch
# ------------------------------------------------------------

batch_mri = example_batch[
    "modalities"
]["mri"]["image"]

batch_mri_masks = example_batch[
    "modalities"
]["mri"]["branch_mask"]

print("\nMRI batch:")

print(
    f"  image shape: {tuple(batch_mri.shape)}"
)

print(
    f"  branch masks: {batch_mri_masks}"
)

print(
    f"  approximate raw MRI batch size: "
    f"{batch_mri.numel() * batch_mri.element_size() / (1024 ** 2):.2f} MB"
)


# ------------------------------------------------------------
# DataLoader summary
# ------------------------------------------------------------

print("\nDataLoader batches:")

print(
    f"  training: {len(train_loader)} batches"
)

print(
    f"  validation: {len(validation_loader)} batches"
)

print(
    f"  test: {len(test_loader)} batches"
)

print(
    "\nThe complete multimodal batch is ready for "
    "modality-specific encoder construction."
)

### 1.6.7. Building the modality-specific encoders

Each modality has a different input structure, so I encode the six branches separately before multimodal interaction.

For the scalar branches, the prepared feature values are combined with their feature-observation masks. This allows the encoders to distinguish an observed standardised value close to zero from a missing-value placeholder.

Categorical variables use trainable embeddings. Encoded index zero remains reserved for missing or unseen values and is handled through embedding padding behaviour.

### 1.6.8. Common latent dimension

Every branch is projected into the same latent dimension:

$$
d_{\mathrm{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore produces:

$$
\mathbf{H}^{(m)}
\in
\mathbb{R}^{B \times 64}.
$$

The shared dimension is required so that all modality representations can later enter the same cascaded cross-modal transformer architecture.

The original interaction pathway implementation used a dimension of \(512\). In this thesis, I begin with a more compact dimension of \(64\) because the main prognosis training folds contain approximately \(369\) participants and the complete model will additionally include independent evidential heads, modality-specific evidence fusion, auxiliary outputs, and the cascaded interaction pathway interaction path.

The embedding dimension remains a model-capacity hyperparameter and may later be compared with larger values using only training and validation data.

### 1.6.9. MRI encoder

The MRI branch follows the general image-encoding structure used by interaction pathway:

1. initial three-dimensional convolutions;
2. residual three-dimensional downsampling blocks;
3. conversion of the final feature map into patch tokens;
4. addition of learned positional embeddings;
5. transformer encoding of patch-wise relationships;
6. global averaging across patch tokens;
7. projection into the shared modality dimension.

The original paper used a 512-dimensional MRI output. Here, the final MRI representation is projected to the common 64-dimensional latent space used by the other branches.

The prepared MRI volumes are larger than those used in the original interaction pathway experiments. I therefore apply stride-two downsampling in the initial convolutional stem before the four residual blocks. This preserves the CNN-transformer design while keeping the number of transformer patch tokens computationally manageable.

No new MRI preprocessing is performed. The encoder receives the complete prepared volume with shape:

$$
(1, 177, 213, 183).
$$

In [ ]:
# ============================================================
# 7. Building the modality-specific encoders
# ============================================================

import math

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Shared representation dimension
# ------------------------------------------------------------

# I project every modality into one common latent space so that
# all branches can later enter the same cross-modal transformers.
MODALITY_EMBEDDING_DIM = 64


# ------------------------------------------------------------
# Reusable scalar encoder
# ------------------------------------------------------------

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        values,
        feature_mask,
    ):
        # I supply both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Demographics encoder
# ------------------------------------------------------------

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# APOE encoder
# ------------------------------------------------------------

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Residual 3D downsampling block
# ------------------------------------------------------------

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original interaction pathway image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()


    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )


# ------------------------------------------------------------
# interaction pathway-style CNN-transformer MRI encoder
# ------------------------------------------------------------

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # --------------------------------------------------------
        # Initial convolutional stem
        # --------------------------------------------------------

        # I use two initial 3D convolutions, following the broad
        # structure shown in the interaction pathway image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original interaction pathway inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )


        # --------------------------------------------------------
        # Four residual downsampling blocks
        # --------------------------------------------------------

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )


        # --------------------------------------------------------
        # Determine the resulting patch grid
        # --------------------------------------------------------

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )


        # --------------------------------------------------------
        # Learned positional embeddings
        # --------------------------------------------------------

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )


        # --------------------------------------------------------
        # Patch-wise transformer encoder
        # --------------------------------------------------------

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )


        # --------------------------------------------------------
        # Projection to the shared modality dimension
        # --------------------------------------------------------

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )


    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # I flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # I add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )


# ------------------------------------------------------------
# Complete set of six modality encoders
# ------------------------------------------------------------

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )


    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations


# ------------------------------------------------------------
# Instantiate the revised encoders
# ------------------------------------------------------------

modality_encoders = ADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Count trainable parameters
# ------------------------------------------------------------

def count_trainable_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print("=" * 72)
print("MODALITY-SPECIFIC ENCODERS")
print("=" * 72)

print(
    f"\nShared modality embedding dimension: "
    f"{MODALITY_EMBEDDING_DIM}"
)

print(
    "\nMRI transformer patch grid: "
    f"{modality_encoders.mri.patch_grid_shape}"
)

print(
    "MRI transformer patch count: "
    f"{modality_encoders.mri.number_of_patches}"
)

print("\nTrainable parameters by encoder:")

for encoder_name in [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]:
    encoder = getattr(
        modality_encoders,
        encoder_name,
    )

    print(
        f"- {encoder_name}: "
        f"{count_trainable_parameters(encoder):,}"
    )

print(
    "\nTotal trainable encoder parameters: "
    f"{count_trainable_parameters(modality_encoders):,}"
)

print(
    "\nThe revised MRI branch now uses a 3D CNN, patch tokens, "
    "positional embeddings, and a transformer encoder."
)

print(
    "No multimodal fusion or classification head has been "
    "added yet."
)

### 1.6.10. Applying branch-availability masks to the encoded modalities

Each modality has a different original input structure, but every modality-specific encoder projects its input into the same latent dimensionality.

In this implementation, each branch produces a representation of size:

$$
d_{\text{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore returns:

$$
\mathbf{H}^{(m)} \in \mathbb{R}^{B \times 64},
$$

where \(m\) denotes one of the six modalities:

- demographics;
- cognitive-functional measures;
- CSF;
- plasma;
- APOE;
- MRI.

The shared dimensionality does not mean that the modalities contain the same information or use equally complex encoders. Each branch has its own input-specific encoder, but the final representations must have a common size so that they can later participate in cross-modal attention and evidential fusion.

After stacking the six modality representations, the model obtains:

$$
\mathbf{H}
\in
\mathbb{R}^{B \times 6 \times 64}.
$$

This follows the general design principle used by interaction pathway, in which heterogeneous modality inputs are first projected into a common transformer embedding space before cross-modal interaction. The original interaction pathway implementation used a larger embedding dimension of \(512\), but \(512\) is an architectural hyperparameter rather than a methodological requirement.

A compact starting dimension of \(64\) is used here because the main MCI prognosis training folds contain only approximately \(369\) participants, while the complete planned model will also contain:

- six modality-specific encoders;
- cascaded cross-modal attention;
- independent evidential heads;
- evidence pathway-style evidential fusion;
- a joint evidential prediction path.

The number of parameters in transformer projections grows approximately with the square of the embedding dimension. For example:

$$
64^2 = 4{,}096,
$$

whereas:

$$
512^2 = 262{,}144.
$$

Thus, increasing the embedding dimension from \(64\) to \(512\) can make several attention and feed-forward parameter blocks approximately \(64\) times larger. A \(512\)-dimensional model would therefore introduce substantially greater overfitting and memory risk for the available prognosis cohort.

The value \(64\) is treated as a compact initial configuration rather than as a permanently fixed optimum. The latent dimensionality can later be compared with alternatives such as \(128\) or \(256\), using only the training and validation subsets.

### 1.6.11. Branch-availability masking

The prepared zero placeholders make missing inputs computationally compatible with neural-network layers, but they do not guarantee that an unavailable modality will produce a zero encoder output.

Linear layers, embeddings, normalisation parameters, and learned biases can generate a non-zero representation even when all supplied inputs are zero. I therefore apply the prepared branch mask after each modality encoder.

For participant \(i\) and modality \(m\), the masked representation is:

$$
\widetilde{\mathbf{h}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{h}_{i}^{(m)},
$$

where:

- \(\mathbf{h}_{i}^{(m)} \in \mathbb{R}^{64}\) is the raw modality representation;
- \(a_{i}^{(m)} \in \{0,1\}\) is the prepared branch-availability mask;
- \(\widetilde{\mathbf{h}}_{i}^{(m)} \in \mathbb{R}^{64}\) is the masked representation passed to later model components.

Therefore:

$$
a_{i}^{(m)} = 1
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{h}_{i}^{(m)},
$$

and:

$$
a_{i}^{(m)} = 0
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{0}.
$$

An unavailable modality consequently contributes an exact zero representation rather than a learned bias-derived vector. The original branch masks are also retained separately so that the later cross-modal attention and evidential-fusion components can explicitly identify which modalities are available for each participant.

In [ ]:
# ============================================================
# 8. Applying branch masks to the encoded modalities
# ============================================================

# ------------------------------------------------------------
# Fixed modality order
# ------------------------------------------------------------

# I use one explicit modality order throughout the architecture.
# This order matches the prepared branch-mask columns.
MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]


# ------------------------------------------------------------
# Mask-aware encoder wrapper
# ------------------------------------------------------------

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )


    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # I add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask


    def forward(self, modalities):
        # I first obtain the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # I then suppress every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # I return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }


# ------------------------------------------------------------
# Instantiate the mask-aware encoder collection
# ------------------------------------------------------------

masked_modality_encoders = MaskedADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Apply the encoders to one complete batch
# ------------------------------------------------------------

# I keep this inspection on the CPU. The training device will be
# configured later when the full model and optimisation loop exist.
masked_modality_encoders.eval()

with torch.no_grad():
    encoded_batch = masked_modality_encoders(
        example_batch["modalities"]
    )


# ------------------------------------------------------------
# Stack modality representations
# ------------------------------------------------------------

# I stack the representations in the fixed modality order.
#
# Resulting shape:
# (batch_size, number_of_modalities, embedding_dimension)
stacked_masked_representations = torch.stack(
    [
        encoded_batch["masked"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_raw_representations = torch.stack(
    [
        encoded_batch["raw"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the encoded structure
# ------------------------------------------------------------

print("=" * 72)
print("MASKED MODALITY REPRESENTATIONS")
print("=" * 72)

print(
    f"\nStacked raw representation shape: "
    f"{tuple(stacked_raw_representations.shape)}"
)

print(
    f"Stacked masked representation shape: "
    f"{tuple(stacked_masked_representations.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, embedding)"
)

print("\nBranch masks in this batch:")

branch_mask_table = pd.DataFrame(
    example_batch["branch_masks"].numpy(),
    columns=MODALITY_ORDER,
)

display(branch_mask_table)


# ------------------------------------------------------------
# Representation norms before and after masking
# ------------------------------------------------------------

# A representation norm summarises the magnitude of each branch
# vector. Missing branches may have non-zero raw norms because of
# learned biases, but their masked norms must be exactly zero.
raw_norms = torch.linalg.vector_norm(
    stacked_raw_representations,
    dim=-1,
)

masked_norms = torch.linalg.vector_norm(
    stacked_masked_representations,
    dim=-1,
)

norm_summary = []

for participant_index in range(
    stacked_masked_representations.shape[0]
):
    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):
        norm_summary.append(
            {
                "BATCH_ROW": participant_index,
                "RID": int(
                    example_batch["rid"][
                        participant_index
                    ].item()
                ),
                "MODALITY": modality_name,
                "BRANCH_MASK": float(
                    example_batch["branch_masks"][
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "RAW_REPRESENTATION_NORM": float(
                    raw_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "MASKED_REPRESENTATION_NORM": float(
                    masked_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
            }
        )

norm_summary = pd.DataFrame(norm_summary)

print("\nRepresentation norms before and after masking:")

display(
    norm_summary.round(6)
)


# ------------------------------------------------------------
# Confirm the representation dimensions
# ------------------------------------------------------------

print("\nEncoded modality shapes:")

for modality_name in MODALITY_ORDER:
    print(
        f"- {modality_name}: "
        f"{tuple(encoded_batch['masked'][modality_name].shape)}"
    )

print(
    "\nOnly the masked representations will enter the "
    "multimodal interaction and evidential-fusion paths."
)

### 1.6.12. Building the availability-gated interaction pathway cascade

The cascade order remains unchanged:

$$
\text{demographics}
\rightarrow
\text{APOE}
\rightarrow
\text{cognitive/functional}
\rightarrow
\text{CSF}
\rightarrow
\text{plasma}
\rightarrow
\text{MRI}.
$$

Each cross-modal interaction block still computes a candidate update using query self-attention followed by cross-attention to the encoded modality token. The branch mask then determines whether that candidate becomes the next cumulative query:

$$
\mathbf{q}_m
=
a_m\mathbf{q}^{\mathrm{candidate}}_m
+
(1-a_m)\mathbf{q}_{m-1}.
$$

For an available modality, $a_m=1$ and the candidate update is used. For an unavailable modality, $a_m=0$ and the stage becomes an exact identity update.

The branch-mask tensor follows `MODALITY_ORDER`, while the cross-modal interaction blocks follow `THREE_MT_CASCADE_ORDER`. The class therefore uses the explicit modality-to-mask index mapping already defined by the fold-0 architecture.

In [ ]:
# ============================================================
# 9. Building the availability-gated interaction pathway cascade
# ============================================================

# ------------------------------------------------------------
# Fixed cascade order
# ------------------------------------------------------------

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]


# ------------------------------------------------------------
# One Cascaded Modality Transformer
# ------------------------------------------------------------

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }


# ------------------------------------------------------------
# Complete six-stage availability-gated cascade
# ------------------------------------------------------------

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )


    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }


# ------------------------------------------------------------
# Instantiate and inspect the gated cascade
# ------------------------------------------------------------

three_mt_cascade = ThreeMTCascade(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    number_of_heads=4,
    dropout=0.10,
)

three_mt_cascade.eval()

with torch.no_grad():
    three_mt_output = three_mt_cascade(
        masked_representations=encoded_batch[
            "masked"
        ],
        branch_masks=example_batch[
            "branch_masks"
        ],
    )


print("=" * 72)
print("AVAILABILITY-GATED 3MT CASCADE")
print("=" * 72)

print(
    f"\nCascade order:\n"
    f"{THREE_MT_CASCADE_ORDER}"
)

print(
    "\nFinal joint representation shape: "
    f"{tuple(three_mt_output['joint_representation'].shape)}"
)


# ------------------------------------------------------------
# Show the actual query update at every stage
# ------------------------------------------------------------

query_change_rows = []

previous_query = (
    three_mt_cascade
    .learned_latent_query
    .expand(
        example_batch["target"].shape[0],
        -1,
        -1,
    )
)

for modality_name in THREE_MT_CASCADE_ORDER:
    current_query = three_mt_output[
        "stage_queries"
    ][modality_name]

    query_change_norm = torch.linalg.vector_norm(
        current_query - previous_query,
        dim=-1,
    ).squeeze(1)

    modality_index = MODALITY_ORDER.index(
        modality_name
    )

    branch_mask = example_batch[
        "branch_masks"
    ][
        :,
        modality_index,
    ]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):
        query_change_rows.append(
            {
                "BATCH_ROW": batch_row,
                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),
                "CASCADE_STAGE": modality_name,
                "BRANCH_MASK": float(
                    branch_mask[
                        batch_row
                    ].item()
                ),
                "QUERY_CHANGE_NORM": float(
                    query_change_norm[
                        batch_row
                    ].item()
                ),
            }
        )

    previous_query = current_query

query_change_summary = pd.DataFrame(
    query_change_rows
)

print(
    "\nQuery changes after availability gating:"
)

display(
    query_change_summary.round(6)
)

print(
    "\nTrainable cascade parameters: "
    f"{count_trainable_parameters(three_mt_cascade):,}"
)


### 1.6.13. Producing independent modality-specific evidential opinions

The interaction pathway cascade produces a cumulative interaction-aware representation, but its intermediate states are not independent modality opinions because every stage contains information inherited from earlier stages.

Trusted Multi-View Classification requires each modality to produce its own class evidence before cross-modal interaction.

For participant \(i\), modality \(m\), and class \(k\), the modality-specific evidential head produces non-negative evidence:

$$
e_{ik}^{(m)}
=
\operatorname{Softplus}
\left(
\mathbf{W}_{m}
\mathbf{z}_{i}^{(m)}
+
\mathbf{b}_{m}
\right),
$$

where:

- \(\mathbf{z}_{i}^{(m)} \in \mathbb{R}^{64}\) is the independently encoded modality representation;
- \(e_{ik}^{(m)} \geq 0\) is the evidence assigned to class \(k\);
- each modality has its own evidential head.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{(m)}
=
e_{ik}^{(m)} + 1.
$$

For the binary prognosis task:

$$
K = 2,
$$

with class order:

$$
[\mathrm{sMCI},\mathrm{pMCI}].
$$

The expected class probabilities are:

$$
p_{ik}^{(m)}
=
\frac{
\alpha_{ik}^{(m)}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{(m)}
}.
$$

The Dirichlet strength is:

$$
S_{i}^{(m)}
=
\sum_{k=1}^{K}
\alpha_{ik}^{(m)}.
$$

The standard evidential uncertainty mass is:

$$
u_{i}^{(m)}
=
\frac{K}{
S_{i}^{(m)}
}.
$$

Low total evidence produces high uncertainty, while stronger evidence produces lower uncertainty.

### 1.6.14. Treatment of unavailable modalities

Only available modalities should contribute an opinion to modality-specific evidence fusion.

For an unavailable branch, I do not interpret the evidential head output as a genuine prediction. Instead, its effective evidence is set to zero:

$$
\widetilde{\mathbf{e}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{e}_{i}^{(m)},
$$

where \(a_{i}^{(m)}\) is the prepared branch mask.

Therefore, an unavailable modality receives:

$$
\widetilde{\boldsymbol{\alpha}}_{i}^{(m)}
=
\mathbf{1},
$$

which is the uniform Dirichlet opinion with:

$$
u_{i}^{(m)} = 1.
$$

The original branch mask is retained so that the next step can exclude unavailable opinions explicitly during evidence pathway/Dempster--Shafer fusion.

At this stage, I construct and inspect the six independent evidential opinions. I do not yet fuse them or combine them with the interaction pathway joint representation.

In [ ]:
# ============================================================
# 10. Producing independent modality-specific evidential opinions
# ============================================================

# ------------------------------------------------------------
# Binary prognosis class definition
# ------------------------------------------------------------

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]


# ------------------------------------------------------------
# One modality-specific evidential head
# ------------------------------------------------------------

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(
        self,
        representation,
        branch_mask,
    ):
        # I use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }


# ------------------------------------------------------------
# Independent evidential heads for all six modalities
# ------------------------------------------------------------

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )


    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions


# ------------------------------------------------------------
# Instantiate the independent evidential path
# ------------------------------------------------------------

independent_evidential_heads = (
    IndependentModalityEvidentialHeads(
        modality_order=MODALITY_ORDER,
        input_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
    )
)


# ------------------------------------------------------------
# Produce one opinion per modality
# ------------------------------------------------------------

independent_evidential_heads.eval()

with torch.no_grad():

    modality_opinions = (
        independent_evidential_heads(
            # I use the independently encoded branch outputs before
            # they enter the interaction pathway cascade.
            modality_representations=encoded_batch[
                "masked"
            ],

            modalities=example_batch[
                "modalities"
            ],
        )
    )


# ------------------------------------------------------------
# Stack the opinion tensors
# ------------------------------------------------------------

stacked_modality_evidence = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["evidence"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_alpha = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["alpha"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_probabilities = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["probabilities"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_uncertainty = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["uncertainty"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the evidential tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("INDEPENDENT MODALITY EVIDENTIAL OPINIONS")
print("=" * 72)

print(
    f"\nClass order: "
    f"{PROGNOSIS_CLASS_ORDER}"
)

print(
    "\nStacked evidence shape: "
    f"{tuple(stacked_modality_evidence.shape)}"
)

print(
    "Stacked Dirichlet-alpha shape: "
    f"{tuple(stacked_modality_alpha.shape)}"
)

print(
    "Stacked probability shape: "
    f"{tuple(stacked_modality_probabilities.shape)}"
)

print(
    "Stacked uncertainty shape: "
    f"{tuple(stacked_modality_uncertainty.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, class)"
)


# ------------------------------------------------------------
# Create a readable modality-opinion summary
# ------------------------------------------------------------

opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):

        branch_mask = float(
            example_batch["branch_masks"][
                batch_row,
                modality_index,
            ].item()
        )

        alpha_values = stacked_modality_alpha[
            batch_row,
            modality_index,
        ]

        probability_values = (
            stacked_modality_probabilities[
                batch_row,
                modality_index,
            ]
        )

        uncertainty_value = (
            stacked_modality_uncertainty[
                batch_row,
                modality_index,
                0,
            ]
        )

        opinion_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "MODALITY": modality_name,

                "BRANCH_MASK": branch_mask,

                "ALPHA_sMCI": float(
                    alpha_values[0].item()
                ),

                "ALPHA_pMCI": float(
                    alpha_values[1].item()
                ),

                "P_sMCI": float(
                    probability_values[0].item()
                ),

                "P_pMCI": float(
                    probability_values[1].item()
                ),

                "UNCERTAINTY": float(
                    uncertainty_value.item()
                ),
            }
        )


modality_opinion_summary = pd.DataFrame(
    opinion_rows
)

print("\nIndependent modality opinions:")

display(
    modality_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable independent evidential-head parameters: "
    f"{count_trainable_parameters(independent_evidential_heads):,}"
)

print(
    "\nUnavailable modalities should have alpha=[1, 1], "
    "probabilities=[0.5, 0.5], and uncertainty=1."
)

print(
    "\nThe available modality opinions are ready for "
    "TMC/Dempster-Shafer fusion."
)

### 1.6.15. Fusing the independent modality opinions with evidence pathway

combine the six independent modality opinions using the reduced Dempster--Shafer rule adopted by Trusted Multi-View Classification.

For modality \(m\), the Dirichlet parameters are:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{e}^{(m)} + \mathbf{1}.
$$

The Dirichlet strength is:

$$
S^{(m)}
=
\sum_{k=1}^{K}
\alpha_{k}^{(m)}.
$$

The class-specific belief masses are:

$$
b_{k}^{(m)}
=
\frac{
\alpha_{k}^{(m)} - 1
}{
S^{(m)}
}
=
\frac{
e_{k}^{(m)}
}{
S^{(m)}
}.
$$

The uncertainty mass is:

$$
u^{(m)}
=
\frac{K}{
S^{(m)}
}.
$$

These masses satisfy:

$$
\sum_{k=1}^{K}
b_{k}^{(m)}
+
u^{(m)}
=
1.
$$

### 1.6.16. Combining two opinions

Consider two opinions, \(A\) and \(B\). Their conflict mass is:

$$
C
=
\sum_{i \neq j}
b_{i}^{A}
b_{j}^{B}.
$$

For each class \(k\), the combined belief mass is:

$$
b_{k}^{A \oplus B}
=
\frac{
b_{k}^{A}b_{k}^{B}
+
b_{k}^{A}u^{B}
+
b_{k}^{B}u^{A}
}{
1-C
}.
$$

The combined uncertainty mass is:

$$
u^{A \oplus B}
=
\frac{
u^{A}u^{B}
}{
1-C
}.
$$

The fused Dirichlet strength is recovered from the fused uncertainty:

$$
S^{A \oplus B}
=
\frac{K}{
u^{A \oplus B}
}.
$$

The fused evidence and Dirichlet parameters are then:

$$
e_{k}^{A \oplus B}
=
b_{k}^{A \oplus B}
S^{A \oplus B},
$$

and:

$$
\alpha_{k}^{A \oplus B}
=
e_{k}^{A \oplus B} + 1.
$$

The rule is applied repeatedly until all six modality opinions have been considered.

### 1.6.17. Missing modalities

An unavailable modality was assigned the vacuous Dirichlet opinion:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{1}.
$$

For this opinion:

$$
\mathbf{b}^{(m)}
=
\mathbf{0},
\qquad
u^{(m)} = 1.
$$

A vacuous opinion acts as an identity element in this combination rule. It adds no class evidence, creates no conflict, and leaves the available opinion unchanged.

Therefore, the same fusion procedure can process all participants without complete-case filtering or synthetic modality imputation.

At this stage, I construct only the evidence pathway-fused independent opinion. The interaction-aware interaction pathway query will receive its own evidential head in a later step.

In [ ]:
# ============================================================
# 11. Fusing the independent modality opinions with evidence pathway
# ============================================================

# ------------------------------------------------------------
# Reduced Dempster-Shafer combination rule
# ------------------------------------------------------------

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )


    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }


    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]


        # --------------------------------------------------------
        # Conflict mass
        # --------------------------------------------------------

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)


        # --------------------------------------------------------
        # Fused belief and uncertainty masses
        # --------------------------------------------------------

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation


        # --------------------------------------------------------
        # Recover the fused Dirichlet opinion
        # --------------------------------------------------------

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }


    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # I retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }


        # --------------------------------------------------------
        # Final fused opinion
        # --------------------------------------------------------

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }


# ------------------------------------------------------------
# Instantiate the modality-specific evidence fusion module
# ------------------------------------------------------------

tmc_fusion = TMCFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
)


# ------------------------------------------------------------
# Fuse the current independent modality opinions
# ------------------------------------------------------------

with torch.no_grad():

    tmc_output = tmc_fusion(
        modality_opinions=modality_opinions
    )


# ------------------------------------------------------------
# Display final fused tensor shapes
# ------------------------------------------------------------

print("=" * 72)
print("TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    f"\nFusion order: "
    f"{MODALITY_ORDER}"
)

print(
    "\nFused evidence shape: "
    f"{tuple(tmc_output['evidence'].shape)}"
)

print(
    "Fused alpha shape: "
    f"{tuple(tmc_output['alpha'].shape)}"
)

print(
    "Fused probability shape: "
    f"{tuple(tmc_output['probabilities'].shape)}"
)

print(
    "Fused uncertainty shape: "
    f"{tuple(tmc_output['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Final participant-level fused opinions
# ------------------------------------------------------------

fused_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    fused_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "AVAILABLE_MODALITIES": int(
                example_batch["branch_masks"][
                    batch_row
                ].sum()
                .item()
            ),

            "ALPHA_sMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                tmc_output["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


fused_opinion_summary = pd.DataFrame(
    fused_opinion_rows
)

print("\nFinal TMC-fused opinions:")

display(
    fused_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Inspect the sequential fusion history
# ------------------------------------------------------------

fusion_history_rows = []

for modality_name in MODALITY_ORDER:

    stage_output = tmc_output[
        "fusion_history"
    ][modality_name]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):

        modality_index = MODALITY_ORDER.index(
            modality_name
        )

        fusion_history_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "FUSED_THROUGH": modality_name,

                "CURRENT_MODALITY_MASK": float(
                    example_batch["branch_masks"][
                        batch_row,
                        modality_index,
                    ].item()
                ),

                "STAGE_CONFLICT": float(
                    stage_output["conflict"][
                        batch_row,
                        0,
                    ].item()
                ),

                "STAGE_UNCERTAINTY": float(
                    stage_output["uncertainty"][
                        batch_row,
                        0,
                    ].item()
                ),
            }
        )


fusion_history_summary = pd.DataFrame(
    fusion_history_rows
)

print("\nSequential fusion history:")

display(
    fusion_history_summary.round(6)
)


print(
    "\nUnavailable modalities should introduce zero conflict "
    "and leave the accumulated opinion unchanged."
)

print(
    "The TMC-fused independent opinion is ready for later "
    "combination with the interaction-aware 3MT opinion."
)

### 1.6.18. Producing the interaction-aware interaction pathway evidential opinion

The modality-specific evidence pathway combines independent modality opinions produced before cross-modal interaction. construct a separate evidential head for the final query produced by the interaction pathway cascade.

The final interaction-pathway representation is:

$$
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\in
\mathbb{R}^{64}.
$$

Unlike the independent modality representations, this vector contains cumulative information learned through the ordered sequence of Cascaded Modality Transformers.

The joint evidential head produces non-negative class evidence:

$$
e_{ik}^{\mathrm{joint}}
=
\operatorname{Softplus}
\left(
f_{\mathrm{joint}}
\left(
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\right)
\right),
$$

where \(k\) denotes either sMCI or pMCI.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{\mathrm{joint}}
=
e_{ik}^{\mathrm{joint}} + 1.
$$

The expected class probabilities are:

$$
p_{ik}^{\mathrm{joint}}
=
\frac{
\alpha_{ik}^{\mathrm{joint}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{joint}}
}.
$$

The joint uncertainty is:

$$
u_{i}^{\mathrm{joint}}
=
\frac{K}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{joint}}
}.
$$

This opinion serves a different purpose from the evidence pathway-fused independent opinion:

- the modality-specific opinion represents agreement and conflict between modality-specific predictions;
- the joint interaction pathway opinion represents the prediction obtained after learning cross-modal interactions.

### 1.6.19. Auxiliary outputs

The original interaction pathway architecture places an auxiliary classifier after each intermediate cross-modal interaction to provide direct training signals to earlier cascade stages.

The notebook preserves that principle by attaching one auxiliary classifier to every intermediate query except the final MRI stage. These auxiliary heads produce ordinary logits for the prognosis classes and are used only during training.

The auxiliary classifiers are not treated as independent modality-specific opinions because each intermediate query already contains information accumulated from all preceding modalities. They support gradient flow through the cascade but do not represent isolated modality evidence.

The final MRI-stage query receives the joint evidential head and produces the interaction-aware Dirichlet opinion.

In [ ]:
# ============================================================
# 12. Producing the interaction-aware interaction pathway evidential opinion
# ============================================================

# ------------------------------------------------------------
# Auxiliary classifier for one intermediate interaction pathway query
# ------------------------------------------------------------

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(self, representation):
        return self.network(
            representation
        )


# ------------------------------------------------------------
# Joint interaction pathway evidential and auxiliary heads
# ------------------------------------------------------------

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )


    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # --------------------------------------------------------
        # Intermediate auxiliary predictions
        # --------------------------------------------------------

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )


        # --------------------------------------------------------
        # Final joint evidential opinion
        # --------------------------------------------------------

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final interaction pathway query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }


# ------------------------------------------------------------
# Instantiate the interaction-pathway prediction heads
# ------------------------------------------------------------

three_mt_prediction_heads = ThreeMTPredictionHeads(
    cascade_order=THREE_MT_CASCADE_ORDER,
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
)


# ------------------------------------------------------------
# Produce the auxiliary and joint outputs
# ------------------------------------------------------------

three_mt_prediction_heads.eval()

with torch.no_grad():

    three_mt_predictions = (
        three_mt_prediction_heads(
            three_mt_output=three_mt_output
        )
    )


joint_opinion = three_mt_predictions[
    "joint_opinion"
]


# ------------------------------------------------------------
# Display the output structure
# ------------------------------------------------------------

print("=" * 72)
print("3MT PREDICTION HEADS")
print("=" * 72)

print(
    f"\nAuxiliary stages: "
    f"{three_mt_prediction_heads.auxiliary_stages}"
)

print(
    f"Final evidential stage: "
    f"{three_mt_prediction_heads.final_stage}"
)

print("\nAuxiliary-logit shapes:")

for stage_name, stage_logits in (
    three_mt_predictions[
        "auxiliary_logits"
    ].items()
):
    print(
        f"- after {stage_name}: "
        f"{tuple(stage_logits.shape)}"
    )


print("\nJoint evidential shapes:")

print(
    "  evidence: "
    f"{tuple(joint_opinion['evidence'].shape)}"
)

print(
    "  alpha: "
    f"{tuple(joint_opinion['alpha'].shape)}"
)

print(
    "  probabilities: "
    f"{tuple(joint_opinion['probabilities'].shape)}"
)

print(
    "  uncertainty: "
    f"{tuple(joint_opinion['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Participant-level joint opinions
# ------------------------------------------------------------

joint_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    joint_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ALPHA_sMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                joint_opinion["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


joint_opinion_summary = pd.DataFrame(
    joint_opinion_rows
)

print("\nInteraction-aware 3MT opinions:")

display(
    joint_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable 3MT prediction-head parameters: "
    f"{count_trainable_parameters(three_mt_prediction_heads):,}"
)

print(
    "\nThe auxiliary outputs will support training-time "
    "gradient flow through the cascade."
)

print(
    "The final joint opinion is ready for combination with "
    "the TMC-fused independent opinion."
)

### 1.6.20. Combining the interaction pathway and modality-specific evidence pathways with a constrained reliability gate

The model currently produces two complementary evidential outputs:

1. the interaction-aware interaction pathway opinion;
2. the independently fused modality-specific opinion.

These opinions are derived from the same underlying participant data and therefore should not be combined using Dempster--Shafer fusion as though they were independent evidence sources.

Instead, This notebook uses a participant-specific convex mixture of their calibrated evidence vectors.

### 1.6.21. Pathway calibration

The evidence magnitudes produced by the two pathways may have different numerical scales. In particular, the modality-specific evidence pathway accumulates evidence across several available modalities, whereas the cross-modal interaction pathway produces evidence from one joint head.

I therefore introduce one positive scalar calibration parameter for each pathway:

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
=
\tau_{\mathrm{interaction pathway}}
\mathbf{e}_{i}^{\mathrm{interaction pathway}},
$$

and

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}
=
\tau_{\mathrm{evidence pathway}}
\mathbf{e}_{i}^{\mathrm{evidence pathway}}.
$$

Both calibration factors are constrained to be positive using the Softplus function:

$$
\tau_r
=
\operatorname{Softplus}(\rho_r),
\qquad
r \in
\{\mathrm{interaction pathway},\mathrm{evidence pathway}\}.
$$

The parameters are initialised so that both calibration factors begin at approximately one.

### 1.6.22. Reliability-gate input

The reliability gate receives only uncertainty, conflict, and modality-availability information. It does not receive the original participant features or the full interaction-pathway representation.

For participant \(i\), the gate input is:

$$
\mathbf{r}_i
=
\left[
u_i^{\mathrm{interaction pathway}},
u_i^{\mathrm{evidence pathway}},
\bar{C}_i^{\mathrm{evidence pathway}},
\frac{n_i^{\mathrm{available}}}{M},
\mathbf{a}_i
\right],
$$

where:

- \(u_i^{\mathrm{interaction pathway}}\) is the uncertainty of the joint interaction pathway opinion;
- \(u_i^{\mathrm{evidence pathway}}\) is the uncertainty of the evidence pathway-fused opinion;
- \(\bar{C}_i^{\mathrm{evidence pathway}}\) is the mean conflict encountered when combining available modality opinions;
- \(n_i^{\mathrm{available}}\) is the number of available branches;
- \(M=6\) is the total number of branches;
- \(\mathbf{a}_i\) is the six-element branch-availability vector.

The gate produces one scalar weight:

$$
w_i
=
\sigma
\left(
\mathbf{w}^{\top}
\mathbf{r}_i+b
\right).
$$

The interpretation is:

$$
w_i \rightarrow 1
\quad
\Longrightarrow
\quad
\text{greater reliance on interaction pathway},
$$

and

$$
w_i \rightarrow 0
\quad
\Longrightarrow
\quad
\text{greater reliance on evidence pathway}.
$$

The gate is initialised with zero weights and zero bias. Therefore, before training:

$$
w_i = 0.5.
$$

This prevents either pathway from being preferred arbitrarily at model initialisation.

### 1.6.23. Final evidential opinion

The final evidence is:

$$
\mathbf{e}_{i}^{\mathrm{final}}
=
w_i
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
+
(1-w_i)
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}.
$$

Because \(w_i \in [0,1]\), this is a convex mixture rather than an addition of two supposedly independent evidence sources.

The final Dirichlet parameters are:

$$
\boldsymbol{\alpha}_{i}^{\mathrm{final}}
=
\mathbf{e}_{i}^{\mathrm{final}}
+
\mathbf{1}.
$$

The final class probabilities and uncertainty are:

$$
p_{ik}^{\mathrm{final}}
=
\frac{
\alpha_{ik}^{\mathrm{final}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{final}}
},
$$

and

$$
u_i^{\mathrm{final}}
=
\frac{
K
}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{final}}
}.
$$

The reliability statistics supplied to the gate are detached from the computational graph. This prevents the upstream pathways from manipulating their uncertainty or conflict values merely to obtain a larger gate weight. The final loss can still train both pathways through their evidence contributions.

In [ ]:
# ============================================================
# 13. Combining interaction pathway and evidence pathway with fixed equal fusion
# ============================================================

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )


class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )


    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )


    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


hybrid_fusion = FixedEqualHybridFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    number_of_modalities=len(
        MODALITY_ORDER
    ),
    initial_three_mt_scale=1.0,
    initial_tmc_scale=1.0,
)


hybrid_fusion.eval()

with torch.no_grad():
    hybrid_output = hybrid_fusion(
        joint_opinion=joint_opinion,
        tmc_output=tmc_output,
        branch_masks=example_batch[
            "branch_masks"
        ],
        modality_order=MODALITY_ORDER,
    )


print("=" * 72)
print("FIXED 50/50 3MT-TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    "\nFinal probability shape: "
    f"{tuple(hybrid_output['probabilities'].shape)}"
)

print(
    "Initial 3MT evidence scale: "
    f"{hybrid_output['three_mt_scale'].item():.6f}"
)

print(
    "Initial TMC evidence scale: "
    f"{hybrid_output['tmc_scale'].item():.6f}"
)

print(
    "Trainable fixed-fusion parameters: "
    f"{count_trainable_parameters(hybrid_fusion):,}"
)

maximum_three_mt_weight_error = (
    hybrid_output[
        "three_mt_weight"
    ]
    - 0.5
).abs().max().item()

maximum_tmc_weight_error = (
    hybrid_output[
        "tmc_weight"
    ]
    - 0.5
).abs().max().item()

probability_sum_error = (
    hybrid_output[
        "probabilities"
    ].sum(
        dim=-1
    )
    - 1.0
).abs().max().item()

print(
    "\nMaximum 3MT-weight deviation from 0.5: "
    f"{maximum_three_mt_weight_error:.10f}"
)

print(
    "Maximum TMC-weight deviation from 0.5: "
    f"{maximum_tmc_weight_error:.10f}"
)

print(
    "Maximum final probability-sum error: "
    f"{probability_sum_error:.10f}"
)

assert maximum_three_mt_weight_error == 0.0
assert maximum_tmc_weight_error == 0.0

print(
    "\nThe participant-specific reliability gate is absent. "
    "Only the two positive pathway evidence scales remain trainable."
)


### 1.6.24. Assembling the complete availability-gated interaction-evidence model

The complete model keeps the same six components used in the original fold-0 run:

1. modality-specific encoders;
2. training-time modality dropout;
3. independent modality evidential heads;
4. modality-specific evidence fusion;
5. the interaction pathway interaction pathway;
6. fixed-equal hybrid evidence fusion.

The effective branch masks are created before encoding. They contain both natural missingness and any additional modality removed by training-time dropout. These exact masks are now passed into the interaction pathway cascade, so every unavailable cross-modal interaction stage preserves the preceding cumulative query.

The modality-specific evidence pathway and fixed equal fusion are unchanged. This isolates the effect of availability-gating the cross-modal interaction updates.

In [ ]:
# ============================================================
# 14. Assembling the complete end-to-end interaction-evidence model
# ============================================================

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )


        # --------------------------------------------------------
        # Modality-specific encoders
        # --------------------------------------------------------

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )


        # --------------------------------------------------------
        # Independent modality evidential pathway
        # --------------------------------------------------------

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )


        # --------------------------------------------------------
        # Final fixed 50/50 hybrid fusion
        # --------------------------------------------------------

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )


    # ------------------------------------------------------------
    # Training-time modality dropout
    # ------------------------------------------------------------

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )


        # --------------------------------------------------------
        # Sample branch-retention indicators
        # --------------------------------------------------------

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )


        # --------------------------------------------------------
        # Prevent complete information removal
        # --------------------------------------------------------

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # I randomly restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0


        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )


    # ------------------------------------------------------------
    # Construct effective modality dictionaries
    # ------------------------------------------------------------

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities


    # ------------------------------------------------------------
    # Complete forward pass
    # ------------------------------------------------------------

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # --------------------------------------------------------
        # Apply training-time modality dropout
        # --------------------------------------------------------

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )


        # --------------------------------------------------------
        # Encode all six modalities
        # --------------------------------------------------------

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. I use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]


        # --------------------------------------------------------
        # Independent modality opinions and modality-specific evidence fusion
        # --------------------------------------------------------

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]


        # --------------------------------------------------------
        # Final fixed-equal hybrid opinion
        # --------------------------------------------------------

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )


        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


# ------------------------------------------------------------
# Instantiate the complete model
# ------------------------------------------------------------

complete_model = ADNIEvidential3MTTMCModel(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    modality_dropout_probability=0.50,
)


# ------------------------------------------------------------
# Evaluation-mode end-to-end forward pass
# ------------------------------------------------------------

# In evaluation mode, modality dropout is disabled.
complete_model.eval()

with torch.no_grad():

    complete_model_output = complete_model(
        modalities=example_batch[
            "modalities"
        ],

        original_branch_masks=example_batch[
            "branch_masks"
        ],
    )


# ------------------------------------------------------------
# Inspect final outputs
# ------------------------------------------------------------

final_output = complete_model_output[
    "final_output"
]

print("=" * 72)
print("COMPLETE AVAILABILITY-GATED 3MT-TMC MODEL")
print("=" * 72)

print(
    "\nModel mode: "
    f"{'training' if complete_model.training else 'evaluation'}"
)

print(
    "Modality-dropout probability: "
    f"{complete_model.modality_dropout_probability:.2f}"
)

print(
    "\nOriginal branch-mask shape: "
    f"{tuple(complete_model_output['original_branch_masks'].shape)}"
)

print(
    "Effective branch-mask shape: "
    f"{tuple(complete_model_output['effective_branch_masks'].shape)}"
)

print(
    "\nFinal alpha shape: "
    f"{tuple(final_output['alpha'].shape)}"
)

print(
    "Final probability shape: "
    f"{tuple(final_output['probabilities'].shape)}"
)

print(
    "Final uncertainty shape: "
    f"{tuple(final_output['uncertainty'].shape)}"
)

print(
    "\nTotal trainable model parameters: "
    f"{count_trainable_parameters(complete_model):,}"
)


# ------------------------------------------------------------
# Verify that evaluation mode preserves genuine availability
# ------------------------------------------------------------

evaluation_mask_difference = (
    complete_model_output[
        "effective_branch_masks"
    ]
    - complete_model_output[
        "original_branch_masks"
    ]
).abs().max().item()

print(
    "\nMaximum evaluation-mode difference between original "
    f"and effective branch masks: "
    f"{evaluation_mask_difference:.10f}"
)


# ------------------------------------------------------------
# Participant-level output summary
# ------------------------------------------------------------

complete_model_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    original_available_count = int(
        complete_model_output[
            "original_branch_masks"
        ][batch_row].sum().item()
    )

    effective_available_count = int(
        complete_model_output[
            "effective_branch_masks"
        ][batch_row].sum().item()
    )

    complete_model_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ORIGINAL_MODALITIES":
                original_available_count,

            "EFFECTIVE_MODALITIES":
                effective_available_count,

            "W_3MT": float(
                final_output[
                    "three_mt_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "W_TMC": float(
                final_output[
                    "tmc_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_sMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    1,
                ].item()
            ),

            "FINAL_UNCERTAINTY": float(
                final_output[
                    "uncertainty"
                ][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


complete_model_summary = pd.DataFrame(
    complete_model_rows
)

print("\nComplete-model evaluation-mode outputs:")

display(
    complete_model_summary.round(6)
)

print(
    "\nThe complete model now passes the effective branch "
    "masks into every CMT stage."
)


### 1.6.25. Defining the joint training objective

The complete architecture produces several supervised outputs with different purposes:

1. the final fixed-equal hybrid opinion;
2. the interaction-aware interaction pathway joint opinion;
3. the evidence pathway-fused independent-modality opinion;
4. six independent modality-specific opinions;
5. five intermediate interaction pathway auxiliary predictions.

These outputs are trained jointly, but the final hybrid prediction remains the primary objective.

The total loss is:

$$
\mathcal{L}_{\mathrm{total}}
=
\mathcal{L}_{\mathrm{final}}
+
\lambda_{\mathrm{joint}}
\mathcal{L}_{\mathrm{joint}}
+
\lambda_{\mathrm{evidence pathway}}
\mathcal{L}_{\mathrm{evidence pathway}}
+
\lambda_{\mathrm{mod}}
\mathcal{L}_{\mathrm{mod}}
+
\lambda_{\mathrm{aux}}
\mathcal{L}_{\mathrm{aux}}
+
\lambda_{\mathrm{gate}}
\mathcal{L}_{\mathrm{gate}}.
$$

The initial loss weights are:

$$
\lambda_{\mathrm{joint}} = 0.5,
\qquad
\lambda_{\mathrm{evidence pathway}} = 0.5,
\qquad
\lambda_{\mathrm{mod}} = 0.1,
$$

$$
\lambda_{\mathrm{aux}} = 0.1,
\qquad
\lambda_{\mathrm{gate}} = 0.01.
$$

The final hybrid loss has coefficient one and therefore remains the dominant objective. The remaining terms provide direct supervision to the two pathways and their intermediate components.

These coefficients are initial modelling choices. Any comparison of alternative values must use only the training and validation partitions.

### 1.6.26. Evidential classification loss

For a Dirichlet prediction:

$$
\boldsymbol{\alpha}_i
=
\mathbf{e}_i+\mathbf{1},
$$

the expected cross-entropy loss is:

$$
\mathcal{L}_{\mathrm{ECE},i}
=
\sum_{k=1}^{K}
y_{ik}
\left[
\psi(S_i)
-
\psi(\alpha_{ik})
\right],
$$

where:

$$
S_i
=
\sum_{k=1}^{K}
\alpha_{ik},
$$

and \(\psi(\cdot)\) denotes the digamma function.

This objective minimises the expected negative log-likelihood under the predicted Dirichlet distribution.

### 1.6.27. Evidence regularisation

An evidential network may become unjustifiably confident by assigning strong evidence to an incorrect class. I therefore add a Kullback--Leibler regularisation term that discourages unsupported evidence.

The adjusted Dirichlet parameters are:

$$
\widetilde{\boldsymbol{\alpha}}_i
=
\mathbf{y}_i
+
(1-\mathbf{y}_i)
\odot
\boldsymbol{\alpha}_i.
$$

This construction removes the evidence assigned to the correct class from the regularisation term while penalising evidence assigned to incorrect classes.

The regularisation term is:

$$
\mathcal{L}_{\mathrm{KL},i}
=
D_{\mathrm{KL}}
\left[
\operatorname{Dir}
\left(
\widetilde{\boldsymbol{\alpha}}_i
\right)
\parallel
\operatorname{Dir}
\left(
\mathbf{1}
\right)
\right].
$$

The complete evidential loss is:

$$
\mathcal{L}_{\mathrm{EDL},i}
=
\mathcal{L}_{\mathrm{ECE},i}
+
\beta_t
\mathcal{L}_{\mathrm{KL},i}.
$$

The regularisation coefficient is annealed during the first training epochs:

$$
\beta_t
=
\min
\left(
1,
\frac{t}{T_{\mathrm{anneal}}}
\right),
$$

where \(t\) is the current epoch and \(T_{\mathrm{anneal}}\) is initially set to ten epochs.

This allows the model to begin learning the classification task before the full evidence penalty is applied.

### 1.6.28. Modality-specific loss

Each independent modality opinion is supervised only when that modality is effectively available after training-time modality dropout.

For branch \(m\), let:

$$
\widetilde{a}_i^{(m)}
\in
\{0,1\}
$$

denote the effective branch mask.

The modality-specific loss is:

$$
\mathcal{L}_{\mathrm{mod}}
=
\frac{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
\mathcal{L}_{\mathrm{EDL},i}^{(m)}
}{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
+
\varepsilon
}.
$$

Naturally unavailable or deliberately dropped branches therefore contribute no modality-specific classification loss.

### 1.6.29. Auxiliary interaction pathway loss

The intermediate interaction pathway heads produce ordinary class logits rather than Dirichlet opinions. Their loss is the mean cross-entropy across the five intermediate cascade stages:

$$
\mathcal{L}_{\mathrm{aux}}
=
\frac{1}{J}
\sum_{j=1}^{J}
\operatorname{CE}
\left(
\mathbf{z}^{(j)},
y
\right),
$$

where \(J=5\).

These losses provide direct gradient signals to earlier stages of the cascaded transformer.

### 1.6.30. Gate regularisation

The reliability gate is initialised at:

$$
w_i=0.5.
$$

A weak early-training regulariser discourages immediate collapse to a single pathway:

$$
\mathcal{L}_{\mathrm{gate}}
=
\left(
\frac{1}{N}
\sum_{i=1}^{N}
w_i
-
0.5
\right)^2.
$$

The gate regularisation is annealed to zero after the initial training period. It therefore stabilises early optimisation without forcing the final trained gate to remain balanced.

This cell defines and validates the training objective only. Parameter updates begin after gradient flow is checked in the following step.

In [ ]:
# ============================================================
# 15. Defining the complete joint training objective
# ============================================================

# ------------------------------------------------------------
# KL divergence between a predicted Dirichlet distribution and
# a uniform Dirichlet distribution
# ------------------------------------------------------------

def dirichlet_kl_to_uniform(
    alpha,
):
    """
    Calculate:

        KL(Dir(alpha) || Dir(1))

    for every participant in the batch.

    Parameters
    ----------
    alpha:
        Positive Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    Returns
    -------
    Tensor with shape:
        (batch_size,)
    """

    number_of_classes = alpha.shape[-1]

    uniform_alpha = torch.ones_like(
        alpha
    )

    alpha_strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    uniform_strength = uniform_alpha.sum(
        dim=-1,
        keepdim=True,
    )

    log_normalisation_ratio = (
        torch.lgamma(alpha_strength)
        - torch.lgamma(uniform_strength)
        - torch.lgamma(alpha).sum(
            dim=-1,
            keepdim=True,
        )
        + torch.lgamma(uniform_alpha).sum(
            dim=-1,
            keepdim=True,
        )
    )

    digamma_difference = (
        torch.digamma(alpha)
        - torch.digamma(alpha_strength)
    )

    parameter_difference = (
        alpha
        - uniform_alpha
    )

    expectation_term = (
        parameter_difference
        * digamma_difference
    ).sum(
        dim=-1,
        keepdim=True,
    )

    kl_divergence = (
        log_normalisation_ratio
        + expectation_term
    )

    return kl_divergence.squeeze(-1)


# ------------------------------------------------------------
# Evidential classification loss
# ------------------------------------------------------------

def evidential_classification_loss(
    alpha,
    targets,
    number_of_classes,
    annealing_coefficient,
    class_weights=None,
    reduction="mean",
):
    """
    Calculate the expected cross-entropy under a Dirichlet
    distribution together with annealed KL regularisation.

    Parameters
    ----------
    alpha:
        Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    targets:
        Integer class labels with shape:
        (batch_size,)

    number_of_classes:
        Number of prognosis classes.

    annealing_coefficient:
        Current coefficient applied to the KL term.

    class_weights:
        Optional class-weight tensor with shape:
        (number_of_classes,)

    reduction:
        "none", "mean", or "sum".
    """

    targets = targets.long()

    one_hot_targets = F.one_hot(
        targets,
        num_classes=number_of_classes,
    ).to(
        dtype=alpha.dtype
    )

    strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )


    # --------------------------------------------------------
    # Expected cross-entropy under the Dirichlet distribution
    # --------------------------------------------------------

    expected_cross_entropy_by_class = (
        torch.digamma(strength)
        - torch.digamma(alpha)
    )

    expected_cross_entropy = (
        one_hot_targets
        * expected_cross_entropy_by_class
    ).sum(
        dim=-1
    )


    # --------------------------------------------------------
    # Optional class weighting
    # --------------------------------------------------------

    if class_weights is not None:

        sample_weights = class_weights[
            targets
        ].to(
            dtype=alpha.dtype,
            device=alpha.device,
        )

        expected_cross_entropy = (
            expected_cross_entropy
            * sample_weights
        )


    # --------------------------------------------------------
    # Remove correct-class evidence from the KL penalty
    # --------------------------------------------------------

    adjusted_alpha = (
        one_hot_targets
        +
        (
            1.0
            - one_hot_targets
        )
        * alpha
    )

    kl_regularisation = (
        dirichlet_kl_to_uniform(
            adjusted_alpha
        )
    )

    per_sample_loss = (
        expected_cross_entropy
        +
        annealing_coefficient
        * kl_regularisation
    )


    # --------------------------------------------------------
    # Requested reduction
    # --------------------------------------------------------

    if reduction == "none":
        reduced_loss = per_sample_loss

    elif reduction == "mean":
        reduced_loss = per_sample_loss.mean()

    elif reduction == "sum":
        reduced_loss = per_sample_loss.sum()

    else:
        raise ValueError(
            "reduction must be 'none', 'mean', or 'sum'."
        )


    return {
        "loss":
            reduced_loss,

        "per_sample_loss":
            per_sample_loss,

        "expected_cross_entropy":
            expected_cross_entropy,

        "kl_regularisation":
            kl_regularisation,
    }


# ------------------------------------------------------------
# Complete multi-output objective
# ------------------------------------------------------------

class HybridEvidentialTrainingLoss(nn.Module):
    """
    Jointly supervise the final hybrid output, both main pathways,
    the individual available modality opinions, and the auxiliary
    3MT classifiers.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        evidential_annealing_epochs=10,
        gate_regularisation_epochs=0,
        joint_loss_weight=0.50,
        tmc_loss_weight=0.50,
        modality_loss_weight=0.10,
        auxiliary_loss_weight=0.10,
        gate_loss_weight=0.0,
        class_weights=None,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.evidential_annealing_epochs = (
            evidential_annealing_epochs
        )

        self.gate_regularisation_epochs = (
            gate_regularisation_epochs
        )

        self.joint_loss_weight = (
            joint_loss_weight
        )

        self.tmc_loss_weight = (
            tmc_loss_weight
        )

        self.modality_loss_weight = (
            modality_loss_weight
        )

        self.auxiliary_loss_weight = (
            auxiliary_loss_weight
        )

        self.gate_loss_weight = (
            gate_loss_weight
        )


        # --------------------------------------------------------
        # Optional training-fold class weights
        # --------------------------------------------------------

        if class_weights is None:

            self.register_buffer(
                "class_weights",
                None,
            )

        else:

            class_weights = torch.as_tensor(
                class_weights,
                dtype=torch.float32,
            )

            if class_weights.shape != (
                number_of_classes,
            ):
                raise ValueError(
                    "class_weights must contain one value "
                    "for each class."
                )

            self.register_buffer(
                "class_weights",
                class_weights,
            )


    # ------------------------------------------------------------
    # Annealing schedules
    # ------------------------------------------------------------

    def _evidential_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Increase the KL coefficient linearly from zero to one.
        """

        if self.evidential_annealing_epochs <= 0:
            return 1.0

        return min(
            1.0,
            float(epoch)
            / float(
                self.evidential_annealing_epochs
            ),
        )


    def _gate_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Reduce the gate-balance penalty to zero after the initial
        optimisation period.
        """

        if self.gate_regularisation_epochs <= 0:
            return 0.0

        return max(
            0.0,
            1.0
            -
            (
                float(epoch - 1)
                /
                float(
                    self.gate_regularisation_epochs
                )
            ),
        )


    # ------------------------------------------------------------
    # Complete loss calculation
    # ------------------------------------------------------------

    def forward(
        self,
        model_output,
        targets,
        epoch,
    ):
        targets = targets.long()

        evidential_annealing = (
            self._evidential_annealing_coefficient(
                epoch
            )
        )

        gate_annealing = (
            self._gate_annealing_coefficient(
                epoch
            )
        )


        # --------------------------------------------------------
        # Final hybrid evidential loss
        # --------------------------------------------------------

        final_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "final_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        final_loss = final_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Interaction-aware interaction pathway evidential loss
        # --------------------------------------------------------

        joint_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "joint_opinion"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        joint_loss = joint_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # evidence pathway-fused evidential loss
        # --------------------------------------------------------

        tmc_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "tmc_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        tmc_loss = tmc_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Independent available-modality evidential loss
        # --------------------------------------------------------

        effective_branch_masks = model_output[
            "effective_branch_masks"
        ]

        weighted_modality_loss_sum = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        available_opinion_count = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        modality_loss_by_name = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):

            modality_alpha = model_output[
                "modality_opinions"
            ][modality_name]["alpha"]

            modality_loss_components = (
                evidential_classification_loss(
                    alpha=modality_alpha,

                    targets=targets,

                    number_of_classes=
                        self.number_of_classes,

                    annealing_coefficient=
                        evidential_annealing,

                    class_weights=
                        self.class_weights,

                    reduction="none",
                )
            )

            per_sample_modality_loss = (
                modality_loss_components[
                    "per_sample_loss"
                ]
            )

            modality_mask = effective_branch_masks[
                :,
                modality_index,
            ].to(
                dtype=per_sample_modality_loss.dtype
            )

            masked_modality_loss_sum = (
                per_sample_modality_loss
                * modality_mask
            ).sum()

            modality_available_count = (
                modality_mask.sum()
            )

            weighted_modality_loss_sum = (
                weighted_modality_loss_sum
                + masked_modality_loss_sum
            )

            available_opinion_count = (
                available_opinion_count
                + modality_available_count
            )

            modality_loss_by_name[
                modality_name
            ] = (
                masked_modality_loss_sum
                /
                modality_available_count.clamp_min(
                    1.0
                )
            )

        modality_loss = (
            weighted_modality_loss_sum
            /
            available_opinion_count.clamp_min(
                1.0
            )
        )


        # --------------------------------------------------------
        # Intermediate interaction pathway auxiliary cross-entropy loss
        # --------------------------------------------------------

        auxiliary_logits = model_output[
            "three_mt_predictions"
        ]["auxiliary_logits"]

        auxiliary_loss_by_stage = {}

        auxiliary_losses = []

        for stage_name, stage_logits in (
            auxiliary_logits.items()
        ):

            stage_loss = F.cross_entropy(
                input=stage_logits,
                target=targets,
                weight=self.class_weights,
            )

            auxiliary_loss_by_stage[
                stage_name
            ] = stage_loss

            auxiliary_losses.append(
                stage_loss
            )

        if auxiliary_losses:

            auxiliary_loss = torch.stack(
                auxiliary_losses
            ).mean()

        else:

            auxiliary_loss = torch.zeros(
                (),
                dtype=final_loss.dtype,
                device=final_loss.device,
            )


        # --------------------------------------------------------
        # Early gate-balance regularisation
        # --------------------------------------------------------

        three_mt_weights = model_output[
            "final_output"
        ]["three_mt_weight"]

        raw_gate_loss = (
            three_mt_weights.mean()
            - 0.5
        ).pow(2)

        annealed_gate_loss = (
            gate_annealing
            * raw_gate_loss
        )


        # --------------------------------------------------------
        # Weighted total objective
        # --------------------------------------------------------

        total_loss = (
            final_loss
            +
            self.joint_loss_weight
            * joint_loss
            +
            self.tmc_loss_weight
            * tmc_loss
            +
            self.modality_loss_weight
            * modality_loss
            +
            self.auxiliary_loss_weight
            * auxiliary_loss
            +
            self.gate_loss_weight
            * annealed_gate_loss
        )


        return {
            "total_loss":
                total_loss,

            "final_loss":
                final_loss,

            "joint_loss":
                joint_loss,

            "tmc_loss":
                tmc_loss,

            "modality_loss":
                modality_loss,

            "auxiliary_loss":
                auxiliary_loss,

            "raw_gate_loss":
                raw_gate_loss,

            "annealed_gate_loss":
                annealed_gate_loss,

            "evidential_annealing":
                torch.tensor(
                    evidential_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "gate_annealing":
                torch.tensor(
                    gate_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "available_opinion_count":
                available_opinion_count,

            "modality_loss_by_name":
                modality_loss_by_name,

            "auxiliary_loss_by_stage":
                auxiliary_loss_by_stage,

            "final_expected_cross_entropy":
                final_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "final_kl_regularisation":
                final_loss_components[
                    "kl_regularisation"
                ].mean(),

            "joint_expected_cross_entropy":
                joint_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "joint_kl_regularisation":
                joint_loss_components[
                    "kl_regularisation"
                ].mean(),

            "tmc_expected_cross_entropy":
                tmc_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "tmc_kl_regularisation":
                tmc_loss_components[
                    "kl_regularisation"
                ].mean(),
        }


# ------------------------------------------------------------
# Instantiate the training objective
# ------------------------------------------------------------

training_objective = HybridEvidentialTrainingLoss(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,

    evidential_annealing_epochs=10,
    gate_regularisation_epochs=0,

    joint_loss_weight=0.50,
    tmc_loss_weight=0.50,
    modality_loss_weight=0.10,
    auxiliary_loss_weight=0.10,
    gate_loss_weight=0.0,

    # I initially leave class weighting disabled. It can be added
    # using weights calculated from each training fold only.
    class_weights=None,
)


# ------------------------------------------------------------
# Test the objective on the complete untrained forward pass
# ------------------------------------------------------------

example_loss_output = training_objective(
    model_output=complete_model_output,

    targets=example_batch[
        "target"
    ],

    epoch=1,
)


# ------------------------------------------------------------
# Display the initial loss structure
# ------------------------------------------------------------

print("=" * 72)
print("JOINT 3MT-TMC TRAINING OBJECTIVE")
print("=" * 72)

print(
    "\nEvidential KL annealing coefficient at epoch 1: "
    f"{example_loss_output['evidential_annealing'].item():.6f}"
)

print(
    "Gate regularisation coefficient at epoch 1: "
    f"{example_loss_output['gate_annealing'].item():.6f}"
)

print("\nMain loss components:")

for loss_name in [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]:
    print(
        f"- {loss_name}: "
        f"{example_loss_output[loss_name].item():.6f}"
    )


print("\nFinal evidential-loss decomposition:")

print(
    "- expected cross-entropy: "
    f"{example_loss_output['final_expected_cross_entropy'].item():.6f}"
)

print(
    "- KL regularisation: "
    f"{example_loss_output['final_kl_regularisation'].item():.6f}"
)


print("\nAvailable modality opinions in this batch: "
      f"{int(example_loss_output['available_opinion_count'].item())}")


# ------------------------------------------------------------
# Modality-specific loss summary
# ------------------------------------------------------------

modality_loss_rows = []

for modality_name in MODALITY_ORDER:

    modality_loss_rows.append(
        {
            "MODALITY": modality_name,

            "MEAN_AVAILABLE_LOSS": float(
                example_loss_output[
                    "modality_loss_by_name"
                ][modality_name].item()
            ),
        }
    )


print("\nModality-specific evidential losses:")

display(
    pd.DataFrame(
        modality_loss_rows
    ).round(6)
)


# ------------------------------------------------------------
# Auxiliary-stage loss summary
# ------------------------------------------------------------

auxiliary_loss_rows = []

for stage_name in THREE_MT_CASCADE_ORDER[:-1]:

    auxiliary_loss_rows.append(
        {
            "AUXILIARY_STAGE": stage_name,

            "CROSS_ENTROPY_LOSS": float(
                example_loss_output[
                    "auxiliary_loss_by_stage"
                ][stage_name].item()
            ),
        }
    )


print("\nIntermediate 3MT auxiliary losses:")

display(
    pd.DataFrame(
        auxiliary_loss_rows
    ).round(6)
)

print(
    "\nThe joint objective is ready for one end-to-end "
    "backward pass."
)


### 1.6.31. Running one end-to-end backward pass

Before configuring the optimiser, I run one training batch through the complete gated model and joint objective.

For this single diagnostic pass, modality dropout is set to zero so that the natural fold-0 availability pattern is used without additional random branch removal. The total loss is backpropagated, but no optimiser step is taken. The cell reports the total loss and the global gradient norm, then restores the configured modality-dropout probability of `0.50`.

In [ ]:
# ============================================================
# 16. Running one end-to-end backward pass
# ============================================================

diagnostic_batch = next(
    iter(train_loader)
)

original_modality_dropout_probability = (
    complete_model.modality_dropout_probability
)

complete_model.modality_dropout_probability = 0.0
complete_model.train()
complete_model.zero_grad(
    set_to_none=True
)


diagnostic_output = complete_model(
    modalities=diagnostic_batch[
        "modalities"
    ],
    original_branch_masks=diagnostic_batch[
        "branch_masks"
    ],
)

diagnostic_losses = training_objective(
    model_output=diagnostic_output,
    targets=diagnostic_batch[
        "target"
    ],
    epoch=1,
)

diagnostic_total_loss = diagnostic_losses[
    "total_loss"
]

diagnostic_total_loss.backward()


global_gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().float().pow(2).sum()
        for parameter in complete_model.parameters()
        if parameter.grad is not None
    )
)

print("=" * 72)
print("END-TO-END BACKWARD PASS")
print("=" * 72)

print(
    "\nTotal loss: "
    f"{diagnostic_total_loss.item():.6f}"
)

print(
    "Global gradient norm: "
    f"{global_gradient_norm.item():.6f}"
)

complete_model.zero_grad(
    set_to_none=True
)

complete_model.modality_dropout_probability = (
    original_modality_dropout_probability
)

complete_model.eval()

print(
    "Modality-dropout probability restored to: "
    f"{complete_model.modality_dropout_probability:.2f}"
)


### 1.6.32. Configuring optimisation and experiment-specific output paths

The original training hyperparameters remain unchanged. Every checkpoint, history file, validation prediction, and test prediction is written only under:

```text
models/3mt_tmc_evidential/experiments/
    gated_cmt_learned_gate_md050/
        mci_prognosis/fold_0/
```

This notebook does not use the older shared `training/mci_prognosis/fold_0/` directory.

In [ ]:
# ============================================================
# 17. Configuring optimisation, checkpoints, and metrics
# ============================================================

import os
import random
from datetime import datetime

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# Current experiment
# ------------------------------------------------------------

# EXPERIMENT_NAME, SELECTED_TASK, and SELECTED_FOLD were fixed when this fold's prepared input was loaded.


# ------------------------------------------------------------
# Reproducibility configuration
# ------------------------------------------------------------

GLOBAL_RANDOM_SEED = 42


def set_global_random_seed(seed):
    """
    Set the random seed used by Python, NumPy, and PyTorch.
    """

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_random_seed(
    GLOBAL_RANDOM_SEED
)


# ------------------------------------------------------------
# PyTorch numerical configuration
# ------------------------------------------------------------

# I allow cuDNN to choose efficient convolution algorithms.
#
# This is appropriate for the computationally expensive 3D MRI
# encoder. Exact bitwise reproducibility can still depend on the
# installed PyTorch, CUDA, cuDNN, and GPU versions.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 72)
print("TRAINING CONFIGURATION")
print("=" * 72)

print(
    f"\nExperiment: {EXPERIMENT_NAME}"
)

print(
    f"Task: {SELECTED_TASK}"
)

print(
    f"Selected fold: {SELECTED_FOLD}"
)

print(
    f"Device: {DEVICE}"
)

if torch.cuda.is_available():

    print(
        "CUDA device: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        "CUDA memory allocated before model transfer: "
        f"{torch.cuda.memory_allocated(0) / (1024 ** 3):.3f} GB"
    )


# ------------------------------------------------------------
# Move the complete model and objective to the selected device
# ------------------------------------------------------------

complete_model = complete_model.to(
    DEVICE
)

training_objective = training_objective.to(
    DEVICE
)


# ------------------------------------------------------------
# Optimisation hyperparameters
# ------------------------------------------------------------

MAXIMUM_EPOCHS = 50

INITIAL_LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAXIMUM_GRADIENT_NORM = 5.0

EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 3

SCHEDULER_REDUCTION_FACTOR = 0.5

MINIMUM_LEARNING_RATE = 1e-6

CLASSIFICATION_THRESHOLD = 0.50


# ------------------------------------------------------------
# AdamW optimiser
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    params=complete_model.parameters(),
    lr=INITIAL_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ------------------------------------------------------------
# Validation-AUC learning-rate scheduler
# ------------------------------------------------------------

learning_rate_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="max",
        factor=SCHEDULER_REDUCTION_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)


# ------------------------------------------------------------
# Checkpoint and history directories
# ------------------------------------------------------------

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
)

FOLD_TRAINING_DIR = (
    EXPERIMENT_ROOT
    / SELECTED_TASK
    / f"fold_{SELECTED_FOLD}"
)

CHECKPOINT_DIR = (
    FOLD_TRAINING_DIR
    / "checkpoints"
)

HISTORY_DIR = (
    FOLD_TRAINING_DIR
    / "history"
)

PREDICTION_DIR = (
    FOLD_TRAINING_DIR
    / "predictions"
)


BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_validation_auc_checkpoint.pt"
)

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last_epoch_checkpoint.pt"
)


# ------------------------------------------------------------
# Fold-output safety policy
# ------------------------------------------------------------

existing_fold_files = []

if FOLD_TRAINING_DIR.exists():
    existing_fold_files = [
        path
        for path in FOLD_TRAINING_DIR.rglob("*")
        if path.is_file()
    ]


if FOLD_RUN_MODE == "fresh" and existing_fold_files:
    raise FileExistsError(
        "Fresh training was requested, but this fold directory "
        "already contains files. Nothing was overwritten. "
        "Use a different experiment name, remove the intentionally "
        "discarded fold directory, or select resume mode only for "
        "an interrupted run of this exact fold.\n"
        f"Fold directory: {FOLD_TRAINING_DIR}"
    )


if FOLD_RUN_MODE == "resume" and not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Resume mode was requested, but this fold has no latest "
        "checkpoint. Nothing was changed.\n"
        f"Expected checkpoint: {LAST_CHECKPOINT_PATH}"
    )


for directory_path in [
    EXPERIMENT_ROOT,
    FOLD_TRAINING_DIR,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    PREDICTION_DIR,
]:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

TRAINING_HISTORY_PATH = (
    HISTORY_DIR
    / "training_history.csv"
)

TRAINING_CONFIGURATION_PATH = (
    HISTORY_DIR
    / "training_configuration.json"
)

VALIDATION_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "best_validation_predictions.csv"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "test_predictions.csv"
)


# ------------------------------------------------------------
# Record the training configuration
# ------------------------------------------------------------

training_configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "fixed_equal_fusion": True,
    "three_mt_weight": 0.5,
    "tmc_weight": 0.5,
    "task": SELECTED_TASK,
    "fold": int(SELECTED_FOLD),
    "random_seed": int(GLOBAL_RANDOM_SEED),

    "maximum_epochs": int(
        MAXIMUM_EPOCHS
    ),

    "batch_size": int(
        train_loader.batch_size
    ),

    "initial_learning_rate": float(
        INITIAL_LEARNING_RATE
    ),

    "weight_decay": float(
        WEIGHT_DECAY
    ),

    "maximum_gradient_norm": float(
        MAXIMUM_GRADIENT_NORM
    ),

    "early_stopping_patience": int(
        EARLY_STOPPING_PATIENCE
    ),

    "scheduler_patience": int(
        SCHEDULER_PATIENCE
    ),

    "scheduler_reduction_factor": float(
        SCHEDULER_REDUCTION_FACTOR
    ),

    "minimum_learning_rate": float(
        MINIMUM_LEARNING_RATE
    ),

    "classification_threshold": float(
        CLASSIFICATION_THRESHOLD
    ),

    "modality_dropout_probability": float(
        complete_model.modality_dropout_probability
    ),

    "embedding_dimension": int(
        MODALITY_EMBEDDING_DIM
    ),

    "number_of_classes": int(
        NUMBER_OF_CLASSES
    ),

    "class_order": list(
        PROGNOSIS_CLASS_ORDER
    ),

    "modality_order": list(
        MODALITY_ORDER
    ),

    "three_mt_cascade_order": list(
        THREE_MT_CASCADE_ORDER
    ),

    "trainable_parameters": int(
        count_trainable_parameters(
            complete_model
        )
    ),

    "device": str(
        DEVICE
    ),

    "cuda_device_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}


with open(
    TRAINING_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        training_configuration,
        configuration_file,
        indent=2,
    )


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def calculate_binary_expected_calibration_error(
    targets,
    positive_class_probabilities,
    number_of_bins=10,
):
    """
    Calculate equal-width binary expected calibration error.

    Confidence is the probability assigned to the predicted class.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        return float("nan")

    predicted_classes = (
        positive_class_probabilities
        >= 0.5
    ).astype(
        np.int64
    )

    predicted_confidences = np.where(
        predicted_classes == 1,
        positive_class_probabilities,
        1.0 - positive_class_probabilities,
    )

    prediction_correctness = (
        predicted_classes
        == targets
    ).astype(
        np.float64
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    expected_calibration_error = 0.0

    sample_count = targets.size

    for bin_index in range(
        number_of_bins
    ):

        lower_edge = bin_edges[
            bin_index
        ]

        upper_edge = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:

            in_bin = (
                predicted_confidences
                >= lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        else:

            in_bin = (
                predicted_confidences
                > lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        bin_count = int(
            in_bin.sum()
        )

        if bin_count == 0:
            continue

        mean_confidence = float(
            predicted_confidences[
                in_bin
            ].mean()
        )

        mean_accuracy = float(
            prediction_correctness[
                in_bin
            ].mean()
        )

        expected_calibration_error += (
            bin_count
            / sample_count
        ) * abs(
            mean_accuracy
            - mean_confidence
        )

    return float(
        expected_calibration_error
    )


# ------------------------------------------------------------
# Safe metric helpers
# ------------------------------------------------------------

def safely_calculate_roc_auc(
    targets,
    probabilities,
):
    """
    Return NaN when ROC AUC is undefined because only one class
    is present in the supplied targets.
    """

    if np.unique(targets).size < 2:
        return float("nan")

    return float(
        roc_auc_score(
            targets,
            probabilities,
        )
    )


def safely_calculate_average_precision(
    targets,
    probabilities,
):
    """
    Return NaN when average precision is not meaningful because
    the supplied targets contain no positive examples.
    """

    if np.sum(targets == 1) == 0:
        return float("nan")

    return float(
        average_precision_score(
            targets,
            probabilities,
        )
    )


# ------------------------------------------------------------
# Complete binary prognosis metrics
# ------------------------------------------------------------

def calculate_binary_classification_metrics(
    targets,
    positive_class_probabilities,
    uncertainties=None,
    three_mt_weights=None,
    classification_threshold=0.50,
):
    """
    Calculate discrimination, classification, calibration, and
    uncertainty summaries for the pMCI-positive prognosis task.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        raise ValueError(
            "At least one target is required to calculate metrics."
        )

    if (
        targets.shape[0]
        != positive_class_probabilities.shape[0]
    ):
        raise ValueError(
            "Targets and probabilities must contain the same "
            "number of participants."
        )

    positive_class_probabilities = np.clip(
        positive_class_probabilities,
        0.0,
        1.0,
    )

    predicted_classes = (
        positive_class_probabilities
        >= classification_threshold
    ).astype(
        np.int64
    )

    (
        true_negative,
        false_positive,
        false_negative,
        true_positive,
    ) = confusion_matrix(
        targets,
        predicted_classes,
        labels=[0, 1],
    ).ravel()


    sensitivity_denominator = (
        true_positive
        + false_negative
    )

    specificity_denominator = (
        true_negative
        + false_positive
    )

    sensitivity = (
        true_positive
        / sensitivity_denominator
        if sensitivity_denominator > 0
        else float("nan")
    )

    specificity = (
        true_negative
        / specificity_denominator
        if specificity_denominator > 0
        else float("nan")
    )


    clipped_probabilities = np.clip(
        positive_class_probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    metrics = {
        "roc_auc":
            safely_calculate_roc_auc(
                targets,
                positive_class_probabilities,
            ),

        "average_precision":
            safely_calculate_average_precision(
                targets,
                positive_class_probabilities,
            ),

        "accuracy":
            float(
                accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "precision":
            float(
                precision_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "f1":
            float(
                f1_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "brier_score":
            float(
                brier_score_loss(
                    targets,
                    positive_class_probabilities,
                )
            ),

        "negative_log_likelihood":
            float(
                log_loss(
                    targets,
                    np.column_stack(
                        [
                            1.0
                            - clipped_probabilities,

                            clipped_probabilities,
                        ]
                    ),
                    labels=[0, 1],
                )
            ),

        "expected_calibration_error":
            calculate_binary_expected_calibration_error(
                targets=targets,

                positive_class_probabilities=
                    positive_class_probabilities,

                number_of_bins=10,
            ),

        "classification_threshold":
            float(
                classification_threshold
            ),

        "true_negative":
            int(
                true_negative
            ),

        "false_positive":
            int(
                false_positive
            ),

        "false_negative":
            int(
                false_negative
            ),

        "true_positive":
            int(
                true_positive
            ),
    }


    if uncertainties is not None:

        uncertainties = np.asarray(
            uncertainties,
            dtype=np.float64,
        )

        metrics[
            "mean_uncertainty"
        ] = float(
            uncertainties.mean()
        )

        metrics[
            "std_uncertainty"
        ] = float(
            uncertainties.std()
        )


    if three_mt_weights is not None:

        three_mt_weights = np.asarray(
            three_mt_weights,
            dtype=np.float64,
        )

        metrics[
            "mean_three_mt_weight"
        ] = float(
            three_mt_weights.mean()
        )

        metrics[
            "std_three_mt_weight"
        ] = float(
            three_mt_weights.std()
        )

        metrics[
            "minimum_three_mt_weight"
        ] = float(
            three_mt_weights.min()
        )

        metrics[
            "maximum_three_mt_weight"
        ] = float(
            three_mt_weights.max()
        )


    return metrics

# ------------------------------------------------------------
# Exact fold-0 parameter and target summaries
# ------------------------------------------------------------

model_parameter_count = sum(
    parameter.numel()
    for parameter in complete_model.parameters()
    if parameter.requires_grad
)

optimised_parameter_count = sum(
    parameter.numel()
    for parameter_group in optimizer.param_groups
    for parameter in parameter_group["params"]
    if parameter.requires_grad
)

training_target_counts = (
    train_loader
    .dataset
    .dataframe[target_column]
    .value_counts()
    .sort_index()
)

training_target_summary = pd.DataFrame(
    {
        "CLASS_INDEX": [0, 1],
        "CLASS_NAME": ["sMCI", "pMCI"],
        "TRAINING_COUNT": [
            int(training_target_counts.loc[0]),
            int(training_target_counts.loc[1]),
        ],
    }
)

training_target_summary[
    "TRAINING_PROPORTION"
] = (
    training_target_summary[
        "TRAINING_COUNT"
    ]
    /
    training_target_summary[
        "TRAINING_COUNT"
    ].sum()
)


print("\nOptimiser: AdamW")
print(
    "Initial learning rate: "
    f"{INITIAL_LEARNING_RATE:.6f}"
)
print(
    "Weight decay: "
    f"{WEIGHT_DECAY:.6f}"
)
print(
    "Maximum epochs: "
    f"{MAXIMUM_EPOCHS}"
)
print(
    "Early-stopping patience: "
    f"{EARLY_STOPPING_PATIENCE} epochs"
)
print(
    "Scheduler patience: "
    f"{SCHEDULER_PATIENCE} epochs"
)
print(
    "Checkpoint-selection metric: validation ROC AUC"
)

print(
    "\nExperiment output directory:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "\nBest-checkpoint path:\n"
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    "\nTrainable model parameters: "
    f"{model_parameter_count:,}"
)

print(
    "Parameters included in optimiser: "
    f"{optimised_parameter_count:,}"
)

print(
    "\nFold-0 training target balance:"
)

display(
    training_target_summary.round(6)
)


### 1.6.33. Defining reusable training and validation epoch functions

define the reusable functions that will execute one complete training epoch and one complete validation epoch.

A single training epoch performs the following operations for every mini-batch:

1. move the nested multimodal batch to the selected device;
2. run the complete interaction pathway--evidence pathway forward pass in training mode;
3. apply training-time modality dropout;
4. calculate the joint objective;
5. backpropagate the total loss;
6. clip the global gradient norm;
7. update all model parameters using AdamW;
8. accumulate predictions, uncertainty estimates, gate weights, and losses.

The validation epoch uses the same complete model but differs in three important ways:

- the model is placed in evaluation mode;
- modality dropout and ordinary neural-network dropout are disabled;
- no gradients or parameter updates are calculated.

For both training and validation, pMCI is treated as the positive class. The epoch functions collect:

- participant identifiers;
- binary targets;
- final pMCI probabilities;
- final uncertainty values;
- interaction pathway and modality-specific evidence pathway weights;
- interaction pathway-only pMCI probabilities;
- evidence pathway-only pMCI probabilities;
- the original and effective numbers of available modalities.

The accumulated participant-level outputs are passed to the previously defined metric function after the entire epoch has completed.

### 1.6.34. Gradient clipping

For every training batch, the total gradient norm is calculated and clipped before the optimiser step:

$$
\left\|
\nabla_{\boldsymbol{\theta}}
\mathcal{L}_{\mathrm{total}}
\right\|_2
\leq 5.
$$

The unclipped norm is retained for monitoring.

### 1.6.35. Epoch-level loss aggregation

For loss component \(\ell\), the epoch-level mean is calculated using the number of participants in each mini-batch:

$$
\overline{\mathcal{L}}_{\ell}
=
\frac{
\sum_{b=1}^{B}
n_b
\mathcal{L}_{\ell,b}
}{
\sum_{b=1}^{B}
n_b
},
$$

where \(n_b\) is the batch size.

This avoids giving the final incomplete mini-batch the same weight as a full mini-batch.

The functions defined in this step do not yet train the model across multiple epochs. The full checkpointed training loop is constructed in the following step.

In [ ]:
# ============================================================
# 18. Defining reusable training and validation epoch functions
# ============================================================

from collections import defaultdict


# ------------------------------------------------------------
# Initial numerical-precision policy
# ------------------------------------------------------------

# I initially train in full float32 precision.
#
# This is computationally feasible on the available A100 GPU and
# avoids introducing mixed-precision instability into the Dirichlet
# digamma, log-gamma, and Dempster-Shafer calculations.
USE_MIXED_PRECISION = False


# ------------------------------------------------------------
# Recursively move a nested batch to the selected device
# ------------------------------------------------------------

def move_nested_batch_to_device(
    value,
    device,
):
    """
    Recursively move tensors in dictionaries, lists, and tuples
    to the selected PyTorch device.

    Non-tensor values are preserved unchanged.
    """

    if isinstance(
        value,
        torch.Tensor,
    ):
        return value.to(
            device,
            non_blocking=True,
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: move_nested_batch_to_device(
                nested_value,
                device,
            )
            for key, nested_value in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        )

    return value


# ------------------------------------------------------------
# Convert one completed forward pass into stored predictions
# ------------------------------------------------------------

def extract_batch_prediction_arrays(
    batch,
    model_output,
):
    """
    Extract participant-level targets, predictions, uncertainty,
    pathway outputs, and modality counts from one mini-batch.
    """

    final_output = model_output[
        "final_output"
    ]

    joint_opinion = model_output[
        "joint_opinion"
    ]

    tmc_output = model_output[
        "tmc_output"
    ]


    extracted = {
        "rid":
            batch["rid"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "target":
            batch["target"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_p_pMCI":
            final_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_uncertainty":
            final_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_weight":
            final_output[
                "three_mt_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_weight":
            final_output[
                "tmc_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_p_pMCI":
            joint_opinion[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_uncertainty":
            joint_opinion[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_p_pMCI":
            tmc_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_uncertainty":
            tmc_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "original_modality_count":
            model_output[
                "original_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "effective_modality_count":
            model_output[
                "effective_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),
    }

    return extracted


# ------------------------------------------------------------
# Concatenate the participant outputs collected across an epoch
# ------------------------------------------------------------

def concatenate_epoch_prediction_storage(
    prediction_storage,
):
    """
    Concatenate a dictionary of mini-batch NumPy arrays.
    """

    concatenated = {}

    for key, value_list in prediction_storage.items():

        if len(value_list) == 0:
            concatenated[key] = np.asarray([])

        else:
            concatenated[key] = np.concatenate(
                value_list,
                axis=0,
            )

    return concatenated


# ------------------------------------------------------------
# Convert stored predictions into a participant-level table
# ------------------------------------------------------------

def build_epoch_prediction_table(
    concatenated_predictions,
    split_name,
    epoch,
):
    """
    Build one participant-level DataFrame for an epoch.
    """

    prediction_table = pd.DataFrame(
        {
            "RID":
                concatenated_predictions[
                    "rid"
                ].astype(
                    np.int64
                ),

            "TARGET":
                concatenated_predictions[
                    "target"
                ].astype(
                    np.int64
                ),

            "FINAL_P_pMCI":
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            "FINAL_UNCERTAINTY":
                concatenated_predictions[
                    "final_uncertainty"
                ],

            "W_3MT":
                concatenated_predictions[
                    "three_mt_weight"
                ],

            "W_TMC":
                concatenated_predictions[
                    "tmc_weight"
                ],

            "THREE_MT_P_pMCI":
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            "THREE_MT_UNCERTAINTY":
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            "TMC_P_pMCI":
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            "TMC_UNCERTAINTY":
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            "ORIGINAL_MODALITY_COUNT":
                concatenated_predictions[
                    "original_modality_count"
                ].astype(
                    np.int64
                ),

            "EFFECTIVE_MODALITY_COUNT":
                concatenated_predictions[
                    "effective_modality_count"
                ].astype(
                    np.int64
                ),
        }
    )

    prediction_table.insert(
        loc=0,
        column="EPOCH",
        value=int(epoch),
    )

    prediction_table.insert(
        loc=1,
        column="SPLIT",
        value=str(split_name),
    )

    prediction_table[
        "FINAL_PREDICTED_CLASS"
    ] = (
        prediction_table[
            "FINAL_P_pMCI"
        ]
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        np.int64
    )

    return prediction_table


# ------------------------------------------------------------
# Calculate metrics for all three prediction outputs
# ------------------------------------------------------------

def calculate_epoch_prediction_metrics(
    concatenated_predictions,
):
    """
    Calculate metrics for:

    1. the final fixed-equal hybrid output;
    2. the 3MT-only joint output;
    3. the TMC-only fused output.
    """

    targets = concatenated_predictions[
        "target"
    ]

    final_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "final_uncertainty"
                ],

            three_mt_weights=
                concatenated_predictions[
                    "three_mt_weight"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    three_mt_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    tmc_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    return {
        "final": final_metrics,
        "three_mt": three_mt_metrics,
        "tmc": tmc_metrics,
    }


# ------------------------------------------------------------
# Initialise the loss accumulator used within an epoch
# ------------------------------------------------------------

EPOCH_LOSS_NAMES = [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]


def initialise_epoch_loss_storage():
    """
    Create participant-weighted loss totals.
    """

    return {
        loss_name: 0.0
        for loss_name in EPOCH_LOSS_NAMES
    }


def update_epoch_loss_storage(
    storage,
    loss_output,
    batch_size,
):
    """
    Add one batch's losses, weighted by the number of participants.
    """

    for loss_name in EPOCH_LOSS_NAMES:

        storage[loss_name] += (
            float(
                loss_output[
                    loss_name
                ]
                .detach()
                .float()
                .item()
            )
            * batch_size
        )


def finalise_epoch_loss_storage(
    storage,
    participant_count,
):
    """
    Convert accumulated loss sums into participant-weighted means.
    """

    if participant_count <= 0:
        raise ValueError(
            "The epoch contained no participants."
        )

    return {
        loss_name:
            loss_sum
            / participant_count

        for loss_name, loss_sum in storage.items()
    }


# ------------------------------------------------------------
# Run one complete training epoch
# ------------------------------------------------------------

def run_training_epoch(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_gradient_norm,
):
    """
    Train the complete model for one epoch.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []


    for batch_index, batch in enumerate(
        data_loader
    ):

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = targets.shape[0]

        participant_count += batch_size


        # --------------------------------------------------------
        # Clear gradients from the previous mini-batch
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # --------------------------------------------------------
        # Complete forward pass
        # --------------------------------------------------------

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )


        # --------------------------------------------------------
        # Complete multi-output objective
        # --------------------------------------------------------

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )


        # --------------------------------------------------------
        # Backpropagation
        # --------------------------------------------------------

        total_loss.backward()


        # --------------------------------------------------------
        # Global gradient clipping
        # --------------------------------------------------------

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )


        # --------------------------------------------------------
        # Parameter update
        # --------------------------------------------------------

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate losses and predictions
        # --------------------------------------------------------

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[key].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


    # ------------------------------------------------------------
    # Finalise the complete training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# Run one complete validation epoch
# ------------------------------------------------------------

def run_validation_epoch(
    model,
    data_loader,
    objective,
    device,
    epoch,
):
    """
    Evaluate the complete model for one epoch without gradients
    or training-time modality dropout.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    with torch.no_grad():

        for batch_index, batch in enumerate(
            data_loader
        ):

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = targets.shape[0]

            participant_count += batch_size


            # ----------------------------------------------------
            # Complete evaluation forward pass
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ----------------------------------------------------
            # Validation objective
            # ----------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_index}."
                )


            # ----------------------------------------------------
            # Accumulate losses and predictions
            # ----------------------------------------------------

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[key].append(
                    values
                )


    # ------------------------------------------------------------
    # Finalise the complete validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Display the configured epoch-function summary
# ------------------------------------------------------------

print("=" * 72)
print("TRAINING AND VALIDATION EPOCH FUNCTIONS")
print("=" * 72)

print(
    f"\nMixed precision enabled: "
    f"{USE_MIXED_PRECISION}"
)

print(
    "Training batches per epoch: "
    f"{len(train_loader)}"
)

print(
    "Validation batches per epoch: "
    f"{len(validation_loader)}"
)

print(
    "Training participants: "
    f"{len(train_loader.dataset)}"
)

print(
    "Validation participants: "
    f"{len(validation_loader.dataset)}"
)

print(
    "\nTraining epoch operations:"
)

print(
    "- forward pass with modality dropout;"
)

print(
    "- complete joint loss calculation;"
)

print(
    "- backpropagation;"
)

print(
    "- global gradient clipping;"
)

print(
    "- AdamW parameter update;"
)

print(
    "- prediction and uncertainty accumulation."
)

print(
    "\nValidation epoch operations:"
)

print(
    "- evaluation mode;"
)

print(
    "- no modality dropout;"
)

print(
    "- no gradient calculation;"
)

print(
    "- complete validation loss and metric calculation."
)

print(
    "\nThe epoch functions are defined."
)

print(
    "No complete training or validation epoch has been run yet."
)

print(
    "The next step will create the checkpointed multi-epoch "
    "training loop and begin model optimisation."
)

### 1.6.36. Training a newly initialised model with validation-based checkpointing

This standalone experiment starts from epoch 1 using the model initialised in this notebook. It does not load a checkpoint from the previous ungated baseline or from an earlier gated run.

The validation partition controls learning-rate reduction, early stopping, and best-checkpoint selection. The test partition remains untouched during training.

In [ ]:
# ============================================================
# 19. Training with live batch and epoch progress
# ============================================================

import time
import traceback
from collections import defaultdict
from datetime import datetime

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Resume and progress-display behaviour
# ------------------------------------------------------------

# The formal first run begins from epoch 1.
# Resume is enabled only when FOLD_RUN_MODE was explicitly set
# to "resume" for this exact experiment and fold.
RESUME_FROM_LAST_CHECKPOINT = (
    FOLD_RUN_MODE == "resume"
)

# I refresh the live progress display after every batch.
PROGRESS_UPDATE_INTERVAL = 1


# ------------------------------------------------------------
# Loss-configuration serialisation
# ------------------------------------------------------------

def obtain_training_objective_configuration(
    objective,
):
    """
    Return the principal loss settings in a checkpoint-safe form.
    """

    return {
        "evidential_annealing_epochs": int(
            objective.evidential_annealing_epochs
        ),

        "gate_regularisation_epochs": int(
            objective.gate_regularisation_epochs
        ),

        "joint_loss_weight": float(
            objective.joint_loss_weight
        ),

        "tmc_loss_weight": float(
            objective.tmc_loss_weight
        ),

        "modality_loss_weight": float(
            objective.modality_loss_weight
        ),

        "auxiliary_loss_weight": float(
            objective.auxiliary_loss_weight
        ),

        "gate_loss_weight": float(
            objective.gate_loss_weight
        ),

        "class_weights": (
            None
            if objective.class_weights is None
            else
            objective.class_weights
            .detach()
            .cpu()
            .tolist()
        ),
    }


# ------------------------------------------------------------
# Flatten one epoch into one history row
# ------------------------------------------------------------

def build_training_history_row(
    epoch,
    learning_rate,
    epoch_duration_seconds,
    training_result,
    validation_result,
    best_validation_auc,
    epochs_without_improvement,
    checkpoint_improved,
):
    """
    Create one flat row containing losses, metrics, optimisation
    diagnostics, and checkpoint information.
    """

    history_row = {
        "epoch":
            int(epoch),

        "learning_rate":
            float(learning_rate),

        "epoch_duration_seconds":
            float(epoch_duration_seconds),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "checkpoint_improved":
            bool(checkpoint_improved),

        "train_participants":
            int(
                training_result[
                    "participant_count"
                ]
            ),

        "validation_participants":
            int(
                validation_result[
                    "participant_count"
                ]
            ),

        "train_batches":
            int(
                training_result[
                    "batch_count"
                ]
            ),

        "validation_batches":
            int(
                validation_result[
                    "batch_count"
                ]
            ),

        "train_mean_gradient_norm":
            float(
                training_result[
                    "gradient_summary"
                ]["mean_gradient_norm"]
            ),

        "train_maximum_gradient_norm_before_clipping":
            float(
                training_result[
                    "gradient_summary"
                ][
                    "maximum_gradient_norm_before_clipping"
                ]
            ),

        "train_mean_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "mean_effective_modality_count"
                ]
            ),

        "train_minimum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "minimum_effective_modality_count"
                ]
            ),

        "train_maximum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "maximum_effective_modality_count"
                ]
            ),

        "validation_maximum_modality_count_difference":
            float(
                validation_result[
                    "maximum_evaluation_modality_count_difference"
                ]
            ),
    }


    # --------------------------------------------------------
    # Add all training and validation losses
    # --------------------------------------------------------

    for loss_name, loss_value in (
        training_result[
            "losses"
        ].items()
    ):
        history_row[
            f"train_{loss_name}"
        ] = float(
            loss_value
        )

    for loss_name, loss_value in (
        validation_result[
            "losses"
        ].items()
    ):
        history_row[
            f"validation_{loss_name}"
        ] = float(
            loss_value
        )


    # --------------------------------------------------------
    # Add metrics for all three prediction outputs
    # --------------------------------------------------------

    for output_name in [
        "final",
        "three_mt",
        "tmc",
    ]:

        for metric_name, metric_value in (
            training_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"train_{output_name}_{metric_name}"
            ] = metric_value

        for metric_name, metric_value in (
            validation_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"validation_{output_name}_{metric_name}"
            ] = metric_value


    return history_row


# ------------------------------------------------------------
# Save a fully recoverable checkpoint
# ------------------------------------------------------------

def save_training_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_validation_auc,
    epochs_without_improvement,
    validation_metrics,
    training_history,
):
    """
    Save the state required to reproduce or continue training.
    """

    checkpoint = {
        "epoch":
            int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "validation_metrics":
            validation_metrics,

        "training_configuration":
            training_configuration,

        "training_objective_configuration":
            obtain_training_objective_configuration(
                training_objective
            ),

        "training_history":
            training_history,

        "random_seed":
            int(GLOBAL_RANDOM_SEED),

        "task":
            SELECTED_TASK,

        "fold":
            int(SELECTED_FOLD),

        "saved_at":
            datetime.now().isoformat(
                timespec="seconds"
            ),
    }

    torch.save(
        checkpoint,
        checkpoint_path,
    )


# ------------------------------------------------------------
# GPU-memory helper for the progress display
# ------------------------------------------------------------

def current_cuda_memory_gb():
    """
    Return currently allocated CUDA memory in gigabytes.
    """

    if not torch.cuda.is_available():
        return 0.0

    return float(
        torch.cuda.memory_allocated(
            DEVICE
        )
        / (1024 ** 3)
    )


# ------------------------------------------------------------
# One training epoch with a live batch progress bar
# ------------------------------------------------------------

def run_training_epoch_with_progress(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_epochs,
    maximum_gradient_norm,
):
    """
    Train for one epoch while displaying batch-level progress.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | training"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    for batch_number, batch in progress_bar:

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = int(
            targets.shape[0]
        )

        participant_count += (
            batch_size
        )


        # --------------------------------------------------------
        # Forward pass and loss
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )


        # --------------------------------------------------------
        # Backpropagation, clipping, and update
        # --------------------------------------------------------

        total_loss.backward()

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate diagnostics
        # --------------------------------------------------------

        current_total_loss = float(
            total_loss.detach().item()
        )

        running_total_loss_sum += (
            current_total_loss
            * batch_size
        )

        running_mean_total_loss = (
            running_total_loss_sum
            / participant_count
        )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[
                key
            ].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


        # --------------------------------------------------------
        # Update the visible progress information
        # --------------------------------------------------------

        if (
            batch_number
            % PROGRESS_UPDATE_INTERVAL
            == 0
            or batch_number
            == len(data_loader)
        ):

            elapsed_minutes = (
                time.time()
                - phase_start_time
            ) / 60.0

            progress_bar.set_postfix(
                {
                    "loss":
                        f"{current_total_loss:.3f}",

                    "avg":
                        f"{running_mean_total_loss:.3f}",

                    "grad":
                        f"{float(gradient_norm):.2f}",

                    "lr":
                        f"{optimizer.param_groups[0]['lr']:.1e}",

                    "GPU":
                        f"{current_cuda_memory_gb():.1f}GB",

                    "elapsed":
                        f"{elapsed_minutes:.1f}m",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise the training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# One validation epoch with a live batch progress bar
# ------------------------------------------------------------

def run_validation_epoch_with_progress(
    model,
    data_loader,
    objective,
    device,
    epoch,
    maximum_epochs,
):
    """
    Validate for one epoch while displaying batch-level progress.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | validation"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ----------------------------------------------------
            # Forward pass and validation loss
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            total_loss = loss_output[
                "total_loss"
            ]


            if not torch.isfinite(
                total_loss
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_number}."
                )


            # ----------------------------------------------------
            # Accumulate diagnostics and predictions
            # ----------------------------------------------------

            current_total_loss = float(
                total_loss.detach().item()
            )

            running_total_loss_sum += (
                current_total_loss
                * batch_size
            )

            running_mean_total_loss = (
                running_total_loss_sum
                / participant_count
            )

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[
                    key
                ].append(
                    values
                )


            # ----------------------------------------------------
            # Update the visible progress information
            # ----------------------------------------------------

            if (
                batch_number
                % PROGRESS_UPDATE_INTERVAL
                == 0
                or batch_number
                == len(data_loader)
            ):

                elapsed_minutes = (
                    time.time()
                    - phase_start_time
                ) / 60.0

                progress_bar.set_postfix(
                    {
                        "loss":
                            f"{current_total_loss:.3f}",

                        "avg":
                            f"{running_mean_total_loss:.3f}",

                        "GPU":
                            f"{current_cuda_memory_gb():.1f}GB",

                        "elapsed":
                            f"{elapsed_minutes:.1f}m",
                    },
                    refresh=True,
                )


    # ------------------------------------------------------------
    # Finalise the validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                concatenated_predictions[
                    "original_modality_count"
                ]
                -
                concatenated_predictions[
                    "effective_modality_count"
                ]
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Initial training state
# ------------------------------------------------------------

training_history = []

starting_epoch = 1

best_validation_auc = float(
    "-inf"
)

epochs_without_improvement = 0


# ------------------------------------------------------------
# Resume from the latest fully completed epoch
# ------------------------------------------------------------

if (
    RESUME_FROM_LAST_CHECKPOINT
    and LAST_CHECKPOINT_PATH.exists()
):

    print(
        "Loading the latest fully completed checkpoint:",
        flush=True,
    )

    print(
        LAST_CHECKPOINT_PATH,
        flush=True,
    )

    resumed_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    complete_model.load_state_dict(
        resumed_checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        resumed_checkpoint[
            "optimizer_state_dict"
        ]
    )

    learning_rate_scheduler.load_state_dict(
        resumed_checkpoint[
            "scheduler_state_dict"
        ]
    )

    completed_epoch = int(
        resumed_checkpoint[
            "epoch"
        ]
    )

    starting_epoch = (
        completed_epoch
        + 1
    )

    best_validation_auc = float(
        resumed_checkpoint[
            "best_validation_auc"
        ]
    )

    epochs_without_improvement = int(
        resumed_checkpoint[
            "epochs_without_improvement"
        ]
    )

    training_history = list(
        resumed_checkpoint.get(
            "training_history",
            [],
        )
    )

    print(
        f"\nResuming after epoch {completed_epoch}.",
        flush=True,
    )

    print(
        f"Next epoch: {starting_epoch}",
        flush=True,
    )

    print(
        "Best validation ROC AUC so far: "
        f"{best_validation_auc:.6f}",
        flush=True,
    )

else:

    print(
        "No fully completed checkpoint was loaded.",
        flush=True,
    )

    print(
        "Fresh training begins from the newly initialised model state.",
        flush=True,
    )


# ------------------------------------------------------------
# Main multi-epoch training loop
# ------------------------------------------------------------

if starting_epoch > MAXIMUM_EPOCHS:

    print(
        "\nTraining has already reached the configured maximum "
        f"of {MAXIMUM_EPOCHS} epochs.",
        flush=True,
    )

else:

    print("\n" + "=" * 72, flush=True)
    print("BEGINNING MODEL TRAINING", flush=True)
    print("=" * 72, flush=True)

    print(
        f"\nEpoch range: {starting_epoch}--{MAXIMUM_EPOCHS}",
        flush=True,
    )

    print(
        f"Training batches per epoch: {len(train_loader)}",
        flush=True,
    )

    print(
        f"Validation batches per epoch: {len(validation_loader)}",
        flush=True,
    )

    print(
        "Each epoch displays separate live training and "
        "validation progress bars.",
        flush=True,
    )

    print(
        "The test partition will not be evaluated.",
        flush=True,
    )


    try:

        for epoch in range(
            starting_epoch,
            MAXIMUM_EPOCHS + 1,
        ):

            epoch_start_time = time.time()

            current_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )


            print("\n" + "=" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d}/{MAXIMUM_EPOCHS}",
                flush=True,
            )

            print("=" * 72, flush=True)

            print(
                "\nPhase 1/4: training batches",
                flush=True,
            )


            # ------------------------------------------------
            # Train on all training participants
            # ------------------------------------------------

            training_result = (
                run_training_epoch_with_progress(
                    model=complete_model,
                    data_loader=train_loader,
                    objective=training_objective,
                    optimizer=optimizer,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                    maximum_gradient_norm=
                        MAXIMUM_GRADIENT_NORM,
                )
            )


            print(
                "\nPhase 2/4: validation batches",
                flush=True,
            )


            # ------------------------------------------------
            # Validate on all validation participants
            # ------------------------------------------------

            validation_result = (
                run_validation_epoch_with_progress(
                    model=complete_model,
                    data_loader=validation_loader,
                    objective=training_objective,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                )
            )


            print(
                "\nPhase 3/4: calculating metrics and "
                "updating the scheduler",
                flush=True,
            )


            # ------------------------------------------------
            # Validation AUC and checkpoint decision
            # ------------------------------------------------

            validation_auc = float(
                validation_result[
                    "metrics"
                ]["final"]["roc_auc"]
            )

            validation_auc_is_valid = bool(
                np.isfinite(
                    validation_auc
                )
            )

            checkpoint_improved = (
                validation_auc_is_valid
                and
                validation_auc
                > best_validation_auc
            )

            if checkpoint_improved:

                best_validation_auc = (
                    validation_auc
                )

                epochs_without_improvement = 0

            else:

                epochs_without_improvement += 1


            scheduler_score = (
                validation_auc
                if validation_auc_is_valid
                else -1.0
            )

            learning_rate_scheduler.step(
                scheduler_score
            )

            updated_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            epoch_duration_seconds = (
                time.time()
                - epoch_start_time
            )


            # ------------------------------------------------
            # Persistent training history
            # ------------------------------------------------

            history_row = build_training_history_row(
                epoch=epoch,

                learning_rate=
                    current_learning_rate,

                epoch_duration_seconds=
                    epoch_duration_seconds,

                training_result=
                    training_result,

                validation_result=
                    validation_result,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                checkpoint_improved=
                    checkpoint_improved,
            )

            history_row[
                "learning_rate_after_scheduler"
            ] = updated_learning_rate

            training_history.append(
                history_row
            )

            pd.DataFrame(
                training_history
            ).to_csv(
                TRAINING_HISTORY_PATH,
                index=False,
            )


            print(
                "\nPhase 4/4: saving checkpoints and history",
                flush=True,
            )


            # ------------------------------------------------
            # Save the latest completed epoch
            # ------------------------------------------------

            save_training_checkpoint(
                checkpoint_path=
                    LAST_CHECKPOINT_PATH,

                epoch=epoch,

                model=complete_model,

                optimizer=optimizer,

                scheduler=
                    learning_rate_scheduler,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                validation_metrics=
                    validation_result[
                        "metrics"
                    ],

                training_history=
                    training_history,
            )


            # ------------------------------------------------
            # Save the best validation checkpoint
            # ------------------------------------------------

            if checkpoint_improved:

                save_training_checkpoint(
                    checkpoint_path=
                        BEST_CHECKPOINT_PATH,

                    epoch=epoch,

                    model=complete_model,

                    optimizer=optimizer,

                    scheduler=
                        learning_rate_scheduler,

                    best_validation_auc=
                        best_validation_auc,

                    epochs_without_improvement=
                        epochs_without_improvement,

                    validation_metrics=
                        validation_result[
                            "metrics"
                        ],

                    training_history=
                        training_history,
                )

                validation_result[
                    "predictions"
                ].to_csv(
                    VALIDATION_PREDICTIONS_PATH,
                    index=False,
                )


            # ------------------------------------------------
            # Readable completed-epoch summary
            # ------------------------------------------------

            train_metrics = training_result[
                "metrics"
            ]["final"]

            validation_metrics = validation_result[
                "metrics"
            ]["final"]


            print("\n" + "-" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d} COMPLETE",
                flush=True,
            )

            print(
                "Duration: "
                f"{epoch_duration_seconds / 60.0:.2f} minutes",
                flush=True,
            )

            print(
                "Learning rate: "
                f"{current_learning_rate:.8f}"
                f" -> {updated_learning_rate:.8f}",
                flush=True,
            )


            print("\nTraining:", flush=True)

            print(
                "  total loss: "
                f"{training_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{train_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{train_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{train_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{train_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )

            print(
                "  mean effective modalities: "
                f"{training_result['modality_dropout_summary']['mean_effective_modality_count']:.3f}",
                flush=True,
            )

            print(
                "  maximum pre-clipping gradient norm: "
                f"{training_result['gradient_summary']['maximum_gradient_norm_before_clipping']:.6f}",
                flush=True,
            )


            print("\nValidation:", flush=True)

            print(
                "  total loss: "
                f"{validation_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{validation_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  average precision: "
                f"{validation_metrics['average_precision']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{validation_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  sensitivity: "
                f"{validation_metrics['sensitivity']:.6f}",
                flush=True,
            )

            print(
                "  specificity: "
                f"{validation_metrics['specificity']:.6f}",
                flush=True,
            )

            print(
                "  Brier score: "
                f"{validation_metrics['brier_score']:.6f}",
                flush=True,
            )

            print(
                "  calibration error: "
                f"{validation_metrics['expected_calibration_error']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{validation_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{validation_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )


            print("\nCheckpoint status:", flush=True)

            print(
                "  improved this epoch: "
                f"{checkpoint_improved}",
                flush=True,
            )

            print(
                "  best validation ROC AUC: "
                f"{best_validation_auc:.6f}",
                flush=True,
            )

            print(
                "  epochs without improvement: "
                f"{epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}",
                flush=True,
            )

            print(
                "  latest completed epoch saved: True",
                flush=True,
            )

            if torch.cuda.is_available():

                print(
                    "  peak CUDA memory: "
                    f"{torch.cuda.max_memory_allocated(0) / (1024 ** 3):.3f} GB",
                    flush=True,
                )


            # ------------------------------------------------
            # Early stopping
            # ------------------------------------------------

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):

                print(
                    "\nEarly stopping activated because "
                    "validation ROC AUC did not improve for "
                    f"{EARLY_STOPPING_PATIENCE} consecutive epochs.",
                    flush=True,
                )

                break


    # --------------------------------------------------------
    # Interruption and error handling
    # --------------------------------------------------------

    except KeyboardInterrupt:

        print(
            "\nTraining was interrupted manually.",
            flush=True,
        )

        print(
            "Only fully completed epochs are recoverable from "
            "the last-epoch checkpoint.",
            flush=True,
        )


    except Exception:

        print(
            "\nTraining stopped because an exception occurred.",
            flush=True,
        )

        print(
            "The latest fully completed epoch remains saved.",
            flush=True,
        )

        traceback.print_exc()

        raise


    # --------------------------------------------------------
    # Final training summary
    # --------------------------------------------------------

    if len(
        training_history
    ) > 0:

        final_history_table = pd.DataFrame(
            training_history
        )

        completed_epochs = int(
            final_history_table[
                "epoch"
            ].max()
        )

        print("\n" + "=" * 72, flush=True)

        print(
            "TRAINING RUN COMPLETE",
            flush=True,
        )

        print("=" * 72, flush=True)

        print(
            f"\nLast completed epoch: {completed_epochs}",
            flush=True,
        )

        print(
            "Best validation ROC AUC: "
            f"{best_validation_auc:.6f}",
            flush=True,
        )

        print(
            "\nBest checkpoint:\n"
            f"{BEST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nLatest checkpoint:\n"
            f"{LAST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nTraining history:\n"
            f"{TRAINING_HISTORY_PATH}",
            flush=True,
        )

        print(
            "\nThe test partition has not been evaluated.",
            flush=True,
        )

### 1.6.37. Evaluating the best gated checkpoint on the untouched test set

After training and validation-based checkpoint selection, the best checkpoint from this named experiment is loaded from its experiment-specific fold directory and evaluated once on the held-out test partition.

The evaluation cell does not update model parameters, the optimiser, the scheduler, or modality-dropout state.

In [ ]:
from datetime import datetime


# ------------------------------------------------------------
# Test-output paths
# ------------------------------------------------------------

TEST_METRICS_PATH = (
    PREDICTION_DIR
    / "test_metrics.json"
)

TEST_SUMMARY_PATH = (
    PREDICTION_DIR
    / "test_metrics_summary.csv"
)


print("=" * 72)
print("FIXED-EQUAL-FUSION TEST-SET EVALUATION")
print("=" * 72)

print(
    "\nLoading the best validation checkpoint:\n"
    f"{BEST_CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# Load the best validation checkpoint
# ------------------------------------------------------------


best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_checkpoint_epoch = int(
    best_checkpoint[
        "epoch"
    ]
)

best_checkpoint_validation_auc = float(
    best_checkpoint[
        "best_validation_auc"
    ]
)


complete_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


print(
    f"\nBest checkpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Best validation ROC AUC stored in checkpoint: "
    f"{best_checkpoint_validation_auc:.6f}"
)


# ------------------------------------------------------------
# Test evaluation function
# ------------------------------------------------------------

def run_test_epoch(
    model,
    data_loader,
    objective,
    device,
    checkpoint_epoch,
):
    """
    Evaluate the frozen model on the untouched test partition.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc="Test evaluation",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ------------------------------------------------
            # Frozen forward pass
            # ------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ------------------------------------------------
            # Test loss
            # ------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=checkpoint_epoch,
            )


            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):

                raise FloatingPointError(
                    "A non-finite test loss was encountered "
                    f"at batch {batch_number}."
                )


            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )


            # ------------------------------------------------
            # Store participant-level outputs
            # ------------------------------------------------

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):

                prediction_storage[
                    key
                ].append(
                    values
                )


            running_average_loss = (
                loss_storage[
                    "total_loss"
                ]
                / participant_count
            )

            progress_bar.set_postfix(
                {
                    "avg_loss":
                        f"{running_average_loss:.3f}",

                    "participants":
                        f"{participant_count}/{len(data_loader.dataset)}",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise test losses and predictions
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="test",

            epoch=checkpoint_epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_modality_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_modality_count_difference,
    }


# ------------------------------------------------------------
# Run the untouched test evaluation
# ------------------------------------------------------------

test_result = run_test_epoch(
    model=complete_model,
    data_loader=test_loader,
    objective=training_objective,
    device=DEVICE,
    checkpoint_epoch=best_checkpoint_epoch,
)


# ------------------------------------------------------------
# Evaluation-mode modality count
# ------------------------------------------------------------

maximum_test_mask_difference = (
    test_result[
        "maximum_evaluation_modality_count_difference"
    ]
)


# ------------------------------------------------------------
# Save participant-level test predictions
# ------------------------------------------------------------


test_prediction_table = test_result[
    "predictions"
]

test_prediction_table.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save all test metrics in JSON format
# ------------------------------------------------------------

serialisable_test_result = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "task":
        SELECTED_TASK,

    "fold":
        int(
            SELECTED_FOLD
        ),

    "checkpoint_epoch":
        int(
            best_checkpoint_epoch
        ),

    "checkpoint_validation_auc":
        float(
            best_checkpoint_validation_auc
        ),

    "classification_threshold":
        float(
            CLASSIFICATION_THRESHOLD
        ),

    "test_participants":
        int(
            test_result[
                "participant_count"
            ]
        ),

    "test_batches":
        int(
            test_result[
                "batch_count"
            ]
        ),

    "test_losses": {
        key:
            float(value)

        for key, value in (
            test_result[
                "losses"
            ].items()
        )
    },

    "test_metrics":
        test_result[
            "metrics"
        ],

    "maximum_modality_count_difference":
        float(
            maximum_test_mask_difference
        ),

    "evaluated_at":
        datetime.now().isoformat(
            timespec="seconds"
        ),
}


with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as test_metrics_file:

    json.dump(
        serialisable_test_result,
        test_metrics_file,
        indent=2,
    )


# ------------------------------------------------------------
# Build a compact comparison table
# ------------------------------------------------------------

test_metric_rows = []

for output_name, display_name in [
    (
        "final",
        "Hybrid",
    ),
    (
        "three_mt",
        "3MT-only",
    ),
    (
        "tmc",
        "TMC-only",
    ),
]:

    output_metrics = test_result[
        "metrics"
    ][
        output_name
    ]

    test_metric_rows.append(
        {
            "OUTPUT":
                display_name,

            "ROC_AUC":
                output_metrics[
                    "roc_auc"
                ],

            "AVERAGE_PRECISION":
                output_metrics[
                    "average_precision"
                ],

            "ACCURACY":
                output_metrics[
                    "accuracy"
                ],

            "BALANCED_ACCURACY":
                output_metrics[
                    "balanced_accuracy"
                ],

            "SENSITIVITY":
                output_metrics[
                    "sensitivity"
                ],

            "SPECIFICITY":
                output_metrics[
                    "specificity"
                ],

            "PRECISION":
                output_metrics[
                    "precision"
                ],

            "F1":
                output_metrics[
                    "f1"
                ],

            "BRIER_SCORE":
                output_metrics[
                    "brier_score"
                ],

            "NEGATIVE_LOG_LIKELIHOOD":
                output_metrics[
                    "negative_log_likelihood"
                ],

            "EXPECTED_CALIBRATION_ERROR":
                output_metrics[
                    "expected_calibration_error"
                ],

            "MEAN_UNCERTAINTY":
                output_metrics[
                    "mean_uncertainty"
                ],
        }
    )


test_metrics_summary = pd.DataFrame(
    test_metric_rows
)

test_metrics_summary.to_csv(
    TEST_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display the final test results
# ------------------------------------------------------------

hybrid_test_metrics = test_result[
    "metrics"
][
    "final"
]


print("\n" + "=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} TEST RESULTS")
print("=" * 72)

print(
    f"\nCheckpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Test participants: "
    f"{test_result['participant_count']}"
)

print(
    "Test batches: "
    f"{test_result['batch_count']}"
)

print(
    "Maximum difference between original and effective "
    "test modality counts: "
    f"{maximum_test_mask_difference:.1f}"
)


print(
    "\nFinal hybrid test performance:"
)

print(
    "  ROC AUC: "
    f"{hybrid_test_metrics['roc_auc']:.6f}"
)

print(
    "  average precision: "
    f"{hybrid_test_metrics['average_precision']:.6f}"
)

print(
    "  accuracy: "
    f"{hybrid_test_metrics['accuracy']:.6f}"
)

print(
    "  balanced accuracy: "
    f"{hybrid_test_metrics['balanced_accuracy']:.6f}"
)

print(
    "  sensitivity: "
    f"{hybrid_test_metrics['sensitivity']:.6f}"
)

print(
    "  specificity: "
    f"{hybrid_test_metrics['specificity']:.6f}"
)

print(
    "  precision: "
    f"{hybrid_test_metrics['precision']:.6f}"
)

print(
    "  F1 score: "
    f"{hybrid_test_metrics['f1']:.6f}"
)

print(
    "  Brier score: "
    f"{hybrid_test_metrics['brier_score']:.6f}"
)

print(
    "  negative log-likelihood: "
    f"{hybrid_test_metrics['negative_log_likelihood']:.6f}"
)

print(
    "  expected calibration error: "
    f"{hybrid_test_metrics['expected_calibration_error']:.6f}"
)

print(
    "  mean uncertainty: "
    f"{hybrid_test_metrics['mean_uncertainty']:.6f}"
)

print(
    "  mean 3MT weight: "
    f"{hybrid_test_metrics['mean_three_mt_weight']:.6f}"
)

print(
    "  standard deviation of 3MT weight: "
    f"{hybrid_test_metrics['std_three_mt_weight']:.6f}"
)


print(
    "\nConfusion matrix counts:"
)

print(
    "  true negatives: "
    f"{hybrid_test_metrics['true_negative']}"
)

print(
    "  false positives: "
    f"{hybrid_test_metrics['false_positive']}"
)

print(
    "  false negatives: "
    f"{hybrid_test_metrics['false_negative']}"
)

print(
    "  true positives: "
    f"{hybrid_test_metrics['true_positive']}"
)


print(
    "\nHybrid, 3MT-only, and TMC-only comparison:"
)

display(
    test_metrics_summary.round(
        6
    )
)


print(
    "\nParticipant-level predictions saved to:\n"
    f"{TEST_PREDICTIONS_PATH}"
)

print(
    "\nComplete test metrics saved to:\n"
    f"{TEST_METRICS_PATH}"
)

print(
    "\nCompact metric summary saved to:\n"
    f"{TEST_SUMMARY_PATH}"
)

print(
    "\nThe model was evaluated without gradient updates, "
    "scheduler changes, or test-time modality dropout."
)

### 1.6.38. Releasing fold-specific memory

The fold-2 checkpoints, histories, validation predictions, test predictions, and metrics are already stored in its own directory. This cleanup removes the in-memory model and DataLoaders before the next fold is created.

In [ ]:
# ============================================================
# 21. Releasing fold-2 memory before the next fold
# ============================================================

import gc

objects_to_release = [
    "complete_model",
    "optimizer",
    "scheduler",
    "training_objective",
    "train_loader",
    "validation_loader",
    "test_loader",
    "train_dataset",
    "validation_dataset",
    "test_dataset",
    "example_batch",
    "diagnostic_batch",
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Fold 2 outputs remain saved under:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "Fold-specific GPU cache has been released."
)


## 1.7. Fold 3

This section trains, validates, and evaluates fold 3. It uses the exact prepared table `mci_prognosis_outer_fold_3_final_task_ready.csv` and writes only to the experiment's `fold_3` directory.


### 1.7.1. Loading the prepared fold-3 input

The model-input columns are defined explicitly in this notebook, matching the original fold-0 implementation. No schema file or output from an earlier experiment is loaded.


In [ ]:
# ============================================================
# Loading the prepared inputs for fold 3
# ============================================================

from pathlib import Path
import json
import pandas as pd

from google.colab import drive


# ------------------------------------------------------------
# Experiment identity
# ------------------------------------------------------------

EXPERIMENT_NAME = "temporal_prebaseline_fixed_equal_fusion_md050"
SELECTED_TASK = "mci_prognosis"
SELECTED_FOLD = 3

# Use "fresh" for the formal first run.
# Change this to "resume" only after an interrupted run of this
# same experiment and fold.
FOLD_RUN_MODE = "fresh"


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():
    drive.mount(
        "/content/drive"
    )


# ------------------------------------------------------------
# Exact project and prepared-input paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

SELECTED_INPUT_PATH = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / "mci_prognosis"
    / f"mci_prognosis_outer_fold_{SELECTED_FOLD}_final_task_ready.csv"
)


# ------------------------------------------------------------
# Fixed model-input columns from the prepared pipeline
# ------------------------------------------------------------

final_model_schema = {
    "identifier_columns": [
        "RID",
        "PTID",
    ],

    "audit_label_columns": [
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ],

    "model_target_column":
        "MODEL_TARGET",

    "split_columns": [
        "OUTER_FOLD",
        "DATA_ROLE",
    ],

    "scaled_continuous_columns": [
        "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
        "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        "ADAS__TOTSCORE__Z",
        "ADAS__TOTAL13__Z",
        "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
        "FAQ__FAQTOTAL__Z",
        "CSF__ABETA40__Z",
        "CSF__ABETA42__Z",
        "CSF__TAU__Z",
        "CSF__PTAU__Z",
        "CSF__ABETA42_40_RATIO__Z",
        "PLASMA__pT217_F__Z",
        "PLASMA__AB42_F__Z",
        "PLASMA__AB40_F__Z",
        "PLASMA__AB42_AB40_F__Z",
        "PLASMA__pT217_AB42_F__Z",
        "PLASMA__NfL_Q__Z",
        "PLASMA__GFAP_Q__Z",
        "PLASMA__NfL_F__Z",
        "PLASMA__GFAP_F__Z",
    ],

    "encoded_categorical_columns": [
        "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
        "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        "APOE__APOE4_ALLELE_COUNT__IDX",
    ],

    "mri_path_columns": [
        "MRI__NORMALIZED_T1_NPY_PATH",
    ],

    "branch_mask_columns": [
        "BRANCH_MASK__DEMOGRAPHICS",
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
        "BRANCH_MASK__CSF",
        "BRANCH_MASK__PLASMA",
        "BRANCH_MASK__APOE",
        "BRANCH_MASK__MRI",
    ],

    "feature_mask_columns": [
        "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
        "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        "FEATURE_MASK__ADAS_TOTSCORE",
        "FEATURE_MASK__ADAS_TOTAL13",
        "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
        "FEATURE_MASK__FAQ_FAQTOTAL",
        "FEATURE_MASK__CSF_ABETA40",
        "FEATURE_MASK__CSF_ABETA42",
        "FEATURE_MASK__CSF_TAU",
        "FEATURE_MASK__CSF_PTAU",
        "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        "FEATURE_MASK__PLASMA_pT217_F",
        "FEATURE_MASK__PLASMA_AB42_F",
        "FEATURE_MASK__PLASMA_AB40_F",
        "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "FEATURE_MASK__PLASMA_NfL_Q",
        "FEATURE_MASK__PLASMA_GFAP_Q",
        "FEATURE_MASK__PLASMA_NfL_F",
        "FEATURE_MASK__PLASMA_GFAP_F",
        "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
    ],
}



# ------------------------------------------------------------
# Load the prepared fold table
# ------------------------------------------------------------

fold_table = pd.read_csv(
    SELECTED_INPUT_PATH
)


print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} INPUT")
print("=" * 72)

print(
    f"\nTask-ready table:\n{SELECTED_INPUT_PATH}"
)

print(
    f"\nLoaded shape: "
    f"{fold_table.shape[0]} rows × "
    f"{fold_table.shape[1]} columns"
)


### 1.7.2. Preparing the fold-3 model-input groups

The continuous, categorical, MRI-path, branch-mask, feature-mask, identifier, target, and split columns are taken from the fixed definitions loaded above.


In [ ]:
# ============================================================
# 2. Reading the prepared schema and describing fold 0
# ============================================================

# ------------------------------------------------------------
# Prepared model-input column groups
# ------------------------------------------------------------

identifier_columns = final_model_schema[
    "identifier_columns"
]

audit_label_columns = final_model_schema[
    "audit_label_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

split_columns = final_model_schema[
    "split_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]


# ------------------------------------------------------------
# Fold-0 structure
# ------------------------------------------------------------

print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} EXPERIMENT")
print("=" * 72)

print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Task: {SELECTED_TASK}")
print(f"Outer fold: {SELECTED_FOLD}")

print("\nPrepared data roles:")
print(
    fold_table["DATA_ROLE"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared target counts by role:")
display(
    pd.crosstab(
        fold_table["DATA_ROLE"],
        fold_table[target_column],
    ).reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared input groups:")
print(
    f"- scaled continuous features: "
    f"{len(scaled_continuous_columns)}"
)
print(
    f"- encoded categorical features: "
    f"{len(encoded_categorical_columns)}"
)
print(
    f"- MRI path columns: "
    f"{len(mri_path_columns)}"
)
print(
    f"- branch masks: "
    f"{len(branch_mask_columns)}"
)
print(
    f"- feature masks: "
    f"{len(feature_mask_columns)}"
)

preview_columns = (
    identifier_columns
    + ["CLINICAL_GROUP"]
    + split_columns
    + [target_column]
    + branch_mask_columns
    + mri_path_columns
)

print("\nExample prepared rows:")
display(
    fold_table[
        preview_columns
    ].head(5)
)


### 1.7.3. Defining the multimodal dataset-output contract

The model will receive each modality as a separate input branch rather than as one combined feature vector.

Continuous and categorical variables are kept separate because they require different encoder operations. Continuous variables will enter small numerical encoders, while categorical variables will later be represented through trainable embeddings.

Each sample will also contain:

- the participant identifier;
- the binary prognosis target;
- branch-level availability masks;
- feature-level observation masks;
- the prepared MRI path.

The dataset will not load MRI arrays yet. At this stage, I define the column organisation and the exact sample structure that the PyTorch dataset will later return.

For participants without MRI, the MRI path remains unavailable and the MRI branch mask remains zero. The participant is retained in the dataset.

In [ ]:
# ============================================================
# 3. Defining the multimodal dataset-output contract
# ============================================================

# ------------------------------------------------------------
# Branch-specific predictor columns
# ------------------------------------------------------------

# I organise the prepared continuous and categorical columns into
# the six modality branches used by the architecture.

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


# ------------------------------------------------------------
# Dataset sample structure
# ------------------------------------------------------------

# One participant will later be returned by the PyTorch dataset
# using the following nested structure.
#
# Continuous features will become float32 tensors.
# Categorical indices will become int64 tensors for embeddings.
# Masks will become float32 tensors containing 0 or 1.
# The target will become an int64 class index.

dataset_output_contract = {
    "rid": "Participant RID as an integer",
    "ptid": "Participant PTID as a string",
    "target": "Binary class index: 0 for sMCI and 1 for pMCI",

    "modalities": {
        "demographics": {
            "continuous": "Shape (2,), float32",
            "categorical": "Shape (2,), int64",
            "feature_mask": "Shape (4,), float32",
            "branch_mask": "Scalar, float32",
        },

        "cognitive_functional": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "csf": {
            "continuous": "Shape (5,), float32",
            "categorical": None,
            "feature_mask": "Shape (5,), float32",
            "branch_mask": "Scalar, float32",
        },

        "plasma": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "apoe": {
            "continuous": None,
            "categorical": "Shape (1,), int64",
            "feature_mask": "Shape (1,), float32",
            "branch_mask": "Scalar, float32",
        },

        "mri": {
            "path": "Prepared NumPy path or None",
            "image": "Later: shape (1, 177, 213, 183), float32",
            "branch_mask": "Scalar, float32",
        },
    },

    "branch_masks": (
        "Shape (6,), float32, ordered as "
        "demographics, cognitive-functional, CSF, "
        "plasma, APOE, MRI"
    ),

    "feature_masks": (
        "Shape (28,), float32, using the authoritative "
        "feature-mask order"
    ),
}


# ------------------------------------------------------------
# Display the agreed contract
# ------------------------------------------------------------

print("=" * 72)
print("MULTIMODAL DATASET CONTRACT")
print("=" * 72)

print("\nBranch-specific input dimensions:")

for branch_name, branch_definition in dataset_column_contract.items():

    continuous_count = len(
        branch_definition.get("continuous", [])
    )

    categorical_count = len(
        branch_definition.get("categorical", [])
    )

    feature_mask_count = len(
        branch_definition.get("feature_masks", [])
    )

    has_mri_path = "path" in branch_definition

    print(
        f"- {branch_name}: "
        f"{continuous_count} continuous, "
        f"{categorical_count} categorical, "
        f"{feature_mask_count} feature masks"
        + (", 1 MRI path" if has_mri_path else "")
    )


print("\nPlanned sample output:")
print(
    json.dumps(
        dataset_output_contract,
        indent=2,
    )
)

print(
    "\nThis contract will be used in the next step to "
    "implement the PyTorch dataset."
)

### 1.7.4. Implementing the multimodal PyTorch dataset

implement a PyTorch dataset that converts each prepared participant row into the agreed multimodal structure.

The dataset preserves the six modality branches and returns continuous variables, categorical indices, observation masks, participant identifiers, and the prognosis target separately.

MRI volumes are loaded only when a sample is requested. The existing preprocessed NumPy array is used directly, and a channel dimension is added to produce the shape required by a three-dimensional neural network.

When MRI is unavailable, the participant remains in the dataset. The dataset returns a zero placeholder volume together with an MRI branch mask of zero, allowing the model to distinguish an unavailable scan from an observed image.

In [ ]:
# ============================================================
# 4. Implementing the multimodal PyTorch dataset
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset


# ------------------------------------------------------------
# Expected prepared MRI shape
# ------------------------------------------------------------

# I preserve the spatial dimensions produced by the completed
# MRI preprocessing pipeline.
MRI_SPATIAL_SHAPE = (177, 213, 183)

# I add one channel dimension when returning an MRI tensor.
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)


# ------------------------------------------------------------
# Multimodal PyTorch dataset
# ------------------------------------------------------------

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # I reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # I retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # I preserve the authoritative mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri


    def __len__(self):
        return len(self.dataframe)


    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # I load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # I ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)


    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # --------------------------------------------------------
        # Demographics
        # --------------------------------------------------------

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Cognitive and functional measures
        # --------------------------------------------------------

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # CSF
        # --------------------------------------------------------

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Plasma
        # --------------------------------------------------------

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # APOE
        # --------------------------------------------------------

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # MRI
        # --------------------------------------------------------

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Complete sample
        # --------------------------------------------------------

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample


# ------------------------------------------------------------
# Create role-specific datasets
# ------------------------------------------------------------

# I retain the prepared role assignments exactly as stored in
# the selected outer-fold table.
train_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "train"
    ]
    .reset_index(drop=True)
)

validation_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "validation"
    ]
    .reset_index(drop=True)
)

test_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "test"
    ]
    .reset_index(drop=True)
)


# I initially disable MRI disk loading so that I can inspect the
# dataset structure quickly before constructing the DataLoaders.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)


# ------------------------------------------------------------
# Concise dataset summary
# ------------------------------------------------------------

print("=" * 72)
print("PYTORCH DATASETS")
print("=" * 72)

print(f"\nTraining participants: {len(train_dataset)}")
print(f"Validation participants: {len(validation_dataset)}")
print(f"Test participants: {len(test_dataset)}")

print(
    "\nThe datasets preserve the prepared role assignments "
    "and return separate modality inputs."
)

print(
    "MRI loading is temporarily disabled for structural "
    "inspection and will be enabled for the DataLoaders."
)

### 1.7.5. Inspecting one multimodal sample and one prepared MRI volume

Before constructing the DataLoaders, The notebook inspects the structure returned for one participant.

use the dataset with MRI loading disabled to confirm the scalar tensors, categorical indices, targets, and masks. I then create a temporary MRI-enabled dataset and load one participant whose MRI branch is available.

This checks the dataset interface required by the model while avoiding unnecessary loading of multiple MRI volumes at this stage.

In [ ]:
# ============================================================
# 5. Inspecting one multimodal sample and one MRI volume
# ============================================================

# ------------------------------------------------------------
# Inspect one scalar-only training sample
# ------------------------------------------------------------

# I retrieve one participant while MRI disk loading remains
# disabled. The returned MRI tensor is therefore only the
# temporary placeholder defined in the current dataset class.
sample = train_dataset[0]

print("=" * 72)
print("EXAMPLE MULTIMODAL SAMPLE")
print("=" * 72)

print(f"\nRID: {sample['rid']}")
print(f"PTID: {sample['ptid']}")
print(f"Target: {sample['target'].item()}")

print("\nModality tensor structure:")

for modality_name, modality_data in sample["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"{value}"
            )

print(
    f"\nComplete branch-mask shape: "
    f"{tuple(sample['branch_masks'].shape)}"
)

print(
    f"Complete feature-mask shape: "
    f"{tuple(sample['feature_masks'].shape)}"
)


# ------------------------------------------------------------
# Find one participant with an available MRI
# ------------------------------------------------------------

# I select the first training participant whose prepared MRI
# branch mask is one.
example_mri_index = train_table.index[
    train_table["BRANCH_MASK__MRI"] == 1
][0]


# ------------------------------------------------------------
# Create a temporary MRI-enabled dataset
# ------------------------------------------------------------

# I enable MRI loading only for this temporary inspection
# dataset. The main role-specific datasets remain unchanged.
mri_inspection_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

mri_sample = mri_inspection_dataset[
    example_mri_index
]

mri_tensor = mri_sample[
    "modalities"
]["mri"]["image"]

mri_branch_mask = mri_sample[
    "modalities"
]["mri"]["branch_mask"]


# ------------------------------------------------------------
# Display the prepared MRI tensor information
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXAMPLE PREPARED MRI")
print("=" * 72)

print(f"\nRID: {mri_sample['rid']}")
print(f"PTID: {mri_sample['ptid']}")

print(
    f"MRI branch mask: "
    f"{mri_branch_mask.item():.0f}"
)

print(
    f"MRI tensor shape: "
    f"{tuple(mri_tensor.shape)}"
)

print(
    f"MRI tensor dtype: "
    f"{mri_tensor.dtype}"
)

print(
    f"MRI intensity minimum: "
    f"{mri_tensor.min().item():.6f}"
)

print(
    f"MRI intensity maximum: "
    f"{mri_tensor.max().item():.6f}"
)

print(
    f"MRI intensity mean: "
    f"{mri_tensor.mean().item():.6f}"
)

print(
    "\nThe sample structure and full prepared MRI volume "
    "are ready for DataLoader construction."
)

### 1.7.6. Constructing the multimodal DataLoaders

create separate DataLoaders for the fixed training, validation, and test subsets.

MRI loading is enabled, so an available scan is read lazily from its prepared NumPy path when its participant enters a batch. Participants without MRI receive a zero placeholder volume and retain an MRI branch mask of zero.

I begin with a small batch size because each sample contains a full three-dimensional MRI volume. The final training batch size will be selected later according to the memory requirements of the complete model.

The training DataLoader shuffles participants. Validation and test DataLoaders preserve a deterministic order.

In [ ]:
# ============================================================
# 6. Constructing the multimodal DataLoaders
# ============================================================

from torch.utils.data import DataLoader


# ------------------------------------------------------------
# Initial DataLoader settings
# ------------------------------------------------------------

# I begin with a small batch because each participant may contain
# a full MRI volume with shape (1, 177, 213, 183).
INITIAL_BATCH_SIZE = 2

# I initially use the main process for data loading. This is the
# most reliable starting configuration when reading NumPy files
# from mounted Google Drive.
NUM_WORKERS = 0

# Pinned memory can speed transfers to a CUDA device.
PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# Recreate the datasets with MRI loading enabled
# ------------------------------------------------------------

# Available MRI volumes will now be read lazily when requested.
# Missing MRI branches will retain their zero placeholders and
# branch masks of zero.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)


# ------------------------------------------------------------
# Create role-specific DataLoaders
# ------------------------------------------------------------

# I shuffle only the training subset.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# Validation order does not need to be shuffled.
validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# The test subset also retains a deterministic order.
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)


# ------------------------------------------------------------
# Retrieve one complete training batch
# ------------------------------------------------------------

example_batch = next(iter(train_loader))


# ------------------------------------------------------------
# Display the batched tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("EXAMPLE MULTIMODAL BATCH")
print("=" * 72)

print(f"\nBatch size: {example_batch['target'].shape[0]}")
print(f"RID values: {example_batch['rid']}")
print(f"PTID values: {example_batch['ptid']}")
print(f"Targets: {example_batch['target']}")

print("\nModality tensors:")

for modality_name, modality_data in example_batch["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"type={type(value).__name__}"
            )


print("\nCombined masks:")

print(
    "  branch_masks: "
    f"shape={tuple(example_batch['branch_masks'].shape)}, "
    f"dtype={example_batch['branch_masks'].dtype}"
)

print(
    "  feature_masks: "
    f"shape={tuple(example_batch['feature_masks'].shape)}, "
    f"dtype={example_batch['feature_masks'].dtype}"
)


# ------------------------------------------------------------
# Show MRI availability in this batch
# ------------------------------------------------------------

batch_mri = example_batch[
    "modalities"
]["mri"]["image"]

batch_mri_masks = example_batch[
    "modalities"
]["mri"]["branch_mask"]

print("\nMRI batch:")

print(
    f"  image shape: {tuple(batch_mri.shape)}"
)

print(
    f"  branch masks: {batch_mri_masks}"
)

print(
    f"  approximate raw MRI batch size: "
    f"{batch_mri.numel() * batch_mri.element_size() / (1024 ** 2):.2f} MB"
)


# ------------------------------------------------------------
# DataLoader summary
# ------------------------------------------------------------

print("\nDataLoader batches:")

print(
    f"  training: {len(train_loader)} batches"
)

print(
    f"  validation: {len(validation_loader)} batches"
)

print(
    f"  test: {len(test_loader)} batches"
)

print(
    "\nThe complete multimodal batch is ready for "
    "modality-specific encoder construction."
)

### 1.7.7. Building the modality-specific encoders

Each modality has a different input structure, so I encode the six branches separately before multimodal interaction.

For the scalar branches, the prepared feature values are combined with their feature-observation masks. This allows the encoders to distinguish an observed standardised value close to zero from a missing-value placeholder.

Categorical variables use trainable embeddings. Encoded index zero remains reserved for missing or unseen values and is handled through embedding padding behaviour.

### 1.7.8. Common latent dimension

Every branch is projected into the same latent dimension:

$$
d_{\mathrm{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore produces:

$$
\mathbf{H}^{(m)}
\in
\mathbb{R}^{B \times 64}.
$$

The shared dimension is required so that all modality representations can later enter the same cascaded cross-modal transformer architecture.

The original interaction pathway implementation used a dimension of \(512\). In this thesis, I begin with a more compact dimension of \(64\) because the main prognosis training folds contain approximately \(369\) participants and the complete model will additionally include independent evidential heads, modality-specific evidence fusion, auxiliary outputs, and the cascaded interaction pathway interaction path.

The embedding dimension remains a model-capacity hyperparameter and may later be compared with larger values using only training and validation data.

### 1.7.9. MRI encoder

The MRI branch follows the general image-encoding structure used by interaction pathway:

1. initial three-dimensional convolutions;
2. residual three-dimensional downsampling blocks;
3. conversion of the final feature map into patch tokens;
4. addition of learned positional embeddings;
5. transformer encoding of patch-wise relationships;
6. global averaging across patch tokens;
7. projection into the shared modality dimension.

The original paper used a 512-dimensional MRI output. Here, the final MRI representation is projected to the common 64-dimensional latent space used by the other branches.

The prepared MRI volumes are larger than those used in the original interaction pathway experiments. I therefore apply stride-two downsampling in the initial convolutional stem before the four residual blocks. This preserves the CNN-transformer design while keeping the number of transformer patch tokens computationally manageable.

No new MRI preprocessing is performed. The encoder receives the complete prepared volume with shape:

$$
(1, 177, 213, 183).
$$

In [ ]:
# ============================================================
# 7. Building the modality-specific encoders
# ============================================================

import math

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Shared representation dimension
# ------------------------------------------------------------

# I project every modality into one common latent space so that
# all branches can later enter the same cross-modal transformers.
MODALITY_EMBEDDING_DIM = 64


# ------------------------------------------------------------
# Reusable scalar encoder
# ------------------------------------------------------------

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        values,
        feature_mask,
    ):
        # I supply both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Demographics encoder
# ------------------------------------------------------------

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# APOE encoder
# ------------------------------------------------------------

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Residual 3D downsampling block
# ------------------------------------------------------------

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original interaction pathway image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()


    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )


# ------------------------------------------------------------
# interaction pathway-style CNN-transformer MRI encoder
# ------------------------------------------------------------

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # --------------------------------------------------------
        # Initial convolutional stem
        # --------------------------------------------------------

        # I use two initial 3D convolutions, following the broad
        # structure shown in the interaction pathway image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original interaction pathway inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )


        # --------------------------------------------------------
        # Four residual downsampling blocks
        # --------------------------------------------------------

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )


        # --------------------------------------------------------
        # Determine the resulting patch grid
        # --------------------------------------------------------

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )


        # --------------------------------------------------------
        # Learned positional embeddings
        # --------------------------------------------------------

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )


        # --------------------------------------------------------
        # Patch-wise transformer encoder
        # --------------------------------------------------------

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )


        # --------------------------------------------------------
        # Projection to the shared modality dimension
        # --------------------------------------------------------

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )


    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # I flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # I add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )


# ------------------------------------------------------------
# Complete set of six modality encoders
# ------------------------------------------------------------

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )


    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations


# ------------------------------------------------------------
# Instantiate the revised encoders
# ------------------------------------------------------------

modality_encoders = ADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Count trainable parameters
# ------------------------------------------------------------

def count_trainable_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print("=" * 72)
print("MODALITY-SPECIFIC ENCODERS")
print("=" * 72)

print(
    f"\nShared modality embedding dimension: "
    f"{MODALITY_EMBEDDING_DIM}"
)

print(
    "\nMRI transformer patch grid: "
    f"{modality_encoders.mri.patch_grid_shape}"
)

print(
    "MRI transformer patch count: "
    f"{modality_encoders.mri.number_of_patches}"
)

print("\nTrainable parameters by encoder:")

for encoder_name in [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]:
    encoder = getattr(
        modality_encoders,
        encoder_name,
    )

    print(
        f"- {encoder_name}: "
        f"{count_trainable_parameters(encoder):,}"
    )

print(
    "\nTotal trainable encoder parameters: "
    f"{count_trainable_parameters(modality_encoders):,}"
)

print(
    "\nThe revised MRI branch now uses a 3D CNN, patch tokens, "
    "positional embeddings, and a transformer encoder."
)

print(
    "No multimodal fusion or classification head has been "
    "added yet."
)

### 1.7.10. Applying branch-availability masks to the encoded modalities

Each modality has a different original input structure, but every modality-specific encoder projects its input into the same latent dimensionality.

In this implementation, each branch produces a representation of size:

$$
d_{\text{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore returns:

$$
\mathbf{H}^{(m)} \in \mathbb{R}^{B \times 64},
$$

where \(m\) denotes one of the six modalities:

- demographics;
- cognitive-functional measures;
- CSF;
- plasma;
- APOE;
- MRI.

The shared dimensionality does not mean that the modalities contain the same information or use equally complex encoders. Each branch has its own input-specific encoder, but the final representations must have a common size so that they can later participate in cross-modal attention and evidential fusion.

After stacking the six modality representations, the model obtains:

$$
\mathbf{H}
\in
\mathbb{R}^{B \times 6 \times 64}.
$$

This follows the general design principle used by interaction pathway, in which heterogeneous modality inputs are first projected into a common transformer embedding space before cross-modal interaction. The original interaction pathway implementation used a larger embedding dimension of \(512\), but \(512\) is an architectural hyperparameter rather than a methodological requirement.

A compact starting dimension of \(64\) is used here because the main MCI prognosis training folds contain only approximately \(369\) participants, while the complete planned model will also contain:

- six modality-specific encoders;
- cascaded cross-modal attention;
- independent evidential heads;
- evidence pathway-style evidential fusion;
- a joint evidential prediction path.

The number of parameters in transformer projections grows approximately with the square of the embedding dimension. For example:

$$
64^2 = 4{,}096,
$$

whereas:

$$
512^2 = 262{,}144.
$$

Thus, increasing the embedding dimension from \(64\) to \(512\) can make several attention and feed-forward parameter blocks approximately \(64\) times larger. A \(512\)-dimensional model would therefore introduce substantially greater overfitting and memory risk for the available prognosis cohort.

The value \(64\) is treated as a compact initial configuration rather than as a permanently fixed optimum. The latent dimensionality can later be compared with alternatives such as \(128\) or \(256\), using only the training and validation subsets.

### 1.7.11. Branch-availability masking

The prepared zero placeholders make missing inputs computationally compatible with neural-network layers, but they do not guarantee that an unavailable modality will produce a zero encoder output.

Linear layers, embeddings, normalisation parameters, and learned biases can generate a non-zero representation even when all supplied inputs are zero. I therefore apply the prepared branch mask after each modality encoder.

For participant \(i\) and modality \(m\), the masked representation is:

$$
\widetilde{\mathbf{h}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{h}_{i}^{(m)},
$$

where:

- \(\mathbf{h}_{i}^{(m)} \in \mathbb{R}^{64}\) is the raw modality representation;
- \(a_{i}^{(m)} \in \{0,1\}\) is the prepared branch-availability mask;
- \(\widetilde{\mathbf{h}}_{i}^{(m)} \in \mathbb{R}^{64}\) is the masked representation passed to later model components.

Therefore:

$$
a_{i}^{(m)} = 1
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{h}_{i}^{(m)},
$$

and:

$$
a_{i}^{(m)} = 0
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{0}.
$$

An unavailable modality consequently contributes an exact zero representation rather than a learned bias-derived vector. The original branch masks are also retained separately so that the later cross-modal attention and evidential-fusion components can explicitly identify which modalities are available for each participant.

In [ ]:
# ============================================================
# 8. Applying branch masks to the encoded modalities
# ============================================================

# ------------------------------------------------------------
# Fixed modality order
# ------------------------------------------------------------

# I use one explicit modality order throughout the architecture.
# This order matches the prepared branch-mask columns.
MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]


# ------------------------------------------------------------
# Mask-aware encoder wrapper
# ------------------------------------------------------------

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )


    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # I add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask


    def forward(self, modalities):
        # I first obtain the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # I then suppress every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # I return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }


# ------------------------------------------------------------
# Instantiate the mask-aware encoder collection
# ------------------------------------------------------------

masked_modality_encoders = MaskedADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Apply the encoders to one complete batch
# ------------------------------------------------------------

# I keep this inspection on the CPU. The training device will be
# configured later when the full model and optimisation loop exist.
masked_modality_encoders.eval()

with torch.no_grad():
    encoded_batch = masked_modality_encoders(
        example_batch["modalities"]
    )


# ------------------------------------------------------------
# Stack modality representations
# ------------------------------------------------------------

# I stack the representations in the fixed modality order.
#
# Resulting shape:
# (batch_size, number_of_modalities, embedding_dimension)
stacked_masked_representations = torch.stack(
    [
        encoded_batch["masked"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_raw_representations = torch.stack(
    [
        encoded_batch["raw"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the encoded structure
# ------------------------------------------------------------

print("=" * 72)
print("MASKED MODALITY REPRESENTATIONS")
print("=" * 72)

print(
    f"\nStacked raw representation shape: "
    f"{tuple(stacked_raw_representations.shape)}"
)

print(
    f"Stacked masked representation shape: "
    f"{tuple(stacked_masked_representations.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, embedding)"
)

print("\nBranch masks in this batch:")

branch_mask_table = pd.DataFrame(
    example_batch["branch_masks"].numpy(),
    columns=MODALITY_ORDER,
)

display(branch_mask_table)


# ------------------------------------------------------------
# Representation norms before and after masking
# ------------------------------------------------------------

# A representation norm summarises the magnitude of each branch
# vector. Missing branches may have non-zero raw norms because of
# learned biases, but their masked norms must be exactly zero.
raw_norms = torch.linalg.vector_norm(
    stacked_raw_representations,
    dim=-1,
)

masked_norms = torch.linalg.vector_norm(
    stacked_masked_representations,
    dim=-1,
)

norm_summary = []

for participant_index in range(
    stacked_masked_representations.shape[0]
):
    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):
        norm_summary.append(
            {
                "BATCH_ROW": participant_index,
                "RID": int(
                    example_batch["rid"][
                        participant_index
                    ].item()
                ),
                "MODALITY": modality_name,
                "BRANCH_MASK": float(
                    example_batch["branch_masks"][
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "RAW_REPRESENTATION_NORM": float(
                    raw_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "MASKED_REPRESENTATION_NORM": float(
                    masked_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
            }
        )

norm_summary = pd.DataFrame(norm_summary)

print("\nRepresentation norms before and after masking:")

display(
    norm_summary.round(6)
)


# ------------------------------------------------------------
# Confirm the representation dimensions
# ------------------------------------------------------------

print("\nEncoded modality shapes:")

for modality_name in MODALITY_ORDER:
    print(
        f"- {modality_name}: "
        f"{tuple(encoded_batch['masked'][modality_name].shape)}"
    )

print(
    "\nOnly the masked representations will enter the "
    "multimodal interaction and evidential-fusion paths."
)

### 1.7.12. Building the availability-gated interaction pathway cascade

The cascade order remains unchanged:

$$
\text{demographics}
\rightarrow
\text{APOE}
\rightarrow
\text{cognitive/functional}
\rightarrow
\text{CSF}
\rightarrow
\text{plasma}
\rightarrow
\text{MRI}.
$$

Each cross-modal interaction block still computes a candidate update using query self-attention followed by cross-attention to the encoded modality token. The branch mask then determines whether that candidate becomes the next cumulative query:

$$
\mathbf{q}_m
=
a_m\mathbf{q}^{\mathrm{candidate}}_m
+
(1-a_m)\mathbf{q}_{m-1}.
$$

For an available modality, $a_m=1$ and the candidate update is used. For an unavailable modality, $a_m=0$ and the stage becomes an exact identity update.

The branch-mask tensor follows `MODALITY_ORDER`, while the cross-modal interaction blocks follow `THREE_MT_CASCADE_ORDER`. The class therefore uses the explicit modality-to-mask index mapping already defined by the fold-0 architecture.

In [ ]:
# ============================================================
# 9. Building the availability-gated interaction pathway cascade
# ============================================================

# ------------------------------------------------------------
# Fixed cascade order
# ------------------------------------------------------------

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]


# ------------------------------------------------------------
# One Cascaded Modality Transformer
# ------------------------------------------------------------

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }


# ------------------------------------------------------------
# Complete six-stage availability-gated cascade
# ------------------------------------------------------------

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )


    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }


# ------------------------------------------------------------
# Instantiate and inspect the gated cascade
# ------------------------------------------------------------

three_mt_cascade = ThreeMTCascade(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    number_of_heads=4,
    dropout=0.10,
)

three_mt_cascade.eval()

with torch.no_grad():
    three_mt_output = three_mt_cascade(
        masked_representations=encoded_batch[
            "masked"
        ],
        branch_masks=example_batch[
            "branch_masks"
        ],
    )


print("=" * 72)
print("AVAILABILITY-GATED 3MT CASCADE")
print("=" * 72)

print(
    f"\nCascade order:\n"
    f"{THREE_MT_CASCADE_ORDER}"
)

print(
    "\nFinal joint representation shape: "
    f"{tuple(three_mt_output['joint_representation'].shape)}"
)


# ------------------------------------------------------------
# Show the actual query update at every stage
# ------------------------------------------------------------

query_change_rows = []

previous_query = (
    three_mt_cascade
    .learned_latent_query
    .expand(
        example_batch["target"].shape[0],
        -1,
        -1,
    )
)

for modality_name in THREE_MT_CASCADE_ORDER:
    current_query = three_mt_output[
        "stage_queries"
    ][modality_name]

    query_change_norm = torch.linalg.vector_norm(
        current_query - previous_query,
        dim=-1,
    ).squeeze(1)

    modality_index = MODALITY_ORDER.index(
        modality_name
    )

    branch_mask = example_batch[
        "branch_masks"
    ][
        :,
        modality_index,
    ]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):
        query_change_rows.append(
            {
                "BATCH_ROW": batch_row,
                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),
                "CASCADE_STAGE": modality_name,
                "BRANCH_MASK": float(
                    branch_mask[
                        batch_row
                    ].item()
                ),
                "QUERY_CHANGE_NORM": float(
                    query_change_norm[
                        batch_row
                    ].item()
                ),
            }
        )

    previous_query = current_query

query_change_summary = pd.DataFrame(
    query_change_rows
)

print(
    "\nQuery changes after availability gating:"
)

display(
    query_change_summary.round(6)
)

print(
    "\nTrainable cascade parameters: "
    f"{count_trainable_parameters(three_mt_cascade):,}"
)


### 1.7.13. Producing independent modality-specific evidential opinions

The interaction pathway cascade produces a cumulative interaction-aware representation, but its intermediate states are not independent modality opinions because every stage contains information inherited from earlier stages.

Trusted Multi-View Classification requires each modality to produce its own class evidence before cross-modal interaction.

For participant \(i\), modality \(m\), and class \(k\), the modality-specific evidential head produces non-negative evidence:

$$
e_{ik}^{(m)}
=
\operatorname{Softplus}
\left(
\mathbf{W}_{m}
\mathbf{z}_{i}^{(m)}
+
\mathbf{b}_{m}
\right),
$$

where:

- \(\mathbf{z}_{i}^{(m)} \in \mathbb{R}^{64}\) is the independently encoded modality representation;
- \(e_{ik}^{(m)} \geq 0\) is the evidence assigned to class \(k\);
- each modality has its own evidential head.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{(m)}
=
e_{ik}^{(m)} + 1.
$$

For the binary prognosis task:

$$
K = 2,
$$

with class order:

$$
[\mathrm{sMCI},\mathrm{pMCI}].
$$

The expected class probabilities are:

$$
p_{ik}^{(m)}
=
\frac{
\alpha_{ik}^{(m)}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{(m)}
}.
$$

The Dirichlet strength is:

$$
S_{i}^{(m)}
=
\sum_{k=1}^{K}
\alpha_{ik}^{(m)}.
$$

The standard evidential uncertainty mass is:

$$
u_{i}^{(m)}
=
\frac{K}{
S_{i}^{(m)}
}.
$$

Low total evidence produces high uncertainty, while stronger evidence produces lower uncertainty.

### 1.7.14. Treatment of unavailable modalities

Only available modalities should contribute an opinion to modality-specific evidence fusion.

For an unavailable branch, I do not interpret the evidential head output as a genuine prediction. Instead, its effective evidence is set to zero:

$$
\widetilde{\mathbf{e}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{e}_{i}^{(m)},
$$

where \(a_{i}^{(m)}\) is the prepared branch mask.

Therefore, an unavailable modality receives:

$$
\widetilde{\boldsymbol{\alpha}}_{i}^{(m)}
=
\mathbf{1},
$$

which is the uniform Dirichlet opinion with:

$$
u_{i}^{(m)} = 1.
$$

The original branch mask is retained so that the next step can exclude unavailable opinions explicitly during evidence pathway/Dempster--Shafer fusion.

At this stage, I construct and inspect the six independent evidential opinions. I do not yet fuse them or combine them with the interaction pathway joint representation.

In [ ]:
# ============================================================
# 10. Producing independent modality-specific evidential opinions
# ============================================================

# ------------------------------------------------------------
# Binary prognosis class definition
# ------------------------------------------------------------

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]


# ------------------------------------------------------------
# One modality-specific evidential head
# ------------------------------------------------------------

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(
        self,
        representation,
        branch_mask,
    ):
        # I use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }


# ------------------------------------------------------------
# Independent evidential heads for all six modalities
# ------------------------------------------------------------

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )


    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions


# ------------------------------------------------------------
# Instantiate the independent evidential path
# ------------------------------------------------------------

independent_evidential_heads = (
    IndependentModalityEvidentialHeads(
        modality_order=MODALITY_ORDER,
        input_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
    )
)


# ------------------------------------------------------------
# Produce one opinion per modality
# ------------------------------------------------------------

independent_evidential_heads.eval()

with torch.no_grad():

    modality_opinions = (
        independent_evidential_heads(
            # I use the independently encoded branch outputs before
            # they enter the interaction pathway cascade.
            modality_representations=encoded_batch[
                "masked"
            ],

            modalities=example_batch[
                "modalities"
            ],
        )
    )


# ------------------------------------------------------------
# Stack the opinion tensors
# ------------------------------------------------------------

stacked_modality_evidence = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["evidence"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_alpha = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["alpha"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_probabilities = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["probabilities"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_uncertainty = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["uncertainty"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the evidential tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("INDEPENDENT MODALITY EVIDENTIAL OPINIONS")
print("=" * 72)

print(
    f"\nClass order: "
    f"{PROGNOSIS_CLASS_ORDER}"
)

print(
    "\nStacked evidence shape: "
    f"{tuple(stacked_modality_evidence.shape)}"
)

print(
    "Stacked Dirichlet-alpha shape: "
    f"{tuple(stacked_modality_alpha.shape)}"
)

print(
    "Stacked probability shape: "
    f"{tuple(stacked_modality_probabilities.shape)}"
)

print(
    "Stacked uncertainty shape: "
    f"{tuple(stacked_modality_uncertainty.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, class)"
)


# ------------------------------------------------------------
# Create a readable modality-opinion summary
# ------------------------------------------------------------

opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):

        branch_mask = float(
            example_batch["branch_masks"][
                batch_row,
                modality_index,
            ].item()
        )

        alpha_values = stacked_modality_alpha[
            batch_row,
            modality_index,
        ]

        probability_values = (
            stacked_modality_probabilities[
                batch_row,
                modality_index,
            ]
        )

        uncertainty_value = (
            stacked_modality_uncertainty[
                batch_row,
                modality_index,
                0,
            ]
        )

        opinion_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "MODALITY": modality_name,

                "BRANCH_MASK": branch_mask,

                "ALPHA_sMCI": float(
                    alpha_values[0].item()
                ),

                "ALPHA_pMCI": float(
                    alpha_values[1].item()
                ),

                "P_sMCI": float(
                    probability_values[0].item()
                ),

                "P_pMCI": float(
                    probability_values[1].item()
                ),

                "UNCERTAINTY": float(
                    uncertainty_value.item()
                ),
            }
        )


modality_opinion_summary = pd.DataFrame(
    opinion_rows
)

print("\nIndependent modality opinions:")

display(
    modality_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable independent evidential-head parameters: "
    f"{count_trainable_parameters(independent_evidential_heads):,}"
)

print(
    "\nUnavailable modalities should have alpha=[1, 1], "
    "probabilities=[0.5, 0.5], and uncertainty=1."
)

print(
    "\nThe available modality opinions are ready for "
    "TMC/Dempster-Shafer fusion."
)

### 1.7.15. Fusing the independent modality opinions with evidence pathway

combine the six independent modality opinions using the reduced Dempster--Shafer rule adopted by Trusted Multi-View Classification.

For modality \(m\), the Dirichlet parameters are:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{e}^{(m)} + \mathbf{1}.
$$

The Dirichlet strength is:

$$
S^{(m)}
=
\sum_{k=1}^{K}
\alpha_{k}^{(m)}.
$$

The class-specific belief masses are:

$$
b_{k}^{(m)}
=
\frac{
\alpha_{k}^{(m)} - 1
}{
S^{(m)}
}
=
\frac{
e_{k}^{(m)}
}{
S^{(m)}
}.
$$

The uncertainty mass is:

$$
u^{(m)}
=
\frac{K}{
S^{(m)}
}.
$$

These masses satisfy:

$$
\sum_{k=1}^{K}
b_{k}^{(m)}
+
u^{(m)}
=
1.
$$

### 1.7.16. Combining two opinions

Consider two opinions, \(A\) and \(B\). Their conflict mass is:

$$
C
=
\sum_{i \neq j}
b_{i}^{A}
b_{j}^{B}.
$$

For each class \(k\), the combined belief mass is:

$$
b_{k}^{A \oplus B}
=
\frac{
b_{k}^{A}b_{k}^{B}
+
b_{k}^{A}u^{B}
+
b_{k}^{B}u^{A}
}{
1-C
}.
$$

The combined uncertainty mass is:

$$
u^{A \oplus B}
=
\frac{
u^{A}u^{B}
}{
1-C
}.
$$

The fused Dirichlet strength is recovered from the fused uncertainty:

$$
S^{A \oplus B}
=
\frac{K}{
u^{A \oplus B}
}.
$$

The fused evidence and Dirichlet parameters are then:

$$
e_{k}^{A \oplus B}
=
b_{k}^{A \oplus B}
S^{A \oplus B},
$$

and:

$$
\alpha_{k}^{A \oplus B}
=
e_{k}^{A \oplus B} + 1.
$$

The rule is applied repeatedly until all six modality opinions have been considered.

### 1.7.17. Missing modalities

An unavailable modality was assigned the vacuous Dirichlet opinion:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{1}.
$$

For this opinion:

$$
\mathbf{b}^{(m)}
=
\mathbf{0},
\qquad
u^{(m)} = 1.
$$

A vacuous opinion acts as an identity element in this combination rule. It adds no class evidence, creates no conflict, and leaves the available opinion unchanged.

Therefore, the same fusion procedure can process all participants without complete-case filtering or synthetic modality imputation.

At this stage, I construct only the evidence pathway-fused independent opinion. The interaction-aware interaction pathway query will receive its own evidential head in a later step.

In [ ]:
# ============================================================
# 11. Fusing the independent modality opinions with evidence pathway
# ============================================================

# ------------------------------------------------------------
# Reduced Dempster-Shafer combination rule
# ------------------------------------------------------------

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )


    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }


    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]


        # --------------------------------------------------------
        # Conflict mass
        # --------------------------------------------------------

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)


        # --------------------------------------------------------
        # Fused belief and uncertainty masses
        # --------------------------------------------------------

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation


        # --------------------------------------------------------
        # Recover the fused Dirichlet opinion
        # --------------------------------------------------------

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }


    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # I retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }


        # --------------------------------------------------------
        # Final fused opinion
        # --------------------------------------------------------

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }


# ------------------------------------------------------------
# Instantiate the modality-specific evidence fusion module
# ------------------------------------------------------------

tmc_fusion = TMCFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
)


# ------------------------------------------------------------
# Fuse the current independent modality opinions
# ------------------------------------------------------------

with torch.no_grad():

    tmc_output = tmc_fusion(
        modality_opinions=modality_opinions
    )


# ------------------------------------------------------------
# Display final fused tensor shapes
# ------------------------------------------------------------

print("=" * 72)
print("TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    f"\nFusion order: "
    f"{MODALITY_ORDER}"
)

print(
    "\nFused evidence shape: "
    f"{tuple(tmc_output['evidence'].shape)}"
)

print(
    "Fused alpha shape: "
    f"{tuple(tmc_output['alpha'].shape)}"
)

print(
    "Fused probability shape: "
    f"{tuple(tmc_output['probabilities'].shape)}"
)

print(
    "Fused uncertainty shape: "
    f"{tuple(tmc_output['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Final participant-level fused opinions
# ------------------------------------------------------------

fused_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    fused_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "AVAILABLE_MODALITIES": int(
                example_batch["branch_masks"][
                    batch_row
                ].sum()
                .item()
            ),

            "ALPHA_sMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                tmc_output["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


fused_opinion_summary = pd.DataFrame(
    fused_opinion_rows
)

print("\nFinal TMC-fused opinions:")

display(
    fused_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Inspect the sequential fusion history
# ------------------------------------------------------------

fusion_history_rows = []

for modality_name in MODALITY_ORDER:

    stage_output = tmc_output[
        "fusion_history"
    ][modality_name]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):

        modality_index = MODALITY_ORDER.index(
            modality_name
        )

        fusion_history_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "FUSED_THROUGH": modality_name,

                "CURRENT_MODALITY_MASK": float(
                    example_batch["branch_masks"][
                        batch_row,
                        modality_index,
                    ].item()
                ),

                "STAGE_CONFLICT": float(
                    stage_output["conflict"][
                        batch_row,
                        0,
                    ].item()
                ),

                "STAGE_UNCERTAINTY": float(
                    stage_output["uncertainty"][
                        batch_row,
                        0,
                    ].item()
                ),
            }
        )


fusion_history_summary = pd.DataFrame(
    fusion_history_rows
)

print("\nSequential fusion history:")

display(
    fusion_history_summary.round(6)
)


print(
    "\nUnavailable modalities should introduce zero conflict "
    "and leave the accumulated opinion unchanged."
)

print(
    "The TMC-fused independent opinion is ready for later "
    "combination with the interaction-aware 3MT opinion."
)

### 1.7.18. Producing the interaction-aware interaction pathway evidential opinion

The modality-specific evidence pathway combines independent modality opinions produced before cross-modal interaction. construct a separate evidential head for the final query produced by the interaction pathway cascade.

The final interaction-pathway representation is:

$$
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\in
\mathbb{R}^{64}.
$$

Unlike the independent modality representations, this vector contains cumulative information learned through the ordered sequence of Cascaded Modality Transformers.

The joint evidential head produces non-negative class evidence:

$$
e_{ik}^{\mathrm{joint}}
=
\operatorname{Softplus}
\left(
f_{\mathrm{joint}}
\left(
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\right)
\right),
$$

where \(k\) denotes either sMCI or pMCI.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{\mathrm{joint}}
=
e_{ik}^{\mathrm{joint}} + 1.
$$

The expected class probabilities are:

$$
p_{ik}^{\mathrm{joint}}
=
\frac{
\alpha_{ik}^{\mathrm{joint}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{joint}}
}.
$$

The joint uncertainty is:

$$
u_{i}^{\mathrm{joint}}
=
\frac{K}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{joint}}
}.
$$

This opinion serves a different purpose from the evidence pathway-fused independent opinion:

- the modality-specific opinion represents agreement and conflict between modality-specific predictions;
- the joint interaction pathway opinion represents the prediction obtained after learning cross-modal interactions.

### 1.7.19. Auxiliary outputs

The original interaction pathway architecture places an auxiliary classifier after each intermediate cross-modal interaction to provide direct training signals to earlier cascade stages.

The notebook preserves that principle by attaching one auxiliary classifier to every intermediate query except the final MRI stage. These auxiliary heads produce ordinary logits for the prognosis classes and are used only during training.

The auxiliary classifiers are not treated as independent modality-specific opinions because each intermediate query already contains information accumulated from all preceding modalities. They support gradient flow through the cascade but do not represent isolated modality evidence.

The final MRI-stage query receives the joint evidential head and produces the interaction-aware Dirichlet opinion.

In [ ]:
# ============================================================
# 12. Producing the interaction-aware interaction pathway evidential opinion
# ============================================================

# ------------------------------------------------------------
# Auxiliary classifier for one intermediate interaction pathway query
# ------------------------------------------------------------

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(self, representation):
        return self.network(
            representation
        )


# ------------------------------------------------------------
# Joint interaction pathway evidential and auxiliary heads
# ------------------------------------------------------------

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )


    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # --------------------------------------------------------
        # Intermediate auxiliary predictions
        # --------------------------------------------------------

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )


        # --------------------------------------------------------
        # Final joint evidential opinion
        # --------------------------------------------------------

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final interaction pathway query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }


# ------------------------------------------------------------
# Instantiate the interaction-pathway prediction heads
# ------------------------------------------------------------

three_mt_prediction_heads = ThreeMTPredictionHeads(
    cascade_order=THREE_MT_CASCADE_ORDER,
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
)


# ------------------------------------------------------------
# Produce the auxiliary and joint outputs
# ------------------------------------------------------------

three_mt_prediction_heads.eval()

with torch.no_grad():

    three_mt_predictions = (
        three_mt_prediction_heads(
            three_mt_output=three_mt_output
        )
    )


joint_opinion = three_mt_predictions[
    "joint_opinion"
]


# ------------------------------------------------------------
# Display the output structure
# ------------------------------------------------------------

print("=" * 72)
print("3MT PREDICTION HEADS")
print("=" * 72)

print(
    f"\nAuxiliary stages: "
    f"{three_mt_prediction_heads.auxiliary_stages}"
)

print(
    f"Final evidential stage: "
    f"{three_mt_prediction_heads.final_stage}"
)

print("\nAuxiliary-logit shapes:")

for stage_name, stage_logits in (
    three_mt_predictions[
        "auxiliary_logits"
    ].items()
):
    print(
        f"- after {stage_name}: "
        f"{tuple(stage_logits.shape)}"
    )


print("\nJoint evidential shapes:")

print(
    "  evidence: "
    f"{tuple(joint_opinion['evidence'].shape)}"
)

print(
    "  alpha: "
    f"{tuple(joint_opinion['alpha'].shape)}"
)

print(
    "  probabilities: "
    f"{tuple(joint_opinion['probabilities'].shape)}"
)

print(
    "  uncertainty: "
    f"{tuple(joint_opinion['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Participant-level joint opinions
# ------------------------------------------------------------

joint_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    joint_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ALPHA_sMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                joint_opinion["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


joint_opinion_summary = pd.DataFrame(
    joint_opinion_rows
)

print("\nInteraction-aware 3MT opinions:")

display(
    joint_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable 3MT prediction-head parameters: "
    f"{count_trainable_parameters(three_mt_prediction_heads):,}"
)

print(
    "\nThe auxiliary outputs will support training-time "
    "gradient flow through the cascade."
)

print(
    "The final joint opinion is ready for combination with "
    "the TMC-fused independent opinion."
)

### 1.7.20. Combining the interaction pathway and modality-specific evidence pathways with a constrained reliability gate

The model currently produces two complementary evidential outputs:

1. the interaction-aware interaction pathway opinion;
2. the independently fused modality-specific opinion.

These opinions are derived from the same underlying participant data and therefore should not be combined using Dempster--Shafer fusion as though they were independent evidence sources.

Instead, This notebook uses a participant-specific convex mixture of their calibrated evidence vectors.

### 1.7.21. Pathway calibration

The evidence magnitudes produced by the two pathways may have different numerical scales. In particular, the modality-specific evidence pathway accumulates evidence across several available modalities, whereas the cross-modal interaction pathway produces evidence from one joint head.

I therefore introduce one positive scalar calibration parameter for each pathway:

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
=
\tau_{\mathrm{interaction pathway}}
\mathbf{e}_{i}^{\mathrm{interaction pathway}},
$$

and

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}
=
\tau_{\mathrm{evidence pathway}}
\mathbf{e}_{i}^{\mathrm{evidence pathway}}.
$$

Both calibration factors are constrained to be positive using the Softplus function:

$$
\tau_r
=
\operatorname{Softplus}(\rho_r),
\qquad
r \in
\{\mathrm{interaction pathway},\mathrm{evidence pathway}\}.
$$

The parameters are initialised so that both calibration factors begin at approximately one.

### 1.7.22. Reliability-gate input

The reliability gate receives only uncertainty, conflict, and modality-availability information. It does not receive the original participant features or the full interaction-pathway representation.

For participant \(i\), the gate input is:

$$
\mathbf{r}_i
=
\left[
u_i^{\mathrm{interaction pathway}},
u_i^{\mathrm{evidence pathway}},
\bar{C}_i^{\mathrm{evidence pathway}},
\frac{n_i^{\mathrm{available}}}{M},
\mathbf{a}_i
\right],
$$

where:

- \(u_i^{\mathrm{interaction pathway}}\) is the uncertainty of the joint interaction pathway opinion;
- \(u_i^{\mathrm{evidence pathway}}\) is the uncertainty of the evidence pathway-fused opinion;
- \(\bar{C}_i^{\mathrm{evidence pathway}}\) is the mean conflict encountered when combining available modality opinions;
- \(n_i^{\mathrm{available}}\) is the number of available branches;
- \(M=6\) is the total number of branches;
- \(\mathbf{a}_i\) is the six-element branch-availability vector.

The gate produces one scalar weight:

$$
w_i
=
\sigma
\left(
\mathbf{w}^{\top}
\mathbf{r}_i+b
\right).
$$

The interpretation is:

$$
w_i \rightarrow 1
\quad
\Longrightarrow
\quad
\text{greater reliance on interaction pathway},
$$

and

$$
w_i \rightarrow 0
\quad
\Longrightarrow
\quad
\text{greater reliance on evidence pathway}.
$$

The gate is initialised with zero weights and zero bias. Therefore, before training:

$$
w_i = 0.5.
$$

This prevents either pathway from being preferred arbitrarily at model initialisation.

### 1.7.23. Final evidential opinion

The final evidence is:

$$
\mathbf{e}_{i}^{\mathrm{final}}
=
w_i
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
+
(1-w_i)
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}.
$$

Because \(w_i \in [0,1]\), this is a convex mixture rather than an addition of two supposedly independent evidence sources.

The final Dirichlet parameters are:

$$
\boldsymbol{\alpha}_{i}^{\mathrm{final}}
=
\mathbf{e}_{i}^{\mathrm{final}}
+
\mathbf{1}.
$$

The final class probabilities and uncertainty are:

$$
p_{ik}^{\mathrm{final}}
=
\frac{
\alpha_{ik}^{\mathrm{final}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{final}}
},
$$

and

$$
u_i^{\mathrm{final}}
=
\frac{
K
}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{final}}
}.
$$

The reliability statistics supplied to the gate are detached from the computational graph. This prevents the upstream pathways from manipulating their uncertainty or conflict values merely to obtain a larger gate weight. The final loss can still train both pathways through their evidence contributions.

In [ ]:
# ============================================================
# 13. Combining interaction pathway and evidence pathway with fixed equal fusion
# ============================================================

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )


class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )


    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )


    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


hybrid_fusion = FixedEqualHybridFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    number_of_modalities=len(
        MODALITY_ORDER
    ),
    initial_three_mt_scale=1.0,
    initial_tmc_scale=1.0,
)


hybrid_fusion.eval()

with torch.no_grad():
    hybrid_output = hybrid_fusion(
        joint_opinion=joint_opinion,
        tmc_output=tmc_output,
        branch_masks=example_batch[
            "branch_masks"
        ],
        modality_order=MODALITY_ORDER,
    )


print("=" * 72)
print("FIXED 50/50 3MT-TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    "\nFinal probability shape: "
    f"{tuple(hybrid_output['probabilities'].shape)}"
)

print(
    "Initial 3MT evidence scale: "
    f"{hybrid_output['three_mt_scale'].item():.6f}"
)

print(
    "Initial TMC evidence scale: "
    f"{hybrid_output['tmc_scale'].item():.6f}"
)

print(
    "Trainable fixed-fusion parameters: "
    f"{count_trainable_parameters(hybrid_fusion):,}"
)

maximum_three_mt_weight_error = (
    hybrid_output[
        "three_mt_weight"
    ]
    - 0.5
).abs().max().item()

maximum_tmc_weight_error = (
    hybrid_output[
        "tmc_weight"
    ]
    - 0.5
).abs().max().item()

probability_sum_error = (
    hybrid_output[
        "probabilities"
    ].sum(
        dim=-1
    )
    - 1.0
).abs().max().item()

print(
    "\nMaximum 3MT-weight deviation from 0.5: "
    f"{maximum_three_mt_weight_error:.10f}"
)

print(
    "Maximum TMC-weight deviation from 0.5: "
    f"{maximum_tmc_weight_error:.10f}"
)

print(
    "Maximum final probability-sum error: "
    f"{probability_sum_error:.10f}"
)

assert maximum_three_mt_weight_error == 0.0
assert maximum_tmc_weight_error == 0.0

print(
    "\nThe participant-specific reliability gate is absent. "
    "Only the two positive pathway evidence scales remain trainable."
)


### 1.7.24. Assembling the complete availability-gated interaction-evidence model

The complete model keeps the same six components used in the original fold-0 run:

1. modality-specific encoders;
2. training-time modality dropout;
3. independent modality evidential heads;
4. modality-specific evidence fusion;
5. the interaction pathway interaction pathway;
6. fixed-equal hybrid evidence fusion.

The effective branch masks are created before encoding. They contain both natural missingness and any additional modality removed by training-time dropout. These exact masks are now passed into the interaction pathway cascade, so every unavailable cross-modal interaction stage preserves the preceding cumulative query.

The modality-specific evidence pathway and fixed equal fusion are unchanged. This isolates the effect of availability-gating the cross-modal interaction updates.

In [ ]:
# ============================================================
# 14. Assembling the complete end-to-end interaction-evidence model
# ============================================================

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )


        # --------------------------------------------------------
        # Modality-specific encoders
        # --------------------------------------------------------

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )


        # --------------------------------------------------------
        # Independent modality evidential pathway
        # --------------------------------------------------------

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )


        # --------------------------------------------------------
        # Final fixed 50/50 hybrid fusion
        # --------------------------------------------------------

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )


    # ------------------------------------------------------------
    # Training-time modality dropout
    # ------------------------------------------------------------

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )


        # --------------------------------------------------------
        # Sample branch-retention indicators
        # --------------------------------------------------------

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )


        # --------------------------------------------------------
        # Prevent complete information removal
        # --------------------------------------------------------

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # I randomly restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0


        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )


    # ------------------------------------------------------------
    # Construct effective modality dictionaries
    # ------------------------------------------------------------

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities


    # ------------------------------------------------------------
    # Complete forward pass
    # ------------------------------------------------------------

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # --------------------------------------------------------
        # Apply training-time modality dropout
        # --------------------------------------------------------

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )


        # --------------------------------------------------------
        # Encode all six modalities
        # --------------------------------------------------------

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. I use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]


        # --------------------------------------------------------
        # Independent modality opinions and modality-specific evidence fusion
        # --------------------------------------------------------

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]


        # --------------------------------------------------------
        # Final fixed-equal hybrid opinion
        # --------------------------------------------------------

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )


        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


# ------------------------------------------------------------
# Instantiate the complete model
# ------------------------------------------------------------

complete_model = ADNIEvidential3MTTMCModel(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    modality_dropout_probability=0.50,
)


# ------------------------------------------------------------
# Evaluation-mode end-to-end forward pass
# ------------------------------------------------------------

# In evaluation mode, modality dropout is disabled.
complete_model.eval()

with torch.no_grad():

    complete_model_output = complete_model(
        modalities=example_batch[
            "modalities"
        ],

        original_branch_masks=example_batch[
            "branch_masks"
        ],
    )


# ------------------------------------------------------------
# Inspect final outputs
# ------------------------------------------------------------

final_output = complete_model_output[
    "final_output"
]

print("=" * 72)
print("COMPLETE AVAILABILITY-GATED 3MT-TMC MODEL")
print("=" * 72)

print(
    "\nModel mode: "
    f"{'training' if complete_model.training else 'evaluation'}"
)

print(
    "Modality-dropout probability: "
    f"{complete_model.modality_dropout_probability:.2f}"
)

print(
    "\nOriginal branch-mask shape: "
    f"{tuple(complete_model_output['original_branch_masks'].shape)}"
)

print(
    "Effective branch-mask shape: "
    f"{tuple(complete_model_output['effective_branch_masks'].shape)}"
)

print(
    "\nFinal alpha shape: "
    f"{tuple(final_output['alpha'].shape)}"
)

print(
    "Final probability shape: "
    f"{tuple(final_output['probabilities'].shape)}"
)

print(
    "Final uncertainty shape: "
    f"{tuple(final_output['uncertainty'].shape)}"
)

print(
    "\nTotal trainable model parameters: "
    f"{count_trainable_parameters(complete_model):,}"
)


# ------------------------------------------------------------
# Verify that evaluation mode preserves genuine availability
# ------------------------------------------------------------

evaluation_mask_difference = (
    complete_model_output[
        "effective_branch_masks"
    ]
    - complete_model_output[
        "original_branch_masks"
    ]
).abs().max().item()

print(
    "\nMaximum evaluation-mode difference between original "
    f"and effective branch masks: "
    f"{evaluation_mask_difference:.10f}"
)


# ------------------------------------------------------------
# Participant-level output summary
# ------------------------------------------------------------

complete_model_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    original_available_count = int(
        complete_model_output[
            "original_branch_masks"
        ][batch_row].sum().item()
    )

    effective_available_count = int(
        complete_model_output[
            "effective_branch_masks"
        ][batch_row].sum().item()
    )

    complete_model_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ORIGINAL_MODALITIES":
                original_available_count,

            "EFFECTIVE_MODALITIES":
                effective_available_count,

            "W_3MT": float(
                final_output[
                    "three_mt_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "W_TMC": float(
                final_output[
                    "tmc_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_sMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    1,
                ].item()
            ),

            "FINAL_UNCERTAINTY": float(
                final_output[
                    "uncertainty"
                ][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


complete_model_summary = pd.DataFrame(
    complete_model_rows
)

print("\nComplete-model evaluation-mode outputs:")

display(
    complete_model_summary.round(6)
)

print(
    "\nThe complete model now passes the effective branch "
    "masks into every CMT stage."
)


### 1.7.25. Defining the joint training objective

The complete architecture produces several supervised outputs with different purposes:

1. the final fixed-equal hybrid opinion;
2. the interaction-aware interaction pathway joint opinion;
3. the evidence pathway-fused independent-modality opinion;
4. six independent modality-specific opinions;
5. five intermediate interaction pathway auxiliary predictions.

These outputs are trained jointly, but the final hybrid prediction remains the primary objective.

The total loss is:

$$
\mathcal{L}_{\mathrm{total}}
=
\mathcal{L}_{\mathrm{final}}
+
\lambda_{\mathrm{joint}}
\mathcal{L}_{\mathrm{joint}}
+
\lambda_{\mathrm{evidence pathway}}
\mathcal{L}_{\mathrm{evidence pathway}}
+
\lambda_{\mathrm{mod}}
\mathcal{L}_{\mathrm{mod}}
+
\lambda_{\mathrm{aux}}
\mathcal{L}_{\mathrm{aux}}
+
\lambda_{\mathrm{gate}}
\mathcal{L}_{\mathrm{gate}}.
$$

The initial loss weights are:

$$
\lambda_{\mathrm{joint}} = 0.5,
\qquad
\lambda_{\mathrm{evidence pathway}} = 0.5,
\qquad
\lambda_{\mathrm{mod}} = 0.1,
$$

$$
\lambda_{\mathrm{aux}} = 0.1,
\qquad
\lambda_{\mathrm{gate}} = 0.01.
$$

The final hybrid loss has coefficient one and therefore remains the dominant objective. The remaining terms provide direct supervision to the two pathways and their intermediate components.

These coefficients are initial modelling choices. Any comparison of alternative values must use only the training and validation partitions.

### 1.7.26. Evidential classification loss

For a Dirichlet prediction:

$$
\boldsymbol{\alpha}_i
=
\mathbf{e}_i+\mathbf{1},
$$

the expected cross-entropy loss is:

$$
\mathcal{L}_{\mathrm{ECE},i}
=
\sum_{k=1}^{K}
y_{ik}
\left[
\psi(S_i)
-
\psi(\alpha_{ik})
\right],
$$

where:

$$
S_i
=
\sum_{k=1}^{K}
\alpha_{ik},
$$

and \(\psi(\cdot)\) denotes the digamma function.

This objective minimises the expected negative log-likelihood under the predicted Dirichlet distribution.

### 1.7.27. Evidence regularisation

An evidential network may become unjustifiably confident by assigning strong evidence to an incorrect class. I therefore add a Kullback--Leibler regularisation term that discourages unsupported evidence.

The adjusted Dirichlet parameters are:

$$
\widetilde{\boldsymbol{\alpha}}_i
=
\mathbf{y}_i
+
(1-\mathbf{y}_i)
\odot
\boldsymbol{\alpha}_i.
$$

This construction removes the evidence assigned to the correct class from the regularisation term while penalising evidence assigned to incorrect classes.

The regularisation term is:

$$
\mathcal{L}_{\mathrm{KL},i}
=
D_{\mathrm{KL}}
\left[
\operatorname{Dir}
\left(
\widetilde{\boldsymbol{\alpha}}_i
\right)
\parallel
\operatorname{Dir}
\left(
\mathbf{1}
\right)
\right].
$$

The complete evidential loss is:

$$
\mathcal{L}_{\mathrm{EDL},i}
=
\mathcal{L}_{\mathrm{ECE},i}
+
\beta_t
\mathcal{L}_{\mathrm{KL},i}.
$$

The regularisation coefficient is annealed during the first training epochs:

$$
\beta_t
=
\min
\left(
1,
\frac{t}{T_{\mathrm{anneal}}}
\right),
$$

where \(t\) is the current epoch and \(T_{\mathrm{anneal}}\) is initially set to ten epochs.

This allows the model to begin learning the classification task before the full evidence penalty is applied.

### 1.7.28. Modality-specific loss

Each independent modality opinion is supervised only when that modality is effectively available after training-time modality dropout.

For branch \(m\), let:

$$
\widetilde{a}_i^{(m)}
\in
\{0,1\}
$$

denote the effective branch mask.

The modality-specific loss is:

$$
\mathcal{L}_{\mathrm{mod}}
=
\frac{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
\mathcal{L}_{\mathrm{EDL},i}^{(m)}
}{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
+
\varepsilon
}.
$$

Naturally unavailable or deliberately dropped branches therefore contribute no modality-specific classification loss.

### 1.7.29. Auxiliary interaction pathway loss

The intermediate interaction pathway heads produce ordinary class logits rather than Dirichlet opinions. Their loss is the mean cross-entropy across the five intermediate cascade stages:

$$
\mathcal{L}_{\mathrm{aux}}
=
\frac{1}{J}
\sum_{j=1}^{J}
\operatorname{CE}
\left(
\mathbf{z}^{(j)},
y
\right),
$$

where \(J=5\).

These losses provide direct gradient signals to earlier stages of the cascaded transformer.

### 1.7.30. Gate regularisation

The reliability gate is initialised at:

$$
w_i=0.5.
$$

A weak early-training regulariser discourages immediate collapse to a single pathway:

$$
\mathcal{L}_{\mathrm{gate}}
=
\left(
\frac{1}{N}
\sum_{i=1}^{N}
w_i
-
0.5
\right)^2.
$$

The gate regularisation is annealed to zero after the initial training period. It therefore stabilises early optimisation without forcing the final trained gate to remain balanced.

This cell defines and validates the training objective only. Parameter updates begin after gradient flow is checked in the following step.

In [ ]:
# ============================================================
# 15. Defining the complete joint training objective
# ============================================================

# ------------------------------------------------------------
# KL divergence between a predicted Dirichlet distribution and
# a uniform Dirichlet distribution
# ------------------------------------------------------------

def dirichlet_kl_to_uniform(
    alpha,
):
    """
    Calculate:

        KL(Dir(alpha) || Dir(1))

    for every participant in the batch.

    Parameters
    ----------
    alpha:
        Positive Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    Returns
    -------
    Tensor with shape:
        (batch_size,)
    """

    number_of_classes = alpha.shape[-1]

    uniform_alpha = torch.ones_like(
        alpha
    )

    alpha_strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    uniform_strength = uniform_alpha.sum(
        dim=-1,
        keepdim=True,
    )

    log_normalisation_ratio = (
        torch.lgamma(alpha_strength)
        - torch.lgamma(uniform_strength)
        - torch.lgamma(alpha).sum(
            dim=-1,
            keepdim=True,
        )
        + torch.lgamma(uniform_alpha).sum(
            dim=-1,
            keepdim=True,
        )
    )

    digamma_difference = (
        torch.digamma(alpha)
        - torch.digamma(alpha_strength)
    )

    parameter_difference = (
        alpha
        - uniform_alpha
    )

    expectation_term = (
        parameter_difference
        * digamma_difference
    ).sum(
        dim=-1,
        keepdim=True,
    )

    kl_divergence = (
        log_normalisation_ratio
        + expectation_term
    )

    return kl_divergence.squeeze(-1)


# ------------------------------------------------------------
# Evidential classification loss
# ------------------------------------------------------------

def evidential_classification_loss(
    alpha,
    targets,
    number_of_classes,
    annealing_coefficient,
    class_weights=None,
    reduction="mean",
):
    """
    Calculate the expected cross-entropy under a Dirichlet
    distribution together with annealed KL regularisation.

    Parameters
    ----------
    alpha:
        Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    targets:
        Integer class labels with shape:
        (batch_size,)

    number_of_classes:
        Number of prognosis classes.

    annealing_coefficient:
        Current coefficient applied to the KL term.

    class_weights:
        Optional class-weight tensor with shape:
        (number_of_classes,)

    reduction:
        "none", "mean", or "sum".
    """

    targets = targets.long()

    one_hot_targets = F.one_hot(
        targets,
        num_classes=number_of_classes,
    ).to(
        dtype=alpha.dtype
    )

    strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )


    # --------------------------------------------------------
    # Expected cross-entropy under the Dirichlet distribution
    # --------------------------------------------------------

    expected_cross_entropy_by_class = (
        torch.digamma(strength)
        - torch.digamma(alpha)
    )

    expected_cross_entropy = (
        one_hot_targets
        * expected_cross_entropy_by_class
    ).sum(
        dim=-1
    )


    # --------------------------------------------------------
    # Optional class weighting
    # --------------------------------------------------------

    if class_weights is not None:

        sample_weights = class_weights[
            targets
        ].to(
            dtype=alpha.dtype,
            device=alpha.device,
        )

        expected_cross_entropy = (
            expected_cross_entropy
            * sample_weights
        )


    # --------------------------------------------------------
    # Remove correct-class evidence from the KL penalty
    # --------------------------------------------------------

    adjusted_alpha = (
        one_hot_targets
        +
        (
            1.0
            - one_hot_targets
        )
        * alpha
    )

    kl_regularisation = (
        dirichlet_kl_to_uniform(
            adjusted_alpha
        )
    )

    per_sample_loss = (
        expected_cross_entropy
        +
        annealing_coefficient
        * kl_regularisation
    )


    # --------------------------------------------------------
    # Requested reduction
    # --------------------------------------------------------

    if reduction == "none":
        reduced_loss = per_sample_loss

    elif reduction == "mean":
        reduced_loss = per_sample_loss.mean()

    elif reduction == "sum":
        reduced_loss = per_sample_loss.sum()

    else:
        raise ValueError(
            "reduction must be 'none', 'mean', or 'sum'."
        )


    return {
        "loss":
            reduced_loss,

        "per_sample_loss":
            per_sample_loss,

        "expected_cross_entropy":
            expected_cross_entropy,

        "kl_regularisation":
            kl_regularisation,
    }


# ------------------------------------------------------------
# Complete multi-output objective
# ------------------------------------------------------------

class HybridEvidentialTrainingLoss(nn.Module):
    """
    Jointly supervise the final hybrid output, both main pathways,
    the individual available modality opinions, and the auxiliary
    3MT classifiers.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        evidential_annealing_epochs=10,
        gate_regularisation_epochs=0,
        joint_loss_weight=0.50,
        tmc_loss_weight=0.50,
        modality_loss_weight=0.10,
        auxiliary_loss_weight=0.10,
        gate_loss_weight=0.0,
        class_weights=None,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.evidential_annealing_epochs = (
            evidential_annealing_epochs
        )

        self.gate_regularisation_epochs = (
            gate_regularisation_epochs
        )

        self.joint_loss_weight = (
            joint_loss_weight
        )

        self.tmc_loss_weight = (
            tmc_loss_weight
        )

        self.modality_loss_weight = (
            modality_loss_weight
        )

        self.auxiliary_loss_weight = (
            auxiliary_loss_weight
        )

        self.gate_loss_weight = (
            gate_loss_weight
        )


        # --------------------------------------------------------
        # Optional training-fold class weights
        # --------------------------------------------------------

        if class_weights is None:

            self.register_buffer(
                "class_weights",
                None,
            )

        else:

            class_weights = torch.as_tensor(
                class_weights,
                dtype=torch.float32,
            )

            if class_weights.shape != (
                number_of_classes,
            ):
                raise ValueError(
                    "class_weights must contain one value "
                    "for each class."
                )

            self.register_buffer(
                "class_weights",
                class_weights,
            )


    # ------------------------------------------------------------
    # Annealing schedules
    # ------------------------------------------------------------

    def _evidential_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Increase the KL coefficient linearly from zero to one.
        """

        if self.evidential_annealing_epochs <= 0:
            return 1.0

        return min(
            1.0,
            float(epoch)
            / float(
                self.evidential_annealing_epochs
            ),
        )


    def _gate_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Reduce the gate-balance penalty to zero after the initial
        optimisation period.
        """

        if self.gate_regularisation_epochs <= 0:
            return 0.0

        return max(
            0.0,
            1.0
            -
            (
                float(epoch - 1)
                /
                float(
                    self.gate_regularisation_epochs
                )
            ),
        )


    # ------------------------------------------------------------
    # Complete loss calculation
    # ------------------------------------------------------------

    def forward(
        self,
        model_output,
        targets,
        epoch,
    ):
        targets = targets.long()

        evidential_annealing = (
            self._evidential_annealing_coefficient(
                epoch
            )
        )

        gate_annealing = (
            self._gate_annealing_coefficient(
                epoch
            )
        )


        # --------------------------------------------------------
        # Final hybrid evidential loss
        # --------------------------------------------------------

        final_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "final_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        final_loss = final_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Interaction-aware interaction pathway evidential loss
        # --------------------------------------------------------

        joint_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "joint_opinion"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        joint_loss = joint_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # evidence pathway-fused evidential loss
        # --------------------------------------------------------

        tmc_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "tmc_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        tmc_loss = tmc_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Independent available-modality evidential loss
        # --------------------------------------------------------

        effective_branch_masks = model_output[
            "effective_branch_masks"
        ]

        weighted_modality_loss_sum = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        available_opinion_count = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        modality_loss_by_name = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):

            modality_alpha = model_output[
                "modality_opinions"
            ][modality_name]["alpha"]

            modality_loss_components = (
                evidential_classification_loss(
                    alpha=modality_alpha,

                    targets=targets,

                    number_of_classes=
                        self.number_of_classes,

                    annealing_coefficient=
                        evidential_annealing,

                    class_weights=
                        self.class_weights,

                    reduction="none",
                )
            )

            per_sample_modality_loss = (
                modality_loss_components[
                    "per_sample_loss"
                ]
            )

            modality_mask = effective_branch_masks[
                :,
                modality_index,
            ].to(
                dtype=per_sample_modality_loss.dtype
            )

            masked_modality_loss_sum = (
                per_sample_modality_loss
                * modality_mask
            ).sum()

            modality_available_count = (
                modality_mask.sum()
            )

            weighted_modality_loss_sum = (
                weighted_modality_loss_sum
                + masked_modality_loss_sum
            )

            available_opinion_count = (
                available_opinion_count
                + modality_available_count
            )

            modality_loss_by_name[
                modality_name
            ] = (
                masked_modality_loss_sum
                /
                modality_available_count.clamp_min(
                    1.0
                )
            )

        modality_loss = (
            weighted_modality_loss_sum
            /
            available_opinion_count.clamp_min(
                1.0
            )
        )


        # --------------------------------------------------------
        # Intermediate interaction pathway auxiliary cross-entropy loss
        # --------------------------------------------------------

        auxiliary_logits = model_output[
            "three_mt_predictions"
        ]["auxiliary_logits"]

        auxiliary_loss_by_stage = {}

        auxiliary_losses = []

        for stage_name, stage_logits in (
            auxiliary_logits.items()
        ):

            stage_loss = F.cross_entropy(
                input=stage_logits,
                target=targets,
                weight=self.class_weights,
            )

            auxiliary_loss_by_stage[
                stage_name
            ] = stage_loss

            auxiliary_losses.append(
                stage_loss
            )

        if auxiliary_losses:

            auxiliary_loss = torch.stack(
                auxiliary_losses
            ).mean()

        else:

            auxiliary_loss = torch.zeros(
                (),
                dtype=final_loss.dtype,
                device=final_loss.device,
            )


        # --------------------------------------------------------
        # Early gate-balance regularisation
        # --------------------------------------------------------

        three_mt_weights = model_output[
            "final_output"
        ]["three_mt_weight"]

        raw_gate_loss = (
            three_mt_weights.mean()
            - 0.5
        ).pow(2)

        annealed_gate_loss = (
            gate_annealing
            * raw_gate_loss
        )


        # --------------------------------------------------------
        # Weighted total objective
        # --------------------------------------------------------

        total_loss = (
            final_loss
            +
            self.joint_loss_weight
            * joint_loss
            +
            self.tmc_loss_weight
            * tmc_loss
            +
            self.modality_loss_weight
            * modality_loss
            +
            self.auxiliary_loss_weight
            * auxiliary_loss
            +
            self.gate_loss_weight
            * annealed_gate_loss
        )


        return {
            "total_loss":
                total_loss,

            "final_loss":
                final_loss,

            "joint_loss":
                joint_loss,

            "tmc_loss":
                tmc_loss,

            "modality_loss":
                modality_loss,

            "auxiliary_loss":
                auxiliary_loss,

            "raw_gate_loss":
                raw_gate_loss,

            "annealed_gate_loss":
                annealed_gate_loss,

            "evidential_annealing":
                torch.tensor(
                    evidential_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "gate_annealing":
                torch.tensor(
                    gate_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "available_opinion_count":
                available_opinion_count,

            "modality_loss_by_name":
                modality_loss_by_name,

            "auxiliary_loss_by_stage":
                auxiliary_loss_by_stage,

            "final_expected_cross_entropy":
                final_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "final_kl_regularisation":
                final_loss_components[
                    "kl_regularisation"
                ].mean(),

            "joint_expected_cross_entropy":
                joint_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "joint_kl_regularisation":
                joint_loss_components[
                    "kl_regularisation"
                ].mean(),

            "tmc_expected_cross_entropy":
                tmc_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "tmc_kl_regularisation":
                tmc_loss_components[
                    "kl_regularisation"
                ].mean(),
        }


# ------------------------------------------------------------
# Instantiate the training objective
# ------------------------------------------------------------

training_objective = HybridEvidentialTrainingLoss(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,

    evidential_annealing_epochs=10,
    gate_regularisation_epochs=0,

    joint_loss_weight=0.50,
    tmc_loss_weight=0.50,
    modality_loss_weight=0.10,
    auxiliary_loss_weight=0.10,
    gate_loss_weight=0.0,

    # I initially leave class weighting disabled. It can be added
    # using weights calculated from each training fold only.
    class_weights=None,
)


# ------------------------------------------------------------
# Test the objective on the complete untrained forward pass
# ------------------------------------------------------------

example_loss_output = training_objective(
    model_output=complete_model_output,

    targets=example_batch[
        "target"
    ],

    epoch=1,
)


# ------------------------------------------------------------
# Display the initial loss structure
# ------------------------------------------------------------

print("=" * 72)
print("JOINT 3MT-TMC TRAINING OBJECTIVE")
print("=" * 72)

print(
    "\nEvidential KL annealing coefficient at epoch 1: "
    f"{example_loss_output['evidential_annealing'].item():.6f}"
)

print(
    "Gate regularisation coefficient at epoch 1: "
    f"{example_loss_output['gate_annealing'].item():.6f}"
)

print("\nMain loss components:")

for loss_name in [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]:
    print(
        f"- {loss_name}: "
        f"{example_loss_output[loss_name].item():.6f}"
    )


print("\nFinal evidential-loss decomposition:")

print(
    "- expected cross-entropy: "
    f"{example_loss_output['final_expected_cross_entropy'].item():.6f}"
)

print(
    "- KL regularisation: "
    f"{example_loss_output['final_kl_regularisation'].item():.6f}"
)


print("\nAvailable modality opinions in this batch: "
      f"{int(example_loss_output['available_opinion_count'].item())}")


# ------------------------------------------------------------
# Modality-specific loss summary
# ------------------------------------------------------------

modality_loss_rows = []

for modality_name in MODALITY_ORDER:

    modality_loss_rows.append(
        {
            "MODALITY": modality_name,

            "MEAN_AVAILABLE_LOSS": float(
                example_loss_output[
                    "modality_loss_by_name"
                ][modality_name].item()
            ),
        }
    )


print("\nModality-specific evidential losses:")

display(
    pd.DataFrame(
        modality_loss_rows
    ).round(6)
)


# ------------------------------------------------------------
# Auxiliary-stage loss summary
# ------------------------------------------------------------

auxiliary_loss_rows = []

for stage_name in THREE_MT_CASCADE_ORDER[:-1]:

    auxiliary_loss_rows.append(
        {
            "AUXILIARY_STAGE": stage_name,

            "CROSS_ENTROPY_LOSS": float(
                example_loss_output[
                    "auxiliary_loss_by_stage"
                ][stage_name].item()
            ),
        }
    )


print("\nIntermediate 3MT auxiliary losses:")

display(
    pd.DataFrame(
        auxiliary_loss_rows
    ).round(6)
)

print(
    "\nThe joint objective is ready for one end-to-end "
    "backward pass."
)


### 1.7.31. Running one end-to-end backward pass

Before configuring the optimiser, I run one training batch through the complete gated model and joint objective.

For this single diagnostic pass, modality dropout is set to zero so that the natural fold-0 availability pattern is used without additional random branch removal. The total loss is backpropagated, but no optimiser step is taken. The cell reports the total loss and the global gradient norm, then restores the configured modality-dropout probability of `0.50`.

In [ ]:
# ============================================================
# 16. Running one end-to-end backward pass
# ============================================================

diagnostic_batch = next(
    iter(train_loader)
)

original_modality_dropout_probability = (
    complete_model.modality_dropout_probability
)

complete_model.modality_dropout_probability = 0.0
complete_model.train()
complete_model.zero_grad(
    set_to_none=True
)


diagnostic_output = complete_model(
    modalities=diagnostic_batch[
        "modalities"
    ],
    original_branch_masks=diagnostic_batch[
        "branch_masks"
    ],
)

diagnostic_losses = training_objective(
    model_output=diagnostic_output,
    targets=diagnostic_batch[
        "target"
    ],
    epoch=1,
)

diagnostic_total_loss = diagnostic_losses[
    "total_loss"
]

diagnostic_total_loss.backward()


global_gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().float().pow(2).sum()
        for parameter in complete_model.parameters()
        if parameter.grad is not None
    )
)

print("=" * 72)
print("END-TO-END BACKWARD PASS")
print("=" * 72)

print(
    "\nTotal loss: "
    f"{diagnostic_total_loss.item():.6f}"
)

print(
    "Global gradient norm: "
    f"{global_gradient_norm.item():.6f}"
)

complete_model.zero_grad(
    set_to_none=True
)

complete_model.modality_dropout_probability = (
    original_modality_dropout_probability
)

complete_model.eval()

print(
    "Modality-dropout probability restored to: "
    f"{complete_model.modality_dropout_probability:.2f}"
)


### 1.7.32. Configuring optimisation and experiment-specific output paths

The original training hyperparameters remain unchanged. Every checkpoint, history file, validation prediction, and test prediction is written only under:

```text
models/3mt_tmc_evidential/experiments/
    gated_cmt_learned_gate_md050/
        mci_prognosis/fold_0/
```

This notebook does not use the older shared `training/mci_prognosis/fold_0/` directory.

In [ ]:
# ============================================================
# 17. Configuring optimisation, checkpoints, and metrics
# ============================================================

import os
import random
from datetime import datetime

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# Current experiment
# ------------------------------------------------------------

# EXPERIMENT_NAME, SELECTED_TASK, and SELECTED_FOLD were fixed when this fold's prepared input was loaded.


# ------------------------------------------------------------
# Reproducibility configuration
# ------------------------------------------------------------

GLOBAL_RANDOM_SEED = 42


def set_global_random_seed(seed):
    """
    Set the random seed used by Python, NumPy, and PyTorch.
    """

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_random_seed(
    GLOBAL_RANDOM_SEED
)


# ------------------------------------------------------------
# PyTorch numerical configuration
# ------------------------------------------------------------

# I allow cuDNN to choose efficient convolution algorithms.
#
# This is appropriate for the computationally expensive 3D MRI
# encoder. Exact bitwise reproducibility can still depend on the
# installed PyTorch, CUDA, cuDNN, and GPU versions.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 72)
print("TRAINING CONFIGURATION")
print("=" * 72)

print(
    f"\nExperiment: {EXPERIMENT_NAME}"
)

print(
    f"Task: {SELECTED_TASK}"
)

print(
    f"Selected fold: {SELECTED_FOLD}"
)

print(
    f"Device: {DEVICE}"
)

if torch.cuda.is_available():

    print(
        "CUDA device: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        "CUDA memory allocated before model transfer: "
        f"{torch.cuda.memory_allocated(0) / (1024 ** 3):.3f} GB"
    )


# ------------------------------------------------------------
# Move the complete model and objective to the selected device
# ------------------------------------------------------------

complete_model = complete_model.to(
    DEVICE
)

training_objective = training_objective.to(
    DEVICE
)


# ------------------------------------------------------------
# Optimisation hyperparameters
# ------------------------------------------------------------

MAXIMUM_EPOCHS = 50

INITIAL_LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAXIMUM_GRADIENT_NORM = 5.0

EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 3

SCHEDULER_REDUCTION_FACTOR = 0.5

MINIMUM_LEARNING_RATE = 1e-6

CLASSIFICATION_THRESHOLD = 0.50


# ------------------------------------------------------------
# AdamW optimiser
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    params=complete_model.parameters(),
    lr=INITIAL_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ------------------------------------------------------------
# Validation-AUC learning-rate scheduler
# ------------------------------------------------------------

learning_rate_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="max",
        factor=SCHEDULER_REDUCTION_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)


# ------------------------------------------------------------
# Checkpoint and history directories
# ------------------------------------------------------------

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
)

FOLD_TRAINING_DIR = (
    EXPERIMENT_ROOT
    / SELECTED_TASK
    / f"fold_{SELECTED_FOLD}"
)

CHECKPOINT_DIR = (
    FOLD_TRAINING_DIR
    / "checkpoints"
)

HISTORY_DIR = (
    FOLD_TRAINING_DIR
    / "history"
)

PREDICTION_DIR = (
    FOLD_TRAINING_DIR
    / "predictions"
)


BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_validation_auc_checkpoint.pt"
)

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last_epoch_checkpoint.pt"
)


# ------------------------------------------------------------
# Fold-output safety policy
# ------------------------------------------------------------

existing_fold_files = []

if FOLD_TRAINING_DIR.exists():
    existing_fold_files = [
        path
        for path in FOLD_TRAINING_DIR.rglob("*")
        if path.is_file()
    ]


if FOLD_RUN_MODE == "fresh" and existing_fold_files:
    raise FileExistsError(
        "Fresh training was requested, but this fold directory "
        "already contains files. Nothing was overwritten. "
        "Use a different experiment name, remove the intentionally "
        "discarded fold directory, or select resume mode only for "
        "an interrupted run of this exact fold.\n"
        f"Fold directory: {FOLD_TRAINING_DIR}"
    )


if FOLD_RUN_MODE == "resume" and not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Resume mode was requested, but this fold has no latest "
        "checkpoint. Nothing was changed.\n"
        f"Expected checkpoint: {LAST_CHECKPOINT_PATH}"
    )


for directory_path in [
    EXPERIMENT_ROOT,
    FOLD_TRAINING_DIR,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    PREDICTION_DIR,
]:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

TRAINING_HISTORY_PATH = (
    HISTORY_DIR
    / "training_history.csv"
)

TRAINING_CONFIGURATION_PATH = (
    HISTORY_DIR
    / "training_configuration.json"
)

VALIDATION_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "best_validation_predictions.csv"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "test_predictions.csv"
)


# ------------------------------------------------------------
# Record the training configuration
# ------------------------------------------------------------

training_configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "fixed_equal_fusion": True,
    "three_mt_weight": 0.5,
    "tmc_weight": 0.5,
    "task": SELECTED_TASK,
    "fold": int(SELECTED_FOLD),
    "random_seed": int(GLOBAL_RANDOM_SEED),

    "maximum_epochs": int(
        MAXIMUM_EPOCHS
    ),

    "batch_size": int(
        train_loader.batch_size
    ),

    "initial_learning_rate": float(
        INITIAL_LEARNING_RATE
    ),

    "weight_decay": float(
        WEIGHT_DECAY
    ),

    "maximum_gradient_norm": float(
        MAXIMUM_GRADIENT_NORM
    ),

    "early_stopping_patience": int(
        EARLY_STOPPING_PATIENCE
    ),

    "scheduler_patience": int(
        SCHEDULER_PATIENCE
    ),

    "scheduler_reduction_factor": float(
        SCHEDULER_REDUCTION_FACTOR
    ),

    "minimum_learning_rate": float(
        MINIMUM_LEARNING_RATE
    ),

    "classification_threshold": float(
        CLASSIFICATION_THRESHOLD
    ),

    "modality_dropout_probability": float(
        complete_model.modality_dropout_probability
    ),

    "embedding_dimension": int(
        MODALITY_EMBEDDING_DIM
    ),

    "number_of_classes": int(
        NUMBER_OF_CLASSES
    ),

    "class_order": list(
        PROGNOSIS_CLASS_ORDER
    ),

    "modality_order": list(
        MODALITY_ORDER
    ),

    "three_mt_cascade_order": list(
        THREE_MT_CASCADE_ORDER
    ),

    "trainable_parameters": int(
        count_trainable_parameters(
            complete_model
        )
    ),

    "device": str(
        DEVICE
    ),

    "cuda_device_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}


with open(
    TRAINING_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        training_configuration,
        configuration_file,
        indent=2,
    )


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def calculate_binary_expected_calibration_error(
    targets,
    positive_class_probabilities,
    number_of_bins=10,
):
    """
    Calculate equal-width binary expected calibration error.

    Confidence is the probability assigned to the predicted class.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        return float("nan")

    predicted_classes = (
        positive_class_probabilities
        >= 0.5
    ).astype(
        np.int64
    )

    predicted_confidences = np.where(
        predicted_classes == 1,
        positive_class_probabilities,
        1.0 - positive_class_probabilities,
    )

    prediction_correctness = (
        predicted_classes
        == targets
    ).astype(
        np.float64
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    expected_calibration_error = 0.0

    sample_count = targets.size

    for bin_index in range(
        number_of_bins
    ):

        lower_edge = bin_edges[
            bin_index
        ]

        upper_edge = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:

            in_bin = (
                predicted_confidences
                >= lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        else:

            in_bin = (
                predicted_confidences
                > lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        bin_count = int(
            in_bin.sum()
        )

        if bin_count == 0:
            continue

        mean_confidence = float(
            predicted_confidences[
                in_bin
            ].mean()
        )

        mean_accuracy = float(
            prediction_correctness[
                in_bin
            ].mean()
        )

        expected_calibration_error += (
            bin_count
            / sample_count
        ) * abs(
            mean_accuracy
            - mean_confidence
        )

    return float(
        expected_calibration_error
    )


# ------------------------------------------------------------
# Safe metric helpers
# ------------------------------------------------------------

def safely_calculate_roc_auc(
    targets,
    probabilities,
):
    """
    Return NaN when ROC AUC is undefined because only one class
    is present in the supplied targets.
    """

    if np.unique(targets).size < 2:
        return float("nan")

    return float(
        roc_auc_score(
            targets,
            probabilities,
        )
    )


def safely_calculate_average_precision(
    targets,
    probabilities,
):
    """
    Return NaN when average precision is not meaningful because
    the supplied targets contain no positive examples.
    """

    if np.sum(targets == 1) == 0:
        return float("nan")

    return float(
        average_precision_score(
            targets,
            probabilities,
        )
    )


# ------------------------------------------------------------
# Complete binary prognosis metrics
# ------------------------------------------------------------

def calculate_binary_classification_metrics(
    targets,
    positive_class_probabilities,
    uncertainties=None,
    three_mt_weights=None,
    classification_threshold=0.50,
):
    """
    Calculate discrimination, classification, calibration, and
    uncertainty summaries for the pMCI-positive prognosis task.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        raise ValueError(
            "At least one target is required to calculate metrics."
        )

    if (
        targets.shape[0]
        != positive_class_probabilities.shape[0]
    ):
        raise ValueError(
            "Targets and probabilities must contain the same "
            "number of participants."
        )

    positive_class_probabilities = np.clip(
        positive_class_probabilities,
        0.0,
        1.0,
    )

    predicted_classes = (
        positive_class_probabilities
        >= classification_threshold
    ).astype(
        np.int64
    )

    (
        true_negative,
        false_positive,
        false_negative,
        true_positive,
    ) = confusion_matrix(
        targets,
        predicted_classes,
        labels=[0, 1],
    ).ravel()


    sensitivity_denominator = (
        true_positive
        + false_negative
    )

    specificity_denominator = (
        true_negative
        + false_positive
    )

    sensitivity = (
        true_positive
        / sensitivity_denominator
        if sensitivity_denominator > 0
        else float("nan")
    )

    specificity = (
        true_negative
        / specificity_denominator
        if specificity_denominator > 0
        else float("nan")
    )


    clipped_probabilities = np.clip(
        positive_class_probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    metrics = {
        "roc_auc":
            safely_calculate_roc_auc(
                targets,
                positive_class_probabilities,
            ),

        "average_precision":
            safely_calculate_average_precision(
                targets,
                positive_class_probabilities,
            ),

        "accuracy":
            float(
                accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "precision":
            float(
                precision_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "f1":
            float(
                f1_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "brier_score":
            float(
                brier_score_loss(
                    targets,
                    positive_class_probabilities,
                )
            ),

        "negative_log_likelihood":
            float(
                log_loss(
                    targets,
                    np.column_stack(
                        [
                            1.0
                            - clipped_probabilities,

                            clipped_probabilities,
                        ]
                    ),
                    labels=[0, 1],
                )
            ),

        "expected_calibration_error":
            calculate_binary_expected_calibration_error(
                targets=targets,

                positive_class_probabilities=
                    positive_class_probabilities,

                number_of_bins=10,
            ),

        "classification_threshold":
            float(
                classification_threshold
            ),

        "true_negative":
            int(
                true_negative
            ),

        "false_positive":
            int(
                false_positive
            ),

        "false_negative":
            int(
                false_negative
            ),

        "true_positive":
            int(
                true_positive
            ),
    }


    if uncertainties is not None:

        uncertainties = np.asarray(
            uncertainties,
            dtype=np.float64,
        )

        metrics[
            "mean_uncertainty"
        ] = float(
            uncertainties.mean()
        )

        metrics[
            "std_uncertainty"
        ] = float(
            uncertainties.std()
        )


    if three_mt_weights is not None:

        three_mt_weights = np.asarray(
            three_mt_weights,
            dtype=np.float64,
        )

        metrics[
            "mean_three_mt_weight"
        ] = float(
            three_mt_weights.mean()
        )

        metrics[
            "std_three_mt_weight"
        ] = float(
            three_mt_weights.std()
        )

        metrics[
            "minimum_three_mt_weight"
        ] = float(
            three_mt_weights.min()
        )

        metrics[
            "maximum_three_mt_weight"
        ] = float(
            three_mt_weights.max()
        )


    return metrics

# ------------------------------------------------------------
# Exact fold-0 parameter and target summaries
# ------------------------------------------------------------

model_parameter_count = sum(
    parameter.numel()
    for parameter in complete_model.parameters()
    if parameter.requires_grad
)

optimised_parameter_count = sum(
    parameter.numel()
    for parameter_group in optimizer.param_groups
    for parameter in parameter_group["params"]
    if parameter.requires_grad
)

training_target_counts = (
    train_loader
    .dataset
    .dataframe[target_column]
    .value_counts()
    .sort_index()
)

training_target_summary = pd.DataFrame(
    {
        "CLASS_INDEX": [0, 1],
        "CLASS_NAME": ["sMCI", "pMCI"],
        "TRAINING_COUNT": [
            int(training_target_counts.loc[0]),
            int(training_target_counts.loc[1]),
        ],
    }
)

training_target_summary[
    "TRAINING_PROPORTION"
] = (
    training_target_summary[
        "TRAINING_COUNT"
    ]
    /
    training_target_summary[
        "TRAINING_COUNT"
    ].sum()
)


print("\nOptimiser: AdamW")
print(
    "Initial learning rate: "
    f"{INITIAL_LEARNING_RATE:.6f}"
)
print(
    "Weight decay: "
    f"{WEIGHT_DECAY:.6f}"
)
print(
    "Maximum epochs: "
    f"{MAXIMUM_EPOCHS}"
)
print(
    "Early-stopping patience: "
    f"{EARLY_STOPPING_PATIENCE} epochs"
)
print(
    "Scheduler patience: "
    f"{SCHEDULER_PATIENCE} epochs"
)
print(
    "Checkpoint-selection metric: validation ROC AUC"
)

print(
    "\nExperiment output directory:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "\nBest-checkpoint path:\n"
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    "\nTrainable model parameters: "
    f"{model_parameter_count:,}"
)

print(
    "Parameters included in optimiser: "
    f"{optimised_parameter_count:,}"
)

print(
    "\nFold-0 training target balance:"
)

display(
    training_target_summary.round(6)
)


### 1.7.33. Defining reusable training and validation epoch functions

define the reusable functions that will execute one complete training epoch and one complete validation epoch.

A single training epoch performs the following operations for every mini-batch:

1. move the nested multimodal batch to the selected device;
2. run the complete interaction pathway--evidence pathway forward pass in training mode;
3. apply training-time modality dropout;
4. calculate the joint objective;
5. backpropagate the total loss;
6. clip the global gradient norm;
7. update all model parameters using AdamW;
8. accumulate predictions, uncertainty estimates, gate weights, and losses.

The validation epoch uses the same complete model but differs in three important ways:

- the model is placed in evaluation mode;
- modality dropout and ordinary neural-network dropout are disabled;
- no gradients or parameter updates are calculated.

For both training and validation, pMCI is treated as the positive class. The epoch functions collect:

- participant identifiers;
- binary targets;
- final pMCI probabilities;
- final uncertainty values;
- interaction pathway and modality-specific evidence pathway weights;
- interaction pathway-only pMCI probabilities;
- evidence pathway-only pMCI probabilities;
- the original and effective numbers of available modalities.

The accumulated participant-level outputs are passed to the previously defined metric function after the entire epoch has completed.

### 1.7.34. Gradient clipping

For every training batch, the total gradient norm is calculated and clipped before the optimiser step:

$$
\left\|
\nabla_{\boldsymbol{\theta}}
\mathcal{L}_{\mathrm{total}}
\right\|_2
\leq 5.
$$

The unclipped norm is retained for monitoring.

### 1.7.35. Epoch-level loss aggregation

For loss component \(\ell\), the epoch-level mean is calculated using the number of participants in each mini-batch:

$$
\overline{\mathcal{L}}_{\ell}
=
\frac{
\sum_{b=1}^{B}
n_b
\mathcal{L}_{\ell,b}
}{
\sum_{b=1}^{B}
n_b
},
$$

where \(n_b\) is the batch size.

This avoids giving the final incomplete mini-batch the same weight as a full mini-batch.

The functions defined in this step do not yet train the model across multiple epochs. The full checkpointed training loop is constructed in the following step.

In [ ]:
# ============================================================
# 18. Defining reusable training and validation epoch functions
# ============================================================

from collections import defaultdict


# ------------------------------------------------------------
# Initial numerical-precision policy
# ------------------------------------------------------------

# I initially train in full float32 precision.
#
# This is computationally feasible on the available A100 GPU and
# avoids introducing mixed-precision instability into the Dirichlet
# digamma, log-gamma, and Dempster-Shafer calculations.
USE_MIXED_PRECISION = False


# ------------------------------------------------------------
# Recursively move a nested batch to the selected device
# ------------------------------------------------------------

def move_nested_batch_to_device(
    value,
    device,
):
    """
    Recursively move tensors in dictionaries, lists, and tuples
    to the selected PyTorch device.

    Non-tensor values are preserved unchanged.
    """

    if isinstance(
        value,
        torch.Tensor,
    ):
        return value.to(
            device,
            non_blocking=True,
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: move_nested_batch_to_device(
                nested_value,
                device,
            )
            for key, nested_value in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        )

    return value


# ------------------------------------------------------------
# Convert one completed forward pass into stored predictions
# ------------------------------------------------------------

def extract_batch_prediction_arrays(
    batch,
    model_output,
):
    """
    Extract participant-level targets, predictions, uncertainty,
    pathway outputs, and modality counts from one mini-batch.
    """

    final_output = model_output[
        "final_output"
    ]

    joint_opinion = model_output[
        "joint_opinion"
    ]

    tmc_output = model_output[
        "tmc_output"
    ]


    extracted = {
        "rid":
            batch["rid"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "target":
            batch["target"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_p_pMCI":
            final_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_uncertainty":
            final_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_weight":
            final_output[
                "three_mt_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_weight":
            final_output[
                "tmc_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_p_pMCI":
            joint_opinion[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_uncertainty":
            joint_opinion[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_p_pMCI":
            tmc_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_uncertainty":
            tmc_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "original_modality_count":
            model_output[
                "original_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "effective_modality_count":
            model_output[
                "effective_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),
    }

    return extracted


# ------------------------------------------------------------
# Concatenate the participant outputs collected across an epoch
# ------------------------------------------------------------

def concatenate_epoch_prediction_storage(
    prediction_storage,
):
    """
    Concatenate a dictionary of mini-batch NumPy arrays.
    """

    concatenated = {}

    for key, value_list in prediction_storage.items():

        if len(value_list) == 0:
            concatenated[key] = np.asarray([])

        else:
            concatenated[key] = np.concatenate(
                value_list,
                axis=0,
            )

    return concatenated


# ------------------------------------------------------------
# Convert stored predictions into a participant-level table
# ------------------------------------------------------------

def build_epoch_prediction_table(
    concatenated_predictions,
    split_name,
    epoch,
):
    """
    Build one participant-level DataFrame for an epoch.
    """

    prediction_table = pd.DataFrame(
        {
            "RID":
                concatenated_predictions[
                    "rid"
                ].astype(
                    np.int64
                ),

            "TARGET":
                concatenated_predictions[
                    "target"
                ].astype(
                    np.int64
                ),

            "FINAL_P_pMCI":
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            "FINAL_UNCERTAINTY":
                concatenated_predictions[
                    "final_uncertainty"
                ],

            "W_3MT":
                concatenated_predictions[
                    "three_mt_weight"
                ],

            "W_TMC":
                concatenated_predictions[
                    "tmc_weight"
                ],

            "THREE_MT_P_pMCI":
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            "THREE_MT_UNCERTAINTY":
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            "TMC_P_pMCI":
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            "TMC_UNCERTAINTY":
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            "ORIGINAL_MODALITY_COUNT":
                concatenated_predictions[
                    "original_modality_count"
                ].astype(
                    np.int64
                ),

            "EFFECTIVE_MODALITY_COUNT":
                concatenated_predictions[
                    "effective_modality_count"
                ].astype(
                    np.int64
                ),
        }
    )

    prediction_table.insert(
        loc=0,
        column="EPOCH",
        value=int(epoch),
    )

    prediction_table.insert(
        loc=1,
        column="SPLIT",
        value=str(split_name),
    )

    prediction_table[
        "FINAL_PREDICTED_CLASS"
    ] = (
        prediction_table[
            "FINAL_P_pMCI"
        ]
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        np.int64
    )

    return prediction_table


# ------------------------------------------------------------
# Calculate metrics for all three prediction outputs
# ------------------------------------------------------------

def calculate_epoch_prediction_metrics(
    concatenated_predictions,
):
    """
    Calculate metrics for:

    1. the final fixed-equal hybrid output;
    2. the 3MT-only joint output;
    3. the TMC-only fused output.
    """

    targets = concatenated_predictions[
        "target"
    ]

    final_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "final_uncertainty"
                ],

            three_mt_weights=
                concatenated_predictions[
                    "three_mt_weight"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    three_mt_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    tmc_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    return {
        "final": final_metrics,
        "three_mt": three_mt_metrics,
        "tmc": tmc_metrics,
    }


# ------------------------------------------------------------
# Initialise the loss accumulator used within an epoch
# ------------------------------------------------------------

EPOCH_LOSS_NAMES = [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]


def initialise_epoch_loss_storage():
    """
    Create participant-weighted loss totals.
    """

    return {
        loss_name: 0.0
        for loss_name in EPOCH_LOSS_NAMES
    }


def update_epoch_loss_storage(
    storage,
    loss_output,
    batch_size,
):
    """
    Add one batch's losses, weighted by the number of participants.
    """

    for loss_name in EPOCH_LOSS_NAMES:

        storage[loss_name] += (
            float(
                loss_output[
                    loss_name
                ]
                .detach()
                .float()
                .item()
            )
            * batch_size
        )


def finalise_epoch_loss_storage(
    storage,
    participant_count,
):
    """
    Convert accumulated loss sums into participant-weighted means.
    """

    if participant_count <= 0:
        raise ValueError(
            "The epoch contained no participants."
        )

    return {
        loss_name:
            loss_sum
            / participant_count

        for loss_name, loss_sum in storage.items()
    }


# ------------------------------------------------------------
# Run one complete training epoch
# ------------------------------------------------------------

def run_training_epoch(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_gradient_norm,
):
    """
    Train the complete model for one epoch.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []


    for batch_index, batch in enumerate(
        data_loader
    ):

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = targets.shape[0]

        participant_count += batch_size


        # --------------------------------------------------------
        # Clear gradients from the previous mini-batch
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # --------------------------------------------------------
        # Complete forward pass
        # --------------------------------------------------------

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )


        # --------------------------------------------------------
        # Complete multi-output objective
        # --------------------------------------------------------

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )


        # --------------------------------------------------------
        # Backpropagation
        # --------------------------------------------------------

        total_loss.backward()


        # --------------------------------------------------------
        # Global gradient clipping
        # --------------------------------------------------------

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )


        # --------------------------------------------------------
        # Parameter update
        # --------------------------------------------------------

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate losses and predictions
        # --------------------------------------------------------

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[key].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


    # ------------------------------------------------------------
    # Finalise the complete training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# Run one complete validation epoch
# ------------------------------------------------------------

def run_validation_epoch(
    model,
    data_loader,
    objective,
    device,
    epoch,
):
    """
    Evaluate the complete model for one epoch without gradients
    or training-time modality dropout.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    with torch.no_grad():

        for batch_index, batch in enumerate(
            data_loader
        ):

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = targets.shape[0]

            participant_count += batch_size


            # ----------------------------------------------------
            # Complete evaluation forward pass
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ----------------------------------------------------
            # Validation objective
            # ----------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_index}."
                )


            # ----------------------------------------------------
            # Accumulate losses and predictions
            # ----------------------------------------------------

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[key].append(
                    values
                )


    # ------------------------------------------------------------
    # Finalise the complete validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Display the configured epoch-function summary
# ------------------------------------------------------------

print("=" * 72)
print("TRAINING AND VALIDATION EPOCH FUNCTIONS")
print("=" * 72)

print(
    f"\nMixed precision enabled: "
    f"{USE_MIXED_PRECISION}"
)

print(
    "Training batches per epoch: "
    f"{len(train_loader)}"
)

print(
    "Validation batches per epoch: "
    f"{len(validation_loader)}"
)

print(
    "Training participants: "
    f"{len(train_loader.dataset)}"
)

print(
    "Validation participants: "
    f"{len(validation_loader.dataset)}"
)

print(
    "\nTraining epoch operations:"
)

print(
    "- forward pass with modality dropout;"
)

print(
    "- complete joint loss calculation;"
)

print(
    "- backpropagation;"
)

print(
    "- global gradient clipping;"
)

print(
    "- AdamW parameter update;"
)

print(
    "- prediction and uncertainty accumulation."
)

print(
    "\nValidation epoch operations:"
)

print(
    "- evaluation mode;"
)

print(
    "- no modality dropout;"
)

print(
    "- no gradient calculation;"
)

print(
    "- complete validation loss and metric calculation."
)

print(
    "\nThe epoch functions are defined."
)

print(
    "No complete training or validation epoch has been run yet."
)

print(
    "The next step will create the checkpointed multi-epoch "
    "training loop and begin model optimisation."
)

### 1.7.36. Training a newly initialised model with validation-based checkpointing

This standalone experiment starts from epoch 1 using the model initialised in this notebook. It does not load a checkpoint from the previous ungated baseline or from an earlier gated run.

The validation partition controls learning-rate reduction, early stopping, and best-checkpoint selection. The test partition remains untouched during training.

In [ ]:
# ============================================================
# 19. Training with live batch and epoch progress
# ============================================================

import time
import traceback
from collections import defaultdict
from datetime import datetime

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Resume and progress-display behaviour
# ------------------------------------------------------------

# The formal first run begins from epoch 1.
# Resume is enabled only when FOLD_RUN_MODE was explicitly set
# to "resume" for this exact experiment and fold.
RESUME_FROM_LAST_CHECKPOINT = (
    FOLD_RUN_MODE == "resume"
)

# I refresh the live progress display after every batch.
PROGRESS_UPDATE_INTERVAL = 1


# ------------------------------------------------------------
# Loss-configuration serialisation
# ------------------------------------------------------------

def obtain_training_objective_configuration(
    objective,
):
    """
    Return the principal loss settings in a checkpoint-safe form.
    """

    return {
        "evidential_annealing_epochs": int(
            objective.evidential_annealing_epochs
        ),

        "gate_regularisation_epochs": int(
            objective.gate_regularisation_epochs
        ),

        "joint_loss_weight": float(
            objective.joint_loss_weight
        ),

        "tmc_loss_weight": float(
            objective.tmc_loss_weight
        ),

        "modality_loss_weight": float(
            objective.modality_loss_weight
        ),

        "auxiliary_loss_weight": float(
            objective.auxiliary_loss_weight
        ),

        "gate_loss_weight": float(
            objective.gate_loss_weight
        ),

        "class_weights": (
            None
            if objective.class_weights is None
            else
            objective.class_weights
            .detach()
            .cpu()
            .tolist()
        ),
    }


# ------------------------------------------------------------
# Flatten one epoch into one history row
# ------------------------------------------------------------

def build_training_history_row(
    epoch,
    learning_rate,
    epoch_duration_seconds,
    training_result,
    validation_result,
    best_validation_auc,
    epochs_without_improvement,
    checkpoint_improved,
):
    """
    Create one flat row containing losses, metrics, optimisation
    diagnostics, and checkpoint information.
    """

    history_row = {
        "epoch":
            int(epoch),

        "learning_rate":
            float(learning_rate),

        "epoch_duration_seconds":
            float(epoch_duration_seconds),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "checkpoint_improved":
            bool(checkpoint_improved),

        "train_participants":
            int(
                training_result[
                    "participant_count"
                ]
            ),

        "validation_participants":
            int(
                validation_result[
                    "participant_count"
                ]
            ),

        "train_batches":
            int(
                training_result[
                    "batch_count"
                ]
            ),

        "validation_batches":
            int(
                validation_result[
                    "batch_count"
                ]
            ),

        "train_mean_gradient_norm":
            float(
                training_result[
                    "gradient_summary"
                ]["mean_gradient_norm"]
            ),

        "train_maximum_gradient_norm_before_clipping":
            float(
                training_result[
                    "gradient_summary"
                ][
                    "maximum_gradient_norm_before_clipping"
                ]
            ),

        "train_mean_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "mean_effective_modality_count"
                ]
            ),

        "train_minimum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "minimum_effective_modality_count"
                ]
            ),

        "train_maximum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "maximum_effective_modality_count"
                ]
            ),

        "validation_maximum_modality_count_difference":
            float(
                validation_result[
                    "maximum_evaluation_modality_count_difference"
                ]
            ),
    }


    # --------------------------------------------------------
    # Add all training and validation losses
    # --------------------------------------------------------

    for loss_name, loss_value in (
        training_result[
            "losses"
        ].items()
    ):
        history_row[
            f"train_{loss_name}"
        ] = float(
            loss_value
        )

    for loss_name, loss_value in (
        validation_result[
            "losses"
        ].items()
    ):
        history_row[
            f"validation_{loss_name}"
        ] = float(
            loss_value
        )


    # --------------------------------------------------------
    # Add metrics for all three prediction outputs
    # --------------------------------------------------------

    for output_name in [
        "final",
        "three_mt",
        "tmc",
    ]:

        for metric_name, metric_value in (
            training_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"train_{output_name}_{metric_name}"
            ] = metric_value

        for metric_name, metric_value in (
            validation_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"validation_{output_name}_{metric_name}"
            ] = metric_value


    return history_row


# ------------------------------------------------------------
# Save a fully recoverable checkpoint
# ------------------------------------------------------------

def save_training_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_validation_auc,
    epochs_without_improvement,
    validation_metrics,
    training_history,
):
    """
    Save the state required to reproduce or continue training.
    """

    checkpoint = {
        "epoch":
            int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "validation_metrics":
            validation_metrics,

        "training_configuration":
            training_configuration,

        "training_objective_configuration":
            obtain_training_objective_configuration(
                training_objective
            ),

        "training_history":
            training_history,

        "random_seed":
            int(GLOBAL_RANDOM_SEED),

        "task":
            SELECTED_TASK,

        "fold":
            int(SELECTED_FOLD),

        "saved_at":
            datetime.now().isoformat(
                timespec="seconds"
            ),
    }

    torch.save(
        checkpoint,
        checkpoint_path,
    )


# ------------------------------------------------------------
# GPU-memory helper for the progress display
# ------------------------------------------------------------

def current_cuda_memory_gb():
    """
    Return currently allocated CUDA memory in gigabytes.
    """

    if not torch.cuda.is_available():
        return 0.0

    return float(
        torch.cuda.memory_allocated(
            DEVICE
        )
        / (1024 ** 3)
    )


# ------------------------------------------------------------
# One training epoch with a live batch progress bar
# ------------------------------------------------------------

def run_training_epoch_with_progress(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_epochs,
    maximum_gradient_norm,
):
    """
    Train for one epoch while displaying batch-level progress.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | training"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    for batch_number, batch in progress_bar:

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = int(
            targets.shape[0]
        )

        participant_count += (
            batch_size
        )


        # --------------------------------------------------------
        # Forward pass and loss
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )


        # --------------------------------------------------------
        # Backpropagation, clipping, and update
        # --------------------------------------------------------

        total_loss.backward()

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate diagnostics
        # --------------------------------------------------------

        current_total_loss = float(
            total_loss.detach().item()
        )

        running_total_loss_sum += (
            current_total_loss
            * batch_size
        )

        running_mean_total_loss = (
            running_total_loss_sum
            / participant_count
        )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[
                key
            ].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


        # --------------------------------------------------------
        # Update the visible progress information
        # --------------------------------------------------------

        if (
            batch_number
            % PROGRESS_UPDATE_INTERVAL
            == 0
            or batch_number
            == len(data_loader)
        ):

            elapsed_minutes = (
                time.time()
                - phase_start_time
            ) / 60.0

            progress_bar.set_postfix(
                {
                    "loss":
                        f"{current_total_loss:.3f}",

                    "avg":
                        f"{running_mean_total_loss:.3f}",

                    "grad":
                        f"{float(gradient_norm):.2f}",

                    "lr":
                        f"{optimizer.param_groups[0]['lr']:.1e}",

                    "GPU":
                        f"{current_cuda_memory_gb():.1f}GB",

                    "elapsed":
                        f"{elapsed_minutes:.1f}m",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise the training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# One validation epoch with a live batch progress bar
# ------------------------------------------------------------

def run_validation_epoch_with_progress(
    model,
    data_loader,
    objective,
    device,
    epoch,
    maximum_epochs,
):
    """
    Validate for one epoch while displaying batch-level progress.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | validation"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ----------------------------------------------------
            # Forward pass and validation loss
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            total_loss = loss_output[
                "total_loss"
            ]


            if not torch.isfinite(
                total_loss
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_number}."
                )


            # ----------------------------------------------------
            # Accumulate diagnostics and predictions
            # ----------------------------------------------------

            current_total_loss = float(
                total_loss.detach().item()
            )

            running_total_loss_sum += (
                current_total_loss
                * batch_size
            )

            running_mean_total_loss = (
                running_total_loss_sum
                / participant_count
            )

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[
                    key
                ].append(
                    values
                )


            # ----------------------------------------------------
            # Update the visible progress information
            # ----------------------------------------------------

            if (
                batch_number
                % PROGRESS_UPDATE_INTERVAL
                == 0
                or batch_number
                == len(data_loader)
            ):

                elapsed_minutes = (
                    time.time()
                    - phase_start_time
                ) / 60.0

                progress_bar.set_postfix(
                    {
                        "loss":
                            f"{current_total_loss:.3f}",

                        "avg":
                            f"{running_mean_total_loss:.3f}",

                        "GPU":
                            f"{current_cuda_memory_gb():.1f}GB",

                        "elapsed":
                            f"{elapsed_minutes:.1f}m",
                    },
                    refresh=True,
                )


    # ------------------------------------------------------------
    # Finalise the validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                concatenated_predictions[
                    "original_modality_count"
                ]
                -
                concatenated_predictions[
                    "effective_modality_count"
                ]
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Initial training state
# ------------------------------------------------------------

training_history = []

starting_epoch = 1

best_validation_auc = float(
    "-inf"
)

epochs_without_improvement = 0


# ------------------------------------------------------------
# Resume from the latest fully completed epoch
# ------------------------------------------------------------

if (
    RESUME_FROM_LAST_CHECKPOINT
    and LAST_CHECKPOINT_PATH.exists()
):

    print(
        "Loading the latest fully completed checkpoint:",
        flush=True,
    )

    print(
        LAST_CHECKPOINT_PATH,
        flush=True,
    )

    resumed_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    complete_model.load_state_dict(
        resumed_checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        resumed_checkpoint[
            "optimizer_state_dict"
        ]
    )

    learning_rate_scheduler.load_state_dict(
        resumed_checkpoint[
            "scheduler_state_dict"
        ]
    )

    completed_epoch = int(
        resumed_checkpoint[
            "epoch"
        ]
    )

    starting_epoch = (
        completed_epoch
        + 1
    )

    best_validation_auc = float(
        resumed_checkpoint[
            "best_validation_auc"
        ]
    )

    epochs_without_improvement = int(
        resumed_checkpoint[
            "epochs_without_improvement"
        ]
    )

    training_history = list(
        resumed_checkpoint.get(
            "training_history",
            [],
        )
    )

    print(
        f"\nResuming after epoch {completed_epoch}.",
        flush=True,
    )

    print(
        f"Next epoch: {starting_epoch}",
        flush=True,
    )

    print(
        "Best validation ROC AUC so far: "
        f"{best_validation_auc:.6f}",
        flush=True,
    )

else:

    print(
        "No fully completed checkpoint was loaded.",
        flush=True,
    )

    print(
        "Fresh training begins from the newly initialised model state.",
        flush=True,
    )


# ------------------------------------------------------------
# Main multi-epoch training loop
# ------------------------------------------------------------

if starting_epoch > MAXIMUM_EPOCHS:

    print(
        "\nTraining has already reached the configured maximum "
        f"of {MAXIMUM_EPOCHS} epochs.",
        flush=True,
    )

else:

    print("\n" + "=" * 72, flush=True)
    print("BEGINNING MODEL TRAINING", flush=True)
    print("=" * 72, flush=True)

    print(
        f"\nEpoch range: {starting_epoch}--{MAXIMUM_EPOCHS}",
        flush=True,
    )

    print(
        f"Training batches per epoch: {len(train_loader)}",
        flush=True,
    )

    print(
        f"Validation batches per epoch: {len(validation_loader)}",
        flush=True,
    )

    print(
        "Each epoch displays separate live training and "
        "validation progress bars.",
        flush=True,
    )

    print(
        "The test partition will not be evaluated.",
        flush=True,
    )


    try:

        for epoch in range(
            starting_epoch,
            MAXIMUM_EPOCHS + 1,
        ):

            epoch_start_time = time.time()

            current_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )


            print("\n" + "=" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d}/{MAXIMUM_EPOCHS}",
                flush=True,
            )

            print("=" * 72, flush=True)

            print(
                "\nPhase 1/4: training batches",
                flush=True,
            )


            # ------------------------------------------------
            # Train on all training participants
            # ------------------------------------------------

            training_result = (
                run_training_epoch_with_progress(
                    model=complete_model,
                    data_loader=train_loader,
                    objective=training_objective,
                    optimizer=optimizer,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                    maximum_gradient_norm=
                        MAXIMUM_GRADIENT_NORM,
                )
            )


            print(
                "\nPhase 2/4: validation batches",
                flush=True,
            )


            # ------------------------------------------------
            # Validate on all validation participants
            # ------------------------------------------------

            validation_result = (
                run_validation_epoch_with_progress(
                    model=complete_model,
                    data_loader=validation_loader,
                    objective=training_objective,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                )
            )


            print(
                "\nPhase 3/4: calculating metrics and "
                "updating the scheduler",
                flush=True,
            )


            # ------------------------------------------------
            # Validation AUC and checkpoint decision
            # ------------------------------------------------

            validation_auc = float(
                validation_result[
                    "metrics"
                ]["final"]["roc_auc"]
            )

            validation_auc_is_valid = bool(
                np.isfinite(
                    validation_auc
                )
            )

            checkpoint_improved = (
                validation_auc_is_valid
                and
                validation_auc
                > best_validation_auc
            )

            if checkpoint_improved:

                best_validation_auc = (
                    validation_auc
                )

                epochs_without_improvement = 0

            else:

                epochs_without_improvement += 1


            scheduler_score = (
                validation_auc
                if validation_auc_is_valid
                else -1.0
            )

            learning_rate_scheduler.step(
                scheduler_score
            )

            updated_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            epoch_duration_seconds = (
                time.time()
                - epoch_start_time
            )


            # ------------------------------------------------
            # Persistent training history
            # ------------------------------------------------

            history_row = build_training_history_row(
                epoch=epoch,

                learning_rate=
                    current_learning_rate,

                epoch_duration_seconds=
                    epoch_duration_seconds,

                training_result=
                    training_result,

                validation_result=
                    validation_result,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                checkpoint_improved=
                    checkpoint_improved,
            )

            history_row[
                "learning_rate_after_scheduler"
            ] = updated_learning_rate

            training_history.append(
                history_row
            )

            pd.DataFrame(
                training_history
            ).to_csv(
                TRAINING_HISTORY_PATH,
                index=False,
            )


            print(
                "\nPhase 4/4: saving checkpoints and history",
                flush=True,
            )


            # ------------------------------------------------
            # Save the latest completed epoch
            # ------------------------------------------------

            save_training_checkpoint(
                checkpoint_path=
                    LAST_CHECKPOINT_PATH,

                epoch=epoch,

                model=complete_model,

                optimizer=optimizer,

                scheduler=
                    learning_rate_scheduler,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                validation_metrics=
                    validation_result[
                        "metrics"
                    ],

                training_history=
                    training_history,
            )


            # ------------------------------------------------
            # Save the best validation checkpoint
            # ------------------------------------------------

            if checkpoint_improved:

                save_training_checkpoint(
                    checkpoint_path=
                        BEST_CHECKPOINT_PATH,

                    epoch=epoch,

                    model=complete_model,

                    optimizer=optimizer,

                    scheduler=
                        learning_rate_scheduler,

                    best_validation_auc=
                        best_validation_auc,

                    epochs_without_improvement=
                        epochs_without_improvement,

                    validation_metrics=
                        validation_result[
                            "metrics"
                        ],

                    training_history=
                        training_history,
                )

                validation_result[
                    "predictions"
                ].to_csv(
                    VALIDATION_PREDICTIONS_PATH,
                    index=False,
                )


            # ------------------------------------------------
            # Readable completed-epoch summary
            # ------------------------------------------------

            train_metrics = training_result[
                "metrics"
            ]["final"]

            validation_metrics = validation_result[
                "metrics"
            ]["final"]


            print("\n" + "-" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d} COMPLETE",
                flush=True,
            )

            print(
                "Duration: "
                f"{epoch_duration_seconds / 60.0:.2f} minutes",
                flush=True,
            )

            print(
                "Learning rate: "
                f"{current_learning_rate:.8f}"
                f" -> {updated_learning_rate:.8f}",
                flush=True,
            )


            print("\nTraining:", flush=True)

            print(
                "  total loss: "
                f"{training_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{train_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{train_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{train_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{train_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )

            print(
                "  mean effective modalities: "
                f"{training_result['modality_dropout_summary']['mean_effective_modality_count']:.3f}",
                flush=True,
            )

            print(
                "  maximum pre-clipping gradient norm: "
                f"{training_result['gradient_summary']['maximum_gradient_norm_before_clipping']:.6f}",
                flush=True,
            )


            print("\nValidation:", flush=True)

            print(
                "  total loss: "
                f"{validation_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{validation_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  average precision: "
                f"{validation_metrics['average_precision']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{validation_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  sensitivity: "
                f"{validation_metrics['sensitivity']:.6f}",
                flush=True,
            )

            print(
                "  specificity: "
                f"{validation_metrics['specificity']:.6f}",
                flush=True,
            )

            print(
                "  Brier score: "
                f"{validation_metrics['brier_score']:.6f}",
                flush=True,
            )

            print(
                "  calibration error: "
                f"{validation_metrics['expected_calibration_error']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{validation_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{validation_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )


            print("\nCheckpoint status:", flush=True)

            print(
                "  improved this epoch: "
                f"{checkpoint_improved}",
                flush=True,
            )

            print(
                "  best validation ROC AUC: "
                f"{best_validation_auc:.6f}",
                flush=True,
            )

            print(
                "  epochs without improvement: "
                f"{epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}",
                flush=True,
            )

            print(
                "  latest completed epoch saved: True",
                flush=True,
            )

            if torch.cuda.is_available():

                print(
                    "  peak CUDA memory: "
                    f"{torch.cuda.max_memory_allocated(0) / (1024 ** 3):.3f} GB",
                    flush=True,
                )


            # ------------------------------------------------
            # Early stopping
            # ------------------------------------------------

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):

                print(
                    "\nEarly stopping activated because "
                    "validation ROC AUC did not improve for "
                    f"{EARLY_STOPPING_PATIENCE} consecutive epochs.",
                    flush=True,
                )

                break


    # --------------------------------------------------------
    # Interruption and error handling
    # --------------------------------------------------------

    except KeyboardInterrupt:

        print(
            "\nTraining was interrupted manually.",
            flush=True,
        )

        print(
            "Only fully completed epochs are recoverable from "
            "the last-epoch checkpoint.",
            flush=True,
        )


    except Exception:

        print(
            "\nTraining stopped because an exception occurred.",
            flush=True,
        )

        print(
            "The latest fully completed epoch remains saved.",
            flush=True,
        )

        traceback.print_exc()

        raise


    # --------------------------------------------------------
    # Final training summary
    # --------------------------------------------------------

    if len(
        training_history
    ) > 0:

        final_history_table = pd.DataFrame(
            training_history
        )

        completed_epochs = int(
            final_history_table[
                "epoch"
            ].max()
        )

        print("\n" + "=" * 72, flush=True)

        print(
            "TRAINING RUN COMPLETE",
            flush=True,
        )

        print("=" * 72, flush=True)

        print(
            f"\nLast completed epoch: {completed_epochs}",
            flush=True,
        )

        print(
            "Best validation ROC AUC: "
            f"{best_validation_auc:.6f}",
            flush=True,
        )

        print(
            "\nBest checkpoint:\n"
            f"{BEST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nLatest checkpoint:\n"
            f"{LAST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nTraining history:\n"
            f"{TRAINING_HISTORY_PATH}",
            flush=True,
        )

        print(
            "\nThe test partition has not been evaluated.",
            flush=True,
        )

### 1.7.37. Evaluating the best gated checkpoint on the untouched test set

After training and validation-based checkpoint selection, the best checkpoint from this named experiment is loaded from its experiment-specific fold directory and evaluated once on the held-out test partition.

The evaluation cell does not update model parameters, the optimiser, the scheduler, or modality-dropout state.

In [ ]:
from datetime import datetime


# ------------------------------------------------------------
# Test-output paths
# ------------------------------------------------------------

TEST_METRICS_PATH = (
    PREDICTION_DIR
    / "test_metrics.json"
)

TEST_SUMMARY_PATH = (
    PREDICTION_DIR
    / "test_metrics_summary.csv"
)


print("=" * 72)
print("FIXED-EQUAL-FUSION TEST-SET EVALUATION")
print("=" * 72)

print(
    "\nLoading the best validation checkpoint:\n"
    f"{BEST_CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# Load the best validation checkpoint
# ------------------------------------------------------------


best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_checkpoint_epoch = int(
    best_checkpoint[
        "epoch"
    ]
)

best_checkpoint_validation_auc = float(
    best_checkpoint[
        "best_validation_auc"
    ]
)


complete_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


print(
    f"\nBest checkpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Best validation ROC AUC stored in checkpoint: "
    f"{best_checkpoint_validation_auc:.6f}"
)


# ------------------------------------------------------------
# Test evaluation function
# ------------------------------------------------------------

def run_test_epoch(
    model,
    data_loader,
    objective,
    device,
    checkpoint_epoch,
):
    """
    Evaluate the frozen model on the untouched test partition.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc="Test evaluation",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ------------------------------------------------
            # Frozen forward pass
            # ------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ------------------------------------------------
            # Test loss
            # ------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=checkpoint_epoch,
            )


            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):

                raise FloatingPointError(
                    "A non-finite test loss was encountered "
                    f"at batch {batch_number}."
                )


            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )


            # ------------------------------------------------
            # Store participant-level outputs
            # ------------------------------------------------

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):

                prediction_storage[
                    key
                ].append(
                    values
                )


            running_average_loss = (
                loss_storage[
                    "total_loss"
                ]
                / participant_count
            )

            progress_bar.set_postfix(
                {
                    "avg_loss":
                        f"{running_average_loss:.3f}",

                    "participants":
                        f"{participant_count}/{len(data_loader.dataset)}",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise test losses and predictions
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="test",

            epoch=checkpoint_epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_modality_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_modality_count_difference,
    }


# ------------------------------------------------------------
# Run the untouched test evaluation
# ------------------------------------------------------------

test_result = run_test_epoch(
    model=complete_model,
    data_loader=test_loader,
    objective=training_objective,
    device=DEVICE,
    checkpoint_epoch=best_checkpoint_epoch,
)


# ------------------------------------------------------------
# Evaluation-mode modality count
# ------------------------------------------------------------

maximum_test_mask_difference = (
    test_result[
        "maximum_evaluation_modality_count_difference"
    ]
)


# ------------------------------------------------------------
# Save participant-level test predictions
# ------------------------------------------------------------


test_prediction_table = test_result[
    "predictions"
]

test_prediction_table.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save all test metrics in JSON format
# ------------------------------------------------------------

serialisable_test_result = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "task":
        SELECTED_TASK,

    "fold":
        int(
            SELECTED_FOLD
        ),

    "checkpoint_epoch":
        int(
            best_checkpoint_epoch
        ),

    "checkpoint_validation_auc":
        float(
            best_checkpoint_validation_auc
        ),

    "classification_threshold":
        float(
            CLASSIFICATION_THRESHOLD
        ),

    "test_participants":
        int(
            test_result[
                "participant_count"
            ]
        ),

    "test_batches":
        int(
            test_result[
                "batch_count"
            ]
        ),

    "test_losses": {
        key:
            float(value)

        for key, value in (
            test_result[
                "losses"
            ].items()
        )
    },

    "test_metrics":
        test_result[
            "metrics"
        ],

    "maximum_modality_count_difference":
        float(
            maximum_test_mask_difference
        ),

    "evaluated_at":
        datetime.now().isoformat(
            timespec="seconds"
        ),
}


with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as test_metrics_file:

    json.dump(
        serialisable_test_result,
        test_metrics_file,
        indent=2,
    )


# ------------------------------------------------------------
# Build a compact comparison table
# ------------------------------------------------------------

test_metric_rows = []

for output_name, display_name in [
    (
        "final",
        "Hybrid",
    ),
    (
        "three_mt",
        "3MT-only",
    ),
    (
        "tmc",
        "TMC-only",
    ),
]:

    output_metrics = test_result[
        "metrics"
    ][
        output_name
    ]

    test_metric_rows.append(
        {
            "OUTPUT":
                display_name,

            "ROC_AUC":
                output_metrics[
                    "roc_auc"
                ],

            "AVERAGE_PRECISION":
                output_metrics[
                    "average_precision"
                ],

            "ACCURACY":
                output_metrics[
                    "accuracy"
                ],

            "BALANCED_ACCURACY":
                output_metrics[
                    "balanced_accuracy"
                ],

            "SENSITIVITY":
                output_metrics[
                    "sensitivity"
                ],

            "SPECIFICITY":
                output_metrics[
                    "specificity"
                ],

            "PRECISION":
                output_metrics[
                    "precision"
                ],

            "F1":
                output_metrics[
                    "f1"
                ],

            "BRIER_SCORE":
                output_metrics[
                    "brier_score"
                ],

            "NEGATIVE_LOG_LIKELIHOOD":
                output_metrics[
                    "negative_log_likelihood"
                ],

            "EXPECTED_CALIBRATION_ERROR":
                output_metrics[
                    "expected_calibration_error"
                ],

            "MEAN_UNCERTAINTY":
                output_metrics[
                    "mean_uncertainty"
                ],
        }
    )


test_metrics_summary = pd.DataFrame(
    test_metric_rows
)

test_metrics_summary.to_csv(
    TEST_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display the final test results
# ------------------------------------------------------------

hybrid_test_metrics = test_result[
    "metrics"
][
    "final"
]


print("\n" + "=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} TEST RESULTS")
print("=" * 72)

print(
    f"\nCheckpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Test participants: "
    f"{test_result['participant_count']}"
)

print(
    "Test batches: "
    f"{test_result['batch_count']}"
)

print(
    "Maximum difference between original and effective "
    "test modality counts: "
    f"{maximum_test_mask_difference:.1f}"
)


print(
    "\nFinal hybrid test performance:"
)

print(
    "  ROC AUC: "
    f"{hybrid_test_metrics['roc_auc']:.6f}"
)

print(
    "  average precision: "
    f"{hybrid_test_metrics['average_precision']:.6f}"
)

print(
    "  accuracy: "
    f"{hybrid_test_metrics['accuracy']:.6f}"
)

print(
    "  balanced accuracy: "
    f"{hybrid_test_metrics['balanced_accuracy']:.6f}"
)

print(
    "  sensitivity: "
    f"{hybrid_test_metrics['sensitivity']:.6f}"
)

print(
    "  specificity: "
    f"{hybrid_test_metrics['specificity']:.6f}"
)

print(
    "  precision: "
    f"{hybrid_test_metrics['precision']:.6f}"
)

print(
    "  F1 score: "
    f"{hybrid_test_metrics['f1']:.6f}"
)

print(
    "  Brier score: "
    f"{hybrid_test_metrics['brier_score']:.6f}"
)

print(
    "  negative log-likelihood: "
    f"{hybrid_test_metrics['negative_log_likelihood']:.6f}"
)

print(
    "  expected calibration error: "
    f"{hybrid_test_metrics['expected_calibration_error']:.6f}"
)

print(
    "  mean uncertainty: "
    f"{hybrid_test_metrics['mean_uncertainty']:.6f}"
)

print(
    "  mean 3MT weight: "
    f"{hybrid_test_metrics['mean_three_mt_weight']:.6f}"
)

print(
    "  standard deviation of 3MT weight: "
    f"{hybrid_test_metrics['std_three_mt_weight']:.6f}"
)


print(
    "\nConfusion matrix counts:"
)

print(
    "  true negatives: "
    f"{hybrid_test_metrics['true_negative']}"
)

print(
    "  false positives: "
    f"{hybrid_test_metrics['false_positive']}"
)

print(
    "  false negatives: "
    f"{hybrid_test_metrics['false_negative']}"
)

print(
    "  true positives: "
    f"{hybrid_test_metrics['true_positive']}"
)


print(
    "\nHybrid, 3MT-only, and TMC-only comparison:"
)

display(
    test_metrics_summary.round(
        6
    )
)


print(
    "\nParticipant-level predictions saved to:\n"
    f"{TEST_PREDICTIONS_PATH}"
)

print(
    "\nComplete test metrics saved to:\n"
    f"{TEST_METRICS_PATH}"
)

print(
    "\nCompact metric summary saved to:\n"
    f"{TEST_SUMMARY_PATH}"
)

print(
    "\nThe model was evaluated without gradient updates, "
    "scheduler changes, or test-time modality dropout."
)

### 1.7.38. Releasing fold-specific memory

The fold-3 checkpoints, histories, validation predictions, test predictions, and metrics are already stored in its own directory. This cleanup removes the in-memory model and DataLoaders before the next fold is created.

In [ ]:
# ============================================================
# 21. Releasing fold-3 memory before the next fold
# ============================================================

import gc

objects_to_release = [
    "complete_model",
    "optimizer",
    "scheduler",
    "training_objective",
    "train_loader",
    "validation_loader",
    "test_loader",
    "train_dataset",
    "validation_dataset",
    "test_dataset",
    "example_batch",
    "diagnostic_batch",
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Fold 3 outputs remain saved under:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "Fold-specific GPU cache has been released."
)


## 1.8. Fold 4

This section trains, validates, and evaluates fold 4. It uses the exact prepared table `mci_prognosis_outer_fold_4_final_task_ready.csv` and writes only to the experiment's `fold_4` directory.


### 1.8.1. Loading the prepared fold-4 input

The model-input columns are defined explicitly in this notebook, matching the original fold-0 implementation. No schema file or output from an earlier experiment is loaded.


In [ ]:
# ============================================================
# Loading the prepared inputs for fold 4
# ============================================================

from pathlib import Path
import json
import pandas as pd

from google.colab import drive


# ------------------------------------------------------------
# Experiment identity
# ------------------------------------------------------------

EXPERIMENT_NAME = "temporal_prebaseline_fixed_equal_fusion_md050"
SELECTED_TASK = "mci_prognosis"
SELECTED_FOLD = 4

# Use "fresh" for the formal first run.
# Change this to "resume" only after an interrupted run of this
# same experiment and fold.
FOLD_RUN_MODE = "fresh"


# ------------------------------------------------------------
# Google Drive
# ------------------------------------------------------------

if not Path(
    "/content/drive/MyDrive"
).exists():
    drive.mount(
        "/content/drive"
    )


# ------------------------------------------------------------
# Exact project and prepared-input paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/adni_mri"
)

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "3mt_tmc_evidential"
)

SELECTED_INPUT_PATH = (
    MODEL_ROOT
    / "temporal_leakage_sensitivity"
    / "prebaseline_only"
    / "final_task_ready_inputs"
    / "mci_prognosis"
    / f"mci_prognosis_outer_fold_{SELECTED_FOLD}_final_task_ready.csv"
)


# ------------------------------------------------------------
# Fixed model-input columns from the prepared pipeline
# ------------------------------------------------------------

final_model_schema = {
    "identifier_columns": [
        "RID",
        "PTID",
    ],

    "audit_label_columns": [
        "CLINICAL_GROUP",
        "MCI_TRAJECTORY_LABEL",
        "MCI_PROGNOSIS_TARGET",
    ],

    "model_target_column":
        "MODEL_TARGET",

    "split_columns": [
        "OUTER_FOLD",
        "DATA_ROLE",
    ],

    "scaled_continuous_columns": [
        "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
        "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        "ADAS__TOTSCORE__Z",
        "ADAS__TOTAL13__Z",
        "MMSE__MMSE_TOTAL_SCORE__Z",
        "MMSE__MMSE_ORIENTATION_SCORE__Z",
        "MMSE__MMSE_REGISTRATION_SCORE__Z",
        "MMSE__MMSE_ATTENTION_SCORE__Z",
        "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
        "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
        "FAQ__FAQTOTAL__Z",
        "CSF__ABETA40__Z",
        "CSF__ABETA42__Z",
        "CSF__TAU__Z",
        "CSF__PTAU__Z",
        "CSF__ABETA42_40_RATIO__Z",
        "PLASMA__pT217_F__Z",
        "PLASMA__AB42_F__Z",
        "PLASMA__AB40_F__Z",
        "PLASMA__AB42_AB40_F__Z",
        "PLASMA__pT217_AB42_F__Z",
        "PLASMA__NfL_Q__Z",
        "PLASMA__GFAP_Q__Z",
        "PLASMA__NfL_F__Z",
        "PLASMA__GFAP_F__Z",
    ],

    "encoded_categorical_columns": [
        "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
        "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        "APOE__APOE4_ALLELE_COUNT__IDX",
    ],

    "mri_path_columns": [
        "MRI__NORMALIZED_T1_NPY_PATH",
    ],

    "branch_mask_columns": [
        "BRANCH_MASK__DEMOGRAPHICS",
        "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
        "BRANCH_MASK__CSF",
        "BRANCH_MASK__PLASMA",
        "BRANCH_MASK__APOE",
        "BRANCH_MASK__MRI",
    ],

    "feature_mask_columns": [
        "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
        "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
        "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        "FEATURE_MASK__ADAS_TOTSCORE",
        "FEATURE_MASK__ADAS_TOTAL13",
        "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
        "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
        "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
        "FEATURE_MASK__FAQ_FAQTOTAL",
        "FEATURE_MASK__CSF_ABETA40",
        "FEATURE_MASK__CSF_ABETA42",
        "FEATURE_MASK__CSF_TAU",
        "FEATURE_MASK__CSF_PTAU",
        "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        "FEATURE_MASK__PLASMA_pT217_F",
        "FEATURE_MASK__PLASMA_AB42_F",
        "FEATURE_MASK__PLASMA_AB40_F",
        "FEATURE_MASK__PLASMA_AB42_AB40_F",
        "FEATURE_MASK__PLASMA_pT217_AB42_F",
        "FEATURE_MASK__PLASMA_NfL_Q",
        "FEATURE_MASK__PLASMA_GFAP_Q",
        "FEATURE_MASK__PLASMA_NfL_F",
        "FEATURE_MASK__PLASMA_GFAP_F",
        "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
    ],
}



# ------------------------------------------------------------
# Load the prepared fold table
# ------------------------------------------------------------

fold_table = pd.read_csv(
    SELECTED_INPUT_PATH
)


print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} INPUT")
print("=" * 72)

print(
    f"\nTask-ready table:\n{SELECTED_INPUT_PATH}"
)

print(
    f"\nLoaded shape: "
    f"{fold_table.shape[0]} rows × "
    f"{fold_table.shape[1]} columns"
)


### 1.8.2. Preparing the fold-4 model-input groups

The continuous, categorical, MRI-path, branch-mask, feature-mask, identifier, target, and split columns are taken from the fixed definitions loaded above.


In [ ]:
# ============================================================
# 2. Reading the prepared schema and describing fold 0
# ============================================================

# ------------------------------------------------------------
# Prepared model-input column groups
# ------------------------------------------------------------

identifier_columns = final_model_schema[
    "identifier_columns"
]

audit_label_columns = final_model_schema[
    "audit_label_columns"
]

target_column = final_model_schema[
    "model_target_column"
]

split_columns = final_model_schema[
    "split_columns"
]

scaled_continuous_columns = final_model_schema[
    "scaled_continuous_columns"
]

encoded_categorical_columns = final_model_schema[
    "encoded_categorical_columns"
]

mri_path_columns = final_model_schema[
    "mri_path_columns"
]

branch_mask_columns = final_model_schema[
    "branch_mask_columns"
]

feature_mask_columns = final_model_schema[
    "feature_mask_columns"
]


# ------------------------------------------------------------
# Fold-0 structure
# ------------------------------------------------------------

print("=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} EXPERIMENT")
print("=" * 72)

print(f"\nExperiment: {EXPERIMENT_NAME}")
print(f"Task: {SELECTED_TASK}")
print(f"Outer fold: {SELECTED_FOLD}")

print("\nPrepared data roles:")
print(
    fold_table["DATA_ROLE"]
    .value_counts()
    .reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared target counts by role:")
display(
    pd.crosstab(
        fold_table["DATA_ROLE"],
        fold_table[target_column],
    ).reindex(
        ["train", "validation", "test"]
    )
)

print("\nPrepared input groups:")
print(
    f"- scaled continuous features: "
    f"{len(scaled_continuous_columns)}"
)
print(
    f"- encoded categorical features: "
    f"{len(encoded_categorical_columns)}"
)
print(
    f"- MRI path columns: "
    f"{len(mri_path_columns)}"
)
print(
    f"- branch masks: "
    f"{len(branch_mask_columns)}"
)
print(
    f"- feature masks: "
    f"{len(feature_mask_columns)}"
)

preview_columns = (
    identifier_columns
    + ["CLINICAL_GROUP"]
    + split_columns
    + [target_column]
    + branch_mask_columns
    + mri_path_columns
)

print("\nExample prepared rows:")
display(
    fold_table[
        preview_columns
    ].head(5)
)


### 1.8.3. Defining the multimodal dataset-output contract

The model will receive each modality as a separate input branch rather than as one combined feature vector.

Continuous and categorical variables are kept separate because they require different encoder operations. Continuous variables will enter small numerical encoders, while categorical variables will later be represented through trainable embeddings.

Each sample will also contain:

- the participant identifier;
- the binary prognosis target;
- branch-level availability masks;
- feature-level observation masks;
- the prepared MRI path.

The dataset will not load MRI arrays yet. At this stage, I define the column organisation and the exact sample structure that the PyTorch dataset will later return.

For participants without MRI, the MRI path remains unavailable and the MRI branch mask remains zero. The participant is retained in the dataset.

In [ ]:
# ============================================================
# 3. Defining the multimodal dataset-output contract
# ============================================================

# ------------------------------------------------------------
# Branch-specific predictor columns
# ------------------------------------------------------------

# I organise the prepared continuous and categorical columns into
# the six modality branches used by the architecture.

dataset_column_contract = {
    "demographics": {
        "continuous": [
            "DEMOGRAPHICS__AGE_AT_BASELINE__Z",
            "DEMOGRAPHICS__PTEDUCAT_CLEAN__Z",
        ],
        "categorical": [
            "DEMOGRAPHICS__PTGENDER_CLEAN__IDX",
            "DEMOGRAPHICS__PTHAND_CLEAN__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__DEMOGRAPHICS_AGE_AT_BASELINE",
            "FEATURE_MASK__DEMOGRAPHICS_PTEDUCAT_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTGENDER_CLEAN",
            "FEATURE_MASK__DEMOGRAPHICS_PTHAND_CLEAN",
        ],
        "branch_mask": "BRANCH_MASK__DEMOGRAPHICS",
    },

    "cognitive_functional": {
        "continuous": [
            "ADAS__TOTSCORE__Z",
            "ADAS__TOTAL13__Z",
            "MMSE__MMSE_TOTAL_SCORE__Z",
            "MMSE__MMSE_ORIENTATION_SCORE__Z",
            "MMSE__MMSE_REGISTRATION_SCORE__Z",
            "MMSE__MMSE_ATTENTION_SCORE__Z",
            "MMSE__MMSE_DELAYED_RECALL_SCORE__Z",
            "MMSE__MMSE_LANGUAGE_COMMAND_SCORE__Z",
            "FAQ__FAQTOTAL__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__ADAS_TOTSCORE",
            "FEATURE_MASK__ADAS_TOTAL13",
            "FEATURE_MASK__MMSE_MMSE_TOTAL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ORIENTATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_REGISTRATION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_ATTENTION_SCORE",
            "FEATURE_MASK__MMSE_MMSE_DELAYED_RECALL_SCORE",
            "FEATURE_MASK__MMSE_MMSE_LANGUAGE_COMMAND_SCORE",
            "FEATURE_MASK__FAQ_FAQTOTAL",
        ],
        "branch_mask": "BRANCH_MASK__COGNITIVE_FUNCTIONAL",
    },

    "csf": {
        "continuous": [
            "CSF__ABETA40__Z",
            "CSF__ABETA42__Z",
            "CSF__TAU__Z",
            "CSF__PTAU__Z",
            "CSF__ABETA42_40_RATIO__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__CSF_ABETA40",
            "FEATURE_MASK__CSF_ABETA42",
            "FEATURE_MASK__CSF_TAU",
            "FEATURE_MASK__CSF_PTAU",
            "FEATURE_MASK__CSF_ABETA42_40_RATIO",
        ],
        "branch_mask": "BRANCH_MASK__CSF",
    },

    "plasma": {
        "continuous": [
            "PLASMA__pT217_F__Z",
            "PLASMA__AB42_F__Z",
            "PLASMA__AB40_F__Z",
            "PLASMA__AB42_AB40_F__Z",
            "PLASMA__pT217_AB42_F__Z",
            "PLASMA__NfL_Q__Z",
            "PLASMA__GFAP_Q__Z",
            "PLASMA__NfL_F__Z",
            "PLASMA__GFAP_F__Z",
        ],
        "categorical": [],
        "feature_masks": [
            "FEATURE_MASK__PLASMA_pT217_F",
            "FEATURE_MASK__PLASMA_AB42_F",
            "FEATURE_MASK__PLASMA_AB40_F",
            "FEATURE_MASK__PLASMA_AB42_AB40_F",
            "FEATURE_MASK__PLASMA_pT217_AB42_F",
            "FEATURE_MASK__PLASMA_NfL_Q",
            "FEATURE_MASK__PLASMA_GFAP_Q",
            "FEATURE_MASK__PLASMA_NfL_F",
            "FEATURE_MASK__PLASMA_GFAP_F",
        ],
        "branch_mask": "BRANCH_MASK__PLASMA",
    },

    "apoe": {
        "continuous": [],
        "categorical": [
            "APOE__APOE4_ALLELE_COUNT__IDX",
        ],
        "feature_masks": [
            "FEATURE_MASK__APOE_APOE4_ALLELE_COUNT",
        ],
        "branch_mask": "BRANCH_MASK__APOE",
    },

    "mri": {
        "path": "MRI__NORMALIZED_T1_NPY_PATH",
        "branch_mask": "BRANCH_MASK__MRI",
    },
}


# ------------------------------------------------------------
# Dataset sample structure
# ------------------------------------------------------------

# One participant will later be returned by the PyTorch dataset
# using the following nested structure.
#
# Continuous features will become float32 tensors.
# Categorical indices will become int64 tensors for embeddings.
# Masks will become float32 tensors containing 0 or 1.
# The target will become an int64 class index.

dataset_output_contract = {
    "rid": "Participant RID as an integer",
    "ptid": "Participant PTID as a string",
    "target": "Binary class index: 0 for sMCI and 1 for pMCI",

    "modalities": {
        "demographics": {
            "continuous": "Shape (2,), float32",
            "categorical": "Shape (2,), int64",
            "feature_mask": "Shape (4,), float32",
            "branch_mask": "Scalar, float32",
        },

        "cognitive_functional": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "csf": {
            "continuous": "Shape (5,), float32",
            "categorical": None,
            "feature_mask": "Shape (5,), float32",
            "branch_mask": "Scalar, float32",
        },

        "plasma": {
            "continuous": "Shape (9,), float32",
            "categorical": None,
            "feature_mask": "Shape (9,), float32",
            "branch_mask": "Scalar, float32",
        },

        "apoe": {
            "continuous": None,
            "categorical": "Shape (1,), int64",
            "feature_mask": "Shape (1,), float32",
            "branch_mask": "Scalar, float32",
        },

        "mri": {
            "path": "Prepared NumPy path or None",
            "image": "Later: shape (1, 177, 213, 183), float32",
            "branch_mask": "Scalar, float32",
        },
    },

    "branch_masks": (
        "Shape (6,), float32, ordered as "
        "demographics, cognitive-functional, CSF, "
        "plasma, APOE, MRI"
    ),

    "feature_masks": (
        "Shape (28,), float32, using the authoritative "
        "feature-mask order"
    ),
}


# ------------------------------------------------------------
# Display the agreed contract
# ------------------------------------------------------------

print("=" * 72)
print("MULTIMODAL DATASET CONTRACT")
print("=" * 72)

print("\nBranch-specific input dimensions:")

for branch_name, branch_definition in dataset_column_contract.items():

    continuous_count = len(
        branch_definition.get("continuous", [])
    )

    categorical_count = len(
        branch_definition.get("categorical", [])
    )

    feature_mask_count = len(
        branch_definition.get("feature_masks", [])
    )

    has_mri_path = "path" in branch_definition

    print(
        f"- {branch_name}: "
        f"{continuous_count} continuous, "
        f"{categorical_count} categorical, "
        f"{feature_mask_count} feature masks"
        + (", 1 MRI path" if has_mri_path else "")
    )


print("\nPlanned sample output:")
print(
    json.dumps(
        dataset_output_contract,
        indent=2,
    )
)

print(
    "\nThis contract will be used in the next step to "
    "implement the PyTorch dataset."
)

### 1.8.4. Implementing the multimodal PyTorch dataset

implement a PyTorch dataset that converts each prepared participant row into the agreed multimodal structure.

The dataset preserves the six modality branches and returns continuous variables, categorical indices, observation masks, participant identifiers, and the prognosis target separately.

MRI volumes are loaded only when a sample is requested. The existing preprocessed NumPy array is used directly, and a channel dimension is added to produce the shape required by a three-dimensional neural network.

When MRI is unavailable, the participant remains in the dataset. The dataset returns a zero placeholder volume together with an MRI branch mask of zero, allowing the model to distinguish an unavailable scan from an observed image.

In [ ]:
# ============================================================
# 4. Implementing the multimodal PyTorch dataset
# ============================================================

import numpy as np
import torch

from torch.utils.data import Dataset


# ------------------------------------------------------------
# Expected prepared MRI shape
# ------------------------------------------------------------

# I preserve the spatial dimensions produced by the completed
# MRI preprocessing pipeline.
MRI_SPATIAL_SHAPE = (177, 213, 183)

# I add one channel dimension when returning an MRI tensor.
MRI_TENSOR_SHAPE = (1, *MRI_SPATIAL_SHAPE)


# ------------------------------------------------------------
# Multimodal PyTorch dataset
# ------------------------------------------------------------

class ADNIMultimodalDataset(Dataset):
    """
    PyTorch dataset for the prepared ADNI multimodal tables.

    Each participant is returned as a dictionary containing:
    - identifiers;
    - target;
    - separate modality inputs;
    - branch-level masks;
    - feature-level masks.

    MRI arrays are loaded lazily from the prepared NumPy paths.
    """

    def __init__(
        self,
        dataframe,
        column_contract,
        branch_mask_order,
        feature_mask_order,
        target_column,
        load_mri=True,
    ):
        # I reset the row index so that PyTorch sample indices map
        # directly to positional rows in this dataset.
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        # I retain the prepared column organisation rather than
        # deriving new feature groups from column-name patterns.
        self.column_contract = column_contract

        # I preserve the authoritative mask order from the final
        # model-input schema.
        self.branch_mask_order = list(branch_mask_order)
        self.feature_mask_order = list(feature_mask_order)

        self.target_column = target_column

        # This option allows scalar-only experiments and dataset
        # inspection without reading the large MRI arrays.
        self.load_mri = load_mri


    def __len__(self):
        return len(self.dataframe)


    @staticmethod
    def _continuous_tensor(row, columns):
        """
        Convert prepared continuous values to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _categorical_tensor(row, columns):
        """
        Convert prepared categorical indices to an int64 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.int64)
        )

        return torch.from_numpy(values)


    @staticmethod
    def _mask_tensor(row, columns):
        """
        Convert prepared binary masks to a float32 tensor.
        """

        if not columns:
            return None

        values = (
            row[columns]
            .to_numpy(dtype=np.float32)
        )

        return torch.from_numpy(values)


    def _load_mri_tensor(
        self,
        mri_path,
        mri_branch_mask,
    ):
        """
        Load one prepared MRI array or return a masked placeholder.
        """

        # A participant without MRI remains in the dataset.
        if float(mri_branch_mask) == 0.0:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # Scalar-only inspection can skip disk loading while
        # preserving the same output structure.
        if not self.load_mri:
            return torch.zeros(
                MRI_TENSOR_SHAPE,
                dtype=torch.float32,
            )

        # An available MRI branch should have a prepared path.
        if pd.isna(mri_path):
            raise ValueError(
                "MRI branch mask is 1, but the MRI path is missing."
            )

        mri_path = Path(str(mri_path))

        if not mri_path.exists():
            raise FileNotFoundError(
                f"Prepared MRI array was not found: {mri_path}"
            )

        # I load the already normalised NumPy volume without
        # applying any additional preprocessing.
        mri_array = np.load(
            mri_path,
            allow_pickle=False,
        )

        if mri_array.shape != MRI_SPATIAL_SHAPE:
            raise ValueError(
                "Unexpected MRI shape for "
                f"{mri_path}: {mri_array.shape}"
            )

        # I ensure float32 representation and add the channel axis:
        # (177, 213, 183) -> (1, 177, 213, 183).
        mri_array = np.asarray(
            mri_array,
            dtype=np.float32,
        )

        mri_array = np.expand_dims(
            mri_array,
            axis=0,
        )

        return torch.from_numpy(mri_array)


    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        modalities = {}

        # --------------------------------------------------------
        # Demographics
        # --------------------------------------------------------

        demographics_contract = self.column_contract[
            "demographics"
        ]

        modalities["demographics"] = {
            "continuous": self._continuous_tensor(
                row,
                demographics_contract["continuous"],
            ),

            "categorical": self._categorical_tensor(
                row,
                demographics_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                demographics_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    demographics_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Cognitive and functional measures
        # --------------------------------------------------------

        cognitive_contract = self.column_contract[
            "cognitive_functional"
        ]

        modalities["cognitive_functional"] = {
            "continuous": self._continuous_tensor(
                row,
                cognitive_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                cognitive_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    cognitive_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # CSF
        # --------------------------------------------------------

        csf_contract = self.column_contract["csf"]

        modalities["csf"] = {
            "continuous": self._continuous_tensor(
                row,
                csf_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                csf_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    csf_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Plasma
        # --------------------------------------------------------

        plasma_contract = self.column_contract["plasma"]

        modalities["plasma"] = {
            "continuous": self._continuous_tensor(
                row,
                plasma_contract["continuous"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                plasma_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    plasma_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # APOE
        # --------------------------------------------------------

        apoe_contract = self.column_contract["apoe"]

        modalities["apoe"] = {
            "categorical": self._categorical_tensor(
                row,
                apoe_contract["categorical"],
            ),

            "feature_mask": self._mask_tensor(
                row,
                apoe_contract["feature_masks"],
            ),

            "branch_mask": torch.tensor(
                row[
                    apoe_contract["branch_mask"]
                ],
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # MRI
        # --------------------------------------------------------

        mri_contract = self.column_contract["mri"]

        mri_branch_mask = row[
            mri_contract["branch_mask"]
        ]

        mri_path = row[
            mri_contract["path"]
        ]

        modalities["mri"] = {
            "image": self._load_mri_tensor(
                mri_path=mri_path,
                mri_branch_mask=mri_branch_mask,
            ),

            "branch_mask": torch.tensor(
                mri_branch_mask,
                dtype=torch.float32,
            ),
        }


        # --------------------------------------------------------
        # Complete sample
        # --------------------------------------------------------

        sample = {
            "rid": int(row["RID"]),
            "ptid": str(row["PTID"]),

            "target": torch.tensor(
                int(row[self.target_column]),
                dtype=torch.long,
            ),

            "modalities": modalities,

            "branch_masks": self._mask_tensor(
                row,
                self.branch_mask_order,
            ),

            "feature_masks": self._mask_tensor(
                row,
                self.feature_mask_order,
            ),
        }

        return sample


# ------------------------------------------------------------
# Create role-specific datasets
# ------------------------------------------------------------

# I retain the prepared role assignments exactly as stored in
# the selected outer-fold table.
train_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "train"
    ]
    .reset_index(drop=True)
)

validation_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "validation"
    ]
    .reset_index(drop=True)
)

test_table = (
    fold_table.loc[
        fold_table["DATA_ROLE"] == "test"
    ]
    .reset_index(drop=True)
)


# I initially disable MRI disk loading so that I can inspect the
# dataset structure quickly before constructing the DataLoaders.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=False,
)


# ------------------------------------------------------------
# Concise dataset summary
# ------------------------------------------------------------

print("=" * 72)
print("PYTORCH DATASETS")
print("=" * 72)

print(f"\nTraining participants: {len(train_dataset)}")
print(f"Validation participants: {len(validation_dataset)}")
print(f"Test participants: {len(test_dataset)}")

print(
    "\nThe datasets preserve the prepared role assignments "
    "and return separate modality inputs."
)

print(
    "MRI loading is temporarily disabled for structural "
    "inspection and will be enabled for the DataLoaders."
)

### 1.8.5. Inspecting one multimodal sample and one prepared MRI volume

Before constructing the DataLoaders, The notebook inspects the structure returned for one participant.

use the dataset with MRI loading disabled to confirm the scalar tensors, categorical indices, targets, and masks. I then create a temporary MRI-enabled dataset and load one participant whose MRI branch is available.

This checks the dataset interface required by the model while avoiding unnecessary loading of multiple MRI volumes at this stage.

In [ ]:
# ============================================================
# 5. Inspecting one multimodal sample and one MRI volume
# ============================================================

# ------------------------------------------------------------
# Inspect one scalar-only training sample
# ------------------------------------------------------------

# I retrieve one participant while MRI disk loading remains
# disabled. The returned MRI tensor is therefore only the
# temporary placeholder defined in the current dataset class.
sample = train_dataset[0]

print("=" * 72)
print("EXAMPLE MULTIMODAL SAMPLE")
print("=" * 72)

print(f"\nRID: {sample['rid']}")
print(f"PTID: {sample['ptid']}")
print(f"Target: {sample['target'].item()}")

print("\nModality tensor structure:")

for modality_name, modality_data in sample["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"{value}"
            )

print(
    f"\nComplete branch-mask shape: "
    f"{tuple(sample['branch_masks'].shape)}"
)

print(
    f"Complete feature-mask shape: "
    f"{tuple(sample['feature_masks'].shape)}"
)


# ------------------------------------------------------------
# Find one participant with an available MRI
# ------------------------------------------------------------

# I select the first training participant whose prepared MRI
# branch mask is one.
example_mri_index = train_table.index[
    train_table["BRANCH_MASK__MRI"] == 1
][0]


# ------------------------------------------------------------
# Create a temporary MRI-enabled dataset
# ------------------------------------------------------------

# I enable MRI loading only for this temporary inspection
# dataset. The main role-specific datasets remain unchanged.
mri_inspection_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

mri_sample = mri_inspection_dataset[
    example_mri_index
]

mri_tensor = mri_sample[
    "modalities"
]["mri"]["image"]

mri_branch_mask = mri_sample[
    "modalities"
]["mri"]["branch_mask"]


# ------------------------------------------------------------
# Display the prepared MRI tensor information
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("EXAMPLE PREPARED MRI")
print("=" * 72)

print(f"\nRID: {mri_sample['rid']}")
print(f"PTID: {mri_sample['ptid']}")

print(
    f"MRI branch mask: "
    f"{mri_branch_mask.item():.0f}"
)

print(
    f"MRI tensor shape: "
    f"{tuple(mri_tensor.shape)}"
)

print(
    f"MRI tensor dtype: "
    f"{mri_tensor.dtype}"
)

print(
    f"MRI intensity minimum: "
    f"{mri_tensor.min().item():.6f}"
)

print(
    f"MRI intensity maximum: "
    f"{mri_tensor.max().item():.6f}"
)

print(
    f"MRI intensity mean: "
    f"{mri_tensor.mean().item():.6f}"
)

print(
    "\nThe sample structure and full prepared MRI volume "
    "are ready for DataLoader construction."
)

### 1.8.6. Constructing the multimodal DataLoaders

create separate DataLoaders for the fixed training, validation, and test subsets.

MRI loading is enabled, so an available scan is read lazily from its prepared NumPy path when its participant enters a batch. Participants without MRI receive a zero placeholder volume and retain an MRI branch mask of zero.

I begin with a small batch size because each sample contains a full three-dimensional MRI volume. The final training batch size will be selected later according to the memory requirements of the complete model.

The training DataLoader shuffles participants. Validation and test DataLoaders preserve a deterministic order.

In [ ]:
# ============================================================
# 6. Constructing the multimodal DataLoaders
# ============================================================

from torch.utils.data import DataLoader


# ------------------------------------------------------------
# Initial DataLoader settings
# ------------------------------------------------------------

# I begin with a small batch because each participant may contain
# a full MRI volume with shape (1, 177, 213, 183).
INITIAL_BATCH_SIZE = 2

# I initially use the main process for data loading. This is the
# most reliable starting configuration when reading NumPy files
# from mounted Google Drive.
NUM_WORKERS = 0

# Pinned memory can speed transfers to a CUDA device.
PIN_MEMORY = torch.cuda.is_available()


# ------------------------------------------------------------
# Recreate the datasets with MRI loading enabled
# ------------------------------------------------------------

# Available MRI volumes will now be read lazily when requested.
# Missing MRI branches will retain their zero placeholders and
# branch masks of zero.
train_dataset = ADNIMultimodalDataset(
    dataframe=train_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

validation_dataset = ADNIMultimodalDataset(
    dataframe=validation_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)

test_dataset = ADNIMultimodalDataset(
    dataframe=test_table,
    column_contract=dataset_column_contract,
    branch_mask_order=branch_mask_columns,
    feature_mask_order=feature_mask_columns,
    target_column=target_column,
    load_mri=True,
)


# ------------------------------------------------------------
# Create role-specific DataLoaders
# ------------------------------------------------------------

# I shuffle only the training subset.
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# Validation order does not need to be shuffled.
validation_loader = DataLoader(
    dataset=validation_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

# The test subset also retains a deterministic order.
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=INITIAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)


# ------------------------------------------------------------
# Retrieve one complete training batch
# ------------------------------------------------------------

example_batch = next(iter(train_loader))


# ------------------------------------------------------------
# Display the batched tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("EXAMPLE MULTIMODAL BATCH")
print("=" * 72)

print(f"\nBatch size: {example_batch['target'].shape[0]}")
print(f"RID values: {example_batch['rid']}")
print(f"PTID values: {example_batch['ptid']}")
print(f"Targets: {example_batch['target']}")

print("\nModality tensors:")

for modality_name, modality_data in example_batch["modalities"].items():

    print(f"\n{modality_name}")

    for input_name, value in modality_data.items():

        if isinstance(value, torch.Tensor):
            print(
                f"  {input_name}: "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )
        else:
            print(
                f"  {input_name}: "
                f"type={type(value).__name__}"
            )


print("\nCombined masks:")

print(
    "  branch_masks: "
    f"shape={tuple(example_batch['branch_masks'].shape)}, "
    f"dtype={example_batch['branch_masks'].dtype}"
)

print(
    "  feature_masks: "
    f"shape={tuple(example_batch['feature_masks'].shape)}, "
    f"dtype={example_batch['feature_masks'].dtype}"
)


# ------------------------------------------------------------
# Show MRI availability in this batch
# ------------------------------------------------------------

batch_mri = example_batch[
    "modalities"
]["mri"]["image"]

batch_mri_masks = example_batch[
    "modalities"
]["mri"]["branch_mask"]

print("\nMRI batch:")

print(
    f"  image shape: {tuple(batch_mri.shape)}"
)

print(
    f"  branch masks: {batch_mri_masks}"
)

print(
    f"  approximate raw MRI batch size: "
    f"{batch_mri.numel() * batch_mri.element_size() / (1024 ** 2):.2f} MB"
)


# ------------------------------------------------------------
# DataLoader summary
# ------------------------------------------------------------

print("\nDataLoader batches:")

print(
    f"  training: {len(train_loader)} batches"
)

print(
    f"  validation: {len(validation_loader)} batches"
)

print(
    f"  test: {len(test_loader)} batches"
)

print(
    "\nThe complete multimodal batch is ready for "
    "modality-specific encoder construction."
)

### 1.8.7. Building the modality-specific encoders

Each modality has a different input structure, so I encode the six branches separately before multimodal interaction.

For the scalar branches, the prepared feature values are combined with their feature-observation masks. This allows the encoders to distinguish an observed standardised value close to zero from a missing-value placeholder.

Categorical variables use trainable embeddings. Encoded index zero remains reserved for missing or unseen values and is handled through embedding padding behaviour.

### 1.8.8. Common latent dimension

Every branch is projected into the same latent dimension:

$$
d_{\mathrm{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore produces:

$$
\mathbf{H}^{(m)}
\in
\mathbb{R}^{B \times 64}.
$$

The shared dimension is required so that all modality representations can later enter the same cascaded cross-modal transformer architecture.

The original interaction pathway implementation used a dimension of \(512\). In this thesis, I begin with a more compact dimension of \(64\) because the main prognosis training folds contain approximately \(369\) participants and the complete model will additionally include independent evidential heads, modality-specific evidence fusion, auxiliary outputs, and the cascaded interaction pathway interaction path.

The embedding dimension remains a model-capacity hyperparameter and may later be compared with larger values using only training and validation data.

### 1.8.9. MRI encoder

The MRI branch follows the general image-encoding structure used by interaction pathway:

1. initial three-dimensional convolutions;
2. residual three-dimensional downsampling blocks;
3. conversion of the final feature map into patch tokens;
4. addition of learned positional embeddings;
5. transformer encoding of patch-wise relationships;
6. global averaging across patch tokens;
7. projection into the shared modality dimension.

The original paper used a 512-dimensional MRI output. Here, the final MRI representation is projected to the common 64-dimensional latent space used by the other branches.

The prepared MRI volumes are larger than those used in the original interaction pathway experiments. I therefore apply stride-two downsampling in the initial convolutional stem before the four residual blocks. This preserves the CNN-transformer design while keeping the number of transformer patch tokens computationally manageable.

No new MRI preprocessing is performed. The encoder receives the complete prepared volume with shape:

$$
(1, 177, 213, 183).
$$

In [ ]:
# ============================================================
# 7. Building the modality-specific encoders
# ============================================================

import math

import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------
# Shared representation dimension
# ------------------------------------------------------------

# I project every modality into one common latent space so that
# all branches can later enter the same cross-modal transformers.
MODALITY_EMBEDDING_DIM = 64


# ------------------------------------------------------------
# Reusable scalar encoder
# ------------------------------------------------------------

class MaskAwareScalarEncoder(nn.Module):
    """
    Encode continuous scalar features together with their
    prepared feature-observation masks.
    """

    def __init__(
        self,
        value_dim,
        mask_dim,
        output_dim,
        hidden_dim=64,
        dropout=0.20,
    ):
        super().__init__()

        input_dim = value_dim + mask_dim

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        values,
        feature_mask,
    ):
        # I supply both the prepared values and their masks so that
        # missing placeholders are not treated as genuine observations.
        inputs = torch.cat(
            [
                values,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Demographics encoder
# ------------------------------------------------------------

class DemographicsEncoder(nn.Module):
    """
    Encode continuous and categorical demographic predictors.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.20,
    ):
        super().__init__()

        # Sex:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.sex_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Handedness:
        # 0 = missing or unseen;
        # 1 and 2 = observed prepared categories.
        self.handedness_embedding = nn.Embedding(
            num_embeddings=3,
            embedding_dim=4,
            padding_idx=0,
        )

        # Input components:
        # 2 continuous values;
        # 4-dimensional sex embedding;
        # 4-dimensional handedness embedding;
        # 4 feature masks.
        input_dim = 2 + 4 + 4 + 4

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                48,
            ),
            nn.LayerNorm(48),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                48,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        continuous,
        categorical,
        feature_mask,
    ):
        sex_index = categorical[:, 0]
        handedness_index = categorical[:, 1]

        sex_representation = self.sex_embedding(
            sex_index
        )

        handedness_representation = (
            self.handedness_embedding(
                handedness_index
            )
        )

        inputs = torch.cat(
            [
                continuous,
                sex_representation,
                handedness_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# APOE encoder
# ------------------------------------------------------------

class APOEEncoder(nn.Module):
    """
    Encode the prepared APOE epsilon-4 allele-count index.
    """

    def __init__(
        self,
        output_dim,
        dropout=0.10,
    ):
        super().__init__()

        # Prepared APOE indices:
        # 0 = missing;
        # 1 = zero epsilon-4 alleles;
        # 2 = one epsilon-4 allele;
        # 3 = two epsilon-4 alleles.
        self.apoe_embedding = nn.Embedding(
            num_embeddings=4,
            embedding_dim=8,
            padding_idx=0,
        )

        self.network = nn.Sequential(
            nn.Linear(
                8 + 1,
                32,
            ),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                32,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
        )


    def forward(
        self,
        categorical,
        feature_mask,
    ):
        apoe_index = categorical[:, 0]

        apoe_representation = self.apoe_embedding(
            apoe_index
        )

        inputs = torch.cat(
            [
                apoe_representation,
                feature_mask,
            ],
            dim=-1,
        )

        return self.network(inputs)


# ------------------------------------------------------------
# Residual 3D downsampling block
# ------------------------------------------------------------

class ResidualDownsampleBlock3D(nn.Module):
    """
    Downsample a three-dimensional feature map and learn a
    residual representation at the new channel width.
    """

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        # The original interaction pathway image encoder applies spatial
        # downsampling before the residual convolutional paths.
        self.pool = nn.MaxPool3d(
            kernel_size=2,
            stride=2,
        )

        self.main_path = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                out_channels,
                affine=True,
            ),
        )

        # A point-wise convolution aligns the residual path with
        # the new number of channels.
        self.residual_path = nn.Conv3d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            bias=False,
        )

        self.activation = nn.GELU()


    def forward(self, inputs):
        pooled_inputs = self.pool(
            inputs
        )

        main_features = self.main_path(
            pooled_inputs
        )

        residual_features = self.residual_path(
            pooled_inputs
        )

        return self.activation(
            main_features
            + residual_features
        )


# ------------------------------------------------------------
# interaction pathway-style CNN-transformer MRI encoder
# ------------------------------------------------------------

class MRIEncoder3D(nn.Module):
    """
    Encode the prepared full-volume MRI using a 3D CNN followed
    by a patch-wise transformer encoder.
    """

    def __init__(
        self,
        output_dim,
        input_shape=MRI_SPATIAL_SHAPE,
        patch_embedding_dim=256,
        transformer_heads=8,
        transformer_layers=1,
        transformer_feedforward_dim=512,
        dropout=0.20,
    ):
        super().__init__()

        if patch_embedding_dim % transformer_heads != 0:
            raise ValueError(
                "The MRI patch-embedding dimension must be "
                "divisible by the number of attention heads."
            )

        self.input_shape = tuple(
            input_shape
        )

        self.patch_embedding_dim = (
            patch_embedding_dim
        )

        # --------------------------------------------------------
        # Initial convolutional stem
        # --------------------------------------------------------

        # I use two initial 3D convolutions, following the broad
        # structure shown in the interaction pathway image encoder.
        #
        # The first convolution uses stride two because the prepared
        # MRI volumes are larger than the original interaction pathway inputs.
        self.stem = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                stride=2,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),

            nn.Conv3d(
                in_channels=16,
                out_channels=16,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.InstanceNorm3d(
                16,
                affine=True,
            ),
            nn.GELU(),
        )


        # --------------------------------------------------------
        # Four residual downsampling blocks
        # --------------------------------------------------------

        self.residual_blocks = nn.Sequential(
            ResidualDownsampleBlock3D(
                in_channels=16,
                out_channels=32,
            ),

            ResidualDownsampleBlock3D(
                in_channels=32,
                out_channels=64,
            ),

            ResidualDownsampleBlock3D(
                in_channels=64,
                out_channels=128,
            ),

            ResidualDownsampleBlock3D(
                in_channels=128,
                out_channels=256,
            ),
        )


        # --------------------------------------------------------
        # Determine the resulting patch grid
        # --------------------------------------------------------

        # The stride-two stem convolution applies ceiling division
        # by two for these kernel and padding settings.
        stem_shape = tuple(
            math.ceil(dimension / 2)
            for dimension in self.input_shape
        )

        # Each of the four MaxPool3d layers applies floor division
        # by two.
        patch_grid_shape = stem_shape

        for _ in range(4):
            patch_grid_shape = tuple(
                dimension // 2
                for dimension in patch_grid_shape
            )

        if any(
            dimension < 1
            for dimension in patch_grid_shape
        ):
            raise ValueError(
                "The MRI input becomes too small after "
                "convolutional downsampling."
            )

        self.patch_grid_shape = (
            patch_grid_shape
        )

        self.number_of_patches = math.prod(
            patch_grid_shape
        )


        # --------------------------------------------------------
        # Learned positional embeddings
        # --------------------------------------------------------

        # Each location in the final 3D feature map becomes one
        # transformer patch token.
        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                self.number_of_patches,
                patch_embedding_dim,
            )
        )

        nn.init.trunc_normal_(
            self.position_embedding,
            std=0.02,
        )


        # --------------------------------------------------------
        # Patch-wise transformer encoder
        # --------------------------------------------------------

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=patch_embedding_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_feedforward_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer=transformer_layer,
            num_layers=transformer_layers,
            norm=nn.LayerNorm(
                patch_embedding_dim
            ),
        )


        # --------------------------------------------------------
        # Projection to the shared modality dimension
        # --------------------------------------------------------

        self.projection = nn.Sequential(
            nn.Linear(
                patch_embedding_dim,
                output_dim,
            ),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )


    def forward(self, image):
        # Expected image shape:
        # (batch_size, 1, 177, 213, 183)
        feature_map = self.stem(
            image
        )

        feature_map = self.residual_blocks(
            feature_map
        )

        # Expected feature-map organisation:
        # (batch_size, 256, depth, height, width)
        batch_size, channels, depth, height, width = (
            feature_map.shape
        )

        actual_patch_count = (
            depth
            * height
            * width
        )

        if actual_patch_count != self.number_of_patches:
            raise ValueError(
                "Unexpected MRI patch count. "
                f"Expected {self.number_of_patches}, "
                f"but obtained {actual_patch_count}."
            )

        # I flatten the spatial locations into patch tokens:
        #
        # (B, C, D, H, W)
        # -> (B, C, N)
        # -> (B, N, C)
        patch_tokens = (
            feature_map
            .flatten(start_dim=2)
            .transpose(1, 2)
        )

        # I add learned positional information before modelling
        # relationships between the 3D patch representations.
        patch_tokens = (
            patch_tokens
            + self.position_embedding
        )

        transformed_tokens = (
            self.transformer_encoder(
                patch_tokens
            )
        )

        # The paper applies patch-wise average pooling before the
        # final linear projection.
        pooled_representation = (
            transformed_tokens.mean(
                dim=1
            )
        )

        return self.projection(
            pooled_representation
        )


# ------------------------------------------------------------
# Complete set of six modality encoders
# ------------------------------------------------------------

class ADNIModalityEncoders(nn.Module):
    """
    Produce one common-dimensional representation per modality.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.demographics = DemographicsEncoder(
            output_dim=output_dim,
        )

        self.cognitive_functional = (
            MaskAwareScalarEncoder(
                value_dim=9,
                mask_dim=9,
                hidden_dim=96,
                output_dim=output_dim,
            )
        )

        self.csf = MaskAwareScalarEncoder(
            value_dim=5,
            mask_dim=5,
            hidden_dim=64,
            output_dim=output_dim,
        )

        self.plasma = MaskAwareScalarEncoder(
            value_dim=9,
            mask_dim=9,
            hidden_dim=96,
            output_dim=output_dim,
        )

        self.apoe = APOEEncoder(
            output_dim=output_dim,
        )

        self.mri = MRIEncoder3D(
            output_dim=output_dim,
            input_shape=MRI_SPATIAL_SHAPE,
            patch_embedding_dim=256,
            transformer_heads=8,
            transformer_layers=1,
            transformer_feedforward_dim=512,
            dropout=0.20,
        )


    def forward(self, modalities):
        representations = {}

        representations["demographics"] = (
            self.demographics(
                continuous=modalities[
                    "demographics"
                ]["continuous"],

                categorical=modalities[
                    "demographics"
                ]["categorical"],

                feature_mask=modalities[
                    "demographics"
                ]["feature_mask"],
            )
        )

        representations["cognitive_functional"] = (
            self.cognitive_functional(
                values=modalities[
                    "cognitive_functional"
                ]["continuous"],

                feature_mask=modalities[
                    "cognitive_functional"
                ]["feature_mask"],
            )
        )

        representations["csf"] = self.csf(
            values=modalities[
                "csf"
            ]["continuous"],

            feature_mask=modalities[
                "csf"
            ]["feature_mask"],
        )

        representations["plasma"] = self.plasma(
            values=modalities[
                "plasma"
            ]["continuous"],

            feature_mask=modalities[
                "plasma"
            ]["feature_mask"],
        )

        representations["apoe"] = self.apoe(
            categorical=modalities[
                "apoe"
            ]["categorical"],

            feature_mask=modalities[
                "apoe"
            ]["feature_mask"],
        )

        representations["mri"] = self.mri(
            modalities[
                "mri"
            ]["image"]
        )

        return representations


# ------------------------------------------------------------
# Instantiate the revised encoders
# ------------------------------------------------------------

modality_encoders = ADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Count trainable parameters
# ------------------------------------------------------------

def count_trainable_parameters(module):
    return sum(
        parameter.numel()
        for parameter in module.parameters()
        if parameter.requires_grad
    )


print("=" * 72)
print("MODALITY-SPECIFIC ENCODERS")
print("=" * 72)

print(
    f"\nShared modality embedding dimension: "
    f"{MODALITY_EMBEDDING_DIM}"
)

print(
    "\nMRI transformer patch grid: "
    f"{modality_encoders.mri.patch_grid_shape}"
)

print(
    "MRI transformer patch count: "
    f"{modality_encoders.mri.number_of_patches}"
)

print("\nTrainable parameters by encoder:")

for encoder_name in [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]:
    encoder = getattr(
        modality_encoders,
        encoder_name,
    )

    print(
        f"- {encoder_name}: "
        f"{count_trainable_parameters(encoder):,}"
    )

print(
    "\nTotal trainable encoder parameters: "
    f"{count_trainable_parameters(modality_encoders):,}"
)

print(
    "\nThe revised MRI branch now uses a 3D CNN, patch tokens, "
    "positional embeddings, and a transformer encoder."
)

print(
    "No multimodal fusion or classification head has been "
    "added yet."
)

### 1.8.10. Applying branch-availability masks to the encoded modalities

Each modality has a different original input structure, but every modality-specific encoder projects its input into the same latent dimensionality.

In this implementation, each branch produces a representation of size:

$$
d_{\text{model}} = 64.
$$

For a batch of size \(B\), each modality encoder therefore returns:

$$
\mathbf{H}^{(m)} \in \mathbb{R}^{B \times 64},
$$

where \(m\) denotes one of the six modalities:

- demographics;
- cognitive-functional measures;
- CSF;
- plasma;
- APOE;
- MRI.

The shared dimensionality does not mean that the modalities contain the same information or use equally complex encoders. Each branch has its own input-specific encoder, but the final representations must have a common size so that they can later participate in cross-modal attention and evidential fusion.

After stacking the six modality representations, the model obtains:

$$
\mathbf{H}
\in
\mathbb{R}^{B \times 6 \times 64}.
$$

This follows the general design principle used by interaction pathway, in which heterogeneous modality inputs are first projected into a common transformer embedding space before cross-modal interaction. The original interaction pathway implementation used a larger embedding dimension of \(512\), but \(512\) is an architectural hyperparameter rather than a methodological requirement.

A compact starting dimension of \(64\) is used here because the main MCI prognosis training folds contain only approximately \(369\) participants, while the complete planned model will also contain:

- six modality-specific encoders;
- cascaded cross-modal attention;
- independent evidential heads;
- evidence pathway-style evidential fusion;
- a joint evidential prediction path.

The number of parameters in transformer projections grows approximately with the square of the embedding dimension. For example:

$$
64^2 = 4{,}096,
$$

whereas:

$$
512^2 = 262{,}144.
$$

Thus, increasing the embedding dimension from \(64\) to \(512\) can make several attention and feed-forward parameter blocks approximately \(64\) times larger. A \(512\)-dimensional model would therefore introduce substantially greater overfitting and memory risk for the available prognosis cohort.

The value \(64\) is treated as a compact initial configuration rather than as a permanently fixed optimum. The latent dimensionality can later be compared with alternatives such as \(128\) or \(256\), using only the training and validation subsets.

### 1.8.11. Branch-availability masking

The prepared zero placeholders make missing inputs computationally compatible with neural-network layers, but they do not guarantee that an unavailable modality will produce a zero encoder output.

Linear layers, embeddings, normalisation parameters, and learned biases can generate a non-zero representation even when all supplied inputs are zero. I therefore apply the prepared branch mask after each modality encoder.

For participant \(i\) and modality \(m\), the masked representation is:

$$
\widetilde{\mathbf{h}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{h}_{i}^{(m)},
$$

where:

- \(\mathbf{h}_{i}^{(m)} \in \mathbb{R}^{64}\) is the raw modality representation;
- \(a_{i}^{(m)} \in \{0,1\}\) is the prepared branch-availability mask;
- \(\widetilde{\mathbf{h}}_{i}^{(m)} \in \mathbb{R}^{64}\) is the masked representation passed to later model components.

Therefore:

$$
a_{i}^{(m)} = 1
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{h}_{i}^{(m)},
$$

and:

$$
a_{i}^{(m)} = 0
\quad\Longrightarrow\quad
\widetilde{\mathbf{h}}_{i}^{(m)}
=
\mathbf{0}.
$$

An unavailable modality consequently contributes an exact zero representation rather than a learned bias-derived vector. The original branch masks are also retained separately so that the later cross-modal attention and evidential-fusion components can explicitly identify which modalities are available for each participant.

In [ ]:
# ============================================================
# 8. Applying branch masks to the encoded modalities
# ============================================================

# ------------------------------------------------------------
# Fixed modality order
# ------------------------------------------------------------

# I use one explicit modality order throughout the architecture.
# This order matches the prepared branch-mask columns.
MODALITY_ORDER = [
    "demographics",
    "cognitive_functional",
    "csf",
    "plasma",
    "apoe",
    "mri",
]


# ------------------------------------------------------------
# Mask-aware encoder wrapper
# ------------------------------------------------------------

class MaskedADNIModalityEncoders(nn.Module):
    """
    Run the six modality encoders and suppress representations
    from unavailable branches.
    """

    def __init__(
        self,
        output_dim=MODALITY_EMBEDDING_DIM,
    ):
        super().__init__()

        self.output_dim = output_dim

        self.encoders = ADNIModalityEncoders(
            output_dim=output_dim,
        )


    @staticmethod
    def _apply_branch_mask(
        representation,
        branch_mask,
    ):
        """
        Multiply each participant's representation by the
        corresponding scalar branch-availability mask.
        """

        # representation:
        #     (batch_size, embedding_dim)
        #
        # branch_mask:
        #     (batch_size,)
        #
        # I add a final dimension so broadcasting is explicit:
        #     (batch_size,) -> (batch_size, 1)
        expanded_mask = branch_mask.unsqueeze(-1)

        return representation * expanded_mask


    def forward(self, modalities):
        # I first obtain the ordinary encoder outputs.
        raw_representations = self.encoders(
            modalities
        )

        masked_representations = {}

        # I then suppress every unavailable branch using its own
        # prepared branch-level mask.
        for modality_name in MODALITY_ORDER:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            masked_representations[modality_name] = (
                self._apply_branch_mask(
                    representation=raw_representations[
                        modality_name
                    ],
                    branch_mask=branch_mask,
                )
            )

        # I return both versions for later interpretation and
        # debugging. Only the masked representations should enter
        # multimodal interaction and fusion.
        return {
            "raw": raw_representations,
            "masked": masked_representations,
        }


# ------------------------------------------------------------
# Instantiate the mask-aware encoder collection
# ------------------------------------------------------------

masked_modality_encoders = MaskedADNIModalityEncoders(
    output_dim=MODALITY_EMBEDDING_DIM,
)


# ------------------------------------------------------------
# Apply the encoders to one complete batch
# ------------------------------------------------------------

# I keep this inspection on the CPU. The training device will be
# configured later when the full model and optimisation loop exist.
masked_modality_encoders.eval()

with torch.no_grad():
    encoded_batch = masked_modality_encoders(
        example_batch["modalities"]
    )


# ------------------------------------------------------------
# Stack modality representations
# ------------------------------------------------------------

# I stack the representations in the fixed modality order.
#
# Resulting shape:
# (batch_size, number_of_modalities, embedding_dimension)
stacked_masked_representations = torch.stack(
    [
        encoded_batch["masked"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_raw_representations = torch.stack(
    [
        encoded_batch["raw"][modality_name]
        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the encoded structure
# ------------------------------------------------------------

print("=" * 72)
print("MASKED MODALITY REPRESENTATIONS")
print("=" * 72)

print(
    f"\nStacked raw representation shape: "
    f"{tuple(stacked_raw_representations.shape)}"
)

print(
    f"Stacked masked representation shape: "
    f"{tuple(stacked_masked_representations.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, embedding)"
)

print("\nBranch masks in this batch:")

branch_mask_table = pd.DataFrame(
    example_batch["branch_masks"].numpy(),
    columns=MODALITY_ORDER,
)

display(branch_mask_table)


# ------------------------------------------------------------
# Representation norms before and after masking
# ------------------------------------------------------------

# A representation norm summarises the magnitude of each branch
# vector. Missing branches may have non-zero raw norms because of
# learned biases, but their masked norms must be exactly zero.
raw_norms = torch.linalg.vector_norm(
    stacked_raw_representations,
    dim=-1,
)

masked_norms = torch.linalg.vector_norm(
    stacked_masked_representations,
    dim=-1,
)

norm_summary = []

for participant_index in range(
    stacked_masked_representations.shape[0]
):
    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):
        norm_summary.append(
            {
                "BATCH_ROW": participant_index,
                "RID": int(
                    example_batch["rid"][
                        participant_index
                    ].item()
                ),
                "MODALITY": modality_name,
                "BRANCH_MASK": float(
                    example_batch["branch_masks"][
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "RAW_REPRESENTATION_NORM": float(
                    raw_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
                "MASKED_REPRESENTATION_NORM": float(
                    masked_norms[
                        participant_index,
                        modality_index,
                    ].item()
                ),
            }
        )

norm_summary = pd.DataFrame(norm_summary)

print("\nRepresentation norms before and after masking:")

display(
    norm_summary.round(6)
)


# ------------------------------------------------------------
# Confirm the representation dimensions
# ------------------------------------------------------------

print("\nEncoded modality shapes:")

for modality_name in MODALITY_ORDER:
    print(
        f"- {modality_name}: "
        f"{tuple(encoded_batch['masked'][modality_name].shape)}"
    )

print(
    "\nOnly the masked representations will enter the "
    "multimodal interaction and evidential-fusion paths."
)

### 1.8.12. Building the availability-gated interaction pathway cascade

The cascade order remains unchanged:

$$
\text{demographics}
\rightarrow
\text{APOE}
\rightarrow
\text{cognitive/functional}
\rightarrow
\text{CSF}
\rightarrow
\text{plasma}
\rightarrow
\text{MRI}.
$$

Each cross-modal interaction block still computes a candidate update using query self-attention followed by cross-attention to the encoded modality token. The branch mask then determines whether that candidate becomes the next cumulative query:

$$
\mathbf{q}_m
=
a_m\mathbf{q}^{\mathrm{candidate}}_m
+
(1-a_m)\mathbf{q}_{m-1}.
$$

For an available modality, $a_m=1$ and the candidate update is used. For an unavailable modality, $a_m=0$ and the stage becomes an exact identity update.

The branch-mask tensor follows `MODALITY_ORDER`, while the cross-modal interaction blocks follow `THREE_MT_CASCADE_ORDER`. The class therefore uses the explicit modality-to-mask index mapping already defined by the fold-0 architecture.

In [ ]:
# ============================================================
# 9. Building the availability-gated interaction pathway cascade
# ============================================================

# ------------------------------------------------------------
# Fixed cascade order
# ------------------------------------------------------------

THREE_MT_CASCADE_ORDER = [
    "demographics",
    "apoe",
    "cognitive_functional",
    "csf",
    "plasma",
    "mri",
]


# ------------------------------------------------------------
# One Cascaded Modality Transformer
# ------------------------------------------------------------

class CascadedModalityTransformer(nn.Module):
    """
    Apply query self-attention and inject one modality through
    cross-attention.
    """

    def __init__(
        self,
        embedding_dim,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.self_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.self_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.self_attention_dropout = nn.Dropout(
            dropout
        )

        self.cross_attention_norm = nn.LayerNorm(
            embedding_dim
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim=embedding_dim,
            num_heads=number_of_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.cross_attention_dropout = nn.Dropout(
            dropout
        )

        self.output_norm = nn.LayerNorm(
            embedding_dim
        )


    def forward(
        self,
        latent_query,
        modality_embedding,
    ):
        normalised_query = self.self_attention_norm(
            latent_query
        )

        self_attention_output, self_attention_weights = (
            self.self_attention(
                query=normalised_query,
                key=normalised_query,
                value=normalised_query,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        self_attended_query = (
            latent_query
            + self.self_attention_dropout(
                self_attention_output
            )
        )

        normalised_self_query = self.cross_attention_norm(
            self_attended_query
        )

        cross_attention_output, cross_attention_weights = (
            self.cross_attention(
                query=normalised_self_query,
                key=modality_embedding,
                value=modality_embedding,
                need_weights=True,
                average_attn_weights=False,
            )
        )

        updated_query = (
            self_attended_query
            + self.cross_attention_dropout(
                cross_attention_output
            )
        )

        updated_query = self.output_norm(
            updated_query
        )

        return {
            "updated_query": updated_query,
            "self_attention_weights": self_attention_weights,
            "cross_attention_weights": cross_attention_weights,
        }


# ------------------------------------------------------------
# Complete six-stage availability-gated cascade
# ------------------------------------------------------------

class ThreeMTCascade(nn.Module):
    """
    Refine one learned latent query through the six CMT stages.

    A stage uses its candidate update only when the corresponding
    effective branch mask is one. Otherwise, the previous query is
    preserved exactly.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        number_of_heads=4,
        dropout=0.10,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.modality_order = list(modality_order)
        self.cascade_order = list(cascade_order)

        self.modality_to_mask_index = {
            modality_name: modality_index
            for modality_index, modality_name in enumerate(
                self.modality_order
            )
        }

        self.learned_latent_query = nn.Parameter(
            torch.empty(
                1,
                1,
                embedding_dim,
            )
        )

        nn.init.normal_(
            self.learned_latent_query,
            mean=0.0,
            std=0.02,
        )

        self.cmt_blocks = nn.ModuleDict(
            {
                modality_name:
                    CascadedModalityTransformer(
                        embedding_dim=embedding_dim,
                        number_of_heads=number_of_heads,
                        dropout=dropout,
                    )

                for modality_name in self.cascade_order
            }
        )


    def forward(
        self,
        masked_representations,
        branch_masks,
    ):
        first_modality = self.cascade_order[0]

        batch_size = masked_representations[
            first_modality
        ].shape[0]

        latent_query = self.learned_latent_query.expand(
            batch_size,
            -1,
            -1,
        )

        stage_queries = {}
        self_attention_weights = {}
        cross_attention_weights = {}

        for modality_name in self.cascade_order:
            previous_query = latent_query

            modality_token = masked_representations[
                modality_name
            ].unsqueeze(1)

            stage_output = self.cmt_blocks[
                modality_name
            ](
                latent_query=previous_query,
                modality_embedding=modality_token,
            )

            candidate_query = stage_output[
                "updated_query"
            ]

            modality_index = self.modality_to_mask_index[
                modality_name
            ]

            availability = branch_masks[
                :,
                modality_index,
            ].view(
                -1,
                1,
                1,
            ).to(
                dtype=previous_query.dtype
            )

            latent_query = (
                availability * candidate_query
                + (1.0 - availability) * previous_query
            )

            stage_queries[modality_name] = latent_query

            self_attention_weights[modality_name] = (
                stage_output[
                    "self_attention_weights"
                ]
            )

            cross_attention_weights[modality_name] = (
                stage_output[
                    "cross_attention_weights"
                ]
            )

        joint_representation = latent_query.squeeze(
            dim=1
        )

        return {
            "joint_representation":
                joint_representation,

            "stage_queries":
                stage_queries,

            "self_attention_weights":
                self_attention_weights,

            "cross_attention_weights":
                cross_attention_weights,
        }


# ------------------------------------------------------------
# Instantiate and inspect the gated cascade
# ------------------------------------------------------------

three_mt_cascade = ThreeMTCascade(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    number_of_heads=4,
    dropout=0.10,
)

three_mt_cascade.eval()

with torch.no_grad():
    three_mt_output = three_mt_cascade(
        masked_representations=encoded_batch[
            "masked"
        ],
        branch_masks=example_batch[
            "branch_masks"
        ],
    )


print("=" * 72)
print("AVAILABILITY-GATED 3MT CASCADE")
print("=" * 72)

print(
    f"\nCascade order:\n"
    f"{THREE_MT_CASCADE_ORDER}"
)

print(
    "\nFinal joint representation shape: "
    f"{tuple(three_mt_output['joint_representation'].shape)}"
)


# ------------------------------------------------------------
# Show the actual query update at every stage
# ------------------------------------------------------------

query_change_rows = []

previous_query = (
    three_mt_cascade
    .learned_latent_query
    .expand(
        example_batch["target"].shape[0],
        -1,
        -1,
    )
)

for modality_name in THREE_MT_CASCADE_ORDER:
    current_query = three_mt_output[
        "stage_queries"
    ][modality_name]

    query_change_norm = torch.linalg.vector_norm(
        current_query - previous_query,
        dim=-1,
    ).squeeze(1)

    modality_index = MODALITY_ORDER.index(
        modality_name
    )

    branch_mask = example_batch[
        "branch_masks"
    ][
        :,
        modality_index,
    ]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):
        query_change_rows.append(
            {
                "BATCH_ROW": batch_row,
                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),
                "CASCADE_STAGE": modality_name,
                "BRANCH_MASK": float(
                    branch_mask[
                        batch_row
                    ].item()
                ),
                "QUERY_CHANGE_NORM": float(
                    query_change_norm[
                        batch_row
                    ].item()
                ),
            }
        )

    previous_query = current_query

query_change_summary = pd.DataFrame(
    query_change_rows
)

print(
    "\nQuery changes after availability gating:"
)

display(
    query_change_summary.round(6)
)

print(
    "\nTrainable cascade parameters: "
    f"{count_trainable_parameters(three_mt_cascade):,}"
)


### 1.8.13. Producing independent modality-specific evidential opinions

The interaction pathway cascade produces a cumulative interaction-aware representation, but its intermediate states are not independent modality opinions because every stage contains information inherited from earlier stages.

Trusted Multi-View Classification requires each modality to produce its own class evidence before cross-modal interaction.

For participant \(i\), modality \(m\), and class \(k\), the modality-specific evidential head produces non-negative evidence:

$$
e_{ik}^{(m)}
=
\operatorname{Softplus}
\left(
\mathbf{W}_{m}
\mathbf{z}_{i}^{(m)}
+
\mathbf{b}_{m}
\right),
$$

where:

- \(\mathbf{z}_{i}^{(m)} \in \mathbb{R}^{64}\) is the independently encoded modality representation;
- \(e_{ik}^{(m)} \geq 0\) is the evidence assigned to class \(k\);
- each modality has its own evidential head.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{(m)}
=
e_{ik}^{(m)} + 1.
$$

For the binary prognosis task:

$$
K = 2,
$$

with class order:

$$
[\mathrm{sMCI},\mathrm{pMCI}].
$$

The expected class probabilities are:

$$
p_{ik}^{(m)}
=
\frac{
\alpha_{ik}^{(m)}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{(m)}
}.
$$

The Dirichlet strength is:

$$
S_{i}^{(m)}
=
\sum_{k=1}^{K}
\alpha_{ik}^{(m)}.
$$

The standard evidential uncertainty mass is:

$$
u_{i}^{(m)}
=
\frac{K}{
S_{i}^{(m)}
}.
$$

Low total evidence produces high uncertainty, while stronger evidence produces lower uncertainty.

### 1.8.14. Treatment of unavailable modalities

Only available modalities should contribute an opinion to modality-specific evidence fusion.

For an unavailable branch, I do not interpret the evidential head output as a genuine prediction. Instead, its effective evidence is set to zero:

$$
\widetilde{\mathbf{e}}_{i}^{(m)}
=
a_{i}^{(m)}
\mathbf{e}_{i}^{(m)},
$$

where \(a_{i}^{(m)}\) is the prepared branch mask.

Therefore, an unavailable modality receives:

$$
\widetilde{\boldsymbol{\alpha}}_{i}^{(m)}
=
\mathbf{1},
$$

which is the uniform Dirichlet opinion with:

$$
u_{i}^{(m)} = 1.
$$

The original branch mask is retained so that the next step can exclude unavailable opinions explicitly during evidence pathway/Dempster--Shafer fusion.

At this stage, I construct and inspect the six independent evidential opinions. I do not yet fuse them or combine them with the interaction pathway joint representation.

In [ ]:
# ============================================================
# 10. Producing independent modality-specific evidential opinions
# ============================================================

# ------------------------------------------------------------
# Binary prognosis class definition
# ------------------------------------------------------------

NUMBER_OF_CLASSES = 2

PROGNOSIS_CLASS_ORDER = [
    "sMCI",
    "pMCI",
]


# ------------------------------------------------------------
# One modality-specific evidential head
# ------------------------------------------------------------

class EvidentialClassificationHead(nn.Module):
    """
    Convert one modality representation into non-negative class
    evidence and the corresponding Dirichlet opinion.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=32,
        dropout=0.10,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(
        self,
        representation,
        branch_mask,
    ):
        # I use Softplus to obtain non-negative evidence while
        # retaining smooth gradients.
        raw_evidence = F.softplus(
            self.network(
                representation
            )
        )

        # An unavailable modality must not contribute evidence.
        effective_evidence = (
            raw_evidence
            * branch_mask.unsqueeze(-1)
        )

        # Evidence plus one defines the Dirichlet parameters.
        alpha = effective_evidence + 1.0

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        probabilities = (
            alpha
            / strength
        )

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "raw_evidence": raw_evidence,
            "evidence": effective_evidence,
            "alpha": alpha,
            "strength": strength,
            "probabilities": probabilities,
            "uncertainty": uncertainty,
        }


# ------------------------------------------------------------
# Independent evidential heads for all six modalities
# ------------------------------------------------------------

class IndependentModalityEvidentialHeads(nn.Module):
    """
    Produce one independent Dirichlet opinion per modality before
    any 3MT cross-modal interaction.
    """

    def __init__(
        self,
        modality_order,
        input_dim,
        number_of_classes,
    ):
        super().__init__()

        self.modality_order = list(
            modality_order
        )

        self.number_of_classes = (
            number_of_classes
        )

        self.heads = nn.ModuleDict(
            {
                modality_name:
                    EvidentialClassificationHead(
                        input_dim=input_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=32,
                        dropout=0.10,
                    )

                for modality_name in self.modality_order
            }
        )


    def forward(
        self,
        modality_representations,
        modalities,
    ):
        opinions = {}

        for modality_name in self.modality_order:

            branch_mask = modalities[
                modality_name
            ]["branch_mask"]

            opinions[modality_name] = self.heads[
                modality_name
            ](
                representation=modality_representations[
                    modality_name
                ],
                branch_mask=branch_mask,
            )

        return opinions


# ------------------------------------------------------------
# Instantiate the independent evidential path
# ------------------------------------------------------------

independent_evidential_heads = (
    IndependentModalityEvidentialHeads(
        modality_order=MODALITY_ORDER,
        input_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
    )
)


# ------------------------------------------------------------
# Produce one opinion per modality
# ------------------------------------------------------------

independent_evidential_heads.eval()

with torch.no_grad():

    modality_opinions = (
        independent_evidential_heads(
            # I use the independently encoded branch outputs before
            # they enter the interaction pathway cascade.
            modality_representations=encoded_batch[
                "masked"
            ],

            modalities=example_batch[
                "modalities"
            ],
        )
    )


# ------------------------------------------------------------
# Stack the opinion tensors
# ------------------------------------------------------------

stacked_modality_evidence = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["evidence"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_alpha = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["alpha"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_probabilities = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["probabilities"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)

stacked_modality_uncertainty = torch.stack(
    [
        modality_opinions[
            modality_name
        ]["uncertainty"]

        for modality_name in MODALITY_ORDER
    ],
    dim=1,
)


# ------------------------------------------------------------
# Display the evidential tensor structure
# ------------------------------------------------------------

print("=" * 72)
print("INDEPENDENT MODALITY EVIDENTIAL OPINIONS")
print("=" * 72)

print(
    f"\nClass order: "
    f"{PROGNOSIS_CLASS_ORDER}"
)

print(
    "\nStacked evidence shape: "
    f"{tuple(stacked_modality_evidence.shape)}"
)

print(
    "Stacked Dirichlet-alpha shape: "
    f"{tuple(stacked_modality_alpha.shape)}"
)

print(
    "Stacked probability shape: "
    f"{tuple(stacked_modality_probabilities.shape)}"
)

print(
    "Stacked uncertainty shape: "
    f"{tuple(stacked_modality_uncertainty.shape)}"
)

print(
    "\nExpected organisation: "
    "(batch, modality, class)"
)


# ------------------------------------------------------------
# Create a readable modality-opinion summary
# ------------------------------------------------------------

opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    for modality_index, modality_name in enumerate(
        MODALITY_ORDER
    ):

        branch_mask = float(
            example_batch["branch_masks"][
                batch_row,
                modality_index,
            ].item()
        )

        alpha_values = stacked_modality_alpha[
            batch_row,
            modality_index,
        ]

        probability_values = (
            stacked_modality_probabilities[
                batch_row,
                modality_index,
            ]
        )

        uncertainty_value = (
            stacked_modality_uncertainty[
                batch_row,
                modality_index,
                0,
            ]
        )

        opinion_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "MODALITY": modality_name,

                "BRANCH_MASK": branch_mask,

                "ALPHA_sMCI": float(
                    alpha_values[0].item()
                ),

                "ALPHA_pMCI": float(
                    alpha_values[1].item()
                ),

                "P_sMCI": float(
                    probability_values[0].item()
                ),

                "P_pMCI": float(
                    probability_values[1].item()
                ),

                "UNCERTAINTY": float(
                    uncertainty_value.item()
                ),
            }
        )


modality_opinion_summary = pd.DataFrame(
    opinion_rows
)

print("\nIndependent modality opinions:")

display(
    modality_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable independent evidential-head parameters: "
    f"{count_trainable_parameters(independent_evidential_heads):,}"
)

print(
    "\nUnavailable modalities should have alpha=[1, 1], "
    "probabilities=[0.5, 0.5], and uncertainty=1."
)

print(
    "\nThe available modality opinions are ready for "
    "TMC/Dempster-Shafer fusion."
)

### 1.8.15. Fusing the independent modality opinions with evidence pathway

combine the six independent modality opinions using the reduced Dempster--Shafer rule adopted by Trusted Multi-View Classification.

For modality \(m\), the Dirichlet parameters are:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{e}^{(m)} + \mathbf{1}.
$$

The Dirichlet strength is:

$$
S^{(m)}
=
\sum_{k=1}^{K}
\alpha_{k}^{(m)}.
$$

The class-specific belief masses are:

$$
b_{k}^{(m)}
=
\frac{
\alpha_{k}^{(m)} - 1
}{
S^{(m)}
}
=
\frac{
e_{k}^{(m)}
}{
S^{(m)}
}.
$$

The uncertainty mass is:

$$
u^{(m)}
=
\frac{K}{
S^{(m)}
}.
$$

These masses satisfy:

$$
\sum_{k=1}^{K}
b_{k}^{(m)}
+
u^{(m)}
=
1.
$$

### 1.8.16. Combining two opinions

Consider two opinions, \(A\) and \(B\). Their conflict mass is:

$$
C
=
\sum_{i \neq j}
b_{i}^{A}
b_{j}^{B}.
$$

For each class \(k\), the combined belief mass is:

$$
b_{k}^{A \oplus B}
=
\frac{
b_{k}^{A}b_{k}^{B}
+
b_{k}^{A}u^{B}
+
b_{k}^{B}u^{A}
}{
1-C
}.
$$

The combined uncertainty mass is:

$$
u^{A \oplus B}
=
\frac{
u^{A}u^{B}
}{
1-C
}.
$$

The fused Dirichlet strength is recovered from the fused uncertainty:

$$
S^{A \oplus B}
=
\frac{K}{
u^{A \oplus B}
}.
$$

The fused evidence and Dirichlet parameters are then:

$$
e_{k}^{A \oplus B}
=
b_{k}^{A \oplus B}
S^{A \oplus B},
$$

and:

$$
\alpha_{k}^{A \oplus B}
=
e_{k}^{A \oplus B} + 1.
$$

The rule is applied repeatedly until all six modality opinions have been considered.

### 1.8.17. Missing modalities

An unavailable modality was assigned the vacuous Dirichlet opinion:

$$
\boldsymbol{\alpha}^{(m)}
=
\mathbf{1}.
$$

For this opinion:

$$
\mathbf{b}^{(m)}
=
\mathbf{0},
\qquad
u^{(m)} = 1.
$$

A vacuous opinion acts as an identity element in this combination rule. It adds no class evidence, creates no conflict, and leaves the available opinion unchanged.

Therefore, the same fusion procedure can process all participants without complete-case filtering or synthetic modality imputation.

At this stage, I construct only the evidence pathway-fused independent opinion. The interaction-aware interaction pathway query will receive its own evidential head in a later step.

In [ ]:
# ============================================================
# 11. Fusing the independent modality opinions with evidence pathway
# ============================================================

# ------------------------------------------------------------
# Reduced Dempster-Shafer combination rule
# ------------------------------------------------------------

class TMCFusion(nn.Module):
    """
    Fuse independent Dirichlet modality opinions using the
    reduced Dempster-Shafer combination rule used by TMC.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        numerical_epsilon=1e-8,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.numerical_epsilon = (
            numerical_epsilon
        )


    def _dirichlet_to_opinion(
        self,
        alpha,
    ):
        """
        Convert Dirichlet parameters into belief masses and
        one uncertainty mass.
        """

        strength = alpha.sum(
            dim=-1,
            keepdim=True,
        )

        evidence = alpha - 1.0

        belief = evidence / strength

        uncertainty = (
            self.number_of_classes
            / strength
        )

        return {
            "evidence": evidence,
            "strength": strength,
            "belief": belief,
            "uncertainty": uncertainty,
        }


    def _combine_two(
        self,
        alpha_a,
        alpha_b,
    ):
        """
        Combine two batches of Dirichlet opinions.

        Both inputs have shape:
        (batch_size, number_of_classes).
        """

        opinion_a = self._dirichlet_to_opinion(
            alpha_a
        )

        opinion_b = self._dirichlet_to_opinion(
            alpha_b
        )

        belief_a = opinion_a["belief"]
        belief_b = opinion_b["belief"]

        uncertainty_a = opinion_a[
            "uncertainty"
        ]

        uncertainty_b = opinion_b[
            "uncertainty"
        ]


        # --------------------------------------------------------
        # Conflict mass
        # --------------------------------------------------------

        # The outer product contains every pairwise combination
        # between class beliefs from the two opinions.
        belief_outer_product = (
            belief_a.unsqueeze(-1)
            * belief_b.unsqueeze(-2)
        )

        total_belief_product = (
            belief_outer_product.sum(
                dim=(-2, -1)
            )
        )

        same_class_agreement = (
            torch.diagonal(
                belief_outer_product,
                dim1=-2,
                dim2=-1,
            )
            .sum(dim=-1)
        )

        # Conflict contains products assigned to different classes.
        conflict = (
            total_belief_product
            - same_class_agreement
        )

        normalisation = (
            1.0
            - conflict
        ).clamp_min(
            self.numerical_epsilon
        ).unsqueeze(-1)


        # --------------------------------------------------------
        # Fused belief and uncertainty masses
        # --------------------------------------------------------

        fused_belief = (
            belief_a * belief_b
            + belief_a * uncertainty_b
            + belief_b * uncertainty_a
        ) / normalisation

        fused_uncertainty = (
            uncertainty_a
            * uncertainty_b
        ) / normalisation


        # --------------------------------------------------------
        # Recover the fused Dirichlet opinion
        # --------------------------------------------------------

        fused_strength = (
            self.number_of_classes
            / fused_uncertainty.clamp_min(
                self.numerical_epsilon
            )
        )

        fused_evidence = (
            fused_belief
            * fused_strength
        )

        fused_alpha = (
            fused_evidence
            + 1.0
        )

        fused_probabilities = (
            fused_alpha
            / fused_alpha.sum(
                dim=-1,
                keepdim=True,
            )
        )

        return {
            "alpha": fused_alpha,
            "evidence": fused_evidence,
            "belief": fused_belief,
            "uncertainty": fused_uncertainty,
            "strength": fused_strength,
            "probabilities": fused_probabilities,
            "conflict": conflict.unsqueeze(-1),
        }


    def forward(
        self,
        modality_opinions,
    ):
        """
        Sequentially combine the modality-specific opinions in
        the fixed modality order.
        """

        first_modality = self.modality_order[0]

        fused_alpha = modality_opinions[
            first_modality
        ]["alpha"]

        fusion_history = {}

        # I retain the starting opinion so that the complete fusion
        # sequence can later be inspected.
        first_opinion = self._dirichlet_to_opinion(
            fused_alpha
        )

        fusion_history[first_modality] = {
            "alpha": fused_alpha,
            "belief": first_opinion["belief"],
            "uncertainty": first_opinion[
                "uncertainty"
            ],
            "conflict": torch.zeros(
                fused_alpha.shape[0],
                1,
                dtype=fused_alpha.dtype,
                device=fused_alpha.device,
            ),
        }

        for modality_name in self.modality_order[1:]:

            next_alpha = modality_opinions[
                modality_name
            ]["alpha"]

            combined = self._combine_two(
                alpha_a=fused_alpha,
                alpha_b=next_alpha,
            )

            fused_alpha = combined["alpha"]

            fusion_history[modality_name] = {
                "alpha": combined["alpha"],
                "belief": combined["belief"],
                "uncertainty": combined[
                    "uncertainty"
                ],
                "conflict": combined["conflict"],
            }


        # --------------------------------------------------------
        # Final fused opinion
        # --------------------------------------------------------

        final_strength = fused_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_evidence = (
            fused_alpha
            - 1.0
        )

        final_belief = (
            final_evidence
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        final_probabilities = (
            fused_alpha
            / final_strength
        )

        return {
            "alpha": fused_alpha,
            "evidence": final_evidence,
            "belief": final_belief,
            "strength": final_strength,
            "uncertainty": final_uncertainty,
            "probabilities": final_probabilities,
            "fusion_history": fusion_history,
        }


# ------------------------------------------------------------
# Instantiate the modality-specific evidence fusion module
# ------------------------------------------------------------

tmc_fusion = TMCFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
)


# ------------------------------------------------------------
# Fuse the current independent modality opinions
# ------------------------------------------------------------

with torch.no_grad():

    tmc_output = tmc_fusion(
        modality_opinions=modality_opinions
    )


# ------------------------------------------------------------
# Display final fused tensor shapes
# ------------------------------------------------------------

print("=" * 72)
print("TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    f"\nFusion order: "
    f"{MODALITY_ORDER}"
)

print(
    "\nFused evidence shape: "
    f"{tuple(tmc_output['evidence'].shape)}"
)

print(
    "Fused alpha shape: "
    f"{tuple(tmc_output['alpha'].shape)}"
)

print(
    "Fused probability shape: "
    f"{tuple(tmc_output['probabilities'].shape)}"
)

print(
    "Fused uncertainty shape: "
    f"{tuple(tmc_output['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Final participant-level fused opinions
# ------------------------------------------------------------

fused_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    fused_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "AVAILABLE_MODALITIES": int(
                example_batch["branch_masks"][
                    batch_row
                ].sum()
                .item()
            ),

            "ALPHA_sMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                tmc_output["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                tmc_output["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                tmc_output["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


fused_opinion_summary = pd.DataFrame(
    fused_opinion_rows
)

print("\nFinal TMC-fused opinions:")

display(
    fused_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Inspect the sequential fusion history
# ------------------------------------------------------------

fusion_history_rows = []

for modality_name in MODALITY_ORDER:

    stage_output = tmc_output[
        "fusion_history"
    ][modality_name]

    for batch_row in range(
        example_batch["target"].shape[0]
    ):

        modality_index = MODALITY_ORDER.index(
            modality_name
        )

        fusion_history_rows.append(
            {
                "BATCH_ROW": batch_row,

                "RID": int(
                    example_batch["rid"][
                        batch_row
                    ].item()
                ),

                "FUSED_THROUGH": modality_name,

                "CURRENT_MODALITY_MASK": float(
                    example_batch["branch_masks"][
                        batch_row,
                        modality_index,
                    ].item()
                ),

                "STAGE_CONFLICT": float(
                    stage_output["conflict"][
                        batch_row,
                        0,
                    ].item()
                ),

                "STAGE_UNCERTAINTY": float(
                    stage_output["uncertainty"][
                        batch_row,
                        0,
                    ].item()
                ),
            }
        )


fusion_history_summary = pd.DataFrame(
    fusion_history_rows
)

print("\nSequential fusion history:")

display(
    fusion_history_summary.round(6)
)


print(
    "\nUnavailable modalities should introduce zero conflict "
    "and leave the accumulated opinion unchanged."
)

print(
    "The TMC-fused independent opinion is ready for later "
    "combination with the interaction-aware 3MT opinion."
)

### 1.8.18. Producing the interaction-aware interaction pathway evidential opinion

The modality-specific evidence pathway combines independent modality opinions produced before cross-modal interaction. construct a separate evidential head for the final query produced by the interaction pathway cascade.

The final interaction-pathway representation is:

$$
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\in
\mathbb{R}^{64}.
$$

Unlike the independent modality representations, this vector contains cumulative information learned through the ordered sequence of Cascaded Modality Transformers.

The joint evidential head produces non-negative class evidence:

$$
e_{ik}^{\mathrm{joint}}
=
\operatorname{Softplus}
\left(
f_{\mathrm{joint}}
\left(
\mathbf{q}_{i}^{\mathrm{interaction pathway}}
\right)
\right),
$$

where \(k\) denotes either sMCI or pMCI.

The corresponding Dirichlet parameters are:

$$
\alpha_{ik}^{\mathrm{joint}}
=
e_{ik}^{\mathrm{joint}} + 1.
$$

The expected class probabilities are:

$$
p_{ik}^{\mathrm{joint}}
=
\frac{
\alpha_{ik}^{\mathrm{joint}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{joint}}
}.
$$

The joint uncertainty is:

$$
u_{i}^{\mathrm{joint}}
=
\frac{K}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{joint}}
}.
$$

This opinion serves a different purpose from the evidence pathway-fused independent opinion:

- the modality-specific opinion represents agreement and conflict between modality-specific predictions;
- the joint interaction pathway opinion represents the prediction obtained after learning cross-modal interactions.

### 1.8.19. Auxiliary outputs

The original interaction pathway architecture places an auxiliary classifier after each intermediate cross-modal interaction to provide direct training signals to earlier cascade stages.

The notebook preserves that principle by attaching one auxiliary classifier to every intermediate query except the final MRI stage. These auxiliary heads produce ordinary logits for the prognosis classes and are used only during training.

The auxiliary classifiers are not treated as independent modality-specific opinions because each intermediate query already contains information accumulated from all preceding modalities. They support gradient flow through the cascade but do not represent isolated modality evidence.

The final MRI-stage query receives the joint evidential head and produces the interaction-aware Dirichlet opinion.

In [ ]:
# ============================================================
# 12. Producing the interaction-aware interaction pathway evidential opinion
# ============================================================

# ------------------------------------------------------------
# Auxiliary classifier for one intermediate interaction pathway query
# ------------------------------------------------------------

class ThreeMTAuxiliaryClassifier(nn.Module):
    """
    Produce ordinary class logits from one intermediate
    cumulative 3MT query.

    These outputs support training only and are not interpreted
    as independent modality opinions.
    """

    def __init__(
        self,
        input_dim,
        number_of_classes,
        hidden_dim=64,
        dropout=0.10,
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LeakyReLU(
                negative_slope=0.01,
            ),
            nn.Dropout(dropout),

            nn.Linear(
                hidden_dim,
                number_of_classes,
            ),
        )


    def forward(self, representation):
        return self.network(
            representation
        )


# ------------------------------------------------------------
# Joint interaction pathway evidential and auxiliary heads
# ------------------------------------------------------------

class ThreeMTPredictionHeads(nn.Module):
    """
    Attach auxiliary classifiers to the intermediate CMT outputs
    and one evidential head to the final 3MT representation.
    """

    def __init__(
        self,
        cascade_order,
        embedding_dim,
        number_of_classes,
    ):
        super().__init__()

        self.cascade_order = list(
            cascade_order
        )

        # The final stage produces the joint evidential opinion.
        self.final_stage = self.cascade_order[-1]

        # Every preceding stage receives an auxiliary classifier.
        self.auxiliary_stages = self.cascade_order[:-1]

        self.auxiliary_heads = nn.ModuleDict(
            {
                stage_name:
                    ThreeMTAuxiliaryClassifier(
                        input_dim=embedding_dim,
                        number_of_classes=number_of_classes,
                        hidden_dim=embedding_dim,
                        dropout=0.10,
                    )

                for stage_name in self.auxiliary_stages
            }
        )

        self.joint_evidential_head = (
            EvidentialClassificationHead(
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
                hidden_dim=32,
                dropout=0.10,
            )
        )


    def forward(
        self,
        three_mt_output,
    ):
        auxiliary_logits = {}

        # --------------------------------------------------------
        # Intermediate auxiliary predictions
        # --------------------------------------------------------

        for stage_name in self.auxiliary_stages:

            # Each stored query has shape:
            # (batch_size, 1, embedding_dim).
            stage_representation = three_mt_output[
                "stage_queries"
            ][stage_name].squeeze(1)

            auxiliary_logits[stage_name] = (
                self.auxiliary_heads[
                    stage_name
                ](
                    stage_representation
                )
            )


        # --------------------------------------------------------
        # Final joint evidential opinion
        # --------------------------------------------------------

        joint_representation = three_mt_output[
            "joint_representation"
        ]

        # The final interaction pathway query always exists, even when some input
        # modalities are unavailable. I therefore use a branch mask
        # of one for the joint interaction-aware opinion.
        joint_presence_mask = torch.ones(
            joint_representation.shape[0],
            dtype=joint_representation.dtype,
            device=joint_representation.device,
        )

        joint_opinion = self.joint_evidential_head(
            representation=joint_representation,
            branch_mask=joint_presence_mask,
        )

        return {
            "auxiliary_logits":
                auxiliary_logits,

            "joint_opinion":
                joint_opinion,
        }


# ------------------------------------------------------------
# Instantiate the interaction-pathway prediction heads
# ------------------------------------------------------------

three_mt_prediction_heads = ThreeMTPredictionHeads(
    cascade_order=THREE_MT_CASCADE_ORDER,
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
)


# ------------------------------------------------------------
# Produce the auxiliary and joint outputs
# ------------------------------------------------------------

three_mt_prediction_heads.eval()

with torch.no_grad():

    three_mt_predictions = (
        three_mt_prediction_heads(
            three_mt_output=three_mt_output
        )
    )


joint_opinion = three_mt_predictions[
    "joint_opinion"
]


# ------------------------------------------------------------
# Display the output structure
# ------------------------------------------------------------

print("=" * 72)
print("3MT PREDICTION HEADS")
print("=" * 72)

print(
    f"\nAuxiliary stages: "
    f"{three_mt_prediction_heads.auxiliary_stages}"
)

print(
    f"Final evidential stage: "
    f"{three_mt_prediction_heads.final_stage}"
)

print("\nAuxiliary-logit shapes:")

for stage_name, stage_logits in (
    three_mt_predictions[
        "auxiliary_logits"
    ].items()
):
    print(
        f"- after {stage_name}: "
        f"{tuple(stage_logits.shape)}"
    )


print("\nJoint evidential shapes:")

print(
    "  evidence: "
    f"{tuple(joint_opinion['evidence'].shape)}"
)

print(
    "  alpha: "
    f"{tuple(joint_opinion['alpha'].shape)}"
)

print(
    "  probabilities: "
    f"{tuple(joint_opinion['probabilities'].shape)}"
)

print(
    "  uncertainty: "
    f"{tuple(joint_opinion['uncertainty'].shape)}"
)


# ------------------------------------------------------------
# Participant-level joint opinions
# ------------------------------------------------------------

joint_opinion_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    joint_opinion_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ALPHA_sMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    0,
                ].item()
            ),

            "ALPHA_pMCI": float(
                joint_opinion["alpha"][
                    batch_row,
                    1,
                ].item()
            ),

            "P_sMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                joint_opinion["probabilities"][
                    batch_row,
                    1,
                ].item()
            ),

            "UNCERTAINTY": float(
                joint_opinion["uncertainty"][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


joint_opinion_summary = pd.DataFrame(
    joint_opinion_rows
)

print("\nInteraction-aware 3MT opinions:")

display(
    joint_opinion_summary.round(6)
)


# ------------------------------------------------------------
# Parameter summary
# ------------------------------------------------------------

print(
    "\nTrainable 3MT prediction-head parameters: "
    f"{count_trainable_parameters(three_mt_prediction_heads):,}"
)

print(
    "\nThe auxiliary outputs will support training-time "
    "gradient flow through the cascade."
)

print(
    "The final joint opinion is ready for combination with "
    "the TMC-fused independent opinion."
)

### 1.8.20. Combining the interaction pathway and modality-specific evidence pathways with a constrained reliability gate

The model currently produces two complementary evidential outputs:

1. the interaction-aware interaction pathway opinion;
2. the independently fused modality-specific opinion.

These opinions are derived from the same underlying participant data and therefore should not be combined using Dempster--Shafer fusion as though they were independent evidence sources.

Instead, This notebook uses a participant-specific convex mixture of their calibrated evidence vectors.

### 1.8.21. Pathway calibration

The evidence magnitudes produced by the two pathways may have different numerical scales. In particular, the modality-specific evidence pathway accumulates evidence across several available modalities, whereas the cross-modal interaction pathway produces evidence from one joint head.

I therefore introduce one positive scalar calibration parameter for each pathway:

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
=
\tau_{\mathrm{interaction pathway}}
\mathbf{e}_{i}^{\mathrm{interaction pathway}},
$$

and

$$
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}
=
\tau_{\mathrm{evidence pathway}}
\mathbf{e}_{i}^{\mathrm{evidence pathway}}.
$$

Both calibration factors are constrained to be positive using the Softplus function:

$$
\tau_r
=
\operatorname{Softplus}(\rho_r),
\qquad
r \in
\{\mathrm{interaction pathway},\mathrm{evidence pathway}\}.
$$

The parameters are initialised so that both calibration factors begin at approximately one.

### 1.8.22. Reliability-gate input

The reliability gate receives only uncertainty, conflict, and modality-availability information. It does not receive the original participant features or the full interaction-pathway representation.

For participant \(i\), the gate input is:

$$
\mathbf{r}_i
=
\left[
u_i^{\mathrm{interaction pathway}},
u_i^{\mathrm{evidence pathway}},
\bar{C}_i^{\mathrm{evidence pathway}},
\frac{n_i^{\mathrm{available}}}{M},
\mathbf{a}_i
\right],
$$

where:

- \(u_i^{\mathrm{interaction pathway}}\) is the uncertainty of the joint interaction pathway opinion;
- \(u_i^{\mathrm{evidence pathway}}\) is the uncertainty of the evidence pathway-fused opinion;
- \(\bar{C}_i^{\mathrm{evidence pathway}}\) is the mean conflict encountered when combining available modality opinions;
- \(n_i^{\mathrm{available}}\) is the number of available branches;
- \(M=6\) is the total number of branches;
- \(\mathbf{a}_i\) is the six-element branch-availability vector.

The gate produces one scalar weight:

$$
w_i
=
\sigma
\left(
\mathbf{w}^{\top}
\mathbf{r}_i+b
\right).
$$

The interpretation is:

$$
w_i \rightarrow 1
\quad
\Longrightarrow
\quad
\text{greater reliance on interaction pathway},
$$

and

$$
w_i \rightarrow 0
\quad
\Longrightarrow
\quad
\text{greater reliance on evidence pathway}.
$$

The gate is initialised with zero weights and zero bias. Therefore, before training:

$$
w_i = 0.5.
$$

This prevents either pathway from being preferred arbitrarily at model initialisation.

### 1.8.23. Final evidential opinion

The final evidence is:

$$
\mathbf{e}_{i}^{\mathrm{final}}
=
w_i
\widetilde{\mathbf{e}}_{i}^{\mathrm{interaction pathway}}
+
(1-w_i)
\widetilde{\mathbf{e}}_{i}^{\mathrm{evidence pathway}}.
$$

Because \(w_i \in [0,1]\), this is a convex mixture rather than an addition of two supposedly independent evidence sources.

The final Dirichlet parameters are:

$$
\boldsymbol{\alpha}_{i}^{\mathrm{final}}
=
\mathbf{e}_{i}^{\mathrm{final}}
+
\mathbf{1}.
$$

The final class probabilities and uncertainty are:

$$
p_{ik}^{\mathrm{final}}
=
\frac{
\alpha_{ik}^{\mathrm{final}}
}{
\sum_{j=1}^{K}
\alpha_{ij}^{\mathrm{final}}
},
$$

and

$$
u_i^{\mathrm{final}}
=
\frac{
K
}{
\sum_{k=1}^{K}
\alpha_{ik}^{\mathrm{final}}
}.
$$

The reliability statistics supplied to the gate are detached from the computational graph. This prevents the upstream pathways from manipulating their uncertainty or conflict values merely to obtain a larger gate weight. The final loss can still train both pathways through their evidence contributions.

In [ ]:
# ============================================================
# 13. Combining interaction pathway and evidence pathway with fixed equal fusion
# ============================================================

def inverse_softplus(value):
    """
    Return an unconstrained value whose Softplus transformation
    is approximately equal to the requested positive value.
    """

    value_tensor = torch.as_tensor(
        value,
        dtype=torch.float32,
    )

    return torch.log(
        torch.expm1(
            value_tensor
        )
    )


class FixedEqualHybridFusion(nn.Module):
    """
    Combine calibrated 3MT and TMC evidence with fixed weights.

    The participant-specific reliability gate is removed:

        w_3MT = 0.5
        w_TMC = 0.5

    The two positive pathway evidence scales remain trainable.
    This isolates the contribution of the learned gate without
    changing the remaining fusion architecture.
    """

    def __init__(
        self,
        number_of_classes,
        number_of_modalities,
        initial_three_mt_scale=1.0,
        initial_tmc_scale=1.0,
    ):
        super().__init__()

        self.number_of_classes = number_of_classes
        self.number_of_modalities = number_of_modalities

        self.three_mt_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_three_mt_scale
            ).clone()
        )

        self.tmc_scale_parameter = nn.Parameter(
            inverse_softplus(
                initial_tmc_scale
            ).clone()
        )


    def _calculate_mean_available_conflict(
        self,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        """
        Calculate the mean TMC conflict across available fusion stages.
        """

        stage_conflicts = []
        stage_masks = []

        for modality_index, modality_name in enumerate(
            modality_order[1:],
            start=1,
        ):
            stage_conflicts.append(
                tmc_output[
                    "fusion_history"
                ][modality_name]["conflict"]
            )

            stage_masks.append(
                branch_masks[
                    :,
                    modality_index,
                ].unsqueeze(-1)
            )

        stacked_conflicts = torch.stack(
            stage_conflicts,
            dim=1,
        )

        stacked_masks = torch.stack(
            stage_masks,
            dim=1,
        )

        conflict_sum = (
            stacked_conflicts
            * stacked_masks
        ).sum(
            dim=1
        )

        available_fusion_count = (
            stacked_masks.sum(
                dim=1
            ).clamp_min(1.0)
        )

        return (
            conflict_sum
            / available_fusion_count
        )


    def forward(
        self,
        joint_opinion,
        tmc_output,
        branch_masks,
        modality_order,
    ):
        three_mt_evidence = joint_opinion[
            "evidence"
        ]

        tmc_evidence = tmc_output[
            "evidence"
        ]

        three_mt_scale = F.softplus(
            self.three_mt_scale_parameter
        )

        tmc_scale = F.softplus(
            self.tmc_scale_parameter
        )

        calibrated_three_mt_evidence = (
            three_mt_scale
            * three_mt_evidence
        )

        calibrated_tmc_evidence = (
            tmc_scale
            * tmc_evidence
        )

        three_mt_uncertainty = joint_opinion[
            "uncertainty"
        ]

        tmc_uncertainty = tmc_output[
            "uncertainty"
        ]

        mean_tmc_conflict = (
            self._calculate_mean_available_conflict(
                tmc_output=tmc_output,
                branch_masks=branch_masks,
                modality_order=modality_order,
            )
        )

        available_modality_count = (
            branch_masks.sum(
                dim=-1,
                keepdim=True,
            )
        )

        available_modality_proportion = (
            available_modality_count
            / float(
                self.number_of_modalities
            )
        )

        three_mt_weight = torch.full_like(
            three_mt_uncertainty,
            fill_value=0.5,
        )

        tmc_weight = torch.full_like(
            tmc_uncertainty,
            fill_value=0.5,
        )

        # These compatibility fields preserve the prediction-table
        # contract used by the learned-gate experiment.
        gate_logit = torch.zeros_like(
            three_mt_weight
        )

        gate_input = torch.cat(
            [
                three_mt_uncertainty.detach(),
                tmc_uncertainty.detach(),
                mean_tmc_conflict.detach(),
                available_modality_proportion,
                branch_masks,
            ],
            dim=-1,
        )

        final_evidence = (
            three_mt_weight
            * calibrated_three_mt_evidence
            +
            tmc_weight
            * calibrated_tmc_evidence
        )

        final_alpha = final_evidence + 1.0

        final_strength = final_alpha.sum(
            dim=-1,
            keepdim=True,
        )

        final_probabilities = (
            final_alpha
            / final_strength
        )

        final_uncertainty = (
            self.number_of_classes
            / final_strength
        )

        return {
            "evidence": final_evidence,
            "alpha": final_alpha,
            "strength": final_strength,
            "probabilities": final_probabilities,
            "uncertainty": final_uncertainty,
            "three_mt_weight": three_mt_weight,
            "tmc_weight": tmc_weight,
            "gate_logit": gate_logit,
            "gate_input": gate_input,
            "mean_tmc_conflict": mean_tmc_conflict,
            "available_modality_count": available_modality_count,
            "three_mt_scale": three_mt_scale,
            "tmc_scale": tmc_scale,
            "calibrated_three_mt_evidence":
                calibrated_three_mt_evidence,
            "calibrated_tmc_evidence":
                calibrated_tmc_evidence,
        }


hybrid_fusion = FixedEqualHybridFusion(
    number_of_classes=NUMBER_OF_CLASSES,
    number_of_modalities=len(
        MODALITY_ORDER
    ),
    initial_three_mt_scale=1.0,
    initial_tmc_scale=1.0,
)


hybrid_fusion.eval()

with torch.no_grad():
    hybrid_output = hybrid_fusion(
        joint_opinion=joint_opinion,
        tmc_output=tmc_output,
        branch_masks=example_batch[
            "branch_masks"
        ],
        modality_order=MODALITY_ORDER,
    )


print("=" * 72)
print("FIXED 50/50 3MT-TMC EVIDENTIAL FUSION")
print("=" * 72)

print(
    "\nFinal probability shape: "
    f"{tuple(hybrid_output['probabilities'].shape)}"
)

print(
    "Initial 3MT evidence scale: "
    f"{hybrid_output['three_mt_scale'].item():.6f}"
)

print(
    "Initial TMC evidence scale: "
    f"{hybrid_output['tmc_scale'].item():.6f}"
)

print(
    "Trainable fixed-fusion parameters: "
    f"{count_trainable_parameters(hybrid_fusion):,}"
)

maximum_three_mt_weight_error = (
    hybrid_output[
        "three_mt_weight"
    ]
    - 0.5
).abs().max().item()

maximum_tmc_weight_error = (
    hybrid_output[
        "tmc_weight"
    ]
    - 0.5
).abs().max().item()

probability_sum_error = (
    hybrid_output[
        "probabilities"
    ].sum(
        dim=-1
    )
    - 1.0
).abs().max().item()

print(
    "\nMaximum 3MT-weight deviation from 0.5: "
    f"{maximum_three_mt_weight_error:.10f}"
)

print(
    "Maximum TMC-weight deviation from 0.5: "
    f"{maximum_tmc_weight_error:.10f}"
)

print(
    "Maximum final probability-sum error: "
    f"{probability_sum_error:.10f}"
)

assert maximum_three_mt_weight_error == 0.0
assert maximum_tmc_weight_error == 0.0

print(
    "\nThe participant-specific reliability gate is absent. "
    "Only the two positive pathway evidence scales remain trainable."
)


### 1.8.24. Assembling the complete availability-gated interaction-evidence model

The complete model keeps the same six components used in the original fold-0 run:

1. modality-specific encoders;
2. training-time modality dropout;
3. independent modality evidential heads;
4. modality-specific evidence fusion;
5. the interaction pathway interaction pathway;
6. fixed-equal hybrid evidence fusion.

The effective branch masks are created before encoding. They contain both natural missingness and any additional modality removed by training-time dropout. These exact masks are now passed into the interaction pathway cascade, so every unavailable cross-modal interaction stage preserves the preceding cumulative query.

The modality-specific evidence pathway and fixed equal fusion are unchanged. This isolates the effect of availability-gating the cross-modal interaction updates.

In [ ]:
# ============================================================
# 14. Assembling the complete end-to-end interaction-evidence model
# ============================================================

class ADNIEvidential3MTTMCModel(nn.Module):
    """
    Complete missing-aware and uncertainty-aware multimodal model.

    The model combines:

    1. six modality-specific encoders;
    2. training-time modality dropout;
    3. independent modality evidential heads;
    4. TMC/Dempster-Shafer fusion;
    5. the availability-gated 3MT interaction pathway;
    6. intermediate 3MT auxiliary classifiers;
    7. a joint 3MT evidential head;
    8. fixed-equal final evidence fusion.
    """

    def __init__(
        self,
        embedding_dim=MODALITY_EMBEDDING_DIM,
        number_of_classes=NUMBER_OF_CLASSES,
        modality_order=MODALITY_ORDER,
        cascade_order=THREE_MT_CASCADE_ORDER,
        modality_dropout_probability=0.50,
    ):
        super().__init__()

        self.embedding_dim = embedding_dim
        self.number_of_classes = number_of_classes

        self.modality_order = list(
            modality_order
        )

        self.cascade_order = list(
            cascade_order
        )

        self.number_of_modalities = len(
            self.modality_order
        )

        self.modality_dropout_probability = (
            modality_dropout_probability
        )


        # --------------------------------------------------------
        # Modality-specific encoders
        # --------------------------------------------------------

        # This wrapper returns both raw and branch-masked modality
        # representations.
        self.modality_encoders = (
            MaskedADNIModalityEncoders(
                output_dim=embedding_dim,
            )
        )


        # --------------------------------------------------------
        # Independent modality evidential pathway
        # --------------------------------------------------------

        self.independent_evidential_heads = (
            IndependentModalityEvidentialHeads(
                modality_order=self.modality_order,
                input_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )

        self.tmc_fusion = TMCFusion(
            number_of_classes=number_of_classes,
            modality_order=self.modality_order,
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        self.three_mt_cascade = ThreeMTCascade(
            embedding_dim=embedding_dim,
            modality_order=self.modality_order,
            cascade_order=self.cascade_order,
            number_of_heads=4,
            dropout=0.10,
        )

        self.three_mt_prediction_heads = (
            ThreeMTPredictionHeads(
                cascade_order=self.cascade_order,
                embedding_dim=embedding_dim,
                number_of_classes=number_of_classes,
            )
        )


        # --------------------------------------------------------
        # Final fixed 50/50 hybrid fusion
        # --------------------------------------------------------

        self.hybrid_fusion = (
            FixedEqualHybridFusion(
                number_of_classes=number_of_classes,
                number_of_modalities=self.number_of_modalities,
                initial_three_mt_scale=1.0,
                initial_tmc_scale=1.0,
            )
        )


    # ------------------------------------------------------------
    # Training-time modality dropout
    # ------------------------------------------------------------

    def _apply_modality_dropout(
        self,
        original_branch_masks,
    ):
        """
        Randomly hide genuinely available modalities during training.

        Parameters
        ----------
        original_branch_masks:
            Tensor of shape:
            (batch_size, number_of_modalities)

        Returns
        -------
        effective_branch_masks:
            Masks after training-time modality dropout.

        dropped_branch_masks:
            Indicators showing which originally available branches
            were hidden by modality dropout.
        """

        # Validation and testing always use the genuine prepared
        # availability pattern.
        if (
            not self.training
            or self.modality_dropout_probability <= 0.0
        ):
            effective_branch_masks = (
                original_branch_masks.clone()
            )

            dropped_branch_masks = torch.zeros_like(
                original_branch_masks
            )

            return (
                effective_branch_masks,
                dropped_branch_masks,
            )


        # --------------------------------------------------------
        # Sample branch-retention indicators
        # --------------------------------------------------------

        retention_probability = (
            1.0
            - self.modality_dropout_probability
        )

        retention_masks = torch.bernoulli(
            torch.full_like(
                original_branch_masks,
                fill_value=retention_probability,
            )
        )

        # A naturally unavailable modality remains unavailable.
        effective_branch_masks = (
            original_branch_masks
            * retention_masks
        )


        # --------------------------------------------------------
        # Prevent complete information removal
        # --------------------------------------------------------

        batch_size = original_branch_masks.shape[0]

        for batch_row in range(batch_size):

            originally_available_indices = torch.nonzero(
                original_branch_masks[
                    batch_row
                ] > 0,
                as_tuple=False,
            ).flatten()

            no_effective_modality = (
                effective_branch_masks[
                    batch_row
                ].sum()
                == 0
            )

            if (
                no_effective_modality
                and originally_available_indices.numel() > 0
            ):
                # I randomly restore one branch that was genuinely
                # available for this participant.
                selected_position = torch.randint(
                    low=0,
                    high=originally_available_indices.numel(),
                    size=(1,),
                    device=original_branch_masks.device,
                )

                selected_modality_index = (
                    originally_available_indices[
                        selected_position
                    ].item()
                )

                effective_branch_masks[
                    batch_row,
                    selected_modality_index,
                ] = 1.0


        dropped_branch_masks = (
            original_branch_masks
            - effective_branch_masks
        ).clamp(
            min=0.0,
            max=1.0,
        )

        return (
            effective_branch_masks,
            dropped_branch_masks,
        )


    # ------------------------------------------------------------
    # Construct effective modality dictionaries
    # ------------------------------------------------------------

    def _replace_branch_masks(
        self,
        modalities,
        effective_branch_masks,
    ):
        """
        Construct a new modality dictionary containing the
        training-time effective branch masks.
        """

        effective_modalities = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):
            effective_modalities[modality_name] = dict(
                modalities[modality_name]
            )

            effective_modalities[
                modality_name
            ]["branch_mask"] = (
                effective_branch_masks[
                    :,
                    modality_index,
                ]
            )

        return effective_modalities


    # ------------------------------------------------------------
    # Complete forward pass
    # ------------------------------------------------------------

    def forward(
        self,
        modalities,
        original_branch_masks,
    ):
        """
        Run the complete multimodal architecture.

        Parameters
        ----------
        modalities:
            Nested modality dictionary produced by the dataset.

        original_branch_masks:
            Genuine prepared modality-availability tensor with shape:
            (batch_size, number_of_modalities).
        """

        # --------------------------------------------------------
        # Apply training-time modality dropout
        # --------------------------------------------------------

        (
            effective_branch_masks,
            dropped_branch_masks,
        ) = self._apply_modality_dropout(
            original_branch_masks
        )

        effective_modalities = (
            self._replace_branch_masks(
                modalities=modalities,
                effective_branch_masks=effective_branch_masks,
            )
        )


        # --------------------------------------------------------
        # Encode all six modalities
        # --------------------------------------------------------

        encoded_modalities = self.modality_encoders(
            effective_modalities
        )

        # The encoders have already applied their effective branch
        # masks. I use these representations for both pathways.
        masked_representations = encoded_modalities[
            "masked"
        ]


        # --------------------------------------------------------
        # Independent modality opinions and modality-specific evidence fusion
        # --------------------------------------------------------

        modality_opinions = (
            self.independent_evidential_heads(
                modality_representations=masked_representations,
                modalities=effective_modalities,
            )
        )

        tmc_output = self.tmc_fusion(
            modality_opinions=modality_opinions
        )


        # --------------------------------------------------------
        # Interaction-aware cross-modal interaction pathway
        # --------------------------------------------------------

        three_mt_output = self.three_mt_cascade(
            masked_representations=masked_representations,
            branch_masks=effective_branch_masks,
        )

        three_mt_predictions = (
            self.three_mt_prediction_heads(
                three_mt_output=three_mt_output
            )
        )

        joint_opinion = three_mt_predictions[
            "joint_opinion"
        ]


        # --------------------------------------------------------
        # Final fixed-equal hybrid opinion
        # --------------------------------------------------------

        final_output = self.hybrid_fusion(
            joint_opinion=joint_opinion,
            tmc_output=tmc_output,
            branch_masks=effective_branch_masks,
            modality_order=self.modality_order,
        )


        return {
            # Final main prediction
            "final_output":
                final_output,

            # Independent uncertainty pathway
            "modality_opinions":
                modality_opinions,

            "tmc_output":
                tmc_output,

            # Interaction-aware pathway
            "three_mt_output":
                three_mt_output,

            "three_mt_predictions":
                three_mt_predictions,

            "joint_opinion":
                joint_opinion,

            # Encoder outputs
            "encoded_modalities":
                encoded_modalities,

            # Missingness and training-time dropout information
            "original_branch_masks":
                original_branch_masks,

            "effective_branch_masks":
                effective_branch_masks,

            "dropped_branch_masks":
                dropped_branch_masks,
        }


# ------------------------------------------------------------
# Instantiate the complete model
# ------------------------------------------------------------

complete_model = ADNIEvidential3MTTMCModel(
    embedding_dim=MODALITY_EMBEDDING_DIM,
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,
    cascade_order=THREE_MT_CASCADE_ORDER,
    modality_dropout_probability=0.50,
)


# ------------------------------------------------------------
# Evaluation-mode end-to-end forward pass
# ------------------------------------------------------------

# In evaluation mode, modality dropout is disabled.
complete_model.eval()

with torch.no_grad():

    complete_model_output = complete_model(
        modalities=example_batch[
            "modalities"
        ],

        original_branch_masks=example_batch[
            "branch_masks"
        ],
    )


# ------------------------------------------------------------
# Inspect final outputs
# ------------------------------------------------------------

final_output = complete_model_output[
    "final_output"
]

print("=" * 72)
print("COMPLETE AVAILABILITY-GATED 3MT-TMC MODEL")
print("=" * 72)

print(
    "\nModel mode: "
    f"{'training' if complete_model.training else 'evaluation'}"
)

print(
    "Modality-dropout probability: "
    f"{complete_model.modality_dropout_probability:.2f}"
)

print(
    "\nOriginal branch-mask shape: "
    f"{tuple(complete_model_output['original_branch_masks'].shape)}"
)

print(
    "Effective branch-mask shape: "
    f"{tuple(complete_model_output['effective_branch_masks'].shape)}"
)

print(
    "\nFinal alpha shape: "
    f"{tuple(final_output['alpha'].shape)}"
)

print(
    "Final probability shape: "
    f"{tuple(final_output['probabilities'].shape)}"
)

print(
    "Final uncertainty shape: "
    f"{tuple(final_output['uncertainty'].shape)}"
)

print(
    "\nTotal trainable model parameters: "
    f"{count_trainable_parameters(complete_model):,}"
)


# ------------------------------------------------------------
# Verify that evaluation mode preserves genuine availability
# ------------------------------------------------------------

evaluation_mask_difference = (
    complete_model_output[
        "effective_branch_masks"
    ]
    - complete_model_output[
        "original_branch_masks"
    ]
).abs().max().item()

print(
    "\nMaximum evaluation-mode difference between original "
    f"and effective branch masks: "
    f"{evaluation_mask_difference:.10f}"
)


# ------------------------------------------------------------
# Participant-level output summary
# ------------------------------------------------------------

complete_model_rows = []

for batch_row in range(
    example_batch["target"].shape[0]
):

    original_available_count = int(
        complete_model_output[
            "original_branch_masks"
        ][batch_row].sum().item()
    )

    effective_available_count = int(
        complete_model_output[
            "effective_branch_masks"
        ][batch_row].sum().item()
    )

    complete_model_rows.append(
        {
            "BATCH_ROW": batch_row,

            "RID": int(
                example_batch["rid"][
                    batch_row
                ].item()
            ),

            "TARGET": int(
                example_batch["target"][
                    batch_row
                ].item()
            ),

            "ORIGINAL_MODALITIES":
                original_available_count,

            "EFFECTIVE_MODALITIES":
                effective_available_count,

            "W_3MT": float(
                final_output[
                    "three_mt_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "W_TMC": float(
                final_output[
                    "tmc_weight"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_sMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    0,
                ].item()
            ),

            "P_pMCI": float(
                final_output[
                    "probabilities"
                ][
                    batch_row,
                    1,
                ].item()
            ),

            "FINAL_UNCERTAINTY": float(
                final_output[
                    "uncertainty"
                ][
                    batch_row,
                    0,
                ].item()
            ),
        }
    )


complete_model_summary = pd.DataFrame(
    complete_model_rows
)

print("\nComplete-model evaluation-mode outputs:")

display(
    complete_model_summary.round(6)
)

print(
    "\nThe complete model now passes the effective branch "
    "masks into every CMT stage."
)


### 1.8.25. Defining the joint training objective

The complete architecture produces several supervised outputs with different purposes:

1. the final fixed-equal hybrid opinion;
2. the interaction-aware interaction pathway joint opinion;
3. the evidence pathway-fused independent-modality opinion;
4. six independent modality-specific opinions;
5. five intermediate interaction pathway auxiliary predictions.

These outputs are trained jointly, but the final hybrid prediction remains the primary objective.

The total loss is:

$$
\mathcal{L}_{\mathrm{total}}
=
\mathcal{L}_{\mathrm{final}}
+
\lambda_{\mathrm{joint}}
\mathcal{L}_{\mathrm{joint}}
+
\lambda_{\mathrm{evidence pathway}}
\mathcal{L}_{\mathrm{evidence pathway}}
+
\lambda_{\mathrm{mod}}
\mathcal{L}_{\mathrm{mod}}
+
\lambda_{\mathrm{aux}}
\mathcal{L}_{\mathrm{aux}}
+
\lambda_{\mathrm{gate}}
\mathcal{L}_{\mathrm{gate}}.
$$

The initial loss weights are:

$$
\lambda_{\mathrm{joint}} = 0.5,
\qquad
\lambda_{\mathrm{evidence pathway}} = 0.5,
\qquad
\lambda_{\mathrm{mod}} = 0.1,
$$

$$
\lambda_{\mathrm{aux}} = 0.1,
\qquad
\lambda_{\mathrm{gate}} = 0.01.
$$

The final hybrid loss has coefficient one and therefore remains the dominant objective. The remaining terms provide direct supervision to the two pathways and their intermediate components.

These coefficients are initial modelling choices. Any comparison of alternative values must use only the training and validation partitions.

### 1.8.26. Evidential classification loss

For a Dirichlet prediction:

$$
\boldsymbol{\alpha}_i
=
\mathbf{e}_i+\mathbf{1},
$$

the expected cross-entropy loss is:

$$
\mathcal{L}_{\mathrm{ECE},i}
=
\sum_{k=1}^{K}
y_{ik}
\left[
\psi(S_i)
-
\psi(\alpha_{ik})
\right],
$$

where:

$$
S_i
=
\sum_{k=1}^{K}
\alpha_{ik},
$$

and \(\psi(\cdot)\) denotes the digamma function.

This objective minimises the expected negative log-likelihood under the predicted Dirichlet distribution.

### 1.8.27. Evidence regularisation

An evidential network may become unjustifiably confident by assigning strong evidence to an incorrect class. I therefore add a Kullback--Leibler regularisation term that discourages unsupported evidence.

The adjusted Dirichlet parameters are:

$$
\widetilde{\boldsymbol{\alpha}}_i
=
\mathbf{y}_i
+
(1-\mathbf{y}_i)
\odot
\boldsymbol{\alpha}_i.
$$

This construction removes the evidence assigned to the correct class from the regularisation term while penalising evidence assigned to incorrect classes.

The regularisation term is:

$$
\mathcal{L}_{\mathrm{KL},i}
=
D_{\mathrm{KL}}
\left[
\operatorname{Dir}
\left(
\widetilde{\boldsymbol{\alpha}}_i
\right)
\parallel
\operatorname{Dir}
\left(
\mathbf{1}
\right)
\right].
$$

The complete evidential loss is:

$$
\mathcal{L}_{\mathrm{EDL},i}
=
\mathcal{L}_{\mathrm{ECE},i}
+
\beta_t
\mathcal{L}_{\mathrm{KL},i}.
$$

The regularisation coefficient is annealed during the first training epochs:

$$
\beta_t
=
\min
\left(
1,
\frac{t}{T_{\mathrm{anneal}}}
\right),
$$

where \(t\) is the current epoch and \(T_{\mathrm{anneal}}\) is initially set to ten epochs.

This allows the model to begin learning the classification task before the full evidence penalty is applied.

### 1.8.28. Modality-specific loss

Each independent modality opinion is supervised only when that modality is effectively available after training-time modality dropout.

For branch \(m\), let:

$$
\widetilde{a}_i^{(m)}
\in
\{0,1\}
$$

denote the effective branch mask.

The modality-specific loss is:

$$
\mathcal{L}_{\mathrm{mod}}
=
\frac{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
\mathcal{L}_{\mathrm{EDL},i}^{(m)}
}{
\sum_{i=1}^{N}
\sum_{m=1}^{M}
\widetilde{a}_i^{(m)}
+
\varepsilon
}.
$$

Naturally unavailable or deliberately dropped branches therefore contribute no modality-specific classification loss.

### 1.8.29. Auxiliary interaction pathway loss

The intermediate interaction pathway heads produce ordinary class logits rather than Dirichlet opinions. Their loss is the mean cross-entropy across the five intermediate cascade stages:

$$
\mathcal{L}_{\mathrm{aux}}
=
\frac{1}{J}
\sum_{j=1}^{J}
\operatorname{CE}
\left(
\mathbf{z}^{(j)},
y
\right),
$$

where \(J=5\).

These losses provide direct gradient signals to earlier stages of the cascaded transformer.

### 1.8.30. Gate regularisation

The reliability gate is initialised at:

$$
w_i=0.5.
$$

A weak early-training regulariser discourages immediate collapse to a single pathway:

$$
\mathcal{L}_{\mathrm{gate}}
=
\left(
\frac{1}{N}
\sum_{i=1}^{N}
w_i
-
0.5
\right)^2.
$$

The gate regularisation is annealed to zero after the initial training period. It therefore stabilises early optimisation without forcing the final trained gate to remain balanced.

This cell defines and validates the training objective only. Parameter updates begin after gradient flow is checked in the following step.

In [ ]:
# ============================================================
# 15. Defining the complete joint training objective
# ============================================================

# ------------------------------------------------------------
# KL divergence between a predicted Dirichlet distribution and
# a uniform Dirichlet distribution
# ------------------------------------------------------------

def dirichlet_kl_to_uniform(
    alpha,
):
    """
    Calculate:

        KL(Dir(alpha) || Dir(1))

    for every participant in the batch.

    Parameters
    ----------
    alpha:
        Positive Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    Returns
    -------
    Tensor with shape:
        (batch_size,)
    """

    number_of_classes = alpha.shape[-1]

    uniform_alpha = torch.ones_like(
        alpha
    )

    alpha_strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )

    uniform_strength = uniform_alpha.sum(
        dim=-1,
        keepdim=True,
    )

    log_normalisation_ratio = (
        torch.lgamma(alpha_strength)
        - torch.lgamma(uniform_strength)
        - torch.lgamma(alpha).sum(
            dim=-1,
            keepdim=True,
        )
        + torch.lgamma(uniform_alpha).sum(
            dim=-1,
            keepdim=True,
        )
    )

    digamma_difference = (
        torch.digamma(alpha)
        - torch.digamma(alpha_strength)
    )

    parameter_difference = (
        alpha
        - uniform_alpha
    )

    expectation_term = (
        parameter_difference
        * digamma_difference
    ).sum(
        dim=-1,
        keepdim=True,
    )

    kl_divergence = (
        log_normalisation_ratio
        + expectation_term
    )

    return kl_divergence.squeeze(-1)


# ------------------------------------------------------------
# Evidential classification loss
# ------------------------------------------------------------

def evidential_classification_loss(
    alpha,
    targets,
    number_of_classes,
    annealing_coefficient,
    class_weights=None,
    reduction="mean",
):
    """
    Calculate the expected cross-entropy under a Dirichlet
    distribution together with annealed KL regularisation.

    Parameters
    ----------
    alpha:
        Dirichlet parameters with shape:
        (batch_size, number_of_classes)

    targets:
        Integer class labels with shape:
        (batch_size,)

    number_of_classes:
        Number of prognosis classes.

    annealing_coefficient:
        Current coefficient applied to the KL term.

    class_weights:
        Optional class-weight tensor with shape:
        (number_of_classes,)

    reduction:
        "none", "mean", or "sum".
    """

    targets = targets.long()

    one_hot_targets = F.one_hot(
        targets,
        num_classes=number_of_classes,
    ).to(
        dtype=alpha.dtype
    )

    strength = alpha.sum(
        dim=-1,
        keepdim=True,
    )


    # --------------------------------------------------------
    # Expected cross-entropy under the Dirichlet distribution
    # --------------------------------------------------------

    expected_cross_entropy_by_class = (
        torch.digamma(strength)
        - torch.digamma(alpha)
    )

    expected_cross_entropy = (
        one_hot_targets
        * expected_cross_entropy_by_class
    ).sum(
        dim=-1
    )


    # --------------------------------------------------------
    # Optional class weighting
    # --------------------------------------------------------

    if class_weights is not None:

        sample_weights = class_weights[
            targets
        ].to(
            dtype=alpha.dtype,
            device=alpha.device,
        )

        expected_cross_entropy = (
            expected_cross_entropy
            * sample_weights
        )


    # --------------------------------------------------------
    # Remove correct-class evidence from the KL penalty
    # --------------------------------------------------------

    adjusted_alpha = (
        one_hot_targets
        +
        (
            1.0
            - one_hot_targets
        )
        * alpha
    )

    kl_regularisation = (
        dirichlet_kl_to_uniform(
            adjusted_alpha
        )
    )

    per_sample_loss = (
        expected_cross_entropy
        +
        annealing_coefficient
        * kl_regularisation
    )


    # --------------------------------------------------------
    # Requested reduction
    # --------------------------------------------------------

    if reduction == "none":
        reduced_loss = per_sample_loss

    elif reduction == "mean":
        reduced_loss = per_sample_loss.mean()

    elif reduction == "sum":
        reduced_loss = per_sample_loss.sum()

    else:
        raise ValueError(
            "reduction must be 'none', 'mean', or 'sum'."
        )


    return {
        "loss":
            reduced_loss,

        "per_sample_loss":
            per_sample_loss,

        "expected_cross_entropy":
            expected_cross_entropy,

        "kl_regularisation":
            kl_regularisation,
    }


# ------------------------------------------------------------
# Complete multi-output objective
# ------------------------------------------------------------

class HybridEvidentialTrainingLoss(nn.Module):
    """
    Jointly supervise the final hybrid output, both main pathways,
    the individual available modality opinions, and the auxiliary
    3MT classifiers.
    """

    def __init__(
        self,
        number_of_classes,
        modality_order,
        evidential_annealing_epochs=10,
        gate_regularisation_epochs=0,
        joint_loss_weight=0.50,
        tmc_loss_weight=0.50,
        modality_loss_weight=0.10,
        auxiliary_loss_weight=0.10,
        gate_loss_weight=0.0,
        class_weights=None,
    ):
        super().__init__()

        self.number_of_classes = (
            number_of_classes
        )

        self.modality_order = list(
            modality_order
        )

        self.evidential_annealing_epochs = (
            evidential_annealing_epochs
        )

        self.gate_regularisation_epochs = (
            gate_regularisation_epochs
        )

        self.joint_loss_weight = (
            joint_loss_weight
        )

        self.tmc_loss_weight = (
            tmc_loss_weight
        )

        self.modality_loss_weight = (
            modality_loss_weight
        )

        self.auxiliary_loss_weight = (
            auxiliary_loss_weight
        )

        self.gate_loss_weight = (
            gate_loss_weight
        )


        # --------------------------------------------------------
        # Optional training-fold class weights
        # --------------------------------------------------------

        if class_weights is None:

            self.register_buffer(
                "class_weights",
                None,
            )

        else:

            class_weights = torch.as_tensor(
                class_weights,
                dtype=torch.float32,
            )

            if class_weights.shape != (
                number_of_classes,
            ):
                raise ValueError(
                    "class_weights must contain one value "
                    "for each class."
                )

            self.register_buffer(
                "class_weights",
                class_weights,
            )


    # ------------------------------------------------------------
    # Annealing schedules
    # ------------------------------------------------------------

    def _evidential_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Increase the KL coefficient linearly from zero to one.
        """

        if self.evidential_annealing_epochs <= 0:
            return 1.0

        return min(
            1.0,
            float(epoch)
            / float(
                self.evidential_annealing_epochs
            ),
        )


    def _gate_annealing_coefficient(
        self,
        epoch,
    ):
        """
        Reduce the gate-balance penalty to zero after the initial
        optimisation period.
        """

        if self.gate_regularisation_epochs <= 0:
            return 0.0

        return max(
            0.0,
            1.0
            -
            (
                float(epoch - 1)
                /
                float(
                    self.gate_regularisation_epochs
                )
            ),
        )


    # ------------------------------------------------------------
    # Complete loss calculation
    # ------------------------------------------------------------

    def forward(
        self,
        model_output,
        targets,
        epoch,
    ):
        targets = targets.long()

        evidential_annealing = (
            self._evidential_annealing_coefficient(
                epoch
            )
        )

        gate_annealing = (
            self._gate_annealing_coefficient(
                epoch
            )
        )


        # --------------------------------------------------------
        # Final hybrid evidential loss
        # --------------------------------------------------------

        final_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "final_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        final_loss = final_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Interaction-aware interaction pathway evidential loss
        # --------------------------------------------------------

        joint_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "joint_opinion"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        joint_loss = joint_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # evidence pathway-fused evidential loss
        # --------------------------------------------------------

        tmc_loss_components = (
            evidential_classification_loss(
                alpha=model_output[
                    "tmc_output"
                ]["alpha"],

                targets=targets,

                number_of_classes=
                    self.number_of_classes,

                annealing_coefficient=
                    evidential_annealing,

                class_weights=
                    self.class_weights,

                reduction="mean",
            )
        )

        tmc_loss = tmc_loss_components[
            "loss"
        ]


        # --------------------------------------------------------
        # Independent available-modality evidential loss
        # --------------------------------------------------------

        effective_branch_masks = model_output[
            "effective_branch_masks"
        ]

        weighted_modality_loss_sum = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        available_opinion_count = torch.zeros(
            (),
            dtype=final_loss.dtype,
            device=final_loss.device,
        )

        modality_loss_by_name = {}

        for modality_index, modality_name in enumerate(
            self.modality_order
        ):

            modality_alpha = model_output[
                "modality_opinions"
            ][modality_name]["alpha"]

            modality_loss_components = (
                evidential_classification_loss(
                    alpha=modality_alpha,

                    targets=targets,

                    number_of_classes=
                        self.number_of_classes,

                    annealing_coefficient=
                        evidential_annealing,

                    class_weights=
                        self.class_weights,

                    reduction="none",
                )
            )

            per_sample_modality_loss = (
                modality_loss_components[
                    "per_sample_loss"
                ]
            )

            modality_mask = effective_branch_masks[
                :,
                modality_index,
            ].to(
                dtype=per_sample_modality_loss.dtype
            )

            masked_modality_loss_sum = (
                per_sample_modality_loss
                * modality_mask
            ).sum()

            modality_available_count = (
                modality_mask.sum()
            )

            weighted_modality_loss_sum = (
                weighted_modality_loss_sum
                + masked_modality_loss_sum
            )

            available_opinion_count = (
                available_opinion_count
                + modality_available_count
            )

            modality_loss_by_name[
                modality_name
            ] = (
                masked_modality_loss_sum
                /
                modality_available_count.clamp_min(
                    1.0
                )
            )

        modality_loss = (
            weighted_modality_loss_sum
            /
            available_opinion_count.clamp_min(
                1.0
            )
        )


        # --------------------------------------------------------
        # Intermediate interaction pathway auxiliary cross-entropy loss
        # --------------------------------------------------------

        auxiliary_logits = model_output[
            "three_mt_predictions"
        ]["auxiliary_logits"]

        auxiliary_loss_by_stage = {}

        auxiliary_losses = []

        for stage_name, stage_logits in (
            auxiliary_logits.items()
        ):

            stage_loss = F.cross_entropy(
                input=stage_logits,
                target=targets,
                weight=self.class_weights,
            )

            auxiliary_loss_by_stage[
                stage_name
            ] = stage_loss

            auxiliary_losses.append(
                stage_loss
            )

        if auxiliary_losses:

            auxiliary_loss = torch.stack(
                auxiliary_losses
            ).mean()

        else:

            auxiliary_loss = torch.zeros(
                (),
                dtype=final_loss.dtype,
                device=final_loss.device,
            )


        # --------------------------------------------------------
        # Early gate-balance regularisation
        # --------------------------------------------------------

        three_mt_weights = model_output[
            "final_output"
        ]["three_mt_weight"]

        raw_gate_loss = (
            three_mt_weights.mean()
            - 0.5
        ).pow(2)

        annealed_gate_loss = (
            gate_annealing
            * raw_gate_loss
        )


        # --------------------------------------------------------
        # Weighted total objective
        # --------------------------------------------------------

        total_loss = (
            final_loss
            +
            self.joint_loss_weight
            * joint_loss
            +
            self.tmc_loss_weight
            * tmc_loss
            +
            self.modality_loss_weight
            * modality_loss
            +
            self.auxiliary_loss_weight
            * auxiliary_loss
            +
            self.gate_loss_weight
            * annealed_gate_loss
        )


        return {
            "total_loss":
                total_loss,

            "final_loss":
                final_loss,

            "joint_loss":
                joint_loss,

            "tmc_loss":
                tmc_loss,

            "modality_loss":
                modality_loss,

            "auxiliary_loss":
                auxiliary_loss,

            "raw_gate_loss":
                raw_gate_loss,

            "annealed_gate_loss":
                annealed_gate_loss,

            "evidential_annealing":
                torch.tensor(
                    evidential_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "gate_annealing":
                torch.tensor(
                    gate_annealing,
                    dtype=final_loss.dtype,
                    device=final_loss.device,
                ),

            "available_opinion_count":
                available_opinion_count,

            "modality_loss_by_name":
                modality_loss_by_name,

            "auxiliary_loss_by_stage":
                auxiliary_loss_by_stage,

            "final_expected_cross_entropy":
                final_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "final_kl_regularisation":
                final_loss_components[
                    "kl_regularisation"
                ].mean(),

            "joint_expected_cross_entropy":
                joint_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "joint_kl_regularisation":
                joint_loss_components[
                    "kl_regularisation"
                ].mean(),

            "tmc_expected_cross_entropy":
                tmc_loss_components[
                    "expected_cross_entropy"
                ].mean(),

            "tmc_kl_regularisation":
                tmc_loss_components[
                    "kl_regularisation"
                ].mean(),
        }


# ------------------------------------------------------------
# Instantiate the training objective
# ------------------------------------------------------------

training_objective = HybridEvidentialTrainingLoss(
    number_of_classes=NUMBER_OF_CLASSES,
    modality_order=MODALITY_ORDER,

    evidential_annealing_epochs=10,
    gate_regularisation_epochs=0,

    joint_loss_weight=0.50,
    tmc_loss_weight=0.50,
    modality_loss_weight=0.10,
    auxiliary_loss_weight=0.10,
    gate_loss_weight=0.0,

    # I initially leave class weighting disabled. It can be added
    # using weights calculated from each training fold only.
    class_weights=None,
)


# ------------------------------------------------------------
# Test the objective on the complete untrained forward pass
# ------------------------------------------------------------

example_loss_output = training_objective(
    model_output=complete_model_output,

    targets=example_batch[
        "target"
    ],

    epoch=1,
)


# ------------------------------------------------------------
# Display the initial loss structure
# ------------------------------------------------------------

print("=" * 72)
print("JOINT 3MT-TMC TRAINING OBJECTIVE")
print("=" * 72)

print(
    "\nEvidential KL annealing coefficient at epoch 1: "
    f"{example_loss_output['evidential_annealing'].item():.6f}"
)

print(
    "Gate regularisation coefficient at epoch 1: "
    f"{example_loss_output['gate_annealing'].item():.6f}"
)

print("\nMain loss components:")

for loss_name in [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]:
    print(
        f"- {loss_name}: "
        f"{example_loss_output[loss_name].item():.6f}"
    )


print("\nFinal evidential-loss decomposition:")

print(
    "- expected cross-entropy: "
    f"{example_loss_output['final_expected_cross_entropy'].item():.6f}"
)

print(
    "- KL regularisation: "
    f"{example_loss_output['final_kl_regularisation'].item():.6f}"
)


print("\nAvailable modality opinions in this batch: "
      f"{int(example_loss_output['available_opinion_count'].item())}")


# ------------------------------------------------------------
# Modality-specific loss summary
# ------------------------------------------------------------

modality_loss_rows = []

for modality_name in MODALITY_ORDER:

    modality_loss_rows.append(
        {
            "MODALITY": modality_name,

            "MEAN_AVAILABLE_LOSS": float(
                example_loss_output[
                    "modality_loss_by_name"
                ][modality_name].item()
            ),
        }
    )


print("\nModality-specific evidential losses:")

display(
    pd.DataFrame(
        modality_loss_rows
    ).round(6)
)


# ------------------------------------------------------------
# Auxiliary-stage loss summary
# ------------------------------------------------------------

auxiliary_loss_rows = []

for stage_name in THREE_MT_CASCADE_ORDER[:-1]:

    auxiliary_loss_rows.append(
        {
            "AUXILIARY_STAGE": stage_name,

            "CROSS_ENTROPY_LOSS": float(
                example_loss_output[
                    "auxiliary_loss_by_stage"
                ][stage_name].item()
            ),
        }
    )


print("\nIntermediate 3MT auxiliary losses:")

display(
    pd.DataFrame(
        auxiliary_loss_rows
    ).round(6)
)

print(
    "\nThe joint objective is ready for one end-to-end "
    "backward pass."
)


### 1.8.31. Running one end-to-end backward pass

Before configuring the optimiser, I run one training batch through the complete gated model and joint objective.

For this single diagnostic pass, modality dropout is set to zero so that the natural fold-0 availability pattern is used without additional random branch removal. The total loss is backpropagated, but no optimiser step is taken. The cell reports the total loss and the global gradient norm, then restores the configured modality-dropout probability of `0.50`.

In [ ]:
# ============================================================
# 16. Running one end-to-end backward pass
# ============================================================

diagnostic_batch = next(
    iter(train_loader)
)

original_modality_dropout_probability = (
    complete_model.modality_dropout_probability
)

complete_model.modality_dropout_probability = 0.0
complete_model.train()
complete_model.zero_grad(
    set_to_none=True
)


diagnostic_output = complete_model(
    modalities=diagnostic_batch[
        "modalities"
    ],
    original_branch_masks=diagnostic_batch[
        "branch_masks"
    ],
)

diagnostic_losses = training_objective(
    model_output=diagnostic_output,
    targets=diagnostic_batch[
        "target"
    ],
    epoch=1,
)

diagnostic_total_loss = diagnostic_losses[
    "total_loss"
]

diagnostic_total_loss.backward()


global_gradient_norm = torch.sqrt(
    sum(
        parameter.grad.detach().float().pow(2).sum()
        for parameter in complete_model.parameters()
        if parameter.grad is not None
    )
)

print("=" * 72)
print("END-TO-END BACKWARD PASS")
print("=" * 72)

print(
    "\nTotal loss: "
    f"{diagnostic_total_loss.item():.6f}"
)

print(
    "Global gradient norm: "
    f"{global_gradient_norm.item():.6f}"
)

complete_model.zero_grad(
    set_to_none=True
)

complete_model.modality_dropout_probability = (
    original_modality_dropout_probability
)

complete_model.eval()

print(
    "Modality-dropout probability restored to: "
    f"{complete_model.modality_dropout_probability:.2f}"
)


### 1.8.32. Configuring optimisation and experiment-specific output paths

The original training hyperparameters remain unchanged. Every checkpoint, history file, validation prediction, and test prediction is written only under:

```text
models/3mt_tmc_evidential/experiments/
    gated_cmt_learned_gate_md050/
        mci_prognosis/fold_0/
```

This notebook does not use the older shared `training/mci_prognosis/fold_0/` directory.

In [ ]:
# ============================================================
# 17. Configuring optimisation, checkpoints, and metrics
# ============================================================

import os
import random
from datetime import datetime

import numpy as np

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    roc_auc_score,
)


# ------------------------------------------------------------
# Current experiment
# ------------------------------------------------------------

# EXPERIMENT_NAME, SELECTED_TASK, and SELECTED_FOLD were fixed when this fold's prepared input was loaded.


# ------------------------------------------------------------
# Reproducibility configuration
# ------------------------------------------------------------

GLOBAL_RANDOM_SEED = 42


def set_global_random_seed(seed):
    """
    Set the random seed used by Python, NumPy, and PyTorch.
    """

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_global_random_seed(
    GLOBAL_RANDOM_SEED
)


# ------------------------------------------------------------
# PyTorch numerical configuration
# ------------------------------------------------------------

# I allow cuDNN to choose efficient convolution algorithms.
#
# This is appropriate for the computationally expensive 3D MRI
# encoder. Exact bitwise reproducibility can still depend on the
# installed PyTorch, CUDA, cuDNN, and GPU versions.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False


# ------------------------------------------------------------
# Device configuration
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print("=" * 72)
print("TRAINING CONFIGURATION")
print("=" * 72)

print(
    f"\nExperiment: {EXPERIMENT_NAME}"
)

print(
    f"Task: {SELECTED_TASK}"
)

print(
    f"Selected fold: {SELECTED_FOLD}"
)

print(
    f"Device: {DEVICE}"
)

if torch.cuda.is_available():

    print(
        "CUDA device: "
        f"{torch.cuda.get_device_name(0)}"
    )

    print(
        "CUDA memory allocated before model transfer: "
        f"{torch.cuda.memory_allocated(0) / (1024 ** 3):.3f} GB"
    )


# ------------------------------------------------------------
# Move the complete model and objective to the selected device
# ------------------------------------------------------------

complete_model = complete_model.to(
    DEVICE
)

training_objective = training_objective.to(
    DEVICE
)


# ------------------------------------------------------------
# Optimisation hyperparameters
# ------------------------------------------------------------

MAXIMUM_EPOCHS = 50

INITIAL_LEARNING_RATE = 5e-4

WEIGHT_DECAY = 1e-4

MAXIMUM_GRADIENT_NORM = 5.0

EARLY_STOPPING_PATIENCE = 10

SCHEDULER_PATIENCE = 3

SCHEDULER_REDUCTION_FACTOR = 0.5

MINIMUM_LEARNING_RATE = 1e-6

CLASSIFICATION_THRESHOLD = 0.50


# ------------------------------------------------------------
# AdamW optimiser
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    params=complete_model.parameters(),
    lr=INITIAL_LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ------------------------------------------------------------
# Validation-AUC learning-rate scheduler
# ------------------------------------------------------------

learning_rate_scheduler = (
    torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer=optimizer,
        mode="max",
        factor=SCHEDULER_REDUCTION_FACTOR,
        patience=SCHEDULER_PATIENCE,
        min_lr=MINIMUM_LEARNING_RATE,
    )
)


# ------------------------------------------------------------
# Checkpoint and history directories
# ------------------------------------------------------------

EXPERIMENT_ROOT = (
    MODEL_ROOT
    / "experiments"
    / EXPERIMENT_NAME
)

FOLD_TRAINING_DIR = (
    EXPERIMENT_ROOT
    / SELECTED_TASK
    / f"fold_{SELECTED_FOLD}"
)

CHECKPOINT_DIR = (
    FOLD_TRAINING_DIR
    / "checkpoints"
)

HISTORY_DIR = (
    FOLD_TRAINING_DIR
    / "history"
)

PREDICTION_DIR = (
    FOLD_TRAINING_DIR
    / "predictions"
)


BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "best_validation_auc_checkpoint.pt"
)

LAST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "last_epoch_checkpoint.pt"
)


# ------------------------------------------------------------
# Fold-output safety policy
# ------------------------------------------------------------

existing_fold_files = []

if FOLD_TRAINING_DIR.exists():
    existing_fold_files = [
        path
        for path in FOLD_TRAINING_DIR.rglob("*")
        if path.is_file()
    ]


if FOLD_RUN_MODE == "fresh" and existing_fold_files:
    raise FileExistsError(
        "Fresh training was requested, but this fold directory "
        "already contains files. Nothing was overwritten. "
        "Use a different experiment name, remove the intentionally "
        "discarded fold directory, or select resume mode only for "
        "an interrupted run of this exact fold.\n"
        f"Fold directory: {FOLD_TRAINING_DIR}"
    )


if FOLD_RUN_MODE == "resume" and not LAST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "Resume mode was requested, but this fold has no latest "
        "checkpoint. Nothing was changed.\n"
        f"Expected checkpoint: {LAST_CHECKPOINT_PATH}"
    )


for directory_path in [
    EXPERIMENT_ROOT,
    FOLD_TRAINING_DIR,
    CHECKPOINT_DIR,
    HISTORY_DIR,
    PREDICTION_DIR,
]:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )

TRAINING_HISTORY_PATH = (
    HISTORY_DIR
    / "training_history.csv"
)

TRAINING_CONFIGURATION_PATH = (
    HISTORY_DIR
    / "training_configuration.json"
)

VALIDATION_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "best_validation_predictions.csv"
)

TEST_PREDICTIONS_PATH = (
    PREDICTION_DIR
    / "test_predictions.csv"
)


# ------------------------------------------------------------
# Record the training configuration
# ------------------------------------------------------------

training_configuration = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "fixed_equal_fusion": True,
    "three_mt_weight": 0.5,
    "tmc_weight": 0.5,
    "task": SELECTED_TASK,
    "fold": int(SELECTED_FOLD),
    "random_seed": int(GLOBAL_RANDOM_SEED),

    "maximum_epochs": int(
        MAXIMUM_EPOCHS
    ),

    "batch_size": int(
        train_loader.batch_size
    ),

    "initial_learning_rate": float(
        INITIAL_LEARNING_RATE
    ),

    "weight_decay": float(
        WEIGHT_DECAY
    ),

    "maximum_gradient_norm": float(
        MAXIMUM_GRADIENT_NORM
    ),

    "early_stopping_patience": int(
        EARLY_STOPPING_PATIENCE
    ),

    "scheduler_patience": int(
        SCHEDULER_PATIENCE
    ),

    "scheduler_reduction_factor": float(
        SCHEDULER_REDUCTION_FACTOR
    ),

    "minimum_learning_rate": float(
        MINIMUM_LEARNING_RATE
    ),

    "classification_threshold": float(
        CLASSIFICATION_THRESHOLD
    ),

    "modality_dropout_probability": float(
        complete_model.modality_dropout_probability
    ),

    "embedding_dimension": int(
        MODALITY_EMBEDDING_DIM
    ),

    "number_of_classes": int(
        NUMBER_OF_CLASSES
    ),

    "class_order": list(
        PROGNOSIS_CLASS_ORDER
    ),

    "modality_order": list(
        MODALITY_ORDER
    ),

    "three_mt_cascade_order": list(
        THREE_MT_CASCADE_ORDER
    ),

    "trainable_parameters": int(
        count_trainable_parameters(
            complete_model
        )
    ),

    "device": str(
        DEVICE
    ),

    "cuda_device_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),
}


with open(
    TRAINING_CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as configuration_file:

    json.dump(
        training_configuration,
        configuration_file,
        indent=2,
    )


# ------------------------------------------------------------
# Expected calibration error
# ------------------------------------------------------------

def calculate_binary_expected_calibration_error(
    targets,
    positive_class_probabilities,
    number_of_bins=10,
):
    """
    Calculate equal-width binary expected calibration error.

    Confidence is the probability assigned to the predicted class.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        return float("nan")

    predicted_classes = (
        positive_class_probabilities
        >= 0.5
    ).astype(
        np.int64
    )

    predicted_confidences = np.where(
        predicted_classes == 1,
        positive_class_probabilities,
        1.0 - positive_class_probabilities,
    )

    prediction_correctness = (
        predicted_classes
        == targets
    ).astype(
        np.float64
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    expected_calibration_error = 0.0

    sample_count = targets.size

    for bin_index in range(
        number_of_bins
    ):

        lower_edge = bin_edges[
            bin_index
        ]

        upper_edge = bin_edges[
            bin_index + 1
        ]

        if bin_index == 0:

            in_bin = (
                predicted_confidences
                >= lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        else:

            in_bin = (
                predicted_confidences
                > lower_edge
            ) & (
                predicted_confidences
                <= upper_edge
            )

        bin_count = int(
            in_bin.sum()
        )

        if bin_count == 0:
            continue

        mean_confidence = float(
            predicted_confidences[
                in_bin
            ].mean()
        )

        mean_accuracy = float(
            prediction_correctness[
                in_bin
            ].mean()
        )

        expected_calibration_error += (
            bin_count
            / sample_count
        ) * abs(
            mean_accuracy
            - mean_confidence
        )

    return float(
        expected_calibration_error
    )


# ------------------------------------------------------------
# Safe metric helpers
# ------------------------------------------------------------

def safely_calculate_roc_auc(
    targets,
    probabilities,
):
    """
    Return NaN when ROC AUC is undefined because only one class
    is present in the supplied targets.
    """

    if np.unique(targets).size < 2:
        return float("nan")

    return float(
        roc_auc_score(
            targets,
            probabilities,
        )
    )


def safely_calculate_average_precision(
    targets,
    probabilities,
):
    """
    Return NaN when average precision is not meaningful because
    the supplied targets contain no positive examples.
    """

    if np.sum(targets == 1) == 0:
        return float("nan")

    return float(
        average_precision_score(
            targets,
            probabilities,
        )
    )


# ------------------------------------------------------------
# Complete binary prognosis metrics
# ------------------------------------------------------------

def calculate_binary_classification_metrics(
    targets,
    positive_class_probabilities,
    uncertainties=None,
    three_mt_weights=None,
    classification_threshold=0.50,
):
    """
    Calculate discrimination, classification, calibration, and
    uncertainty summaries for the pMCI-positive prognosis task.
    """

    targets = np.asarray(
        targets,
        dtype=np.int64,
    )

    positive_class_probabilities = np.asarray(
        positive_class_probabilities,
        dtype=np.float64,
    )

    if targets.size == 0:
        raise ValueError(
            "At least one target is required to calculate metrics."
        )

    if (
        targets.shape[0]
        != positive_class_probabilities.shape[0]
    ):
        raise ValueError(
            "Targets and probabilities must contain the same "
            "number of participants."
        )

    positive_class_probabilities = np.clip(
        positive_class_probabilities,
        0.0,
        1.0,
    )

    predicted_classes = (
        positive_class_probabilities
        >= classification_threshold
    ).astype(
        np.int64
    )

    (
        true_negative,
        false_positive,
        false_negative,
        true_positive,
    ) = confusion_matrix(
        targets,
        predicted_classes,
        labels=[0, 1],
    ).ravel()


    sensitivity_denominator = (
        true_positive
        + false_negative
    )

    specificity_denominator = (
        true_negative
        + false_positive
    )

    sensitivity = (
        true_positive
        / sensitivity_denominator
        if sensitivity_denominator > 0
        else float("nan")
    )

    specificity = (
        true_negative
        / specificity_denominator
        if specificity_denominator > 0
        else float("nan")
    )


    clipped_probabilities = np.clip(
        positive_class_probabilities,
        1e-7,
        1.0 - 1e-7,
    )


    metrics = {
        "roc_auc":
            safely_calculate_roc_auc(
                targets,
                positive_class_probabilities,
            ),

        "average_precision":
            safely_calculate_average_precision(
                targets,
                positive_class_probabilities,
            ),

        "accuracy":
            float(
                accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    targets,
                    predicted_classes,
                )
            ),

        "sensitivity":
            float(
                sensitivity
            ),

        "specificity":
            float(
                specificity
            ),

        "precision":
            float(
                precision_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "f1":
            float(
                f1_score(
                    targets,
                    predicted_classes,
                    zero_division=0,
                )
            ),

        "brier_score":
            float(
                brier_score_loss(
                    targets,
                    positive_class_probabilities,
                )
            ),

        "negative_log_likelihood":
            float(
                log_loss(
                    targets,
                    np.column_stack(
                        [
                            1.0
                            - clipped_probabilities,

                            clipped_probabilities,
                        ]
                    ),
                    labels=[0, 1],
                )
            ),

        "expected_calibration_error":
            calculate_binary_expected_calibration_error(
                targets=targets,

                positive_class_probabilities=
                    positive_class_probabilities,

                number_of_bins=10,
            ),

        "classification_threshold":
            float(
                classification_threshold
            ),

        "true_negative":
            int(
                true_negative
            ),

        "false_positive":
            int(
                false_positive
            ),

        "false_negative":
            int(
                false_negative
            ),

        "true_positive":
            int(
                true_positive
            ),
    }


    if uncertainties is not None:

        uncertainties = np.asarray(
            uncertainties,
            dtype=np.float64,
        )

        metrics[
            "mean_uncertainty"
        ] = float(
            uncertainties.mean()
        )

        metrics[
            "std_uncertainty"
        ] = float(
            uncertainties.std()
        )


    if three_mt_weights is not None:

        three_mt_weights = np.asarray(
            three_mt_weights,
            dtype=np.float64,
        )

        metrics[
            "mean_three_mt_weight"
        ] = float(
            three_mt_weights.mean()
        )

        metrics[
            "std_three_mt_weight"
        ] = float(
            three_mt_weights.std()
        )

        metrics[
            "minimum_three_mt_weight"
        ] = float(
            three_mt_weights.min()
        )

        metrics[
            "maximum_three_mt_weight"
        ] = float(
            three_mt_weights.max()
        )


    return metrics

# ------------------------------------------------------------
# Exact fold-0 parameter and target summaries
# ------------------------------------------------------------

model_parameter_count = sum(
    parameter.numel()
    for parameter in complete_model.parameters()
    if parameter.requires_grad
)

optimised_parameter_count = sum(
    parameter.numel()
    for parameter_group in optimizer.param_groups
    for parameter in parameter_group["params"]
    if parameter.requires_grad
)

training_target_counts = (
    train_loader
    .dataset
    .dataframe[target_column]
    .value_counts()
    .sort_index()
)

training_target_summary = pd.DataFrame(
    {
        "CLASS_INDEX": [0, 1],
        "CLASS_NAME": ["sMCI", "pMCI"],
        "TRAINING_COUNT": [
            int(training_target_counts.loc[0]),
            int(training_target_counts.loc[1]),
        ],
    }
)

training_target_summary[
    "TRAINING_PROPORTION"
] = (
    training_target_summary[
        "TRAINING_COUNT"
    ]
    /
    training_target_summary[
        "TRAINING_COUNT"
    ].sum()
)


print("\nOptimiser: AdamW")
print(
    "Initial learning rate: "
    f"{INITIAL_LEARNING_RATE:.6f}"
)
print(
    "Weight decay: "
    f"{WEIGHT_DECAY:.6f}"
)
print(
    "Maximum epochs: "
    f"{MAXIMUM_EPOCHS}"
)
print(
    "Early-stopping patience: "
    f"{EARLY_STOPPING_PATIENCE} epochs"
)
print(
    "Scheduler patience: "
    f"{SCHEDULER_PATIENCE} epochs"
)
print(
    "Checkpoint-selection metric: validation ROC AUC"
)

print(
    "\nExperiment output directory:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "\nBest-checkpoint path:\n"
    f"{BEST_CHECKPOINT_PATH}"
)

print(
    "\nTrainable model parameters: "
    f"{model_parameter_count:,}"
)

print(
    "Parameters included in optimiser: "
    f"{optimised_parameter_count:,}"
)

print(
    "\nFold-0 training target balance:"
)

display(
    training_target_summary.round(6)
)


### 1.8.33. Defining reusable training and validation epoch functions

define the reusable functions that will execute one complete training epoch and one complete validation epoch.

A single training epoch performs the following operations for every mini-batch:

1. move the nested multimodal batch to the selected device;
2. run the complete interaction pathway--evidence pathway forward pass in training mode;
3. apply training-time modality dropout;
4. calculate the joint objective;
5. backpropagate the total loss;
6. clip the global gradient norm;
7. update all model parameters using AdamW;
8. accumulate predictions, uncertainty estimates, gate weights, and losses.

The validation epoch uses the same complete model but differs in three important ways:

- the model is placed in evaluation mode;
- modality dropout and ordinary neural-network dropout are disabled;
- no gradients or parameter updates are calculated.

For both training and validation, pMCI is treated as the positive class. The epoch functions collect:

- participant identifiers;
- binary targets;
- final pMCI probabilities;
- final uncertainty values;
- interaction pathway and modality-specific evidence pathway weights;
- interaction pathway-only pMCI probabilities;
- evidence pathway-only pMCI probabilities;
- the original and effective numbers of available modalities.

The accumulated participant-level outputs are passed to the previously defined metric function after the entire epoch has completed.

### 1.8.34. Gradient clipping

For every training batch, the total gradient norm is calculated and clipped before the optimiser step:

$$
\left\|
\nabla_{\boldsymbol{\theta}}
\mathcal{L}_{\mathrm{total}}
\right\|_2
\leq 5.
$$

The unclipped norm is retained for monitoring.

### 1.8.35. Epoch-level loss aggregation

For loss component \(\ell\), the epoch-level mean is calculated using the number of participants in each mini-batch:

$$
\overline{\mathcal{L}}_{\ell}
=
\frac{
\sum_{b=1}^{B}
n_b
\mathcal{L}_{\ell,b}
}{
\sum_{b=1}^{B}
n_b
},
$$

where \(n_b\) is the batch size.

This avoids giving the final incomplete mini-batch the same weight as a full mini-batch.

The functions defined in this step do not yet train the model across multiple epochs. The full checkpointed training loop is constructed in the following step.

In [ ]:
# ============================================================
# 18. Defining reusable training and validation epoch functions
# ============================================================

from collections import defaultdict


# ------------------------------------------------------------
# Initial numerical-precision policy
# ------------------------------------------------------------

# I initially train in full float32 precision.
#
# This is computationally feasible on the available A100 GPU and
# avoids introducing mixed-precision instability into the Dirichlet
# digamma, log-gamma, and Dempster-Shafer calculations.
USE_MIXED_PRECISION = False


# ------------------------------------------------------------
# Recursively move a nested batch to the selected device
# ------------------------------------------------------------

def move_nested_batch_to_device(
    value,
    device,
):
    """
    Recursively move tensors in dictionaries, lists, and tuples
    to the selected PyTorch device.

    Non-tensor values are preserved unchanged.
    """

    if isinstance(
        value,
        torch.Tensor,
    ):
        return value.to(
            device,
            non_blocking=True,
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: move_nested_batch_to_device(
                nested_value,
                device,
            )
            for key, nested_value in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            move_nested_batch_to_device(
                nested_value,
                device,
            )
            for nested_value in value
        )

    return value


# ------------------------------------------------------------
# Convert one completed forward pass into stored predictions
# ------------------------------------------------------------

def extract_batch_prediction_arrays(
    batch,
    model_output,
):
    """
    Extract participant-level targets, predictions, uncertainty,
    pathway outputs, and modality counts from one mini-batch.
    """

    final_output = model_output[
        "final_output"
    ]

    joint_opinion = model_output[
        "joint_opinion"
    ]

    tmc_output = model_output[
        "tmc_output"
    ]


    extracted = {
        "rid":
            batch["rid"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "target":
            batch["target"]
            .detach()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_p_pMCI":
            final_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "final_uncertainty":
            final_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_weight":
            final_output[
                "three_mt_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_weight":
            final_output[
                "tmc_weight"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_p_pMCI":
            joint_opinion[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "three_mt_uncertainty":
            joint_opinion[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_p_pMCI":
            tmc_output[
                "probabilities"
            ][
                :,
                1,
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "tmc_uncertainty":
            tmc_output[
                "uncertainty"
            ]
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "original_modality_count":
            model_output[
                "original_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),

        "effective_modality_count":
            model_output[
                "effective_branch_masks"
            ]
            .sum(
                dim=-1
            )
            .detach()
            .float()
            .cpu()
            .numpy()
            .reshape(-1),
    }

    return extracted


# ------------------------------------------------------------
# Concatenate the participant outputs collected across an epoch
# ------------------------------------------------------------

def concatenate_epoch_prediction_storage(
    prediction_storage,
):
    """
    Concatenate a dictionary of mini-batch NumPy arrays.
    """

    concatenated = {}

    for key, value_list in prediction_storage.items():

        if len(value_list) == 0:
            concatenated[key] = np.asarray([])

        else:
            concatenated[key] = np.concatenate(
                value_list,
                axis=0,
            )

    return concatenated


# ------------------------------------------------------------
# Convert stored predictions into a participant-level table
# ------------------------------------------------------------

def build_epoch_prediction_table(
    concatenated_predictions,
    split_name,
    epoch,
):
    """
    Build one participant-level DataFrame for an epoch.
    """

    prediction_table = pd.DataFrame(
        {
            "RID":
                concatenated_predictions[
                    "rid"
                ].astype(
                    np.int64
                ),

            "TARGET":
                concatenated_predictions[
                    "target"
                ].astype(
                    np.int64
                ),

            "FINAL_P_pMCI":
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            "FINAL_UNCERTAINTY":
                concatenated_predictions[
                    "final_uncertainty"
                ],

            "W_3MT":
                concatenated_predictions[
                    "three_mt_weight"
                ],

            "W_TMC":
                concatenated_predictions[
                    "tmc_weight"
                ],

            "THREE_MT_P_pMCI":
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            "THREE_MT_UNCERTAINTY":
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            "TMC_P_pMCI":
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            "TMC_UNCERTAINTY":
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            "ORIGINAL_MODALITY_COUNT":
                concatenated_predictions[
                    "original_modality_count"
                ].astype(
                    np.int64
                ),

            "EFFECTIVE_MODALITY_COUNT":
                concatenated_predictions[
                    "effective_modality_count"
                ].astype(
                    np.int64
                ),
        }
    )

    prediction_table.insert(
        loc=0,
        column="EPOCH",
        value=int(epoch),
    )

    prediction_table.insert(
        loc=1,
        column="SPLIT",
        value=str(split_name),
    )

    prediction_table[
        "FINAL_PREDICTED_CLASS"
    ] = (
        prediction_table[
            "FINAL_P_pMCI"
        ]
        >= CLASSIFICATION_THRESHOLD
    ).astype(
        np.int64
    )

    return prediction_table


# ------------------------------------------------------------
# Calculate metrics for all three prediction outputs
# ------------------------------------------------------------

def calculate_epoch_prediction_metrics(
    concatenated_predictions,
):
    """
    Calculate metrics for:

    1. the final fixed-equal hybrid output;
    2. the 3MT-only joint output;
    3. the TMC-only fused output.
    """

    targets = concatenated_predictions[
        "target"
    ]

    final_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "final_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "final_uncertainty"
                ],

            three_mt_weights=
                concatenated_predictions[
                    "three_mt_weight"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    three_mt_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "three_mt_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "three_mt_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    tmc_metrics = (
        calculate_binary_classification_metrics(
            targets=targets,

            positive_class_probabilities=
                concatenated_predictions[
                    "tmc_p_pMCI"
                ],

            uncertainties=
                concatenated_predictions[
                    "tmc_uncertainty"
                ],

            classification_threshold=
                CLASSIFICATION_THRESHOLD,
        )
    )

    return {
        "final": final_metrics,
        "three_mt": three_mt_metrics,
        "tmc": tmc_metrics,
    }


# ------------------------------------------------------------
# Initialise the loss accumulator used within an epoch
# ------------------------------------------------------------

EPOCH_LOSS_NAMES = [
    "total_loss",
    "final_loss",
    "joint_loss",
    "tmc_loss",
    "modality_loss",
    "auxiliary_loss",
    "raw_gate_loss",
    "annealed_gate_loss",
]


def initialise_epoch_loss_storage():
    """
    Create participant-weighted loss totals.
    """

    return {
        loss_name: 0.0
        for loss_name in EPOCH_LOSS_NAMES
    }


def update_epoch_loss_storage(
    storage,
    loss_output,
    batch_size,
):
    """
    Add one batch's losses, weighted by the number of participants.
    """

    for loss_name in EPOCH_LOSS_NAMES:

        storage[loss_name] += (
            float(
                loss_output[
                    loss_name
                ]
                .detach()
                .float()
                .item()
            )
            * batch_size
        )


def finalise_epoch_loss_storage(
    storage,
    participant_count,
):
    """
    Convert accumulated loss sums into participant-weighted means.
    """

    if participant_count <= 0:
        raise ValueError(
            "The epoch contained no participants."
        )

    return {
        loss_name:
            loss_sum
            / participant_count

        for loss_name, loss_sum in storage.items()
    }


# ------------------------------------------------------------
# Run one complete training epoch
# ------------------------------------------------------------

def run_training_epoch(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_gradient_norm,
):
    """
    Train the complete model for one epoch.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []


    for batch_index, batch in enumerate(
        data_loader
    ):

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = targets.shape[0]

        participant_count += batch_size


        # --------------------------------------------------------
        # Clear gradients from the previous mini-batch
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )


        # --------------------------------------------------------
        # Complete forward pass
        # --------------------------------------------------------

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )


        # --------------------------------------------------------
        # Complete multi-output objective
        # --------------------------------------------------------

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )


        # --------------------------------------------------------
        # Backpropagation
        # --------------------------------------------------------

        total_loss.backward()


        # --------------------------------------------------------
        # Global gradient clipping
        # --------------------------------------------------------

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_index}."
            )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )


        # --------------------------------------------------------
        # Parameter update
        # --------------------------------------------------------

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate losses and predictions
        # --------------------------------------------------------

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[key].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


    # ------------------------------------------------------------
    # Finalise the complete training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# Run one complete validation epoch
# ------------------------------------------------------------

def run_validation_epoch(
    model,
    data_loader,
    objective,
    device,
    epoch,
):
    """
    Evaluate the complete model for one epoch without gradients
    or training-time modality dropout.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    with torch.no_grad():

        for batch_index, batch in enumerate(
            data_loader
        ):

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = targets.shape[0]

            participant_count += batch_size


            # ----------------------------------------------------
            # Complete evaluation forward pass
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ----------------------------------------------------
            # Validation objective
            # ----------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_index}."
                )


            # ----------------------------------------------------
            # Accumulate losses and predictions
            # ----------------------------------------------------

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[key].append(
                    values
                )


    # ------------------------------------------------------------
    # Finalise the complete validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Display the configured epoch-function summary
# ------------------------------------------------------------

print("=" * 72)
print("TRAINING AND VALIDATION EPOCH FUNCTIONS")
print("=" * 72)

print(
    f"\nMixed precision enabled: "
    f"{USE_MIXED_PRECISION}"
)

print(
    "Training batches per epoch: "
    f"{len(train_loader)}"
)

print(
    "Validation batches per epoch: "
    f"{len(validation_loader)}"
)

print(
    "Training participants: "
    f"{len(train_loader.dataset)}"
)

print(
    "Validation participants: "
    f"{len(validation_loader.dataset)}"
)

print(
    "\nTraining epoch operations:"
)

print(
    "- forward pass with modality dropout;"
)

print(
    "- complete joint loss calculation;"
)

print(
    "- backpropagation;"
)

print(
    "- global gradient clipping;"
)

print(
    "- AdamW parameter update;"
)

print(
    "- prediction and uncertainty accumulation."
)

print(
    "\nValidation epoch operations:"
)

print(
    "- evaluation mode;"
)

print(
    "- no modality dropout;"
)

print(
    "- no gradient calculation;"
)

print(
    "- complete validation loss and metric calculation."
)

print(
    "\nThe epoch functions are defined."
)

print(
    "No complete training or validation epoch has been run yet."
)

print(
    "The next step will create the checkpointed multi-epoch "
    "training loop and begin model optimisation."
)

### 1.8.36. Training a newly initialised model with validation-based checkpointing

This standalone experiment starts from epoch 1 using the model initialised in this notebook. It does not load a checkpoint from the previous ungated baseline or from an earlier gated run.

The validation partition controls learning-rate reduction, early stopping, and best-checkpoint selection. The test partition remains untouched during training.

In [ ]:
# ============================================================
# 19. Training with live batch and epoch progress
# ============================================================

import time
import traceback
from collections import defaultdict
from datetime import datetime

from tqdm.auto import tqdm


# ------------------------------------------------------------
# Resume and progress-display behaviour
# ------------------------------------------------------------

# The formal first run begins from epoch 1.
# Resume is enabled only when FOLD_RUN_MODE was explicitly set
# to "resume" for this exact experiment and fold.
RESUME_FROM_LAST_CHECKPOINT = (
    FOLD_RUN_MODE == "resume"
)

# I refresh the live progress display after every batch.
PROGRESS_UPDATE_INTERVAL = 1


# ------------------------------------------------------------
# Loss-configuration serialisation
# ------------------------------------------------------------

def obtain_training_objective_configuration(
    objective,
):
    """
    Return the principal loss settings in a checkpoint-safe form.
    """

    return {
        "evidential_annealing_epochs": int(
            objective.evidential_annealing_epochs
        ),

        "gate_regularisation_epochs": int(
            objective.gate_regularisation_epochs
        ),

        "joint_loss_weight": float(
            objective.joint_loss_weight
        ),

        "tmc_loss_weight": float(
            objective.tmc_loss_weight
        ),

        "modality_loss_weight": float(
            objective.modality_loss_weight
        ),

        "auxiliary_loss_weight": float(
            objective.auxiliary_loss_weight
        ),

        "gate_loss_weight": float(
            objective.gate_loss_weight
        ),

        "class_weights": (
            None
            if objective.class_weights is None
            else
            objective.class_weights
            .detach()
            .cpu()
            .tolist()
        ),
    }


# ------------------------------------------------------------
# Flatten one epoch into one history row
# ------------------------------------------------------------

def build_training_history_row(
    epoch,
    learning_rate,
    epoch_duration_seconds,
    training_result,
    validation_result,
    best_validation_auc,
    epochs_without_improvement,
    checkpoint_improved,
):
    """
    Create one flat row containing losses, metrics, optimisation
    diagnostics, and checkpoint information.
    """

    history_row = {
        "epoch":
            int(epoch),

        "learning_rate":
            float(learning_rate),

        "epoch_duration_seconds":
            float(epoch_duration_seconds),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "checkpoint_improved":
            bool(checkpoint_improved),

        "train_participants":
            int(
                training_result[
                    "participant_count"
                ]
            ),

        "validation_participants":
            int(
                validation_result[
                    "participant_count"
                ]
            ),

        "train_batches":
            int(
                training_result[
                    "batch_count"
                ]
            ),

        "validation_batches":
            int(
                validation_result[
                    "batch_count"
                ]
            ),

        "train_mean_gradient_norm":
            float(
                training_result[
                    "gradient_summary"
                ]["mean_gradient_norm"]
            ),

        "train_maximum_gradient_norm_before_clipping":
            float(
                training_result[
                    "gradient_summary"
                ][
                    "maximum_gradient_norm_before_clipping"
                ]
            ),

        "train_mean_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "mean_effective_modality_count"
                ]
            ),

        "train_minimum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "minimum_effective_modality_count"
                ]
            ),

        "train_maximum_effective_modality_count":
            float(
                training_result[
                    "modality_dropout_summary"
                ][
                    "maximum_effective_modality_count"
                ]
            ),

        "validation_maximum_modality_count_difference":
            float(
                validation_result[
                    "maximum_evaluation_modality_count_difference"
                ]
            ),
    }


    # --------------------------------------------------------
    # Add all training and validation losses
    # --------------------------------------------------------

    for loss_name, loss_value in (
        training_result[
            "losses"
        ].items()
    ):
        history_row[
            f"train_{loss_name}"
        ] = float(
            loss_value
        )

    for loss_name, loss_value in (
        validation_result[
            "losses"
        ].items()
    ):
        history_row[
            f"validation_{loss_name}"
        ] = float(
            loss_value
        )


    # --------------------------------------------------------
    # Add metrics for all three prediction outputs
    # --------------------------------------------------------

    for output_name in [
        "final",
        "three_mt",
        "tmc",
    ]:

        for metric_name, metric_value in (
            training_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"train_{output_name}_{metric_name}"
            ] = metric_value

        for metric_name, metric_value in (
            validation_result[
                "metrics"
            ][output_name].items()
        ):
            history_row[
                f"validation_{output_name}_{metric_name}"
            ] = metric_value


    return history_row


# ------------------------------------------------------------
# Save a fully recoverable checkpoint
# ------------------------------------------------------------

def save_training_checkpoint(
    checkpoint_path,
    epoch,
    model,
    optimizer,
    scheduler,
    best_validation_auc,
    epochs_without_improvement,
    validation_metrics,
    training_history,
):
    """
    Save the state required to reproduce or continue training.
    """

    checkpoint = {
        "epoch":
            int(epoch),

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "best_validation_auc":
            float(best_validation_auc),

        "epochs_without_improvement":
            int(epochs_without_improvement),

        "validation_metrics":
            validation_metrics,

        "training_configuration":
            training_configuration,

        "training_objective_configuration":
            obtain_training_objective_configuration(
                training_objective
            ),

        "training_history":
            training_history,

        "random_seed":
            int(GLOBAL_RANDOM_SEED),

        "task":
            SELECTED_TASK,

        "fold":
            int(SELECTED_FOLD),

        "saved_at":
            datetime.now().isoformat(
                timespec="seconds"
            ),
    }

    torch.save(
        checkpoint,
        checkpoint_path,
    )


# ------------------------------------------------------------
# GPU-memory helper for the progress display
# ------------------------------------------------------------

def current_cuda_memory_gb():
    """
    Return currently allocated CUDA memory in gigabytes.
    """

    if not torch.cuda.is_available():
        return 0.0

    return float(
        torch.cuda.memory_allocated(
            DEVICE
        )
        / (1024 ** 3)
    )


# ------------------------------------------------------------
# One training epoch with a live batch progress bar
# ------------------------------------------------------------

def run_training_epoch_with_progress(
    model,
    data_loader,
    objective,
    optimizer,
    device,
    epoch,
    maximum_epochs,
    maximum_gradient_norm,
):
    """
    Train for one epoch while displaying batch-level progress.
    """

    model.train()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    batch_gradient_norms = []

    effective_modality_counts = []

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | training"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    for batch_number, batch in progress_bar:

        batch = move_nested_batch_to_device(
            batch,
            device,
        )

        targets = batch[
            "target"
        ].long()

        batch_size = int(
            targets.shape[0]
        )

        participant_count += (
            batch_size
        )


        # --------------------------------------------------------
        # Forward pass and loss
        # --------------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        model_output = model(
            modalities=batch[
                "modalities"
            ],

            original_branch_masks=batch[
                "branch_masks"
            ],
        )

        loss_output = objective(
            model_output=model_output,
            targets=targets,
            epoch=epoch,
        )

        total_loss = loss_output[
            "total_loss"
        ]


        if not torch.isfinite(
            total_loss
        ):
            raise FloatingPointError(
                "A non-finite training loss was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )


        # --------------------------------------------------------
        # Backpropagation, clipping, and update
        # --------------------------------------------------------

        total_loss.backward()

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                parameters=model.parameters(),
                max_norm=maximum_gradient_norm,
            )
        )

        if not torch.isfinite(
            gradient_norm
        ):
            raise FloatingPointError(
                "A non-finite gradient norm was encountered "
                f"at epoch {epoch}, batch {batch_number}."
            )

        optimizer.step()


        # --------------------------------------------------------
        # Accumulate diagnostics
        # --------------------------------------------------------

        current_total_loss = float(
            total_loss.detach().item()
        )

        running_total_loss_sum += (
            current_total_loss
            * batch_size
        )

        running_mean_total_loss = (
            running_total_loss_sum
            / participant_count
        )

        batch_gradient_norms.append(
            float(
                gradient_norm.detach().item()
            )
        )

        update_epoch_loss_storage(
            storage=loss_storage,
            loss_output=loss_output,
            batch_size=batch_size,
        )

        extracted_predictions = (
            extract_batch_prediction_arrays(
                batch=batch,
                model_output=model_output,
            )
        )

        for key, values in (
            extracted_predictions.items()
        ):
            prediction_storage[
                key
            ].append(
                values
            )

        effective_modality_counts.extend(
            extracted_predictions[
                "effective_modality_count"
            ].tolist()
        )


        # --------------------------------------------------------
        # Update the visible progress information
        # --------------------------------------------------------

        if (
            batch_number
            % PROGRESS_UPDATE_INTERVAL
            == 0
            or batch_number
            == len(data_loader)
        ):

            elapsed_minutes = (
                time.time()
                - phase_start_time
            ) / 60.0

            progress_bar.set_postfix(
                {
                    "loss":
                        f"{current_total_loss:.3f}",

                    "avg":
                        f"{running_mean_total_loss:.3f}",

                    "grad":
                        f"{float(gradient_norm):.2f}",

                    "lr":
                        f"{optimizer.param_groups[0]['lr']:.1e}",

                    "GPU":
                        f"{current_cuda_memory_gb():.1f}GB",

                    "elapsed":
                        f"{elapsed_minutes:.1f}m",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise the training epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="train",

            epoch=epoch,
        )
    )


    gradient_summary = {
        "mean_gradient_norm":
            float(
                np.mean(
                    batch_gradient_norms
                )
            ),

        "maximum_gradient_norm_before_clipping":
            float(
                np.max(
                    batch_gradient_norms
                )
            ),

        "minimum_gradient_norm":
            float(
                np.min(
                    batch_gradient_norms
                )
            ),
    }


    modality_dropout_summary = {
        "mean_effective_modality_count":
            float(
                np.mean(
                    effective_modality_counts
                )
            ),

        "minimum_effective_modality_count":
            float(
                np.min(
                    effective_modality_counts
                )
            ),

        "maximum_effective_modality_count":
            float(
                np.max(
                    effective_modality_counts
                )
            ),
    }


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "gradient_summary":
            gradient_summary,

        "modality_dropout_summary":
            modality_dropout_summary,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),
    }


# ------------------------------------------------------------
# One validation epoch with a live batch progress bar
# ------------------------------------------------------------

def run_validation_epoch_with_progress(
    model,
    data_loader,
    objective,
    device,
    epoch,
    maximum_epochs,
):
    """
    Validate for one epoch while displaying batch-level progress.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0

    running_total_loss_sum = 0.0

    phase_start_time = time.time()


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc=(
            f"Epoch {epoch:02d}/{maximum_epochs} | validation"
        ),
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ----------------------------------------------------
            # Forward pass and validation loss
            # ----------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=epoch,
            )

            total_loss = loss_output[
                "total_loss"
            ]


            if not torch.isfinite(
                total_loss
            ):
                raise FloatingPointError(
                    "A non-finite validation loss was encountered "
                    f"at epoch {epoch}, batch {batch_number}."
                )


            # ----------------------------------------------------
            # Accumulate diagnostics and predictions
            # ----------------------------------------------------

            current_total_loss = float(
                total_loss.detach().item()
            )

            running_total_loss_sum += (
                current_total_loss
                * batch_size
            )

            running_mean_total_loss = (
                running_total_loss_sum
                / participant_count
            )

            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):
                prediction_storage[
                    key
                ].append(
                    values
                )


            # ----------------------------------------------------
            # Update the visible progress information
            # ----------------------------------------------------

            if (
                batch_number
                % PROGRESS_UPDATE_INTERVAL
                == 0
                or batch_number
                == len(data_loader)
            ):

                elapsed_minutes = (
                    time.time()
                    - phase_start_time
                ) / 60.0

                progress_bar.set_postfix(
                    {
                        "loss":
                            f"{current_total_loss:.3f}",

                        "avg":
                            f"{running_mean_total_loss:.3f}",

                        "GPU":
                            f"{current_cuda_memory_gb():.1f}GB",

                        "elapsed":
                            f"{elapsed_minutes:.1f}m",
                    },
                    refresh=True,
                )


    # ------------------------------------------------------------
    # Finalise the validation epoch
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="validation",

            epoch=epoch,
        )
    )


    maximum_mask_count_difference = float(
        np.max(
            np.abs(
                concatenated_predictions[
                    "original_modality_count"
                ]
                -
                concatenated_predictions[
                    "effective_modality_count"
                ]
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_mask_count_difference,
    }


# ------------------------------------------------------------
# Initial training state
# ------------------------------------------------------------

training_history = []

starting_epoch = 1

best_validation_auc = float(
    "-inf"
)

epochs_without_improvement = 0


# ------------------------------------------------------------
# Resume from the latest fully completed epoch
# ------------------------------------------------------------

if (
    RESUME_FROM_LAST_CHECKPOINT
    and LAST_CHECKPOINT_PATH.exists()
):

    print(
        "Loading the latest fully completed checkpoint:",
        flush=True,
    )

    print(
        LAST_CHECKPOINT_PATH,
        flush=True,
    )

    resumed_checkpoint = torch.load(
        LAST_CHECKPOINT_PATH,
        map_location=DEVICE,
        weights_only=False,
    )

    complete_model.load_state_dict(
        resumed_checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        resumed_checkpoint[
            "optimizer_state_dict"
        ]
    )

    learning_rate_scheduler.load_state_dict(
        resumed_checkpoint[
            "scheduler_state_dict"
        ]
    )

    completed_epoch = int(
        resumed_checkpoint[
            "epoch"
        ]
    )

    starting_epoch = (
        completed_epoch
        + 1
    )

    best_validation_auc = float(
        resumed_checkpoint[
            "best_validation_auc"
        ]
    )

    epochs_without_improvement = int(
        resumed_checkpoint[
            "epochs_without_improvement"
        ]
    )

    training_history = list(
        resumed_checkpoint.get(
            "training_history",
            [],
        )
    )

    print(
        f"\nResuming after epoch {completed_epoch}.",
        flush=True,
    )

    print(
        f"Next epoch: {starting_epoch}",
        flush=True,
    )

    print(
        "Best validation ROC AUC so far: "
        f"{best_validation_auc:.6f}",
        flush=True,
    )

else:

    print(
        "No fully completed checkpoint was loaded.",
        flush=True,
    )

    print(
        "Fresh training begins from the newly initialised model state.",
        flush=True,
    )


# ------------------------------------------------------------
# Main multi-epoch training loop
# ------------------------------------------------------------

if starting_epoch > MAXIMUM_EPOCHS:

    print(
        "\nTraining has already reached the configured maximum "
        f"of {MAXIMUM_EPOCHS} epochs.",
        flush=True,
    )

else:

    print("\n" + "=" * 72, flush=True)
    print("BEGINNING MODEL TRAINING", flush=True)
    print("=" * 72, flush=True)

    print(
        f"\nEpoch range: {starting_epoch}--{MAXIMUM_EPOCHS}",
        flush=True,
    )

    print(
        f"Training batches per epoch: {len(train_loader)}",
        flush=True,
    )

    print(
        f"Validation batches per epoch: {len(validation_loader)}",
        flush=True,
    )

    print(
        "Each epoch displays separate live training and "
        "validation progress bars.",
        flush=True,
    )

    print(
        "The test partition will not be evaluated.",
        flush=True,
    )


    try:

        for epoch in range(
            starting_epoch,
            MAXIMUM_EPOCHS + 1,
        ):

            epoch_start_time = time.time()

            current_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )


            print("\n" + "=" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d}/{MAXIMUM_EPOCHS}",
                flush=True,
            )

            print("=" * 72, flush=True)

            print(
                "\nPhase 1/4: training batches",
                flush=True,
            )


            # ------------------------------------------------
            # Train on all training participants
            # ------------------------------------------------

            training_result = (
                run_training_epoch_with_progress(
                    model=complete_model,
                    data_loader=train_loader,
                    objective=training_objective,
                    optimizer=optimizer,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                    maximum_gradient_norm=
                        MAXIMUM_GRADIENT_NORM,
                )
            )


            print(
                "\nPhase 2/4: validation batches",
                flush=True,
            )


            # ------------------------------------------------
            # Validate on all validation participants
            # ------------------------------------------------

            validation_result = (
                run_validation_epoch_with_progress(
                    model=complete_model,
                    data_loader=validation_loader,
                    objective=training_objective,
                    device=DEVICE,
                    epoch=epoch,
                    maximum_epochs=MAXIMUM_EPOCHS,
                )
            )


            print(
                "\nPhase 3/4: calculating metrics and "
                "updating the scheduler",
                flush=True,
            )


            # ------------------------------------------------
            # Validation AUC and checkpoint decision
            # ------------------------------------------------

            validation_auc = float(
                validation_result[
                    "metrics"
                ]["final"]["roc_auc"]
            )

            validation_auc_is_valid = bool(
                np.isfinite(
                    validation_auc
                )
            )

            checkpoint_improved = (
                validation_auc_is_valid
                and
                validation_auc
                > best_validation_auc
            )

            if checkpoint_improved:

                best_validation_auc = (
                    validation_auc
                )

                epochs_without_improvement = 0

            else:

                epochs_without_improvement += 1


            scheduler_score = (
                validation_auc
                if validation_auc_is_valid
                else -1.0
            )

            learning_rate_scheduler.step(
                scheduler_score
            )

            updated_learning_rate = float(
                optimizer.param_groups[
                    0
                ]["lr"]
            )

            epoch_duration_seconds = (
                time.time()
                - epoch_start_time
            )


            # ------------------------------------------------
            # Persistent training history
            # ------------------------------------------------

            history_row = build_training_history_row(
                epoch=epoch,

                learning_rate=
                    current_learning_rate,

                epoch_duration_seconds=
                    epoch_duration_seconds,

                training_result=
                    training_result,

                validation_result=
                    validation_result,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                checkpoint_improved=
                    checkpoint_improved,
            )

            history_row[
                "learning_rate_after_scheduler"
            ] = updated_learning_rate

            training_history.append(
                history_row
            )

            pd.DataFrame(
                training_history
            ).to_csv(
                TRAINING_HISTORY_PATH,
                index=False,
            )


            print(
                "\nPhase 4/4: saving checkpoints and history",
                flush=True,
            )


            # ------------------------------------------------
            # Save the latest completed epoch
            # ------------------------------------------------

            save_training_checkpoint(
                checkpoint_path=
                    LAST_CHECKPOINT_PATH,

                epoch=epoch,

                model=complete_model,

                optimizer=optimizer,

                scheduler=
                    learning_rate_scheduler,

                best_validation_auc=
                    best_validation_auc,

                epochs_without_improvement=
                    epochs_without_improvement,

                validation_metrics=
                    validation_result[
                        "metrics"
                    ],

                training_history=
                    training_history,
            )


            # ------------------------------------------------
            # Save the best validation checkpoint
            # ------------------------------------------------

            if checkpoint_improved:

                save_training_checkpoint(
                    checkpoint_path=
                        BEST_CHECKPOINT_PATH,

                    epoch=epoch,

                    model=complete_model,

                    optimizer=optimizer,

                    scheduler=
                        learning_rate_scheduler,

                    best_validation_auc=
                        best_validation_auc,

                    epochs_without_improvement=
                        epochs_without_improvement,

                    validation_metrics=
                        validation_result[
                            "metrics"
                        ],

                    training_history=
                        training_history,
                )

                validation_result[
                    "predictions"
                ].to_csv(
                    VALIDATION_PREDICTIONS_PATH,
                    index=False,
                )


            # ------------------------------------------------
            # Readable completed-epoch summary
            # ------------------------------------------------

            train_metrics = training_result[
                "metrics"
            ]["final"]

            validation_metrics = validation_result[
                "metrics"
            ]["final"]


            print("\n" + "-" * 72, flush=True)

            print(
                f"EPOCH {epoch:02d} COMPLETE",
                flush=True,
            )

            print(
                "Duration: "
                f"{epoch_duration_seconds / 60.0:.2f} minutes",
                flush=True,
            )

            print(
                "Learning rate: "
                f"{current_learning_rate:.8f}"
                f" -> {updated_learning_rate:.8f}",
                flush=True,
            )


            print("\nTraining:", flush=True)

            print(
                "  total loss: "
                f"{training_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{train_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{train_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{train_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{train_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )

            print(
                "  mean effective modalities: "
                f"{training_result['modality_dropout_summary']['mean_effective_modality_count']:.3f}",
                flush=True,
            )

            print(
                "  maximum pre-clipping gradient norm: "
                f"{training_result['gradient_summary']['maximum_gradient_norm_before_clipping']:.6f}",
                flush=True,
            )


            print("\nValidation:", flush=True)

            print(
                "  total loss: "
                f"{validation_result['losses']['total_loss']:.6f}",
                flush=True,
            )

            print(
                "  ROC AUC: "
                f"{validation_metrics['roc_auc']:.6f}",
                flush=True,
            )

            print(
                "  average precision: "
                f"{validation_metrics['average_precision']:.6f}",
                flush=True,
            )

            print(
                "  balanced accuracy: "
                f"{validation_metrics['balanced_accuracy']:.6f}",
                flush=True,
            )

            print(
                "  sensitivity: "
                f"{validation_metrics['sensitivity']:.6f}",
                flush=True,
            )

            print(
                "  specificity: "
                f"{validation_metrics['specificity']:.6f}",
                flush=True,
            )

            print(
                "  Brier score: "
                f"{validation_metrics['brier_score']:.6f}",
                flush=True,
            )

            print(
                "  calibration error: "
                f"{validation_metrics['expected_calibration_error']:.6f}",
                flush=True,
            )

            print(
                "  mean uncertainty: "
                f"{validation_metrics['mean_uncertainty']:.6f}",
                flush=True,
            )

            print(
                "  mean 3MT weight: "
                f"{validation_metrics['mean_three_mt_weight']:.6f}",
                flush=True,
            )


            print("\nCheckpoint status:", flush=True)

            print(
                "  improved this epoch: "
                f"{checkpoint_improved}",
                flush=True,
            )

            print(
                "  best validation ROC AUC: "
                f"{best_validation_auc:.6f}",
                flush=True,
            )

            print(
                "  epochs without improvement: "
                f"{epochs_without_improvement}"
                f"/{EARLY_STOPPING_PATIENCE}",
                flush=True,
            )

            print(
                "  latest completed epoch saved: True",
                flush=True,
            )

            if torch.cuda.is_available():

                print(
                    "  peak CUDA memory: "
                    f"{torch.cuda.max_memory_allocated(0) / (1024 ** 3):.3f} GB",
                    flush=True,
                )


            # ------------------------------------------------
            # Early stopping
            # ------------------------------------------------

            if (
                epochs_without_improvement
                >= EARLY_STOPPING_PATIENCE
            ):

                print(
                    "\nEarly stopping activated because "
                    "validation ROC AUC did not improve for "
                    f"{EARLY_STOPPING_PATIENCE} consecutive epochs.",
                    flush=True,
                )

                break


    # --------------------------------------------------------
    # Interruption and error handling
    # --------------------------------------------------------

    except KeyboardInterrupt:

        print(
            "\nTraining was interrupted manually.",
            flush=True,
        )

        print(
            "Only fully completed epochs are recoverable from "
            "the last-epoch checkpoint.",
            flush=True,
        )


    except Exception:

        print(
            "\nTraining stopped because an exception occurred.",
            flush=True,
        )

        print(
            "The latest fully completed epoch remains saved.",
            flush=True,
        )

        traceback.print_exc()

        raise


    # --------------------------------------------------------
    # Final training summary
    # --------------------------------------------------------

    if len(
        training_history
    ) > 0:

        final_history_table = pd.DataFrame(
            training_history
        )

        completed_epochs = int(
            final_history_table[
                "epoch"
            ].max()
        )

        print("\n" + "=" * 72, flush=True)

        print(
            "TRAINING RUN COMPLETE",
            flush=True,
        )

        print("=" * 72, flush=True)

        print(
            f"\nLast completed epoch: {completed_epochs}",
            flush=True,
        )

        print(
            "Best validation ROC AUC: "
            f"{best_validation_auc:.6f}",
            flush=True,
        )

        print(
            "\nBest checkpoint:\n"
            f"{BEST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nLatest checkpoint:\n"
            f"{LAST_CHECKPOINT_PATH}",
            flush=True,
        )

        print(
            "\nTraining history:\n"
            f"{TRAINING_HISTORY_PATH}",
            flush=True,
        )

        print(
            "\nThe test partition has not been evaluated.",
            flush=True,
        )

### 1.8.37. Evaluating the best gated checkpoint on the untouched test set

After training and validation-based checkpoint selection, the best checkpoint from this named experiment is loaded from its experiment-specific fold directory and evaluated once on the held-out test partition.

The evaluation cell does not update model parameters, the optimiser, the scheduler, or modality-dropout state.

In [ ]:
from datetime import datetime


# ------------------------------------------------------------
# Test-output paths
# ------------------------------------------------------------

TEST_METRICS_PATH = (
    PREDICTION_DIR
    / "test_metrics.json"
)

TEST_SUMMARY_PATH = (
    PREDICTION_DIR
    / "test_metrics_summary.csv"
)


print("=" * 72)
print("FIXED-EQUAL-FUSION TEST-SET EVALUATION")
print("=" * 72)

print(
    "\nLoading the best validation checkpoint:\n"
    f"{BEST_CHECKPOINT_PATH}"
)


# ------------------------------------------------------------
# Load the best validation checkpoint
# ------------------------------------------------------------


best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False,
)

best_checkpoint_epoch = int(
    best_checkpoint[
        "epoch"
    ]
)

best_checkpoint_validation_auc = float(
    best_checkpoint[
        "best_validation_auc"
    ]
)


complete_model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


print(
    f"\nBest checkpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Best validation ROC AUC stored in checkpoint: "
    f"{best_checkpoint_validation_auc:.6f}"
)


# ------------------------------------------------------------
# Test evaluation function
# ------------------------------------------------------------

def run_test_epoch(
    model,
    data_loader,
    objective,
    device,
    checkpoint_epoch,
):
    """
    Evaluate the frozen model on the untouched test partition.
    """

    model.eval()

    loss_storage = (
        initialise_epoch_loss_storage()
    )

    prediction_storage = defaultdict(
        list
    )

    participant_count = 0


    progress_bar = tqdm(
        enumerate(
            data_loader,
            start=1,
        ),
        total=len(
            data_loader
        ),
        desc="Test evaluation",
        unit="batch",
        dynamic_ncols=True,
        leave=True,
    )


    with torch.no_grad():

        for batch_number, batch in progress_bar:

            batch = move_nested_batch_to_device(
                batch,
                device,
            )

            targets = batch[
                "target"
            ].long()

            batch_size = int(
                targets.shape[0]
            )

            participant_count += (
                batch_size
            )


            # ------------------------------------------------
            # Frozen forward pass
            # ------------------------------------------------

            model_output = model(
                modalities=batch[
                    "modalities"
                ],

                original_branch_masks=batch[
                    "branch_masks"
                ],
            )


            # ------------------------------------------------
            # Test loss
            # ------------------------------------------------

            loss_output = objective(
                model_output=model_output,
                targets=targets,
                epoch=checkpoint_epoch,
            )


            if not torch.isfinite(
                loss_output[
                    "total_loss"
                ]
            ):

                raise FloatingPointError(
                    "A non-finite test loss was encountered "
                    f"at batch {batch_number}."
                )


            update_epoch_loss_storage(
                storage=loss_storage,
                loss_output=loss_output,
                batch_size=batch_size,
            )


            # ------------------------------------------------
            # Store participant-level outputs
            # ------------------------------------------------

            extracted_predictions = (
                extract_batch_prediction_arrays(
                    batch=batch,
                    model_output=model_output,
                )
            )

            for key, values in (
                extracted_predictions.items()
            ):

                prediction_storage[
                    key
                ].append(
                    values
                )


            running_average_loss = (
                loss_storage[
                    "total_loss"
                ]
                / participant_count
            )

            progress_bar.set_postfix(
                {
                    "avg_loss":
                        f"{running_average_loss:.3f}",

                    "participants":
                        f"{participant_count}/{len(data_loader.dataset)}",
                },
                refresh=True,
            )


    # ------------------------------------------------------------
    # Finalise test losses and predictions
    # ------------------------------------------------------------

    mean_losses = finalise_epoch_loss_storage(
        storage=loss_storage,
        participant_count=participant_count,
    )

    concatenated_predictions = (
        concatenate_epoch_prediction_storage(
            prediction_storage
        )
    )

    prediction_metrics = (
        calculate_epoch_prediction_metrics(
            concatenated_predictions
        )
    )

    prediction_table = (
        build_epoch_prediction_table(
            concatenated_predictions=
                concatenated_predictions,

            split_name="test",

            epoch=checkpoint_epoch,
        )
    )


    original_modality_counts = (
        concatenated_predictions[
            "original_modality_count"
        ]
    )

    effective_modality_counts = (
        concatenated_predictions[
            "effective_modality_count"
        ]
    )

    maximum_modality_count_difference = float(
        np.max(
            np.abs(
                original_modality_counts
                - effective_modality_counts
            )
        )
    )


    return {
        "losses":
            mean_losses,

        "metrics":
            prediction_metrics,

        "predictions":
            prediction_table,

        "participant_count":
            int(
                participant_count
            ),

        "batch_count":
            int(
                len(data_loader)
            ),

        "maximum_evaluation_modality_count_difference":
            maximum_modality_count_difference,
    }


# ------------------------------------------------------------
# Run the untouched test evaluation
# ------------------------------------------------------------

test_result = run_test_epoch(
    model=complete_model,
    data_loader=test_loader,
    objective=training_objective,
    device=DEVICE,
    checkpoint_epoch=best_checkpoint_epoch,
)


# ------------------------------------------------------------
# Evaluation-mode modality count
# ------------------------------------------------------------

maximum_test_mask_difference = (
    test_result[
        "maximum_evaluation_modality_count_difference"
    ]
)


# ------------------------------------------------------------
# Save participant-level test predictions
# ------------------------------------------------------------


test_prediction_table = test_result[
    "predictions"
]

test_prediction_table.to_csv(
    TEST_PREDICTIONS_PATH,
    index=False,
)


# ------------------------------------------------------------
# Save all test metrics in JSON format
# ------------------------------------------------------------

serialisable_test_result = {
    "experiment_name": EXPERIMENT_NAME,
    "availability_gated_cmt": True,
    "task":
        SELECTED_TASK,

    "fold":
        int(
            SELECTED_FOLD
        ),

    "checkpoint_epoch":
        int(
            best_checkpoint_epoch
        ),

    "checkpoint_validation_auc":
        float(
            best_checkpoint_validation_auc
        ),

    "classification_threshold":
        float(
            CLASSIFICATION_THRESHOLD
        ),

    "test_participants":
        int(
            test_result[
                "participant_count"
            ]
        ),

    "test_batches":
        int(
            test_result[
                "batch_count"
            ]
        ),

    "test_losses": {
        key:
            float(value)

        for key, value in (
            test_result[
                "losses"
            ].items()
        )
    },

    "test_metrics":
        test_result[
            "metrics"
        ],

    "maximum_modality_count_difference":
        float(
            maximum_test_mask_difference
        ),

    "evaluated_at":
        datetime.now().isoformat(
            timespec="seconds"
        ),
}


with open(
    TEST_METRICS_PATH,
    "w",
    encoding="utf-8",
) as test_metrics_file:

    json.dump(
        serialisable_test_result,
        test_metrics_file,
        indent=2,
    )


# ------------------------------------------------------------
# Build a compact comparison table
# ------------------------------------------------------------

test_metric_rows = []

for output_name, display_name in [
    (
        "final",
        "Hybrid",
    ),
    (
        "three_mt",
        "3MT-only",
    ),
    (
        "tmc",
        "TMC-only",
    ),
]:

    output_metrics = test_result[
        "metrics"
    ][
        output_name
    ]

    test_metric_rows.append(
        {
            "OUTPUT":
                display_name,

            "ROC_AUC":
                output_metrics[
                    "roc_auc"
                ],

            "AVERAGE_PRECISION":
                output_metrics[
                    "average_precision"
                ],

            "ACCURACY":
                output_metrics[
                    "accuracy"
                ],

            "BALANCED_ACCURACY":
                output_metrics[
                    "balanced_accuracy"
                ],

            "SENSITIVITY":
                output_metrics[
                    "sensitivity"
                ],

            "SPECIFICITY":
                output_metrics[
                    "specificity"
                ],

            "PRECISION":
                output_metrics[
                    "precision"
                ],

            "F1":
                output_metrics[
                    "f1"
                ],

            "BRIER_SCORE":
                output_metrics[
                    "brier_score"
                ],

            "NEGATIVE_LOG_LIKELIHOOD":
                output_metrics[
                    "negative_log_likelihood"
                ],

            "EXPECTED_CALIBRATION_ERROR":
                output_metrics[
                    "expected_calibration_error"
                ],

            "MEAN_UNCERTAINTY":
                output_metrics[
                    "mean_uncertainty"
                ],
        }
    )


test_metrics_summary = pd.DataFrame(
    test_metric_rows
)

test_metrics_summary.to_csv(
    TEST_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# Display the final test results
# ------------------------------------------------------------

hybrid_test_metrics = test_result[
    "metrics"
][
    "final"
]


print("\n" + "=" * 72)
print(f"FIXED-EQUAL-FUSION FOLD-{SELECTED_FOLD} TEST RESULTS")
print("=" * 72)

print(
    f"\nCheckpoint epoch: "
    f"{best_checkpoint_epoch}"
)

print(
    "Test participants: "
    f"{test_result['participant_count']}"
)

print(
    "Test batches: "
    f"{test_result['batch_count']}"
)

print(
    "Maximum difference between original and effective "
    "test modality counts: "
    f"{maximum_test_mask_difference:.1f}"
)


print(
    "\nFinal hybrid test performance:"
)

print(
    "  ROC AUC: "
    f"{hybrid_test_metrics['roc_auc']:.6f}"
)

print(
    "  average precision: "
    f"{hybrid_test_metrics['average_precision']:.6f}"
)

print(
    "  accuracy: "
    f"{hybrid_test_metrics['accuracy']:.6f}"
)

print(
    "  balanced accuracy: "
    f"{hybrid_test_metrics['balanced_accuracy']:.6f}"
)

print(
    "  sensitivity: "
    f"{hybrid_test_metrics['sensitivity']:.6f}"
)

print(
    "  specificity: "
    f"{hybrid_test_metrics['specificity']:.6f}"
)

print(
    "  precision: "
    f"{hybrid_test_metrics['precision']:.6f}"
)

print(
    "  F1 score: "
    f"{hybrid_test_metrics['f1']:.6f}"
)

print(
    "  Brier score: "
    f"{hybrid_test_metrics['brier_score']:.6f}"
)

print(
    "  negative log-likelihood: "
    f"{hybrid_test_metrics['negative_log_likelihood']:.6f}"
)

print(
    "  expected calibration error: "
    f"{hybrid_test_metrics['expected_calibration_error']:.6f}"
)

print(
    "  mean uncertainty: "
    f"{hybrid_test_metrics['mean_uncertainty']:.6f}"
)

print(
    "  mean 3MT weight: "
    f"{hybrid_test_metrics['mean_three_mt_weight']:.6f}"
)

print(
    "  standard deviation of 3MT weight: "
    f"{hybrid_test_metrics['std_three_mt_weight']:.6f}"
)


print(
    "\nConfusion matrix counts:"
)

print(
    "  true negatives: "
    f"{hybrid_test_metrics['true_negative']}"
)

print(
    "  false positives: "
    f"{hybrid_test_metrics['false_positive']}"
)

print(
    "  false negatives: "
    f"{hybrid_test_metrics['false_negative']}"
)

print(
    "  true positives: "
    f"{hybrid_test_metrics['true_positive']}"
)


print(
    "\nHybrid, 3MT-only, and TMC-only comparison:"
)

display(
    test_metrics_summary.round(
        6
    )
)


print(
    "\nParticipant-level predictions saved to:\n"
    f"{TEST_PREDICTIONS_PATH}"
)

print(
    "\nComplete test metrics saved to:\n"
    f"{TEST_METRICS_PATH}"
)

print(
    "\nCompact metric summary saved to:\n"
    f"{TEST_SUMMARY_PATH}"
)

print(
    "\nThe model was evaluated without gradient updates, "
    "scheduler changes, or test-time modality dropout."
)

### 1.8.38. Releasing fold-specific memory

The fold-4 checkpoints, histories, validation predictions, test predictions, and metrics are already stored in its own directory. This cleanup removes the in-memory model and DataLoaders before the next fold is created.

In [ ]:
# ============================================================
# 21. Releasing fold-4 memory before the next fold
# ============================================================

import gc

objects_to_release = [
    "complete_model",
    "optimizer",
    "scheduler",
    "training_objective",
    "train_loader",
    "validation_loader",
    "test_loader",
    "train_dataset",
    "validation_dataset",
    "test_dataset",
    "example_batch",
    "diagnostic_batch",
]

for object_name in objects_to_release:
    if object_name in globals():
        del globals()[object_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(
    "Fold 4 outputs remain saved under:\n"
    f"{FOLD_TRAINING_DIR}"
)

print(
    "Fold-specific GPU cache has been released."
)


## 1.9. Five-fold completion summary

This final section reads the saved test metrics from the five fixed-equal-fusion fold directories and displays one row per fold. It does not retrain or reevaluate any model.


In [ ]:
# ============================================================
# Five-fold completion and metric summary
# ============================================================

from pathlib import Path
import json
import pandas as pd


SUMMARY_MODEL_ROOT = Path(
    "/content/drive/MyDrive/adni_mri/models/3mt_tmc_evidential"
)

SUMMARY_EXPERIMENT_ROOT = (
    SUMMARY_MODEL_ROOT
    / "experiments"
    / "temporal_prebaseline_fixed_equal_fusion_md050"
    / "mci_prognosis"
)


summary_rows = []

for fold_number in range(5):

    fold_root = (
        SUMMARY_EXPERIMENT_ROOT
        / f"fold_{fold_number}"
    )

    metrics_path = (
        fold_root
        / "predictions"
        / "test_metrics.json"
    )

    if not metrics_path.exists():
        summary_rows.append(
            {
                "FOLD": fold_number,
                "STATUS": "not complete",
            }
        )
        continue

    with metrics_path.open(
        "r",
        encoding="utf-8",
    ) as metrics_file:
        metrics_document = json.load(
            metrics_file
        )

    final_metrics = metrics_document[
        "test_metrics"
    ]["final"]

    summary_rows.append(
        {
            "FOLD": fold_number,
            "STATUS": "complete",
            "BEST_EPOCH":
                metrics_document[
                    "checkpoint_epoch"
                ],
            "VALIDATION_ROC_AUC":
                metrics_document[
                    "checkpoint_validation_auc"
                ],
            "TEST_ROC_AUC":
                final_metrics[
                    "roc_auc"
                ],
            "TEST_AVERAGE_PRECISION":
                final_metrics[
                    "average_precision"
                ],
            "TEST_BALANCED_ACCURACY":
                final_metrics[
                    "balanced_accuracy"
                ],
            "TEST_BRIER_SCORE":
                final_metrics[
                    "brier_score"
                ],
            "TEST_ECE":
                final_metrics[
                    "expected_calibration_error"
                ],
            "TEST_MEAN_UNCERTAINTY":
                final_metrics[
                    "mean_uncertainty"
                ],
            "TEST_PARTICIPANTS":
                metrics_document[
                    "test_participants"
                ],
        }
    )


five_fold_summary = pd.DataFrame(
    summary_rows
)

display(
    five_fold_summary
)


complete_fold_count = int(
    (
        five_fold_summary[
            "STATUS"
        ]
        == "complete"
    ).sum()
)

print(
    f"\nComplete folds: {complete_fold_count}/5"
)

if complete_fold_count == 5:
    print(
        "All five fixed-equal-fusion folds are ready "
        "for cross-validation aggregation."
    )
